In [1]:
import copy
import json
import sys
import time
import types
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from IPython.core.magic import register_cell_magic
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 120)
ROOT = Path.cwd()                       # data and outputs are read and written relative to this folder


@register_cell_magic
def module(line, cell):
    """Run the cell as module `line` of the in-notebook `meridian` package, so that the cells import each
    other as the files of a package would (`from .data import load_bundle`). `package` marks a package."""
    name, *flags = line.split()
    parent, _, child = name.rpartition(".")
    mod = sys.modules.get(name) if "package" in flags else None
    mod = mod or types.ModuleType(name)
    mod.__package__ = name if "package" in flags else parent
    if "package" in flags:
        mod.__path__ = []
    sys.modules[name] = mod
    if parent:
        setattr(sys.modules[parent], child, mod)
    exec(compile(cell, f"<{name}>", "exec"), mod.__dict__)


for _name in ["meridian", "meridian.detectors"]:    # empty packages first; their modules fill them below
    sys.modules[_name] = types.ModuleType(_name)
    sys.modules[_name].__path__, sys.modules[_name].__package__ = [], _name
sys.modules["meridian"].detectors = sys.modules["meridian.detectors"]

# Graph-based detection

**Version v1.3** (2026-11). The sponsor scored three submissions blind: v1.0 (this notebook at commit `67553b4`), v1.1 (`9fbf4fc`) and v1.2 (`1b276d6`); the versions and their scores are in `README.md`. With the third score the sponsor released the labelled VALIDATION cases of the main data (43 cases, `Full Dataset/validation_student_safe.csv`), and v1.3 is the first version developed on them (section 9; the TEST labels stay blind). Each change is marked `v1.3` in the code or in section 0.1: (1) **the period filter**, where the labels showed the structural blind spot: a case also belongs to the period when a finding that covers it emerged in it (ranked with weight 0.5), and a structure still holding at the period start is submitted behind the period's own cases (weight 0.1); 8 of the 43 validation cases had been found and then dropped as older than the period; (2) an approver group without sponsor coupling is ranked with the open-set prior, since every labelled approver case is a coupled sponsor–approver pair; (3) a detector for cross-border corridors, the geography typology with the lowest validation recall; (4) the public-institution expert cluster of v1.2 leaves the method again (the labels use the 14 tags only); (5) a group of synchronised spend spikes that chance explains in a large sponsor portfolio is reported as `portfolio_spikes` and ranked with the open-set prior (none of the 117 spike groups of the validation period is a labelled case); (6) a detector for a burst of new vendors under one sponsor (`sponsor_burst`, RAPID_ONBOARDING_LINKED), which the relation detectors missed when the sponsor has many vendors; (7) a common-owner group whose owner is also engaged as an individual expert, with no sponsor engaging both, carries a benign note. The variant is now chosen on the main-data validation labels (section 8).

This notebook runs the graph-based detection method end to end on the starter data and evaluates it. **Run all cells from the folder that holds `student_bundle/`** (about a minute). Every number below is computed here; nothing is read from earlier runs.

The method in one line: the source tables become one time-aware heterogeneous graph; typology detectors look for meta-path patterns in it; their findings are merged into cases, dated, filtered to the period and ranked into a review queue. The notebook is self-contained: section 0 holds all the method code and section 0.1 the method settings (identified by the method hash printed there). It needs only the data folder and the Python packages pandas, numpy, scipy, scikit-learn, networkx, pyarrow, pyyaml and unidecode.

**The deliverable**, in `output/v1.3/` (one folder per method version, `output/<VERSION>/`; `output/v1.0/` to `output/v1.2/` hold the first three submissions and the client's scores): `submission_TEST.csv`, the ranked cases of the TEST period on the full data, every case the method assembles except near-duplicates and lone open-set findings (older structures that still stand come last), from the variant chosen on the main-data VALIDATION labels (section 8), in the format of the sponsor's `submission_template.csv` (case_rank, member_record_ids, typology_tags, score, reason_codes); and `policy_findings_TEST.csv`, the single-record policy findings of the same period, ranked as their own list in the same format. They are the only files written to `output/v1.3/`.

Everything else, in `results/graph/v1.3/`:

| file | content |
|---|---|
| `cases/starter/submission_<PERIOD>.csv`, `cases/starter/submission_ALL.csv` | the ranked cases of each period, and at the snapshot, on the starter data |
| `cases/starter/policy_<PERIOD>.csv` | the single-record policy findings of each period, as their own list |
| `cases/starter/layer2_variants/{all_cases,top_100}/submission_<VARIANT>_<PERIOD>.csv`, `.../policy_<VARIANT>_<PERIOD>.csv` | the variants of section 7, and their policy lists |
| `cases/full_dataset/{all_cases,top_100}/submission_<VARIANT>_<PERIOD>.csv` | the variants on the full data (section 8) |
| `full_dataset/dropped_TEST.csv` | the TEST cases left out of the deliverable, and why |
| `full_dataset/validation_*.csv` | v1.3: the main-data validation labels: every variant's VALIDATION list scored, v1.1 / v1.2 / v1.3 compared, recall by typology and by difficulty tier, where each case lands |
| `metrics_by_period.csv`, `metrics_all.csv` | the scores below |
| `case_landing.csv` | where each labelled case lands: rank and IoU, by period |
| `ap_by_threshold.csv` | AP at other matching thresholds |
| `graph/` | the graph at the snapshot (nodes, edges, activity), its content hash, and `ownership_paths.parquet`: every ownership path at the snapshot with where it stops |

## 1. Data split

| choice | dossier |
|---|---|
| **Three periods by emergence date:** TRAINING (cases emerging up to 2024-06-30), VALIDATION (2024-07-01 – 2025-06-30), TEST (2025-07-01 – 2026-06-30). A case sits entirely in one period. | §8.1; App. B "Case partition" |
| **Case dates** (the labels have none): the day the last member was onboarded. CASE-0016 is a payment-after-termination case and is dated by the termination its label gives (2025-01-25). | inferred; to confirm with the sponsor |
| **Data as of the period end:** each period runs on the data known at its end: a row enters once it is recorded (observed) by t, and end dates after t are blanked. | §8.1 "A graph as of time t should contain only relationships observed by t and valid at t" |
| **Only what emerged or grew in the period is submitted.** A case emerged in the period when its earliest finding falls in it; an older case grew in the period when one of its vendor entities was first onboarded in it. | §8.1; an older structure that gains members in the period is a case of that period |
| **No training on labels.** Detection, case assembly and ranking read no labels; labels are read by the scorer only. | §8.2 |
| **Metrics:** one-to-one matching down the ranking; a submitted case matches the unmatched labelled case it overlaps most if IoU ≥ 0.5. AP, P@k and R@k for k = 10, 25, 50, 100, mean IoU of matches. **AP is the headline metric.** | §9.1; the official threshold is not published |

## 0. Method code

Each cell below is one module of the method. The `%%module` line runs the cell as that module, so the modules import each other exactly as files of a package would; nothing is read from `src/`. Run them in order: each module's imports come before it.

| module | role |
|---|---|
| `meridian.data` | Loading the Meridian bundle, parsing the registries and the as_of(t) view |
| `meridian.resolution` | Normalisation and entity resolution |
| `meridian.context` | Per-time-point context shared by all detectors |
| `meridian.graph` | Canonical heterogeneous graph: typed node and edge tables plus a content hash |
| `meridian.graph_ops` | Graph operations used by the detectors: typed edges and meta-paths |
| `meridian.graph_views` | Relational views derived from the canonical graph |
| `meridian.checks` | Data checks run on every load (Dossier Appendix B) |
| `meridian.scorer` | Local development scorer |
| `meridian.detectors.base` | Shared types for detectors |
| `meridian.detectors.relations` | Entity-level shared-relation items, weighted by rarity |
| `meridian.detectors.evidence` | Look up the graph edges that support a detector's finding |
| `meridian.detectors.shared_address` | SHARED_ADDRESS: several unrelated vendors registered at one address at the same time |
| `meridian.detectors.sponsor` | SPONSOR_CONCENTRATION: a sponsor takes on many new vendors in a short window, relative to role peers (v1.3: RAPID_ONBOARDING_LINKED through one sponsor's burst) |
| `meridian.detectors.approval` | SPONSOR_APPROVER_COUPLING and APPROVER_CONCENTRATION |
| `meridian.detectors.cluster` | DENSE_MULTI_RELATION_CLUSTER and RAPID_ONBOARDING_LINKED |
| `meridian.detectors.market` | BRIDGE_ENTITY and GEOGRAPHY_LINKED_CONCENTRATION (a market pocket; v1.3: a cross-border corridor) |
| `meridian.detectors.activity` | ACTIVITY_TIMING_CONCENTRATION: synchronised spend spikes under one sponsor; payments after termination (v1.3: groups that chance explains are `portfolio_spikes`) |
| `meridian.detectors.ownership` | The five ownership typologies, built on registry R paths |
| `meridian.detectors.policy` | Single-record policy findings, scored separately from the typologies (Dossier §4) |
| `meridian.detectors.record` | Record-level outliers (ladder level A): no relation to any other vendor is used |
| `meridian.detectors.open_set` | Open-set detection. Evaluated by leave-one-type-out in experiments.py |
| `meridian.detectors.layer2_typologies` | The new typologies proposed in Layer 2, added only in the variants of section 7 |
| `meridian.detectors` | Detector registry. Each detector takes a Context and returns a list of Candidates |
| `meridian.features` | Explicit features for the representation ladder |
| `meridian.cases` | Case assembly: merge candidates, emergence dates, benign explanations, window filter (v1.3: covering findings, standing structures) |
| `meridian.ranking` | Unified ranking: calibration, type interleaving, submission table |
| `meridian.calibrate` | Detector hit-rate calibration from labelled cases |
| `meridian.run` | Command-line entry point: one command from raw data to submission and local score |

In [2]:
%%module meridian package
"""Graph-based detection of anomalous third-party relationship patterns."""
__version__ = "0.1.0"

In [3]:
%%module meridian.data
"""Loading the Meridian bundle, parsing the registries and the as_of(t) view.

Labels are deliberately not loaded here: they belong to `scorer.py` only, so
nothing that builds features can reach them by accident.
"""
from __future__ import annotations

import hashlib
import json
from dataclasses import dataclass, field, fields, replace
from pathlib import Path

import pandas as pd

SNAPSHOT = pd.Timestamp("2026-06-30")

# Business tables and reference tables, in the order of the data dictionary.
TABLES = [
    "third_parties",
    "tp_due_diligence",
    "tp_address_history",
    "tp_services",
    "tp_engagements",
    "tp_engagement_approvals",
    "tp_payments_monthly",
    "tp_institution_links",
    "healthcare_institutions",
    "employees",
    "employee_roles",
    "employee_delegations",
    "markets",
    "paying_entities",
    "service_categories",
    "approval_policy_matrix",
    "fmv_rate_cards",
    "country_risk_context",
    "business_events",
]

# Tables the loader tolerates as missing: employee_delegations has 0 rows and no
# file in the starter bundle (the full data has it, see _delegations).
OPTIONAL_TABLES = {"employee_delegations"}

DELEGATION_COLUMNS = [
    "delegation_id", "from_employee_id", "to_employee_id",
    "approver_level", "valid_from", "valid_to", "observed_at",
]

DATE_COLUMNS = {
    "third_parties": ["onboarding_date", "created_date", "terminated_date"],
    "tp_due_diligence": ["dd_completed", "dd_refresh_due"],
    "tp_address_history": ["valid_from", "valid_to", "observed_at"],
    "tp_services": ["valid_from", "valid_to"],
    "tp_engagements": ["start_date", "end_date", "observed_at"],
    "tp_engagement_approvals": ["approval_date", "observed_at"],
    "tp_payments_monthly": ["month"],
    "tp_institution_links": ["valid_from", "valid_to"],
    "employee_roles": ["valid_from", "valid_to"],
    "employee_delegations": ["valid_from", "valid_to", "observed_at"],
    "fmv_rate_cards": ["effective_from", "effective_to"],
    "country_risk_context": ["as_of_date", "valid_to"],
    "business_events": ["start_date", "end_date"],
    "registry_l": ["relationship_start"],
    "reg_entities": ["statement_date"],
    "reg_persons": ["statement_date"],
    "reg_interests": ["statement_date", "start_date", "end_date"],
}


@dataclass
class Bundle:
    """All input tables as DataFrames. `as_of` is the knowledge date of this view."""

    third_parties: pd.DataFrame
    tp_due_diligence: pd.DataFrame
    tp_address_history: pd.DataFrame
    tp_services: pd.DataFrame
    tp_engagements: pd.DataFrame
    tp_engagement_approvals: pd.DataFrame
    tp_payments_monthly: pd.DataFrame
    tp_institution_links: pd.DataFrame
    healthcare_institutions: pd.DataFrame
    employees: pd.DataFrame
    employee_roles: pd.DataFrame
    employee_delegations: pd.DataFrame
    markets: pd.DataFrame
    paying_entities: pd.DataFrame
    service_categories: pd.DataFrame
    approval_policy_matrix: pd.DataFrame
    fmv_rate_cards: pd.DataFrame
    country_risk_context: pd.DataFrame
    business_events: pd.DataFrame
    registry_l: pd.DataFrame
    reg_entities: pd.DataFrame
    reg_persons: pd.DataFrame
    reg_interests: pd.DataFrame
    as_of: pd.Timestamp = SNAPSHOT
    source_dir: str = ""
    input_hashes: dict = field(default_factory=dict)

    def tables(self) -> dict[str, pd.DataFrame]:
        return {f.name: getattr(self, f.name) for f in fields(self)
                if isinstance(getattr(self, f.name), pd.DataFrame)}


def _file_hash(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()[:16]


def _read_table(bundle_dir: Path, name: str, hashes: dict) -> pd.DataFrame | None:
    for ext in (".parquet", ".csv"):
        p = bundle_dir / f"{name}{ext}"
        if p.exists():
            hashes[p.name] = _file_hash(p)
            return pd.read_parquet(p) if ext == ".parquet" else pd.read_csv(p)
    return None


def _to_dates(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce").astype("datetime64[ns]")
    return df


def parse_registry_r(path: Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Flatten the BODS-style statements into entity, person and interest tables.

    The file contains the non-standard JSON value NaN; Python's json
    module accepts it, and NaN identifiers become None here.
    """
    with open(path) as fh:
        statements = json.load(fh)

    def clean(v):
        return None if v is None or (isinstance(v, float) and v != v) else v

    ents, pers, ints = [], [], []
    for s in statements:
        rd = s.get("recordDetails", {})
        base = {"statement_id": s["statementId"], "statement_date": s.get("statementDate")}
        if s["recordType"] == "entity":
            ids = {i["scheme"]: clean(i.get("id")) for i in rd.get("identifiers", [])}
            reg_scheme = next((k for k in ids if k.endswith("-REG")), None)
            ents.append({
                **base,
                "reg_record_id": s.get("recordId"),
                "entity_type": rd.get("entityType", {}).get("type"),
                "entity_subtype": rd.get("entityType", {}).get("subtype"),
                "name": rd.get("name"),
                "jurisdiction": (rd.get("jurisdiction") or {}).get("code"),
                "reg_scheme": reg_scheme,
                "registry_no": ids.get(reg_scheme) if reg_scheme else None,
                "lei": ids.get("XI-LEI"),
                "listed": (rd.get("publicListing") or {}).get("hasPublicListing"),
            })
        elif s["recordType"] == "person":
            names = rd.get("names") or [{}]
            nats = rd.get("nationalities") or [{}]
            pers.append({
                **base,
                "reg_record_id": s.get("recordId"),
                "person_type": rd.get("personType"),
                "full_name": names[0].get("fullName"),
                "nationality": nats[0].get("code"),
                "pep_status": (rd.get("politicalExposure") or {}).get("status"),
            })
        elif s["recordType"] == "relationship":
            ip = rd.get("interestedParty", {})
            if "describedByEntityStatement" in ip:
                kind, party = "entity", ip["describedByEntityStatement"]
            elif "describedByPersonStatement" in ip:
                kind, party = "person", ip["describedByPersonStatement"]
            else:
                kind, party = "unspecified", None
            row = {
                **base,
                "subject_sid": rd.get("subject", {}).get("describedByEntityStatement"),
                "party_sid": party,
                "party_kind": kind,
                "unspecified_reason": ip.get("unspecifiedReason"),
            }
            interests = rd.get("interests") or [{}]
            for i in interests:
                share = i.get("share") or {}
                ints.append({
                    **row,
                    "interest_type": i.get("type"),
                    "details": i.get("details"),
                    "share": clean(share.get("exact")),
                    "start_date": i.get("startDate"),
                    "end_date": i.get("endDate"),
                })
    ents = _to_dates(pd.DataFrame(ents), DATE_COLUMNS["reg_entities"])
    pers = _to_dates(pd.DataFrame(pers), DATE_COLUMNS["reg_persons"])
    ints = _to_dates(pd.DataFrame(ints), DATE_COLUMNS["reg_interests"])
    return ents, pers, ints


# Every table is put in a fixed order defined by its content, so that results never
# depend on the row order of the source files (Dossier Appendix B: same inputs give
# the same results), and tables rebuilt from the graph line up with the originals.
CANONICAL_ORDER = {
    "third_parties": ["record_id"],
    "tp_due_diligence": ["record_id"],
    "tp_address_history": ["record_id", "valid_from", "observed_at", "raw_address"],
    "tp_services": ["record_id", "valid_from", "taxonomy_version", "category_code"],
    "tp_engagements": ["engagement_id"],
    "tp_engagement_approvals": ["approval_id"],
    "tp_payments_monthly": ["engagement_id", "month", "expense_classification", "channel"],
    "tp_institution_links": ["link_id"],
    "healthcare_institutions": ["institution_id"],
    "employees": ["employee_id"],
    "employee_roles": ["employee_id", "valid_from"],
    "markets": ["market_id"],
    "paying_entities": ["paying_entity_id"],
    "service_categories": ["category_code_v2"],
    "approval_policy_matrix": ["value_band"],
    "fmv_rate_cards": ["card_version", "country_band", "expert_role", "activity_type"],
    "country_risk_context": ["iso2", "index", "edition_year"],
    "business_events": ["start_date", "event_name"],
    "registry_l": ["child_lei", "relationship_start", "parent_legal_name"],
    "reg_entities": ["statement_id"],
    "reg_persons": ["statement_id"],
    "reg_interests": ["statement_id", "start_date", "details"],
}


def canonical(name: str, df: pd.DataFrame) -> pd.DataFrame:
    keys = [k for k in CANONICAL_ORDER.get(name, []) if k in df.columns]
    if not keys or df.empty:
        return df.reset_index(drop=True)
    return df.sort_values(keys, kind="stable", na_position="last").reset_index(drop=True)


def _delegations(df: pd.DataFrame, roles: pd.DataFrame) -> pd.DataFrame:
    """The delegation table in the loader's columns.

    The data dictionary documents from_employee_id, to_employee_id, start, end and reason. There is no
    observation time, so a delegation is taken to be known from its start (as for the other tables
    without observed_at), and the authority delegated is the approver level of the delegating
    employee's role on the start date.
    """
    if df.empty or "start" not in df.columns:
        return df
    d = _to_dates(df.rename(columns={"start": "valid_from", "end": "valid_to"}), ["valid_from", "valid_to"])
    d["observed_at"] = d["valid_from"]
    if "delegation_id" not in d.columns:
        d.insert(0, "delegation_id", [f"DL-{i:04d}" for i in range(len(d))])
    if "approver_level" not in d.columns:
        r = roles[["employee_id", "approver_level", "valid_from", "valid_to"]]
        levels = []
        for row in d.itertuples():
            m = r[(r["employee_id"] == row.from_employee_id) & (r["valid_from"] <= row.valid_from)
                  & (r["valid_to"].isna() | (r["valid_to"] >= row.valid_from))]
            levels.append(pd.to_numeric(m["approver_level"], errors="coerce").max() if len(m) else float("nan"))
        d["approver_level"] = levels
    return d


def load_bundle(bundle_dir: str | Path) -> Bundle:
    bundle_dir = Path(bundle_dir)
    hashes: dict[str, str] = {}
    tables = {}
    for name in TABLES:
        df = _read_table(bundle_dir, name, hashes)
        if df is None:
            if name not in OPTIONAL_TABLES:
                raise FileNotFoundError(f"missing required table {name} in {bundle_dir}")
            df = pd.DataFrame(columns=DELEGATION_COLUMNS)
        tables[name] = _to_dates(df, DATE_COLUMNS.get(name, []))
    tables["employee_delegations"] = _delegations(tables["employee_delegations"], tables["employee_roles"])

    reg_l = bundle_dir / "registry_L_parents.csv"
    hashes[reg_l.name] = _file_hash(reg_l)
    tables["registry_l"] = _to_dates(pd.read_csv(reg_l), DATE_COLUMNS["registry_l"])

    reg_r = bundle_dir / "registry_R" / "statements.json"
    hashes["registry_R/statements.json"] = _file_hash(reg_r)
    tables["reg_entities"], tables["reg_persons"], tables["reg_interests"] = parse_registry_r(reg_r)
    tables = {name: canonical(name, df) for name, df in tables.items()}
    return Bundle(**tables, as_of=SNAPSHOT, source_dir=str(bundle_dir), input_hashes=hashes)


# ---------------------------------------------------------------------------
# as_of(t)
# ---------------------------------------------------------------------------
# For each table: the column that says when the row became known, and the
# end-of-validity columns that must be blanked when they fall after t.
# Tables without an observation time use a documented proxy.
AS_OF_RULES: dict[str, tuple[str | None, list[str], str]] = {
    # table: (knowledge column, end columns to censor, assumption note)
    "third_parties": ("onboarding_date", ["terminated_date"], "record known from onboarding"),
    "tp_address_history": ("observed_at", ["valid_to"], "observed_at"),
    "tp_services": ("valid_from", ["valid_to"], "no observed_at: known from valid_from"),
    "tp_engagements": ("observed_at", ["end_date"], "observed_at"),
    "tp_engagement_approvals": ("observed_at", [], "observed_at"),
    "tp_institution_links": ("valid_from", ["valid_to"], "no observed_at: known from valid_from"),
    "employee_roles": ("valid_from", ["valid_to"], "no observed_at: known from valid_from"),
    "employee_delegations": ("observed_at", ["valid_to"], "observed_at"),
    "country_risk_context": ("as_of_date", ["valid_to"], "edition as_of_date"),
    "business_events": ("start_date", [], "event known from its start"),
    "registry_l": ("relationship_start", [], "no observed_at: known from relationship_start"),
    "reg_entities": ("statement_date", [], "statementDate = publication"),
    "reg_persons": ("statement_date", [], "statementDate = publication"),
    "reg_interests": ("statement_date", ["end_date"], "statementDate = publication"),
}


def month_available(month: pd.Series) -> pd.Series:
    """Monthly spend is treated as available the day after the month ends (our convention)."""
    return month + pd.offsets.MonthBegin(1)


def as_of(bundle: Bundle, t) -> Bundle:
    """The history as it was known at time t.

    Keeps rows recorded by t, and blanks end dates that fall after t (at time t
    those relationships were still open). Detectors decide for themselves
    whether they need rows valid *at* t (`valid_at`) or the full history up to t.
    """
    t = pd.Timestamp(t)
    out = {}
    for name, df in bundle.tables().items():
        rule = AS_OF_RULES.get(name)
        if rule is None:
            out[name] = df
            continue
        known_col, end_cols, _ = rule
        keep = df[known_col].isna() | (df[known_col] <= t)
        d = df.loc[keep].copy()
        for c in end_cols:
            d.loc[d[c] > t, c] = pd.NaT
        out[name] = d

    # Interests only count once they have started.
    ri = out["reg_interests"]
    out["reg_interests"] = ri[ri["start_date"].isna() | (ri["start_date"] <= t)]

    # Status is a snapshot field: recompute it from the censored termination date.
    tp = out["third_parties"].copy()
    tp["status"] = tp["terminated_date"].notna().map({True: "terminated", False: "active"})
    out["third_parties"] = tp
    live = set(tp["record_id"])
    for name in ["tp_due_diligence", "tp_address_history", "tp_services", "tp_engagements",
                 "tp_institution_links"]:
        out[name] = out[name][out[name]["record_id"].isin(live)]

    eng = set(out["tp_engagements"]["engagement_id"])
    ap = out["tp_engagement_approvals"]
    out["tp_engagement_approvals"] = ap[ap["engagement_id"].isin(eng)]

    pay = out["tp_payments_monthly"]
    out["tp_payments_monthly"] = pay[(month_available(pay["month"]) <= t + pd.Timedelta(days=1))
                                     & pay["engagement_id"].isin(eng)]

    ents = set(out["reg_entities"]["statement_id"]) | set(out["reg_persons"]["statement_id"])
    ri = out["reg_interests"]
    out["reg_interests"] = ri[ri["subject_sid"].isin(ents)
                              & (ri["party_sid"].isna() | ri["party_sid"].isin(ents))]

    return replace(bundle, **out, as_of=t)


def valid_at(df: pd.DataFrame, t, start: str = "valid_from", end: str = "valid_to") -> pd.DataFrame:
    """Rows whose validity interval contains t. End dates are treated as exclusive."""
    t = pd.Timestamp(t)
    return df[(df[start] <= t) & (df[end].isna() | (df[end] > t))]

In [4]:
%%module meridian.resolution
"""Normalisation and entity resolution.

Every match keeps its method and a confidence. Only unambiguous matches are
accepted; ambiguous ones are kept as `candidate` rows and never merged.
"""
from __future__ import annotations

import re
from collections import defaultdict
from difflib import SequenceMatcher

import pandas as pd
from unidecode import unidecode

from .data import Bundle

# ---------------------------------------------------------------------------
# Names
# ---------------------------------------------------------------------------
# Multi-token legal forms first so "co ltd" is removed as one unit.
LEGAL_FORMS = [
    "sp z o o", "s a r l", "s de r l", "s a de c v", "pte ltd", "pty ltd", "sdn bhd", "co ltd", "company limited",
    "s p a", "s r l", "s a s", "s a", "b v", "n v", "d o o", "a s", "o o o",
    "ltd", "limited", "inc", "incorporated", "corp", "corporation", "llc", "llp", "lp", "plc", "gmbh", "ag",
    "sa", "sas", "sarl", "srl", "bv", "nv", "spa", "oy", "ab", "as", "kk", "ltda", "sl", "kft", "zrt", "ooo",
    "ao", "doo", "se", "co", "company", "pte", "pty", "bhd", "tbk", "pt", "cv", "sc", "sae", "jsc", "pjsc",
    "llc", "fzco", "fze", "wll", "ou", "uab", "sia", "eood", "ad", "dd", "sro", "as", "aps", "asa",
]
_LEGAL_TOKENS = sorted({tuple(f.split()) for f in LEGAL_FORMS}, key=len, reverse=True)
# Duplicate-record markers seen in the vendor master.
_SUFFIX_RE = re.compile(r"\((?:branch|use this one|old|duplicate|dup|do not use|\d+)\)", re.I)
_PUNCT_RE = re.compile(r"[^0-9a-z]+")
# Honorifics on individual names.
_TITLES = {"dr", "drg", "prof", "mr", "mrs", "ms", "mx", "sr", "sra", "dra", "ir", "h", "hj", "dott", "ing"}


def translit(s: str) -> str:
    return unidecode(s or "")


def name_tokens(name: str) -> list[str]:
    s = _SUFFIX_RE.sub(" ", translit(name).lower())
    return _PUNCT_RE.sub(" ", s).split()


def strip_legal_form(tokens: list[str]) -> tuple[list[str], str]:
    """Remove trailing legal forms; return (core tokens, removed legal form)."""
    removed = []
    changed = True
    while changed and tokens:
        changed = False
        for form in _LEGAL_TOKENS:
            n = len(form)
            if len(tokens) > n and tuple(tokens[-n:]) == form:
                removed = list(form) + removed
                tokens = tokens[:-n]
                changed = True
                break
    return tokens, " ".join(removed)


def name_key(name: str) -> str:
    """Name without punctuation, script, case, legal form or duplicate markers."""
    core, _ = strip_legal_form(name_tokens(name))
    return " ".join(core)


def person_key(name: str) -> str:
    """Order-insensitive person name key without honorifics."""
    toks = [t for t in name_tokens(name) if t not in _TITLES]
    return " ".join(sorted(toks))


# ---------------------------------------------------------------------------
# Addresses and contacts
# ---------------------------------------------------------------------------
_ADDR_ABBREV = {
    "ste": "suite", "st": "street", "rd": "road", "ave": "avenue", "av": "avenue", "blvd": "boulevard",
    "bd": "boulevard", "jl": "jalan", "jln": "jalan", "fl": "floor", "flr": "floor", "apt": "apartment",
    "no": "", "nr": "", "piso": "floor", "etage": "floor", "bldg": "building", "dr": "drive",
    "ln": "lane", "hwy": "highway", "sq": "square", "pl": "place", "ct": "court", "cres": "crescent",
    "ul": "ulica", "str": "strasse", "c": "calle",
}


def address_tokens(raw: str) -> list[str]:
    s = _PUNCT_RE.sub(" ", translit(raw).lower())
    out = []
    for tok in s.split():
        tok = _ADDR_ABBREV.get(tok, tok)
        if tok:
            out.append(tok)
    return out


def address_key(raw: str, country: str | None = None) -> str:
    """Order-insensitive key: sorted tokens after expanding abbreviations.

    Unit numbers stay in the key (keep door, building and suite).
    Field order differs across records ("Suite 307, Camino ..." vs
    "CAMINO ... STE 307"), so tokens are sorted.
    """
    toks = sorted(address_tokens(raw))
    return (country or "").upper() + ":" + " ".join(toks)


def phone_key(phone: str | None) -> str | None:
    if not phone or not isinstance(phone, str):
        return None
    main = re.split(r"(?:x|ext\.?)\s*\d+$", phone.strip(), flags=re.I)[0]
    digits = re.sub(r"\D", "", main)
    if len(digits) < 7:
        return None
    return digits[-9:]  # national significant part; tolerant of +CC / trunk-0 variants


def normalise_addresses(b: Bundle) -> pd.DataFrame:
    a = b.tp_address_history.copy()
    a["addr_key"] = [address_key(r, c) for r, c in zip(a["raw_address"], a["address_country"])]
    return a


def normalise_contacts(b: Bundle) -> pd.DataFrame:
    tp = b.third_parties[["record_id", "phone", "email_domain"]].copy()
    tp["phone_key"] = tp["phone"].map(phone_key)
    tp["domain_key"] = tp["email_domain"].str.lower().str.strip()
    return tp


# ---------------------------------------------------------------------------
# Vendor <-> registry matching
# ---------------------------------------------------------------------------
def match_registry(b: Bundle) -> pd.DataFrame:
    """Link vendor records to registry R entity statements.

    Order: registry number, then LEI, then transliterated name within the same
    jurisdiction. A name match is accepted only if it is unique on both sides
    within the jurisdiction; otherwise every pair is kept as a candidate.
    Returns one row per (record_id, statement_id) with method, confidence, status.
    """
    tp = b.third_parties[b.third_parties["party_type"] == "organization"]
    re_ = b.reg_entities
    rows = []
    done: set[str] = set()

    by_reg = tp.dropna(subset=["registry_no"]).merge(
        re_.dropna(subset=["registry_no"])[["registry_no", "statement_id"]], on="registry_no")
    for r in by_reg.itertuples():
        rows.append((r.record_id, r.statement_id, "registry_no", 1.0, "accepted"))
        done.add(r.record_id)

    by_lei = tp[~tp["record_id"].isin(done)].dropna(subset=["lei"]).merge(
        re_.dropna(subset=["lei"])[["lei", "statement_id"]], on="lei")
    for r in by_lei.itertuples():
        rows.append((r.record_id, r.statement_id, "lei", 1.0, "accepted"))
        done.add(r.record_id)

    rows += match_registry_names(tp[~tp["record_id"].isin(done)], re_)
    return pd.DataFrame(rows, columns=["record_id", "statement_id", "method", "confidence", "status"])


_FORM_WORDS = {t for f in LEGAL_FORMS for t in f.split() if len(t) >= 3}


def fuzzy_key(name: str) -> str:
    """name_key, also without misspelt trailing legal forms ("Limietd", "gLtd")."""
    core = name_key(name).split()
    while len(core) > 1 and len(core[-1]) >= 3 and max(
            SequenceMatcher(None, core[-1], f).ratio() for f in _FORM_WORDS) >= 0.8:
        core = core[:-1]
    return " ".join(core)


def _word_sim(a: str, b: str) -> float:
    if a == b:
        return 1.0
    return SequenceMatcher(None, a, b).ratio() if min(len(a), len(b)) >= 4 else 0.0


def match_registry_names(tp: pd.DataFrame, re_: pd.DataFrame, fuzzy: float | None = 0.9,
                         margin: float = 0.05, candidate: float = 0.85) -> list[tuple]:
    """Transliterated name match within the jurisdiction; accepted only when the name is unique there.

    Vendors without an exact match are compared with the jurisdiction's entities
    that share a name word (`fuzzy` set): the best name is accepted when its
    similarity is at least `fuzzy` and at least `margin` above any other name;
    names scoring at least `candidate` stay candidates. Typos, truncations and
    abbreviations in the register ("Thopmson Freight", "Hua Medical Svcs") are
    matched this way.
    """
    ent = re_.assign(key=re_["name"].map(name_key), fkey=re_["name"].map(fuzzy_key))
    ent_idx = defaultdict(list)
    word_idx = defaultdict(set)
    for e in ent.itertuples():
        if e.key:
            ent_idx[(e.jurisdiction, e.key)].append(e.statement_id)
        for w in e.fkey.split():
            word_idx[(e.jurisdiction, w)].add(e.statement_id)
    fkey_of = dict(zip(ent["statement_id"], ent["fkey"]))
    rows = []
    for v in tp.assign(key=lambda d: d["legal_name"].map(name_key)).itertuples():
        ents = ent_idx.get((v.country, v.key), [])
        # Duplicate vendor records of one entity legitimately share a registry entity.
        accepted = len(ents) == 1
        for sid in ents:
            rows.append((v.record_id, sid, "name_jurisdiction", 0.9 if accepted else 0.5,
                         "accepted" if accepted else "candidate"))
        if ents or fuzzy is None or not v.key:
            continue
        fk = fuzzy_key(v.legal_name)
        words = fk.split()
        pool = set().union(*(word_idx.get((v.country, w), set()) for w in words))
        # The vendor's rarest word in the jurisdiction (its distinctive name) must match:
        # "Hayes Clinical Research Center" is not "Ellis Clinical Research Center".
        rare = min(words, key=lambda w: (len(word_idx.get((v.country, w), ())), -len(w)))
        by_name = defaultdict(list)
        for sid in pool:
            if any(_word_sim(rare, w) >= 0.8 for w in fkey_of[sid].split()):
                by_name[fkey_of[sid]].append(sid)
        scored = sorted(((SequenceMatcher(None, fk, k).ratio(), k) for k in by_name), reverse=True)
        if not scored or scored[0][0] < candidate:
            continue
        best, name = scored[0]
        ok = best >= fuzzy and (len(scored) == 1 or best - scored[1][0] >= margin) and len(by_name[name]) == 1
        for sid in by_name[name]:
            rows.append((v.record_id, sid, "name_fuzzy", round(0.8 * best, 3) if ok else 0.4,
                         "accepted" if ok else "candidate"))
    return rows


def name_match_check(b: Bundle) -> dict:
    """Label-free check of name matching: re-match the identifier-linked vendors by name alone.

    The identifier link is taken as the answer. Returns how many of those vendors
    the name match accepts, and how many of the accepted matches agree with it.
    """
    m = match_registry(b)
    by_id = m[m["method"].isin(["registry_no", "lei"])]
    truth = by_id.groupby("record_id")["statement_id"].agg(set)
    tp = b.third_parties[b.third_parties["record_id"].isin(truth.index)]
    names = pd.DataFrame(match_registry_names(tp, b.reg_entities),
                         columns=["record_id", "statement_id", "method", "confidence", "status"])
    acc = names[names["status"] == "accepted"]
    out = {"identifier_linked": len(truth)}
    for method, g in acc.groupby("method"):
        out[f"{method}_accepted"] = len(g)
        out[f"{method}_agrees"] = int(sum(sid in truth[r] for r, sid in zip(g["record_id"], g["statement_id"])))
    out["candidate_only"] = int(len(set(names["record_id"]) - set(acc["record_id"])))
    out["no_name_match"] = int(len(set(truth.index) - set(names["record_id"])))
    return out


def match_persons(b: Bundle) -> pd.DataFrame:
    """Link individual experts to registry natural persons by name and nationality."""
    tp = b.third_parties[b.third_parties["party_type"] == "individual"]
    pers = b.reg_persons.assign(key=b.reg_persons["full_name"].map(person_key))
    idx = defaultdict(list)
    for p in pers.itertuples():
        idx[(p.nationality, p.key)].append(p.statement_id)
    rows = []
    for v in tp.itertuples():
        hits = idx.get((v.country, person_key(v.legal_name)), [])
        for sid in hits:
            rows.append((v.record_id, sid, "person_name_nationality", 0.9 if len(hits) == 1 else 0.5,
                         "accepted" if len(hits) == 1 else "candidate"))
    return pd.DataFrame(rows, columns=["record_id", "statement_id", "method", "confidence", "status"])


# ---------------------------------------------------------------------------
# Duplicate vendor records -> ResolvedEntity
# ---------------------------------------------------------------------------
def resolve_duplicates(b: Bundle, name_merge: str = "corroborated") -> tuple[pd.DataFrame, pd.DataFrame]:
    """Group vendor-master records that are the same real third party.

    Merge evidence:
      - same tax_id, registry number or LEI (confidence 1.0)
      - same name key in the same country, for organisations, when no
        identifier conflicts (both present and different) and the records
        also share a normalised address or phone (confidence 0.8)
    Same-name pairs without that corroboration are kept as IDENTITY_CANDIDATE
    and not merged: on the starter data they are mostly sibling companies of
    one declared group, some with different registry numbers.
    `name_merge` sets the strictness for sensitivity runs: "none" (identifiers
    only), "corroborated" (default), "any" (same name alone, unless identifiers conflict).
    Shared address, sponsor or owner alone is never enough.
    Returns (entity map record_id -> entity_id, pair evidence table).
    """
    tp = b.third_parties.copy()
    tp["key"] = tp["legal_name"].map(name_key)
    tp["phone_key"] = tp["phone"].map(phone_key)
    addrs = normalise_addresses(b).groupby("record_id")["addr_key"].apply(set).to_dict()
    info = tp.set_index("record_id")
    parent = {r: r for r in tp["record_id"]}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, c):
        ra, rb = find(a), find(c)
        if ra != rb:
            parent[max(ra, rb)] = min(ra, rb)

    pairs = []
    for col in ["tax_id", "registry_no", "lei"]:
        for _, g in tp.dropna(subset=[col]).groupby(col)["record_id"]:
            recs = sorted(set(g))
            for other in recs[1:]:
                pairs.append((recs[0], other, col, 1.0, "SAME_AS"))
                union(recs[0], other)

    def conflict(a, c) -> bool:
        for col in ["registry_no", "lei"]:
            va, vc = info.at[a, col], info.at[c, col]
            if isinstance(va, str) and isinstance(vc, str) and va != vc:
                return True
        return False

    orgs = tp[(tp["party_type"] == "organization") & (tp["key"] != "")]
    for _, g in orgs.groupby(["country", "key"])["record_id"]:
        recs = sorted(set(g))
        for i, a in enumerate(recs):
            for c in recs[i + 1:]:
                if find(a) == find(c):
                    continue
                shared_addr = bool(addrs.get(a, set()) & addrs.get(c, set()))
                pa, pc = info.at[a, "phone_key"], info.at[c, "phone_key"]
                shared_phone = isinstance(pa, str) and pa == pc
                if name_merge == "none":
                    ok = False
                elif name_merge == "any":
                    ok = not conflict(a, c)
                else:
                    ok = not conflict(a, c) and (shared_addr or shared_phone)
                if ok:
                    pairs.append((a, c, "name_country+address/phone", 0.8, "SAME_AS"))
                    union(a, c)
                else:
                    why = "identifier conflict" if conflict(a, c) else "no shared address/phone"
                    pairs.append((a, c, f"name_country ({why})", 0.4, "IDENTITY_CANDIDATE"))

    emap = pd.DataFrame({"record_id": tp["record_id"]})
    emap["entity_id"] = emap["record_id"].map(lambda r: "ENT-" + find(r).removeprefix("TP-"))
    emap["n_records"] = emap.groupby("entity_id")["record_id"].transform("size")
    return emap, pd.DataFrame(pairs, columns=["record_a", "record_b", "reason", "confidence", "relation"])

In [5]:
%%module meridian.context
"""Per-time-point context shared by all detectors.

Two ways to build it:
  Context(bundle)                    from source tables; used to build the graph
                                     (normalisation and entity resolution happen here)
  Context.from_graph(graph, ref)     from the canonical graph; used by detection, so
                                     detectors see only what the graph holds

Derived views are computed lazily and cached, so detectors only pay for what they use.
"""
from __future__ import annotations

import bisect
from collections import defaultdict
from functools import cached_property

import numpy as np
import pandas as pd

from . import resolution as res
from .data import Bundle


# Which business events can explain which pattern kinds (from the event descriptions in
# business_events). Kinds: onboarding, sponsor, spend, termination, category.
EVENT_RULES = {
    "covid": {"kinds": {"onboarding", "sponsor", "spend"}, "parents": {"SCM"}, "categories": {"RND.CSL"}},
    "erp migration": {"kinds": {"sponsor"}, "roles": {"system_default"}},
    "demerger": {"kinds": {"termination"}, "units": {"CONSUMER"}},
    "acquisition integration": {"kinds": {"onboarding", "sponsor"}},
    "vaccine launch": {"kinds": {"onboarding", "sponsor", "spend"}, "parents": {"HCP", "EVT", "MED", "PAT"},
                       "categories": {"COM.MACC"}},
    "taxonomy recode": {"kinds": {"category"}},
    "rationalization": {"kinds": {"onboarding", "sponsor", "termination"}, "categories": {"GNA.IT"}},
}


class Context:
    def __init__(self, bundle: Bundle, cfg: dict | None = None, graph=None, precomputed: dict | None = None):
        self.b = bundle
        self.t = bundle.as_of
        self.cfg = cfg or {}
        self.graph = graph
        self._pre = precomputed or {}

    @classmethod
    def for_detection(cls, view: Bundle, cfg: dict | None = None, drop_relations=(), drop_activity: bool = False):
        """Build the canonical graph from an as_of view, then a Context that reads only the graph.

        `drop_relations` / `drop_activity` remove edge types or the payment activity from
        the graph before detection (relation ablation); entity resolution is unaffected.
        """
        from .graph import build_graph, without
        graph = build_graph(cls(view, cfg))
        if drop_relations or drop_activity:
            graph = without(graph, drop_relations, drop_activity)
        return cls.from_graph(graph, view, cfg)

    @classmethod
    def from_graph(cls, graph, reference: Bundle, cfg: dict | None = None) -> "Context":
        """Context whose tables, entity resolution and normalised keys all come from `graph`.

        `reference` supplies only the reference tables (taxonomy, policy, rate cards,
        country context, business events) and the input hashes.
        """
        from .graph_views import bundle_from_graph
        bundle, pre = bundle_from_graph(graph, reference)
        return cls(bundle, cfg, graph=graph, precomputed=pre)

    @property
    def use_ownership(self) -> bool:
        return bool(self.cfg.get("features", {}).get("ownership", True))

    def param(self, detector: str, key: str, default):
        return self.cfg.get("params", {}).get(detector, {}).get(key, default)

    # ------------------------------------------------------------------
    # Vendors and entity resolution
    # ------------------------------------------------------------------
    @cached_property
    def vendors(self) -> pd.DataFrame:
        return self.b.third_parties.set_index("record_id", drop=False)

    @property
    def rcfg(self) -> dict:
        """Entity-resolution settings (strictness and perturbation for sensitivity runs)."""
        return self.cfg.get("resolution", {})

    @cached_property
    def _resolution(self):
        if "entity_map" in self._pre:
            return self._pre["entity_map"], self._pre["identity_pairs"]
        return res.resolve_duplicates(self.b, self.rcfg.get("name_merge", "corroborated"))

    @cached_property
    def entity_map(self) -> pd.DataFrame:
        return self._resolution[0]

    @cached_property
    def identity_pairs(self) -> pd.DataFrame:
        return self._resolution[1]

    @cached_property
    def entity_of(self) -> dict:
        return dict(zip(self.entity_map["record_id"], self.entity_map["entity_id"]))

    @cached_property
    def records_of(self) -> dict:
        out = defaultdict(set)
        for r, e in self.entity_of.items():
            out[e].add(r)
        return out

    def expand(self, records) -> frozenset:
        """All vendor-master records of the entities behind `records`; cases list every record of an entity."""
        out = set()
        for r in records:
            out |= self.records_of.get(self.entity_of.get(r, r), {r})
        return frozenset(out)

    def n_entities(self, records) -> int:
        return len({self.entity_of.get(r, r) for r in records})

    # ------------------------------------------------------------------
    # Addresses and contacts
    # ------------------------------------------------------------------
    @cached_property
    def addresses(self) -> pd.DataFrame:
        if "addresses" in self._pre:
            return self._pre["addresses"]
        return res.normalise_addresses(self.b)

    @cached_property
    def contacts(self) -> pd.DataFrame:
        if "contacts" in self._pre:
            return self._pre["contacts"]
        return res.normalise_contacts(self.b)

    # ------------------------------------------------------------------
    # Service categories (v1 -> v2 disambiguation: 11 of 25 v1 codes map to several v2 codes)
    # ------------------------------------------------------------------
    @cached_property
    def categories(self) -> pd.DataFrame:
        sc = self.b.service_categories.copy()
        sc["flag_set"] = sc["flags"].fillna("").map(lambda s: frozenset(f for f in s.split(";") if f))
        return sc.set_index("category_code_v2")

    @cached_property
    def _v1_to_v2(self) -> dict:
        return self.b.service_categories.groupby("category_code_v1")["category_code_v2"].apply(tuple).to_dict()

    @cached_property
    def _vendor_v2(self) -> dict:
        s = self.b.tp_services
        return s[s["taxonomy_version"] == "v2"].groupby("record_id")["category_code"].apply(set).to_dict()

    def to_v2(self, record_id: str, code: str, version: str) -> tuple[tuple, bool]:
        """Map a category to v2. Returns (candidate v2 codes, certain)."""
        if version == "v2":
            return (code,), True
        cands = self._v1_to_v2.get(code, (code,))
        if len(cands) == 1:
            return cands, True
        own = tuple(c for c in cands if c in self._vendor_v2.get(record_id, ()))
        if len(own) == 1:
            return own, True
        return (own or cands), False

    def flags_of(self, v2_codes) -> tuple[frozenset, frozenset]:
        """(certain flags, possible flags) over candidate v2 codes."""
        sets = [self.categories.at[c, "flag_set"] for c in v2_codes if c in self.categories.index]
        if not sets:
            return frozenset(), frozenset()
        return frozenset.intersection(*sets), frozenset.union(*sets)

    def parent_of(self, v2_codes) -> str | None:
        parents = {self.categories.at[c, "parent"] for c in v2_codes if c in self.categories.index}
        return parents.pop() if len(parents) == 1 else None

    @cached_property
    def engagements(self) -> pd.DataFrame:
        """Engagements with v2 category, flags, parent and sponsor role attached."""
        e = self.b.tp_engagements.copy()
        mapped = [self.to_v2(r, c, v) for r, c, v in zip(e["record_id"], e["service_category"], e["taxonomy_version"])]
        e["cat_v2"] = [m[0] for m in mapped]
        e["cat_certain"] = [m[1] for m in mapped]
        e["cat_primary"] = [m[0][0] if m[1] else None for m in mapped]
        fl = [self.flags_of(m[0]) for m in mapped]
        e["cat_flags"] = [f[0] for f in fl]
        e["cat_flags_possible"] = [f[1] for f in fl]
        e["parent"] = [self.parent_of(m[0]) for m in mapped]
        roles = self.roles[["employee_id", "role", "home_market", "scope_markets", "scope_categories",
                            "approver_level", "franchise"]].add_prefix("sp_")
        e = e.merge(roles, left_on="sponsor_employee_id", right_on="sp_employee_id", how="left")
        return e.drop(columns=["sp_employee_id"])

    @cached_property
    def services(self) -> pd.DataFrame:
        s = self.b.tp_services.copy()
        mapped = [self.to_v2(r, c, v) for r, c, v in zip(s["record_id"], s["category_code"], s["taxonomy_version"])]
        s["cat_v2"] = [m[0] for m in mapped]
        s["cat_certain"] = [m[1] for m in mapped]
        s["cat_flags"] = [self.flags_of(m[0])[0] for m in mapped]
        s["parent"] = [self.parent_of(m[0]) for m in mapped]
        return s

    # ------------------------------------------------------------------
    # People and approvals
    # ------------------------------------------------------------------
    @cached_property
    def roles(self) -> pd.DataFrame:
        return self.b.employee_roles

    @cached_property
    def role_of(self) -> dict:
        return self.roles.set_index("employee_id")["role"].to_dict()

    @cached_property
    def approvals(self) -> pd.DataFrame:
        """Approvals joined to engagement, band requirement and approver role."""
        ap = self.b.tp_engagement_approvals.merge(
            self.engagements[["engagement_id", "record_id", "value_band", "sponsor_employee_id", "market",
                              "start_date", "cat_flags", "parent", "sp_role"]], on="engagement_id")
        ap = ap.merge(self.b.approval_policy_matrix[["value_band", "required_approver_level"]], on="value_band",
                      how="left")
        r = self.roles[["employee_id", "role", "approver_level", "home_market", "scope_markets", "valid_from",
                        "valid_to"]].add_prefix("ap_")
        return ap.merge(r, left_on="approver_employee_id", right_on="ap_employee_id", how="left").drop(
            columns=["ap_employee_id"])

    @cached_property
    def markets(self) -> pd.DataFrame:
        return self.b.markets.set_index("market_id")

    def in_scope(self, scope: str | None, market: str) -> bool:
        """Does an employee scope string ('*', 'R:EUROPE', 'S:EU_WEST', 'US;CA') cover market?"""
        if not isinstance(scope, str):
            return False
        m = self.markets.loc[market] if market in self.markets.index else None
        for s in scope.split(";"):
            s = s.strip()
            if s == "*" or s == market:
                return True
            if m is not None and s.startswith("R:") and s[2:] == m["region"]:
                return True
            if m is not None and s.startswith("S:") and s[2:] == m["subregion"]:
                return True
        return False

    # ------------------------------------------------------------------
    # Business events (benign explanations)
    # ------------------------------------------------------------------
    def events_overlapping(self, lo, hi, market: str | None = None) -> list[str]:
        ev = self.b.business_events
        lo, hi = pd.Timestamp(lo), pd.Timestamp(hi)
        end = ev["end_date"].fillna(ev["start_date"] + pd.Timedelta(days=90))
        hit = ev[(ev["start_date"] <= hi) & (end >= lo)]
        if market and len(hit):
            hit = hit[hit["markets"].map(lambda s: s == "ALL" or market in str(s).split("|")).astype(bool)]
        return list(hit["event_name"])

    def benign_events(self, kind: str, lo, hi, market: str | None = None, cats=(), roles=(), units=()) -> list[str]:
        """Business events that plausibly explain a pattern of this kind.

        `cats` is a list of (parent, v2 category) pairs for the vendors involved.
        An event only counts when its rule covers the pattern kind and, where
        the rule names categories, business units or roles, most of the
        involved vendors fall under them. Without that, a two-year event such
        as COVID would excuse every burst in the period.
        """
        out = []
        cats = list(cats)
        for name in self.events_overlapping(lo, hi, market):
            rule = next((r for key, r in EVENT_RULES.items() if key in name.lower()), None)
            if rule is None or kind not in rule["kinds"]:
                continue
            ok = True
            if "parents" in rule or "categories" in rule:
                hits = [p in rule.get("parents", ()) or c in rule.get("categories", ()) for p, c in cats]
                ok = bool(hits) and sum(hits) >= 0.5 * len(hits)
            if ok and "roles" in rule:
                ok = any(r in rule["roles"] for r in roles)
            if ok and "units" in rule:
                ok = any(u in rule["units"] for u in units)
            if ok:
                out.append(f"business event: {name}")
        return out

    # ------------------------------------------------------------------
    # Country context
    # ------------------------------------------------------------------
    def country_index(self, iso2: str, index: str, at=None):
        """Tier of the edition current at `at` (default t)."""
        at = pd.Timestamp(at) if at is not None else self.t
        c = self.b.country_risk_context
        c = c[(c["iso2"] == iso2) & (c["index"] == index) & (c["as_of_date"] <= at)]
        if len(c) == 0:
            return None
        return c.sort_values("as_of_date").iloc[-1]["tier"]

    # ------------------------------------------------------------------
    # Registry and ownership
    # ------------------------------------------------------------------
    @cached_property
    def reg_matches(self) -> pd.DataFrame:
        if "reg_matches" in self._pre:  # already shaped by the resolution settings when the graph was built
            return self._pre["reg_matches"]
        m = pd.concat([res.match_registry(self.b), res.match_persons(self.b)], ignore_index=True)
        rc = self.rcfg
        by_name = ~m["method"].isin(["registry_no", "lei"])
        if not rc.get("registry_name_match", True):
            m = m[~by_name]
            by_name = by_name[~by_name]
        elif not rc.get("fuzzy_names", True):
            keep = m["method"] != "name_fuzzy"
            m, by_name = m[keep], by_name[keep]
        if rc.get("accept_candidates", False):
            m = m.assign(status="accepted")
        drop = rc.get("drop_name_matches", 0.0)
        if drop:
            rng = np.random.default_rng(rc.get("perturb_seed", 0))
            m = m[~(by_name & (m["status"] == "accepted") & (rng.random(len(m)) < drop))]
        return m

    @cached_property
    def reg_entity_of(self) -> dict:
        """record_id -> accepted registry entity statement id (organisations)."""
        m = self.reg_matches
        m = m[(m["status"] == "accepted") & ~m["method"].str.startswith("person")]
        return m.drop_duplicates("record_id").set_index("record_id")["statement_id"].to_dict()

    @cached_property
    def reg_person_of(self) -> dict:
        m = self.reg_matches
        m = m[(m["status"] == "accepted") & m["method"].str.startswith("person")]
        return m.drop_duplicates("record_id").set_index("record_id")["statement_id"].to_dict()

    @cached_property
    def records_of_reg(self) -> dict:
        out = defaultdict(set)
        for r, s in {**self.reg_entity_of, **self.reg_person_of}.items():
            out[s].add(r)
        return out

    @cached_property
    def reg_nodes(self) -> pd.DataFrame:
        e = self.b.reg_entities.assign(kind="entity")
        p = self.b.reg_persons.assign(kind="person", name=self.b.reg_persons["full_name"],
                                      jurisdiction=self.b.reg_persons["nationality"])
        cols = ["statement_id", "kind", "name", "jurisdiction", "entity_type", "entity_subtype", "pep_status",
                "statement_date", "listed"]
        return pd.concat([e.reindex(columns=cols), p.reindex(columns=cols)]).set_index("statement_id")

    @cached_property
    def _interest_rows(self) -> list[dict]:
        return self.b.reg_interests.to_dict("records")

    @cached_property
    def _interests_by_subject(self) -> dict:
        out = defaultdict(list)
        for r in self._interest_rows:
            out[r["subject_sid"]].append(r)
        return out

    def owners_at(self, at=None) -> dict:
        """subject statement -> interest rows valid at `at` (valid time; default t).

        Knowledge is already cut at t by as_of, so an earlier `at` shows the
        structure as it stood then, as far as it was published by t.
        """
        at = self.t if at is None else pd.Timestamp(at)
        # The valid set only changes on a start or end date, so dates between two
        # consecutive change dates share one result.
        changes = self._interest_changes
        epoch = bisect.bisect_right(changes, at) - 1
        key = changes[epoch] if epoch >= 0 else None
        cache = self.__dict__.setdefault("_owners_cache", {})
        if key not in cache:
            ref = at if key is None else key
            out = defaultdict(list)
            for r in self._interest_rows:
                s, e = r["start_date"], r["end_date"]
                if (pd.isna(s) or s <= ref) and (pd.isna(e) or e > ref):
                    out[r["subject_sid"]].append(r)
            cache[key] = out
        return cache[key]

    @cached_property
    def _interest_changes(self) -> list:
        return sorted({d for r in self._interest_rows for d in (r["start_date"], r["end_date"]) if pd.notna(d)})

    def ownership_paths(self, sid: str, at=None, max_depth: int = 12) -> list[list[dict]]:
        """All upward chains from an entity to a person, a chain stop, or a dead end.

        Each path is a list of interest rows, bottom-up. Cycles are cut and
        marked. A path ending on an entity with no recorded owner is a gap
        (missing data), not evidence of concealment.
        """
        owners = self.owners_at(at)
        paths = []

        def walk(node, path, seen):
            ups = owners.get(node, [])
            if not ups or len(path) >= max_depth:
                paths.append(path)
                return
            for u in ups:
                nxt = u["party_sid"]
                if nxt is None or u["party_kind"] != "entity":
                    paths.append(path + [u])
                elif nxt in seen:
                    paths.append(path + [dict(u, cycle=True)])
                else:
                    walk(nxt, path + [u], seen | {nxt})

        walk(sid, [], {sid})
        return [p for p in paths if p]

    def ultimate_persons(self, sid: str, at=None) -> dict:
        """person statement -> shortest path length from entity `sid`."""
        out = {}
        for p in self.ownership_paths(sid, at):
            last = p[-1]
            if last["party_kind"] == "person":
                out[last["party_sid"]] = min(out.get(last["party_sid"], 99), len(p))
        return out

    def ownership_dates(self, sid: str) -> list[pd.Timestamp]:
        """Dates at which the upward structure of `sid` may change (interest starts and ends)."""
        seen, frontier, dates = {sid}, [sid], set()
        while frontier:
            node = frontier.pop()
            for r in self._interests_by_subject.get(node, []):
                for d in (r["start_date"], r["end_date"]):
                    if pd.notna(d):
                        dates.add(pd.Timestamp(d))
                nxt = r["party_sid"]
                if nxt and r["party_kind"] == "entity" and nxt not in seen:
                    seen.add(nxt)
                    frontier.append(nxt)
        return sorted(dates)

    # ------------------------------------------------------------------
    # Activity
    # ------------------------------------------------------------------
    @cached_property
    def monthly_spend(self) -> pd.DataFrame:
        """record x month total spend."""
        p = self.b.tp_payments_monthly
        return p.groupby(["record_id", "month"], as_index=False)["amount_usd"].sum()

    @cached_property
    def links(self) -> pd.DataFrame:
        li = self.b.tp_institution_links.merge(
            self.b.healthcare_institutions.rename(columns={"country": "inst_country", "name": "inst_name"}),
            on="institution_id", how="left")
        return li

In [6]:
%%module meridian.graph
"""Canonical heterogeneous graph: typed node and edge tables plus a content hash.

The graph is the single input of detection: normalisation and entity
resolution read the source tables to build it, and the detectors then read
only the graph (through Context.from_graph) plus reference tables such as
FMV rate cards and the approval policy matrix. It can be inspected,
exported to graph tools and fingerprinted (Dossier Appendix B: the same
inputs, configuration and seed give the same graph hash).

Monthly payments are a time series per engagement, not a relation: they are
kept as an activity table attached to Engagement nodes.

Every edge keeps its valid time (valid_from, valid_to), observation time
(observed_at), source table and source record, and the entity-resolution
confidence where a match is involved. Missing times stay empty: no
observation date or probability is invented. The graph is built from an
as_of(t) view, so it holds only what was known at t.
"""
from __future__ import annotations

import hashlib
import io
import json
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

from .context import Context

NODE_COLUMNS = ["node_id", "node_type", "label", "source_table", "attributes"]
EDGE_COLUMNS = ["edge_id", "source_node_id", "relation_type", "target_node_id", "valid_from", "valid_to",
                "observed_at", "source_table", "source_record_id", "dataset_version", "resolution_confidence",
                "relation_properties"]

# Permitted (source type, relation, target type) triples (Dossier Appendix B, type constraints).
ALLOWED = {
    ("VendorRecord", "PART_OF", "ResolvedEntity"),
    ("VendorRecord", "SAME_AS", "VendorRecord"),
    ("VendorRecord", "IDENTITY_CANDIDATE", "VendorRecord"),
    ("VendorRecord", "LOCATED_AT", "Address"),
    ("VendorRecord", "HAS_CONTACT", "ContactPoint"),
    ("VendorRecord", "DECLARED_MEMBER_OF", "DeclaredGroup"),
    ("VendorRecord", "CONTRACTED_UNDER", "Engagement"),
    ("Engagement", "SPONSORED_BY", "Employee"),
    ("Engagement", "APPROVED_BY", "Employee"),
    ("Engagement", "FOR_SERVICE", "ServiceCategory"),
    ("VendorRecord", "PROVIDES_SERVICE", "ServiceCategory"),
    ("VendorRecord", "SERVES", "HealthcareInstitution"),
    ("VendorRecord", "SITE_OF", "HealthcareInstitution"),
    ("VendorRecord", "AFFILIATED_WITH", "HealthcareInstitution"),
    ("VendorRecord", "REGISTERED_AS", "LegalEntity"),
    ("VendorRecord", "IS_PERSON", "NaturalPerson"),
    ("LegalEntity", "OWNS", "LegalEntity"),
    ("NaturalPerson", "OWNS", "LegalEntity"),
    ("LegalEntity", "CONTROLS", "LegalEntity"),
    ("NaturalPerson", "CONTROLS", "LegalEntity"),
    ("VendorRecord", "CONSOLIDATED_BY", "AccountingParent"),
    ("VendorRecord", "ULTIMATE_OWNER", "NaturalPerson"),   # derived from OWNS/CONTROLS paths at t
}
# Derived relations summarise other edges; detection never reads them.
DERIVED = {"ULTIMATE_OWNER"}


ACTIVITY_KEY = ["node_id", "month", "expense_classification", "channel"]


@dataclass
class Graph:
    nodes: pd.DataFrame
    edges: pd.DataFrame
    activity: pd.DataFrame          # monthly payments attached to Engagement nodes
    dataset_version: str
    snapshot: pd.Timestamp
    ownership_paths: pd.DataFrame | None = None   # review table behind the ULTIMATE_OWNER edges

    @property
    def hash(self) -> str:
        return graph_hash(self.nodes, self.edges, self.activity)

    def summary(self) -> dict:
        return {
            "graph_hash": self.hash, "dataset_version": self.dataset_version, "snapshot": str(self.snapshot.date()),
            "n_nodes": len(self.nodes), "n_edges": len(self.edges), "n_activity_rows": len(self.activity),
            "nodes_by_type": self.nodes["node_type"].value_counts().sort_index().to_dict(),
            "edges_by_relation": self.edges["relation_type"].value_counts().sort_index().to_dict(),
        }


def _json(d: dict) -> str:
    """Attributes as sorted JSON; missing values (None, NaN, NaT) are left out, not invented."""
    clean = {}
    for k, v in d.items():
        if v is None or (pd.api.types.is_scalar(v) and pd.isna(v)):
            continue
        if isinstance(v, pd.Timestamp):
            v = v.date().isoformat()
        elif hasattr(v, "item"):
            v = v.item()
        elif isinstance(v, list):
            v = [json.loads(_json(x)) if isinstance(x, dict) else x for x in v]
        clean[k] = v
    return json.dumps(clean, sort_keys=True, ensure_ascii=False, default=str)


def _date(v):
    return None if v is None or pd.isna(v) else pd.Timestamp(v).date().isoformat()


class _Builder:
    def __init__(self, dataset_version: str):
        self.version = dataset_version
        self.nodes: dict[str, dict] = {}
        self.edges: list[dict] = []

    def node(self, node_id: str, node_type: str, label, source_table: str, **attrs):
        if node_id not in self.nodes:
            self.nodes[node_id] = {"node_id": node_id, "node_type": node_type, "label": str(label),
                                   "source_table": source_table, "attributes": _json(attrs)}
        return node_id

    def edge(self, src: str, rel: str, dst: str, source_table: str, source_record_id, valid_from=None, valid_to=None,
             observed_at=None, confidence=None, **props):
        self.edges.append({
            "source_node_id": src, "relation_type": rel, "target_node_id": dst,
            "valid_from": _date(valid_from), "valid_to": _date(valid_to), "observed_at": _date(observed_at),
            "source_table": source_table, "source_record_id": str(source_record_id),
            "dataset_version": self.version,
            "resolution_confidence": None if confidence is None else float(confidence),
            "relation_properties": _json(props),
        })

    def frames(self) -> tuple[pd.DataFrame, pd.DataFrame]:
        nodes = pd.DataFrame(list(self.nodes.values()), columns=NODE_COLUMNS).sort_values("node_id")
        edges = pd.DataFrame(self.edges, columns=EDGE_COLUMNS[1:])
        key = edges[["source_node_id", "relation_type", "target_node_id", "valid_from", "source_table",
                     "source_record_id", "relation_properties"]].astype(str).agg("|".join, axis=1)
        edges.insert(0, "edge_id", ["E" + hashlib.sha1(k.encode()).hexdigest()[:16] for k in key])
        edges = edges.drop_duplicates("edge_id").sort_values("edge_id")
        return nodes.reset_index(drop=True), edges.reset_index(drop=True)


def dataset_version(input_hashes: dict) -> str:
    return hashlib.sha256(json.dumps(input_hashes, sort_keys=True).encode()).hexdigest()[:12]


def build_graph(ctx: Context) -> Graph:
    """Materialise the typed relations of an as_of(t) context as node and edge tables."""
    b = _Builder(dataset_version(ctx.b.input_hashes))
    v = ctx.vendors

    # Vendor records (with their due-diligence answers) and resolved entities.
    dd = ctx.b.tp_due_diligence.set_index("record_id")
    for r in v.itertuples(index=False):
        d = dd.loc[r.record_id].to_dict() if r.record_id in dd.index else {}
        b.node(f"vendor:{r.record_id}", "VendorRecord", r.legal_name, "third_parties",
               party_type=r.party_type, country=r.country, status=r.status, onboarding_date=r.onboarding_date,
               created_date=r.created_date, terminated_date=r.terminated_date, expert_type=r.expert_type,
               remit_to_country=r.remit_to_country, tax_id=r.tax_id, lei=r.lei, registry_no=r.registry_no,
               declared_parent_group=r.declared_parent_group, phone=r.phone, email_domain=r.email_domain,
               **{f"dd_{k}" if not k.startswith("dd_") else k: val for k, val in d.items()})
    em = ctx.entity_map
    for r in em.itertuples(index=False):
        ent = b.node(f"entity:{r.entity_id}", "ResolvedEntity", r.entity_id, "resolution", n_records=r.n_records)
        b.edge(f"vendor:{r.record_id}", "PART_OF", ent, "resolution", r.record_id)
    for r in ctx.identity_pairs.itertuples(index=False):
        b.edge(f"vendor:{r.record_a}", r.relation, f"vendor:{r.record_b}", "resolution", f"{r.record_a}|{r.record_b}",
               confidence=r.confidence, evidence=r.reason)

    # Addresses, contacts, declared groups.
    a = ctx.addresses
    types = a.groupby("addr_key")["address_type"].agg(lambda s: s.dropna().mode().iloc[0] if s.notna().any() else None)
    for r in a.itertuples(index=False):
        node = b.node(f"address:{r.addr_key}", "Address", r.raw_address, "tp_address_history",
                      country=r.address_country, address_type=types.get(r.addr_key))
        b.edge(f"vendor:{r.record_id}", "LOCATED_AT", node, "tp_address_history", f"{r.record_id}@{_date(r.valid_from)}",
               valid_from=r.valid_from, valid_to=r.valid_to, observed_at=r.observed_at, raw_address=r.raw_address,
               address_type=r.address_type, address_role=r.address_role)
    for r in ctx.contacts.itertuples(index=False):
        if isinstance(r.phone_key, str):
            node = b.node(f"phone:{r.phone_key}", "ContactPoint", r.phone_key, "third_parties", kind="phone")
            b.edge(f"vendor:{r.record_id}", "HAS_CONTACT", node, "third_parties", r.record_id, raw=r.phone)
        if isinstance(r.domain_key, str):
            node = b.node(f"domain:{r.domain_key}", "ContactPoint", r.domain_key, "third_parties", kind="email_domain")
            b.edge(f"vendor:{r.record_id}", "HAS_CONTACT", node, "third_parties", r.record_id)
    for r in v.dropna(subset=["declared_parent_group"]).itertuples(index=False):
        node = b.node(f"group:{r.declared_parent_group}", "DeclaredGroup", r.declared_parent_group, "third_parties")
        b.edge(f"vendor:{r.record_id}", "DECLARED_MEMBER_OF", node, "third_parties", r.record_id)

    # Engagements, people and categories.
    # Employees exist independently of their roles. A role not yet recorded at t
    # leaves the attributes empty; in the starter data some engagements name a
    # sponsor or approver before that person's role starts.
    roles = ctx.roles.set_index("employee_id")
    names = ctx.b.employees.set_index("employee_id")["name"]
    for emp in ctx.b.employees["employee_id"]:
        r = roles.loc[emp] if emp in roles.index else None
        attrs = {} if r is None else dict(
            role=r["role"], franchise=r["franchise"], home_market=r["home_market"], scope_markets=r["scope_markets"],
            scope_categories=r["scope_categories"], approver_level=r["approver_level"],
            role_valid_from=r["valid_from"], role_valid_to=r["valid_to"])
        b.node(f"employee:{emp}", "Employee", emp, "employees", name=names.get(emp), **attrs)
    cats = ctx.categories
    for code, r in cats.iterrows():
        b.node(f"category:{code}", "ServiceCategory", r["name"], "service_categories", parent=r["parent"],
               flags=r["flags"], category_code_v1=r["category_code_v1"])
    # Approval events with no recorded approver cannot be an edge (there is no
    # endpoint); they stay on the engagement so nothing is dropped.
    ap = ctx.b.tp_engagement_approvals
    unattributed = {eid: [dict(approval_id=x.approval_id, approval_type=x.approval_type,
                               approval_date=x.approval_date, observed_at=x.observed_at) for x in g.itertuples()]
                    for eid, g in ap[ap["approver_employee_id"].isna()].groupby("engagement_id")}
    for r in ctx.engagements.itertuples(index=False):
        en = b.node(f"engagement:{r.engagement_id}", "Engagement", r.engagement_id, "tp_engagements",
                    service_category=r.service_category, taxonomy_version=r.taxonomy_version,
                    business_unit=r.business_unit, market=r.market, contract_type=r.contract_type,
                    value_band=r.value_band, start_date=r.start_date, end_date=r.end_date,
                    approvals_without_approver=unattributed.get(r.engagement_id))
        b.edge(f"vendor:{r.record_id}", "CONTRACTED_UNDER", en, "tp_engagements", r.engagement_id,
               valid_from=r.start_date, valid_to=r.end_date, observed_at=r.observed_at)
        if isinstance(r.sponsor_employee_id, str):
            b.edge(en, "SPONSORED_BY", f"employee:{r.sponsor_employee_id}", "tp_engagements", r.engagement_id,
                   valid_from=r.start_date, valid_to=r.end_date, observed_at=r.observed_at)
        # An ambiguous v1 code keeps one edge per candidate v2 code, with its share as confidence.
        for code in r.cat_v2:
            if code in cats.index:
                b.edge(en, "FOR_SERVICE", f"category:{code}", "tp_engagements", r.engagement_id,
                       confidence=1.0 if r.cat_certain else 1.0 / len(r.cat_v2), recorded_code=r.service_category)
    for r in ctx.b.tp_engagement_approvals.dropna(subset=["approver_employee_id"]).itertuples(index=False):
        b.edge(f"engagement:{r.engagement_id}", "APPROVED_BY", f"employee:{r.approver_employee_id}",
               "tp_engagement_approvals", r.approval_id, valid_from=r.approval_date, observed_at=r.observed_at,
               approval_type=r.approval_type)
    for r in ctx.services.itertuples(index=False):
        for code in r.cat_v2:
            if code in cats.index:
                b.edge(f"vendor:{r.record_id}", "PROVIDES_SERVICE", f"category:{code}", "tp_services",
                       f"{r.record_id}|{r.category_code}|{_date(r.valid_from)}", valid_from=r.valid_from,
                       valid_to=r.valid_to, confidence=1.0 if r.cat_certain else 1.0 / len(r.cat_v2),
                       recorded_code=r.category_code, taxonomy_version=r.taxonomy_version)

    # Healthcare institutions.
    for r in ctx.b.healthcare_institutions.itertuples(index=False):
        b.node(f"institution:{r.institution_id}", "HealthcareInstitution", r.name, "healthcare_institutions",
               country=r.country, inst_type=r.inst_type, public_sector=bool(r.public_sector))
    for r in ctx.b.tp_institution_links.itertuples(index=False):
        b.edge(f"vendor:{r.record_id}", r.link_type, f"institution:{r.institution_id}", "tp_institution_links",
               r.link_id, valid_from=r.valid_from, valid_to=r.valid_to)

    # Ownership (registry R) and accounting parents (registry L).
    if ctx.use_ownership:
        # Statements where the chain stops (no named owner) belong to the owned entity.
        unspecified = ctx.b.reg_interests[ctx.b.reg_interests["party_kind"] == "unspecified"]
        stops = {sid: [dict(statement_id=x.statement_id, reason=x.unspecified_reason, published=x.statement_date,
                            interest_type=x.interest_type, details=x.details, share=x.share,
                            start_date=x.start_date, end_date=x.end_date) for x in g.itertuples()]
                 for sid, g in unspecified.groupby("subject_sid")}
        for r in ctx.b.reg_entities.itertuples(index=False):
            b.node(f"legal_entity:{r.statement_id}", "LegalEntity", r.name, "registry_R", jurisdiction=r.jurisdiction,
                   entity_type=r.entity_type, entity_subtype=r.entity_subtype, registry_no=r.registry_no, lei=r.lei,
                   listed=r.listed, published=r.statement_date, chain_stops=stops.get(r.statement_id))
        for r in ctx.b.reg_persons.itertuples(index=False):
            b.node(f"person:{r.statement_id}", "NaturalPerson", r.full_name, "registry_R",
                   nationality=r.nationality, pep_status=r.pep_status, published=r.statement_date)
        for r in ctx.reg_matches.itertuples(index=False):
            if r.method.startswith("person"):
                b.edge(f"vendor:{r.record_id}", "IS_PERSON", f"person:{r.statement_id}", "registry_R",
                       r.statement_id, confidence=r.confidence, method=r.method, status=r.status)
            else:
                b.edge(f"vendor:{r.record_id}", "REGISTERED_AS", f"legal_entity:{r.statement_id}", "registry_R",
                       r.statement_id, confidence=r.confidence, method=r.method, status=r.status)
        ri = ctx.b.reg_interests.dropna(subset=["party_sid"])
        for r in ri.itertuples(index=False):
            owner = f"{'person' if r.party_kind == 'person' else 'legal_entity'}:{r.party_sid}"
            rel = "OWNS" if r.interest_type == "shareholding" else "CONTROLS"
            b.edge(owner, rel, f"legal_entity:{r.subject_sid}", "registry_R", r.statement_id,
                   valid_from=r.start_date, valid_to=r.end_date, observed_at=r.statement_date,
                   share=r.share, details=r.details, interest_type=r.interest_type)
        lei_to_records = v.dropna(subset=["lei"]).groupby("lei")["record_id"].apply(list).to_dict()
        for r in ctx.b.registry_l.itertuples(index=False):
            parent_key = r.parent_lei if isinstance(r.parent_lei, str) else r.parent_legal_name
            node = b.node(f"accounting_parent:{parent_key}", "AccountingParent", r.parent_legal_name, "registry_L",
                          lei=r.parent_lei if isinstance(r.parent_lei, str) else None)
            for rec in lei_to_records.get(r.child_lei, []):
                b.edge(f"vendor:{rec}", "CONSOLIDATED_BY", node, "registry_L", r.child_lei,
                       valid_from=r.relationship_start, status=r.relationship_status)

        # Ultimate owners (Dossier §7.3), derived from the paths above as they stand at t. One edge
        # per vendor and natural person reached; it cites the statements of every path behind it.
        paths = ownership_path_table(ctx)
        for (rec, person), g in paths[paths["terminal_reason"] == "natural_person"].groupby(["record_id", "end"]):
            shares = g["path_share"].dropna()
            b.edge(f"vendor:{rec}", "ULTIMATE_OWNER", person, "derived:registry_R", f"{rec}|{person}",
                   valid_from=g["valid_from"].min(), observed_at=g["observed_at"].min(),
                   confidence=g["match_confidence"].min(), n_paths=len(g),
                   indirect_share=round(float(shares.sum()), 4) if len(shares) else None,
                   control_only=bool(shares.empty), min_depth=int(g["depth"].min()), max_depth=int(g["depth"].max()),
                   statements=sorted({s for x in g["statements"] for s in x.split(";")}))
    else:
        paths = None

    nodes, edges = b.frames()
    activity = ctx.b.tp_payments_monthly.assign(node_id="engagement:" + ctx.b.tp_payments_monthly["engagement_id"])
    activity = activity[["node_id"] + [c for c in activity.columns if c != "node_id"]]
    activity = activity.sort_values(ACTIVITY_KEY).reset_index(drop=True)
    return Graph(nodes, edges, activity, b.version, ctx.t, paths)


def ownership_path_table(ctx: Context, max_depth: int = 12) -> pd.DataFrame:
    """Every upward ownership path at t of each registry-linked vendor, with where it stops.

    Dossier §7.4: a path ends at a natural person, a stated unknown owner, an
    entity with no recorded owner (or only an accounting parent), a cycle or the
    depth cap; the terminal reason is recorded, never read as concealment.
    path_share multiplies the shares along a path of shareholdings only (None
    when a hop is control without a share).
    """
    nodes = ctx.reg_nodes
    owners = ctx.owners_at(ctx.t)
    m = ctx.reg_matches
    m = m[(m["status"] == "accepted") & ~m["method"].str.startswith("person")]
    confidence = {(r.record_id, r.statement_id): r.confidence for r in m.itertuples()}
    l_children = set(ctx.b.registry_l["child_lei"].dropna())
    entity_lei = ctx.b.reg_entities.set_index("statement_id")["lei"]

    def no_owner(sid):
        """An entity without a recorded owner: only an accounting parent (registry L) when it has one there."""
        return "accounting_parent_only" if entity_lei.get(sid) in l_children else "no_owner_recorded"

    def jur(sid):
        return nodes.at[sid, "jurisdiction"] if sid in nodes.index else None

    rows = []
    for rec, sid in sorted(ctx.reg_entity_of.items()):
        base = {"record_id": rec, "entity": f"legal_entity:{sid}", "match_confidence": confidence.get((rec, sid))}
        paths = ctx.ownership_paths(sid, ctx.t, max_depth)
        if not paths:
            reason = no_owner(sid)
            rows.append({**base, "path": 0, "depth": 0, "end": f"legal_entity:{sid}", "terminal_reason": reason,
                         "path_share": None, "jurisdictions": jur(sid), "statements": "", "valid_from": None,
                         "observed_at": None})
            continue
        for i, p in enumerate(paths):
            last = p[-1]
            if last.get("cycle"):
                reason, end = "cycle", f"legal_entity:{last['party_sid']}"
            elif last["party_kind"] == "person":
                reason, end = "natural_person", f"person:{last['party_sid']}"
            elif last["party_kind"] != "entity":
                reason, end = f"stated:{last.get('unspecified_reason')}", f"legal_entity:{last['subject_sid']}"
            elif owners.get(last["party_sid"]):
                reason, end = "depth_cap", f"legal_entity:{last['party_sid']}"
            else:
                reason, end = no_owner(last["party_sid"]), f"legal_entity:{last['party_sid']}"
            known = all(x["interest_type"] == "shareholding" and pd.notna(x["share"]) for x in p)
            share = float(np.prod([x["share"] / 100 for x in p])) if known else None
            chain = [sid] + [x["party_sid"] for x in p if x["party_kind"] == "entity"]
            rows.append({**base, "path": i, "depth": len(p), "end": end, "terminal_reason": reason, "path_share": share,
                         "jurisdictions": ">".join(j for j in map(jur, chain) if isinstance(j, str)),
                         "statements": ";".join(x["statement_id"] for x in p),
                         "valid_from": max((x["start_date"] for x in p if pd.notna(x["start_date"])), default=None),
                         "observed_at": max(x["statement_date"] for x in p)})
    # Organisations known only to registry L (by LEI) reach an accounting parent and no owner.
    orgs = ctx.vendors[(ctx.vendors["party_type"] == "organization") & ctx.vendors["lei"].isin(l_children)]
    for rec in sorted(set(orgs["record_id"]) - set(ctx.reg_entity_of)):
        rows.append({"record_id": rec, "entity": None, "match_confidence": None, "path": 0, "depth": 0, "end": None,
                     "terminal_reason": "accounting_parent_only", "path_share": None, "jurisdictions": None,
                     "statements": "", "valid_from": None, "observed_at": None})
    return pd.DataFrame(rows, columns=["record_id", "entity", "match_confidence", "path", "depth", "end",
                                       "terminal_reason", "path_share", "jurisdictions", "statements", "valid_from",
                                       "observed_at"])


def without(g: Graph, relations=(), activity: bool = False) -> Graph:
    """A copy of the graph without the given edge types (and payments if `activity`)."""
    edges = g.edges[~g.edges["relation_type"].isin(set(relations))].reset_index(drop=True)
    act = g.activity.iloc[0:0] if activity else g.activity
    return Graph(g.nodes, edges, act, g.dataset_version, g.snapshot, g.ownership_paths)


def graph_hash(nodes: pd.DataFrame, edges: pd.DataFrame, activity: pd.DataFrame | None = None) -> str:
    """SHA-256 over a canonical serialisation (rows sorted by key, fixed column order)."""
    h = hashlib.sha256()
    parts = [(nodes, ["node_id"], NODE_COLUMNS), (edges, ["edge_id"], EDGE_COLUMNS)]
    if activity is not None:
        parts.append((activity, ACTIVITY_KEY, list(activity.columns)))
    for df, key, cols in parts:
        buf = io.StringIO()
        df.sort_values(key)[cols].to_csv(buf, index=False, lineterminator="\n")
        h.update(buf.getvalue().encode())
    return h.hexdigest()[:16]


def validate(g: Graph) -> list[str]:
    """Appendix B checks on the graph itself. Returns a list of problems (empty when clean)."""
    problems = []
    if g.nodes["node_id"].duplicated().any():
        problems.append("duplicate node ids")
    types = g.nodes.set_index("node_id")["node_type"]
    missing = ~g.edges["source_node_id"].isin(types.index) | ~g.edges["target_node_id"].isin(types.index)
    if missing.any():
        problems.append(f"{int(missing.sum())} edges with an endpoint missing from the node table")
    ok = g.edges[~missing]
    triples = set(zip(types[ok["source_node_id"]], ok["relation_type"], types[ok["target_node_id"]]))
    for t in sorted(triples - ALLOWED):
        problems.append(f"relation not allowed: {t}")
    vf, vt = pd.to_datetime(g.edges["valid_from"]), pd.to_datetime(g.edges["valid_to"])
    if (vt.notna() & vf.notna() & (vt < vf)).any():
        problems.append("edges with valid_to before valid_from")
    if (pd.to_datetime(g.edges["observed_at"]) > g.snapshot).any():
        problems.append("edges observed after the snapshot")
    engagements = set(g.nodes.loc[g.nodes["node_type"] == "Engagement", "node_id"])
    if not g.activity["node_id"].isin(engagements).all():
        problems.append("activity rows attached to unknown engagements")
    return problems


def export(g: Graph, out_dir: str | Path) -> dict:
    """Write nodes, edges and activity (parquet) and graph.json (hash, version, counts)."""
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    g.nodes.to_parquet(out / "nodes.parquet", index=False)
    g.edges.to_parquet(out / "edges.parquet", index=False)
    g.activity.to_parquet(out / "activity.parquet", index=False)
    info = g.summary()
    if g.ownership_paths is not None:
        g.ownership_paths.to_parquet(out / "ownership_paths.parquet", index=False)
        info["ownership_terminal_reasons"] = g.ownership_paths["terminal_reason"].value_counts().sort_index().to_dict()
    (out / "graph.json").write_text(json.dumps(info, indent=2))
    return info


In [7]:
%%module meridian.graph_ops
"""Graph operations used by the detectors: typed edges and meta-paths.

A meta-path is a sequence of typed relations, for example
    VendorRecord -CONTRACTED_UNDER-> Engagement -SPONSORED_BY-> Employee.
`metapath` returns every instance of one, with the node at each step, the
edge id of each hop (the evidence a case can cite) and the edge attributes.
Two vendors are related through a meta-path when their instances end on
the same node: that is how shared addresses, contacts, sponsors, approvers
and institutions are found.
"""
from __future__ import annotations

import json
from functools import cached_property

import numpy as np
import pandas as pd

from .graph import Graph

DATE_FIELDS = ("valid_from", "valid_to", "observed_at")


class HeteroGraph:
    """Indexed view of a canonical Graph for traversal."""

    def __init__(self, g: Graph):
        self.g = g

    @cached_property
    def node_type(self) -> dict:
        return dict(zip(self.g.nodes["node_id"], self.g.nodes["node_type"]))

    @cached_property
    def _by_relation(self) -> dict[str, pd.DataFrame]:
        e = self.g.edges.copy()
        for c in DATE_FIELDS:
            e[c] = pd.to_datetime(e[c]).astype("datetime64[ns]")
        e["props"] = [json.loads(p) for p in e["relation_properties"]]
        return {rel: df.reset_index(drop=True) for rel, df in e.groupby("relation_type", sort=True)}

    @cached_property
    def _edges_cache(self) -> dict:
        return {}

    def edges(self, *relations: str) -> pd.DataFrame:
        """Edges of the given relation types (all columns, dates parsed, props as dicts).

        The frame is shared between callers: do not modify it."""
        if relations not in self._edges_cache:
            parts = [self._by_relation[r] for r in relations if r in self._by_relation]
            if not parts:
                out = self.g.edges.iloc[0:0].assign(props=[])
            else:
                out = pd.concat(parts, ignore_index=True) if len(parts) > 1 else parts[0]
            self._edges_cache[relations] = out
        return self._edges_cache[relations]

    @cached_property
    def _path_cache(self) -> dict:
        return {}

    def metapath(self, *steps: str | tuple, where=None) -> pd.DataFrame:
        if where is None:
            if steps not in self._path_cache:
                self._path_cache[steps] = self._metapath(*steps)
            return self._path_cache[steps]
        return self._metapath(*steps, where=where)

    def _metapath(self, *steps: str | tuple, where=None) -> pd.DataFrame:
        """Instances of a meta-path.

        Each step is a relation name (followed source -> target) or a tuple of
        relation names (any of them). Returns columns n0..nk (node ids),
        e1..ek (edge ids), r1..rk (relation), and per hop the edge's
        valid_from/valid_to/observed_at/source_record_id/props as `<field><i>`.
        `where` is an optional dict {hop index: predicate(edge_row_frame) -> mask}.
        """
        out = None
        for i, step in enumerate(steps, start=1):
            rels = (step,) if isinstance(step, str) else tuple(step)
            e = self.edges(*rels)
            if where and i in where:
                # an explicit boolean row mask: a plain list would select columns on an empty frame
                e = e.loc[np.asarray(where[i](e), dtype=bool)]
            hop = pd.DataFrame({
                f"n{i - 1}": e["source_node_id"].to_numpy(), f"n{i}": e["target_node_id"].to_numpy(),
                f"e{i}": e["edge_id"].to_numpy(), f"r{i}": e["relation_type"].to_numpy(),
                f"valid_from{i}": e["valid_from"].to_numpy(), f"valid_to{i}": e["valid_to"].to_numpy(),
                f"observed_at{i}": e["observed_at"].to_numpy(), f"source_record_id{i}": e["source_record_id"].to_numpy(),
                f"props{i}": e["props"].to_numpy(),
            })
            out = hop if out is None else out.merge(hop, on=f"n{i - 1}", how="inner")
        return out

    def shared(self, *steps, where=None) -> dict[str, list]:
        """End node -> start nodes of the meta-path instances that reach it (in instance order)."""
        mp = self.metapath(*steps, where=where)
        k = len(steps)
        out: dict[str, list] = {}
        for start, end in zip(mp["n0"], mp[f"n{k}"]):
            out.setdefault(end, []).append(start)
        return out


def strip(node_id: str) -> str:
    """'vendor:TP-123' -> 'TP-123'."""
    return node_id.split(":", 1)[1]


def hetero(ctx) -> HeteroGraph:
    """The HeteroGraph of a graph-backed Context (cached on the context)."""
    if ctx.graph is None:
        raise ValueError("this context was not built from a graph (use Context.from_graph)")
    if "_hetero" not in ctx.__dict__:
        ctx.__dict__["_hetero"] = HeteroGraph(ctx.graph)
    return ctx.__dict__["_hetero"]

In [8]:
%%module meridian.graph_views
"""Relational views derived from the canonical graph.

Detection reads the graph, not the source files: `bundle_from_graph` rebuilds
every table the detectors use from node attributes, typed edges and the
activity table, and takes entity resolution, normalised addresses and
contact keys straight from the graph instead of recomputing them. Only
reference tables (markets, service taxonomy, approval policy, FMV rate cards,
country context, business events) come from outside the graph.

A round-trip test checks that these views equal the source tables, which is
the evidence that building the graph loses nothing the detectors need.
"""
from __future__ import annotations

import json

import pandas as pd

from .data import DATE_COLUMNS, Bundle, canonical
from .graph import Graph

REFERENCE_TABLES = ["markets", "paying_entities", "service_categories", "approval_policy_matrix", "fmv_rate_cards",
                    "country_risk_context", "business_events", "employee_delegations"]
DD_COLUMNS = ["dd_level", "dd_completed", "dd_refresh_due", "q_gov_ownership", "q_pep_connected",
              "q_public_official_interaction", "q_former_public_official", "screening_result", "adverse_media",
              "risk_rating"]
LINK_TYPES = ("SERVES", "SITE_OF", "AFFILIATED_WITH")


def _dd_attr(col: str) -> str:
    return col if col.startswith("dd_") else f"dd_{col}"


def _strip(ids: pd.Series, prefix: str) -> pd.Series:
    return ids.str.slice(len(prefix) + 1)


def _frame(rows: list[dict], columns: list[str], dates=(), ints=(), bools=()) -> pd.DataFrame:
    df = pd.DataFrame(rows, columns=columns)
    for c in dates:
        df[c] = pd.to_datetime(df[c]).astype("datetime64[ns]")
    for c in ints:
        df[c] = df[c].astype("int64")
    for c in bools:
        df[c] = df[c].astype(bool)
    return df


class _Index:
    def __init__(self, g: Graph):
        self.g = g
        self.attrs = {nid: json.loads(a) for nid, a in zip(g.nodes["node_id"], g.nodes["attributes"])}
        self.label = dict(zip(g.nodes["node_id"], g.nodes["label"]))
        self.type = dict(zip(g.nodes["node_id"], g.nodes["node_type"]))

    def nodes(self, node_type: str) -> pd.DataFrame:
        return self.g.nodes[self.g.nodes["node_type"] == node_type]

    def edges(self, *relations: str) -> pd.DataFrame:
        e = self.g.edges[self.g.edges["relation_type"].isin(relations)].copy()
        e["props"] = [json.loads(p) for p in e["relation_properties"]]
        return e


def bundle_from_graph(g: Graph, reference: Bundle) -> tuple[Bundle, dict]:
    """(Bundle of graph-derived tables plus reference tables, precomputed resolution views)."""
    ix = _Index(g)
    t = {}

    # Vendors and due diligence.
    vend = ix.nodes("VendorRecord")
    rows, dd_rows = [], []
    for nid, label in zip(vend["node_id"], vend["label"]):
        a = ix.attrs[nid]
        rid = nid[len("vendor:"):]
        rows.append(dict(record_id=rid, party_type=a.get("party_type"), legal_name=label, tax_id=a.get("tax_id"),
                         lei=a.get("lei"), registry_no=a.get("registry_no"), country=a.get("country"),
                         status=a.get("status"), onboarding_date=a.get("onboarding_date"),
                         created_date=a.get("created_date"), terminated_date=a.get("terminated_date"),
                         declared_parent_group=a.get("declared_parent_group"), phone=a.get("phone"),
                         email_domain=a.get("email_domain"), remit_to_country=a.get("remit_to_country"),
                         expert_type=a.get("expert_type")))
        if any(_dd_attr(c) in a for c in DD_COLUMNS):
            dd_rows.append({**{c: a.get(_dd_attr(c)) for c in DD_COLUMNS}, "record_id": rid})
    t["third_parties"] = _frame(rows, list(rows[0]) if rows else [], dates=DATE_COLUMNS["third_parties"])
    t["tp_due_diligence"] = _frame(dd_rows, DD_COLUMNS + ["record_id"], dates=DATE_COLUMNS["tp_due_diligence"],
                                   bools=[c for c in DD_COLUMNS if c.startswith("q_") or c == "adverse_media"])

    # Addresses (the address node id carries the normalised key).
    e = ix.edges("LOCATED_AT")
    addr = pd.DataFrame({
        "record_id": _strip(e["source_node_id"], "vendor"),
        "raw_address": [p.get("raw_address") for p in e["props"]],
        "address_country": [ix.attrs[n].get("country") for n in e["target_node_id"]],
        "address_type": [p.get("address_type") for p in e["props"]],
        "address_role": [p.get("address_role") for p in e["props"]],
        "valid_from": e["valid_from"], "valid_to": e["valid_to"], "observed_at": e["observed_at"],
        "addr_key": _strip(e["target_node_id"], "address"),
    })
    addr = _frame(addr.to_dict("records"), list(addr.columns), dates=DATE_COLUMNS["tp_address_history"])
    addr = canonical("tp_address_history", addr)
    t["tp_address_history"] = addr.drop(columns=["addr_key"])

    # Service registrations: one source row may map to several candidate v2 categories.
    e = ix.edges("PROVIDES_SERVICE")
    e["record_id"] = _strip(e["source_node_id"], "vendor")
    svc = e.groupby("source_record_id", sort=False).agg(
        record_id=("record_id", "first"), category_code=("props", lambda s: s.iloc[0].get("recorded_code")),
        taxonomy_version=("props", lambda s: s.iloc[0].get("taxonomy_version")),
        valid_from=("valid_from", "first"), valid_to=("valid_to", "first")).reset_index(drop=True)
    t["tp_services"] = _frame(svc.to_dict("records"), ["record_id", "category_code", "taxonomy_version",
                                                       "valid_from", "valid_to"], dates=DATE_COLUMNS["tp_services"])

    # Engagements, sponsors and approvals.
    con = ix.edges("CONTRACTED_UNDER").set_index("target_node_id")
    spo = ix.edges("SPONSORED_BY").set_index("source_node_id")["target_node_id"]
    rows, unatt = [], []
    for nid in ix.nodes("Engagement")["node_id"]:
        a = ix.attrs[nid]
        eid = nid[len("engagement:"):]
        rows.append(dict(engagement_id=eid, record_id=con.at[nid, "source_node_id"][len("vendor:"):],
                         service_category=a.get("service_category"), taxonomy_version=a.get("taxonomy_version"),
                         business_unit=a.get("business_unit"), market=a.get("market"),
                         contract_type=a.get("contract_type"), start_date=a.get("start_date"),
                         end_date=a.get("end_date"), value_band=a.get("value_band"),
                         sponsor_employee_id=spo[nid][len("employee:"):] if nid in spo.index else None,
                         observed_at=con.at[nid, "observed_at"]))
        for x in a.get("approvals_without_approver") or []:
            unatt.append(dict(approval_id=x["approval_id"], engagement_id=eid, approver_employee_id=None,
                              approval_type=x.get("approval_type"), approval_date=x.get("approval_date"),
                              observed_at=x.get("observed_at")))
    t["tp_engagements"] = _frame(rows, list(rows[0]) if rows else [], dates=DATE_COLUMNS["tp_engagements"],
                                 ints=["value_band"])
    e = ix.edges("APPROVED_BY")
    appr = [dict(approval_id=r.source_record_id, engagement_id=r.source_node_id[len("engagement:"):],
                 approver_employee_id=r.target_node_id[len("employee:"):], approval_type=r.props.get("approval_type"),
                 approval_date=r.valid_from, observed_at=r.observed_at) for r in e.itertuples()]
    t["tp_engagement_approvals"] = _frame(appr + unatt, ["approval_id", "engagement_id", "approver_employee_id",
                                                        "approval_type", "approval_date", "observed_at"],
                                          dates=DATE_COLUMNS["tp_engagement_approvals"])
    t["tp_payments_monthly"] = g.activity.drop(columns=["node_id"]).reset_index(drop=True)

    # Institutions and links.
    inst = ix.nodes("HealthcareInstitution")
    t["healthcare_institutions"] = _frame(
        [dict(institution_id=n[len("institution:"):], name=lab, country=ix.attrs[n].get("country"),
              inst_type=ix.attrs[n].get("inst_type"), public_sector=ix.attrs[n].get("public_sector"))
         for n, lab in zip(inst["node_id"], inst["label"])],
        ["institution_id", "name", "country", "inst_type", "public_sector"], bools=["public_sector"])
    e = ix.edges(*LINK_TYPES)
    t["tp_institution_links"] = _frame(
        [dict(link_id=r.source_record_id, record_id=r.source_node_id[len("vendor:"):],
              institution_id=r.target_node_id[len("institution:"):], link_type=r.relation_type,
              valid_from=r.valid_from, valid_to=r.valid_to) for r in e.itertuples()],
        ["link_id", "record_id", "institution_id", "link_type", "valid_from", "valid_to"],
        dates=DATE_COLUMNS["tp_institution_links"])

    # Employees and their dated roles.
    emp = ix.nodes("Employee")["node_id"]
    t["employees"] = _frame([dict(employee_id=n[len("employee:"):], name=ix.attrs[n].get("name")) for n in emp],
                            ["employee_id", "name"])
    role_cols = ["employee_id", "role", "franchise", "home_market", "scope_markets", "scope_categories",
                 "approver_level", "valid_from", "valid_to"]
    t["employee_roles"] = _frame(
        [dict(employee_id=n[len("employee:"):], role=a["role"], franchise=a.get("franchise"),
              home_market=a.get("home_market"), scope_markets=a.get("scope_markets"),
              scope_categories=a.get("scope_categories"), approver_level=a.get("approver_level"),
              valid_from=a.get("role_valid_from"), valid_to=a.get("role_valid_to"))
         for n in emp if "role" in (a := ix.attrs[n])],
        role_cols, dates=DATE_COLUMNS["employee_roles"], ints=["approver_level"])

    # Registries.
    ent = ix.nodes("LegalEntity")
    t["reg_entities"] = _frame(
        [dict(statement_id=n[len("legal_entity:"):], statement_date=a.get("published"), reg_record_id=None,
              entity_type=a.get("entity_type"), entity_subtype=a.get("entity_subtype"), name=lab,
              jurisdiction=a.get("jurisdiction"), reg_scheme=None, registry_no=a.get("registry_no"),
              lei=a.get("lei"), listed=a.get("listed"))
         for n, lab in zip(ent["node_id"], ent["label"]) if (a := ix.attrs[n]) is not None],
        ["statement_id", "statement_date", "reg_record_id", "entity_type", "entity_subtype", "name", "jurisdiction",
         "reg_scheme", "registry_no", "lei", "listed"], dates=DATE_COLUMNS["reg_entities"])
    per = ix.nodes("NaturalPerson")
    t["reg_persons"] = _frame(
        [dict(statement_id=n[len("person:"):], statement_date=ix.attrs[n].get("published"), reg_record_id=None,
              person_type=None, full_name=lab, nationality=ix.attrs[n].get("nationality"),
              pep_status=ix.attrs[n].get("pep_status")) for n, lab in zip(per["node_id"], per["label"])],
        ["statement_id", "statement_date", "reg_record_id", "person_type", "full_name", "nationality", "pep_status"],
        dates=DATE_COLUMNS["reg_persons"])
    e = ix.edges("OWNS", "CONTROLS")
    ints = [dict(statement_id=r.source_record_id, statement_date=r.observed_at,
                 subject_sid=r.target_node_id[len("legal_entity:"):],
                 party_sid=r.source_node_id.split(":", 1)[1],
                 party_kind="person" if r.source_node_id.startswith("person:") else "entity",
                 unspecified_reason=None, interest_type=r.props.get("interest_type"), details=r.props.get("details"),
                 share=r.props.get("share"), start_date=r.valid_from, end_date=r.valid_to) for r in e.itertuples()]
    for n in ent["node_id"]:
        for x in ix.attrs[n].get("chain_stops") or []:
            ints.append(dict(statement_id=x["statement_id"], statement_date=x.get("published"),
                             subject_sid=n[len("legal_entity:"):], party_sid=None, party_kind="unspecified",
                             unspecified_reason=x.get("reason"), interest_type=x.get("interest_type"),
                             details=x.get("details"), share=x.get("share"), start_date=x.get("start_date"),
                             end_date=x.get("end_date")))
    int_cols = ["statement_id", "statement_date", "subject_sid", "party_sid", "party_kind", "unspecified_reason",
                "interest_type", "details", "share", "start_date", "end_date"]
    t["reg_interests"] = _frame(ints, int_cols, dates=DATE_COLUMNS["reg_interests"])
    t["reg_interests"]["share"] = t["reg_interests"]["share"].astype("float64")
    e = ix.edges("CONSOLIDATED_BY")
    t["registry_l"] = _frame(
        [dict(child_lei=r.source_record_id, child_legal_name=ix.label[r.source_node_id],
              parent_lei=ix.attrs[r.target_node_id].get("lei"), parent_legal_name=ix.label[r.target_node_id],
              relationship_type="IS_DIRECTLY_CONSOLIDATED_BY", relationship_start=r.valid_from,
              relationship_status=r.props.get("status")) for r in e.itertuples()],
        ["child_lei", "child_legal_name", "parent_lei", "parent_legal_name", "relationship_type",
         "relationship_start", "relationship_status"], dates=DATE_COLUMNS["registry_l"])

    for name in REFERENCE_TABLES:
        t[name] = getattr(reference, name)
    t = {name: canonical(name, df) for name, df in t.items()}
    bundle = Bundle(**t, as_of=reference.as_of, source_dir=reference.source_dir, input_hashes=reference.input_hashes)

    # Entity resolution and normalisation, read from the graph rather than recomputed.
    e = ix.edges("PART_OF")
    n_records = {n: ix.attrs[n].get("n_records") for n in e["target_node_id"].unique()}
    entity_map = pd.DataFrame({"record_id": _strip(e["source_node_id"], "vendor"),
                               "entity_id": _strip(e["target_node_id"], "entity"),
                               "n_records": [n_records[n] for n in e["target_node_id"]]})
    entity_map = entity_map.sort_values("record_id").reset_index(drop=True)
    e = ix.edges("SAME_AS", "IDENTITY_CANDIDATE")
    identity_pairs = pd.DataFrame({
        "record_a": _strip(e["source_node_id"], "vendor"), "record_b": _strip(e["target_node_id"], "vendor"),
        "reason": [p.get("evidence") for p in e["props"]], "confidence": e["resolution_confidence"].astype(float),
        "relation": e["relation_type"]}).astype({"record_a": object, "record_b": object, "reason": object,
                                                 "relation": object})
    identity_pairs = identity_pairs.sort_values(["relation", "record_a", "record_b", "reason"]).reset_index(drop=True)
    e = ix.edges("REGISTERED_AS", "IS_PERSON")
    reg_matches = pd.DataFrame({
        "record_id": _strip(e["source_node_id"], "vendor"), "statement_id": e["source_record_id"].astype(object),
        "method": [p.get("method") for p in e["props"]], "confidence": e["resolution_confidence"].astype(float),
        "status": [p.get("status") for p in e["props"]],
    }).astype({"record_id": object, "statement_id": object, "method": object, "status": object})
    reg_matches = reg_matches.sort_values(["record_id", "statement_id"]).reset_index(drop=True)
    tp = bundle.third_parties
    keys = ix.edges("HAS_CONTACT")
    keys["record_id"] = _strip(keys["source_node_id"], "vendor")
    phone = keys[keys["target_node_id"].str.startswith("phone:")].set_index("record_id")["target_node_id"]
    domain = keys[keys["target_node_id"].str.startswith("domain:")].set_index("record_id")["target_node_id"]
    contacts = tp[["record_id", "phone", "email_domain"]].copy()
    for col, keys, prefix in (("phone_key", phone, "phone"), ("domain_key", domain, "domain")):
        mapped = contacts["record_id"].map(_strip(keys, prefix)).astype(object)
        contacts[col] = mapped.where(mapped.notna(), None)  # missing stays None, as in the source
    pre = {"entity_map": entity_map, "identity_pairs": identity_pairs, "reg_matches": reg_matches,
           "addresses": addr, "contacts": contacts}
    return bundle, pre

In [9]:
%%module meridian.checks
"""Data checks run on every load (Dossier Appendix B).

Each check returns a row: area, check, status (ok / warn / fail), value, detail.
`warn` marks a known deviation we handle downstream; `fail` means the input
contract is broken and the run should not be trusted.
"""
from __future__ import annotations

import networkx as nx
import pandas as pd

from .data import SNAPSHOT, Bundle

# Row counts and missingness stated in DATA_DICTIONARY.md, to compare with what we measure.
DICTIONARY_ROWS = {
    "third_parties": 1002, "tp_due_diligence": 1002, "tp_address_history": 1091,
    "tp_services": 1791, "tp_engagements": 2638, "tp_engagement_approvals": 4298,
    "tp_payments_monthly": 34467, "tp_institution_links": 403, "healthcare_institutions": 213,
    "employees": 1502, "employee_roles": 1502, "employee_delegations": 0, "markets": 74,
    "paying_entities": 74, "service_categories": 43, "approval_policy_matrix": 5,
    "fmv_rate_cards": 144, "country_risk_context": 1120, "business_events": 8,
}
DICTIONARY_NULL_RATE = {
    ("tp_engagements", "sponsor_employee_id"): 0.03,
    ("tp_engagement_approvals", "approver_employee_id"): 0.05,
    ("third_parties", "lei"): 0.75,
    ("tp_address_history", "address_type"): 0.40,
}
PRIMARY_KEYS = {
    "third_parties": ["record_id"], "tp_due_diligence": ["record_id"],
    "tp_engagements": ["engagement_id"], "tp_engagement_approvals": ["approval_id"],
    "tp_institution_links": ["link_id"], "healthcare_institutions": ["institution_id"],
    "employees": ["employee_id"], "markets": ["market_id"], "paying_entities": ["paying_entity_id"],
    "service_categories": ["category_code_v2"], "approval_policy_matrix": ["value_band"],
    "reg_entities": ["statement_id"], "reg_persons": ["statement_id"],
    "tp_payments_monthly": ["engagement_id", "month", "expense_classification", "channel"],
}
VALIDITY = {
    "tp_address_history": ("valid_from", "valid_to"), "tp_services": ("valid_from", "valid_to"),
    "tp_engagements": ("start_date", "end_date"), "tp_institution_links": ("valid_from", "valid_to"),
    "employee_roles": ("valid_from", "valid_to"), "third_parties": ("onboarding_date", "terminated_date"),
    "reg_interests": ("start_date", "end_date"), "fmv_rate_cards": ("effective_from", "effective_to"),
}
OBSERVED = {
    "tp_address_history": "observed_at", "tp_engagements": "observed_at",
    "tp_engagement_approvals": "observed_at", "reg_entities": "statement_date",
    "reg_persons": "statement_date", "reg_interests": "statement_date",
}


def _row(area, check, ok, value=None, detail="", warn=False):
    status = "ok" if ok else ("warn" if warn else "fail")
    return {"area": area, "check": check, "status": status, "value": value, "detail": detail}


def _fk(rows, child, ccol, parent, pcol, name, allow_null=True):
    vals = child[ccol].dropna() if allow_null else child[ccol]
    bad = ~vals.isin(set(parent[pcol].dropna()))
    rows.append(_row("referential", name, bad.sum() == 0, int(bad.sum()),
                     f"{bad.sum()} of {len(vals)} {ccol} values not in {pcol}"))


def run_checks(b: Bundle, snapshot=SNAPSHOT) -> pd.DataFrame:
    rows = []
    tables = b.tables()

    # Row counts against the dictionary.
    for name, n in DICTIONARY_ROWS.items():
        got = len(tables[name])
        rows.append(_row("contract", f"rows:{name}", got == n, got, f"dictionary says {n}", warn=True))

    # Missingness against the dictionary.
    for (tname, col), expected in DICTIONARY_NULL_RATE.items():
        rate = tables[tname][col].isna().mean()
        ok = abs(rate - expected) <= 0.02
        rows.append(_row("contract", f"null_rate:{tname}.{col}", ok, round(rate, 3),
                         f"dictionary implies ~{expected:.0%}", warn=True))

    # Primary keys.
    for tname, key in PRIMARY_KEYS.items():
        d = tables[tname]
        dup = d.duplicated(key).sum()
        blank = d[key].isna().any(axis=1).sum()
        rows.append(_row("identifiers", f"pk:{tname}", dup == 0 and blank == 0, int(dup + blank),
                         f"{dup} duplicate and {blank} blank keys on {key}"))

    # Foreign keys.
    tp, en, ap = b.third_parties, b.tp_engagements, b.tp_engagement_approvals
    for tname in ["tp_due_diligence", "tp_address_history", "tp_services", "tp_engagements",
                  "tp_payments_monthly", "tp_institution_links"]:
        _fk(rows, tables[tname], "record_id", tp, "record_id", f"fk:{tname}.record_id", allow_null=False)
    _fk(rows, en, "sponsor_employee_id", b.employees, "employee_id", "fk:engagements.sponsor")
    _fk(rows, en, "market", b.markets, "market_id", "fk:engagements.market")
    _fk(rows, ap, "engagement_id", en, "engagement_id", "fk:approvals.engagement_id", allow_null=False)
    _fk(rows, ap, "approver_employee_id", b.employees, "employee_id", "fk:approvals.approver")
    _fk(rows, b.tp_payments_monthly, "engagement_id", en, "engagement_id", "fk:payments.engagement_id")
    _fk(rows, b.tp_payments_monthly, "paying_entity_id", b.paying_entities, "paying_entity_id",
        "fk:payments.paying_entity")
    _fk(rows, b.tp_institution_links, "institution_id", b.healthcare_institutions, "institution_id",
        "fk:links.institution")
    _fk(rows, b.employee_roles, "employee_id", b.employees, "employee_id", "fk:roles.employee")
    _fk(rows, tp, "country", b.markets, "market_id", "fk:third_parties.country")
    all_codes = pd.DataFrame({"c": pd.concat([b.service_categories["category_code_v2"],
                                              b.service_categories["category_code_v1"]])})
    _fk(rows, b.tp_services, "category_code", all_codes, "c", "fk:services.category_code")
    _fk(rows, en, "service_category", all_codes, "c", "fk:engagements.service_category")
    reg_sids = pd.DataFrame({"s": pd.concat([b.reg_entities["statement_id"], b.reg_persons["statement_id"]])})
    _fk(rows, b.reg_interests, "subject_sid", reg_sids, "s", "fk:registry_R.subject")
    _fk(rows, b.reg_interests, "party_sid", reg_sids, "s", "fk:registry_R.interested_party")
    pay_en = b.tp_payments_monthly.merge(en[["engagement_id", "record_id"]], on="engagement_id",
                                         suffixes=("", "_en"))
    bad = (pay_en["record_id"] != pay_en["record_id_en"]).sum()
    rows.append(_row("referential", "payments.record_id matches engagement", bad == 0, int(bad)))
    unmatched_l = ~b.registry_l["child_lei"].isin(set(tp["lei"].dropna()))
    rows.append(_row("referential", "registry_L.child_lei in third_parties.lei", unmatched_l.sum() == 0,
                     int(unmatched_l.sum()), "children not in the vendor master", warn=True))

    # Temporal validity.
    for tname, (s, e) in VALIDITY.items():
        d = tables[tname]
        bad = (d[e].notna() & (d[e] < d[s])).sum()
        # Starter has one vendor terminated before onboarding (no engagements): a data glitch, not a broken contract.
        rows.append(_row("temporal", f"from<=to:{tname}", bad == 0, int(bad), f"{s} <= {e}",
                         warn=tname == "third_parties"))
    for tname, col in OBSERVED.items():
        late = (tables[tname][col] > snapshot).sum()
        rows.append(_row("temporal", f"observed<=snapshot:{tname}", late == 0, int(late)))
    st = (tp["status"] == "terminated") != tp["terminated_date"].notna()
    rows.append(_row("temporal", "status consistent with terminated_date", st.sum() == 0, int(st.sum())))
    ap_before = ap.merge(en[["engagement_id", "observed_at"]], on="engagement_id", suffixes=("", "_en"))
    rows.append(_row("temporal", "created_date spike (ERP migration)", True,
                     int((tp["created_date"] == pd.Timestamp("2022-05-01")).sum()),
                     "records with created_date = 2022-05-01; use onboarding_date"))
    early = (ap_before["approval_date"] > ap_before["observed_at_en"] + pd.Timedelta(days=365)).sum()
    rows.append(_row("temporal", "approval long after engagement recorded", True, int(early)))

    # Overlapping address validity for the same record.
    a = b.tp_address_history.sort_values(["record_id", "valid_from"])
    nxt = a.groupby("record_id")["valid_from"].shift(-1)
    ovl = (a["valid_to"].isna() & nxt.notna()) | (a["valid_to"] > nxt)
    rows.append(_row("overlap", "overlapping addresses per record", ovl.sum() == 0, int(ovl.sum()), warn=True))

    # Employees: one role per employee (succession = new employee id).
    per_emp = b.employee_roles.groupby("employee_id").size()
    rows.append(_row("contract", "roles per employee", per_emp.max() == 1, int(per_emp.max()),
                     "max role periods per employee", warn=True))

    # Service taxonomy: one-to-many v1 -> v2 mappings.
    sc = b.service_categories
    fan = sc.groupby("category_code_v1")["category_code_v2"].nunique()
    multi = fan[fan > 1]
    affected = (en["taxonomy_version"].eq("v1") & en["service_category"].isin(multi.index)).sum()
    rows.append(_row("contract", "v1 codes mapping to several v2 codes", True, int(len(multi)),
                     f"{len(multi)} of {len(fan)} v1 codes; {affected} v1 engagements affected"))

    # Country context: which indices are loaded.
    idx = sorted(b.country_risk_context["index"].unique())
    rows.append(_row("contract", "country_risk_context indices", {"FSI", "FATF"} <= set(idx), ",".join(idx),
                     "FSI and FATF expected when loaded", warn=True))

    # Ownership percentages and cycles.
    sh = b.reg_interests["share"].dropna()
    rows.append(_row("ownership", "share in [0,100]", sh.between(0, 100).all(), int((~sh.between(0, 100)).sum())))
    g = nx.DiGraph()
    ri = b.reg_interests[b.reg_interests["party_kind"] == "entity"]
    g.add_edges_from(zip(ri["party_sid"], ri["subject_sid"]))
    cycles = list(nx.simple_cycles(g))
    rows.append(_row("ownership", "ownership cycles", len(cycles) == 0, len(cycles), warn=True))
    reg_lag = (b.reg_interests["statement_date"] - b.reg_interests["start_date"]).dt.days.dropna()
    rows.append(_row("ownership", "registry R publication lag (days, median/max)", True,
                     f"{reg_lag.median():.0f}/{reg_lag.max():.0f}"))

    # Duplicates: shared tax_id.
    dup_tax = tp[tp["tax_id"].notna()].groupby("tax_id")["record_id"].nunique()
    rows.append(_row("duplicates", "tax_id groups with >1 record", True, int((dup_tax > 1).sum()),
                     f"{int(dup_tax[dup_tax > 1].sum())} records"))
    shared_dom = tp.groupby("email_domain")["record_id"].nunique()
    rows.append(_row("duplicates", "email domains shared by >1 record", True, int((shared_dom > 1).sum()),
                     f"{int(shared_dom[shared_dom > 1].sum())} records"))

    # Approval authority: approver level below the band requirement.
    under = approval_authority_gaps(b)
    rows.append(_row("policy", "financial approvals below required level", True,
                     int(under["engagement_id"].nunique()),
                     f"{under['record_id'].nunique()} records"))

    return pd.DataFrame(rows)


def approval_authority_gaps(b: Bundle) -> pd.DataFrame:
    """Financial approvals where the approver's level is below the band's requirement.

    Temporary delegations (when provided) excuse the gap.
    """
    ap = b.tp_engagement_approvals
    fin = ap[ap["approval_type"] == "financial"].merge(
        b.tp_engagements[["engagement_id", "record_id", "value_band", "sponsor_employee_id"]], on="engagement_id")
    fin = fin.merge(b.approval_policy_matrix[["value_band", "required_approver_level"]], on="value_band")
    fin = fin.merge(b.employee_roles[["employee_id", "approver_level", "valid_from", "valid_to", "role"]],
                    left_on="approver_employee_id", right_on="employee_id", how="left")
    under = fin[fin["approver_employee_id"].notna() & (fin["approver_level"] < fin["required_approver_level"])]
    dl = b.employee_delegations
    if len(dl):
        m = under.merge(dl, left_on="approver_employee_id", right_on="to_employee_id", how="left",
                        suffixes=("", "_dl"))
        covered = (m["approval_date"] >= m["valid_from_dl"]) & (m["valid_to_dl"].isna()
                                                                | (m["approval_date"] <= m["valid_to_dl"])) \
            & (m["approver_level_dl"] >= m["required_approver_level"])
        under = under[~under["approval_id"].isin(m.loc[covered, "approval_id"])]
    return under


def to_markdown(report: pd.DataFrame) -> str:
    lines = ["| area | check | status | value | detail |", "|---|---|---|---|---|"]
    for r in report.itertuples():
        lines.append(f"| {r.area} | {r.check} | {r.status} | {r.value} | {r.detail} |")
    return "\n".join(lines)

In [10]:
%%module meridian.scorer
"""Local development scorer.

Dossier §9.1 describes the official matching only in words: work down the
submitted ranking and match each submitted case one-to-one to a planted case
whose member set overlaps "sufficiently". The overlap measure and threshold
here are our own, public and configurable; results are labelled as coming
from the development scorer.

This is the only module that reads labels.
"""
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

K_VALUES = (10, 25, 50, 100)

# Dossier §8.1: every planted case sits in the one period in which it emerges.
PERIODS = {
    "TRAINING": (None, "2024-06-30"),
    "VALIDATION": ("2024-07-01", "2025-06-30"),
    "TEST": ("2025-07-01", "2026-06-30"),
}
# Starter cases that emerge after their last member was onboarded, with the date their label gives.
EMERGENCE_OVERRIDES = {
    "CASE-0016": ("2025-01-25", "terminated 2025-01-25, payments continue afterwards"),
}


def split_ids(s) -> list[str]:
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return []
    return [x for x in str(s).split("|") if x]


def load_labels(path: str | Path) -> pd.DataFrame:
    """Labelled cases: the starter practice labels, or the sponsor's validation labels of the main data
    (v1.3: `validation_student_safe.csv`, released 2026-11, with no case ids and an `emergence_date` per case).
    `difficulty_tier` and `market` are kept for stratified evaluation only; nothing in the method reads them."""
    path = Path(path)
    df = pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)
    df = df.copy()
    if "case_id" not in df.columns:   # v1.3: number the cases in file order (VAL-01, ...)
        split = set(df["split"].astype(str).str.lower()) if "split" in df.columns else set()
        prefix = "VAL" if split == {"validation"} else "LBL"
        df.insert(0, "case_id", [f"{prefix}-{i + 1:02d}" for i in range(len(df))])
    if "emergence_date" in df.columns:
        df["emergence_date"] = pd.to_datetime(df["emergence_date"])
    df["members"] = df["member_record_ids"].map(lambda s: frozenset(split_ids(s)))
    df["tags"] = df["typology_tags"].map(lambda s: frozenset(split_ids(s)))
    return df


def case_periods(truth: pd.DataFrame, third_parties: pd.DataFrame) -> pd.DataFrame:
    """Emergence date and period of each planted case.

    The starter labels carry no dates. A case is taken to emerge when its last
    member is onboarded, except for the cases in EMERGENCE_OVERRIDES. Labels
    that carry an `emergence_date` (the main-data validation labels, v1.3) use it.
    Labels only: detection never sees these dates.
    """
    onboarded = third_parties.set_index("record_id")["onboarding_date"]
    rows = []
    for r in truth.itertuples():
        if r.case_id in EMERGENCE_OVERRIDES:
            date, basis = EMERGENCE_OVERRIDES[r.case_id]
            emerged = pd.Timestamp(date)
        elif pd.notna(getattr(r, "emergence_date", None)):
            emerged, basis = pd.Timestamp(r.emergence_date), "label emergence date"
        else:
            emerged = onboarded.reindex(list(r.members)).max()
            basis = "last member onboarded"
        period = next(p for p, (lo, hi) in PERIODS.items()
                      if (lo is None or emerged >= pd.Timestamp(lo)) and emerged <= pd.Timestamp(hi))
        rows.append({"case_id": r.case_id, "emerged": emerged, "period": period, "basis": basis})
    return pd.DataFrame(rows)


def shortcut_checks(truth: pd.DataFrame, bundle, min_support: int = 5) -> pd.DataFrame:
    """Single record attributes that separate planted members from the rest.

    A value held by at least `min_support` records that are all planted members
    is a likely artefact of how cases were generated: it may not carry over to
    other data, so detectors should use it as corroboration at most. Record ids
    are random tokens; their order should not separate members (AUC near 0.5).
    """
    tp = bundle.third_parties.merge(bundle.tp_due_diligence, on="record_id", how="left")
    pos = tp["record_id"].isin(set().union(*truth["members"]))
    rows = []
    for col in tp.columns:
        if col == "record_id" or tp[col].nunique() > 100 or pd.api.types.is_datetime64_any_dtype(tp[col]):
            continue
        for value, g in pos.groupby(tp[col].astype(str)):
            if len(g) >= min_support and g.all():
                rows.append({"attribute": col, "value": value, "records": len(g), "all_planted": True})
    rank = tp["record_id"].rank()
    n_pos, n_neg = int(pos.sum()), int((~pos).sum())
    auc = (rank[pos].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg) if n_pos and n_neg else np.nan
    rows.append({"attribute": "record_id order (AUC)", "value": f"{auc:.3f}", "records": len(tp),
                 "all_planted": abs(auc - 0.5) >= 0.1})
    return pd.DataFrame(rows)


def read_submission(path: str | Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    return prepare_submission(df)


def prepare_submission(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if df["case_rank"].duplicated().any():
        raise ValueError("submission ranks must be unique")
    df = df.sort_values("case_rank").reset_index(drop=True)
    df["members"] = df["member_record_ids"].map(lambda s: frozenset(split_ids(s)))
    df["tags"] = df["typology_tags"].map(lambda s: frozenset(split_ids(s)))
    return df


def overlap(a: frozenset, b: frozenset, measure: str) -> float:
    inter = len(a & b)
    if inter == 0:
        return 0.0
    if measure == "iou":
        return inter / len(a | b)
    if measure == "overlap_coef":
        return inter / min(len(a), len(b))
    raise ValueError(measure)


@dataclass
class ScorerConfig:
    measure: str = "iou"
    threshold: float = 0.5
    k_values: tuple = K_VALUES


def match(sub: pd.DataFrame, truth: pd.DataFrame, cfg: ScorerConfig) -> pd.DataFrame:
    """Greedy one-to-one matching down the submitted ranking.

    Each submitted case takes the still-unmatched planted case with the highest
    overlap at or above the threshold (ties broken by case_id). A planted case
    matched once cannot be matched again, so resubmitting it scores nothing.
    """
    truth_members = list(truth["members"])
    truth_ids = list(truth["case_id"])
    taken: set[int] = set()
    rows = []
    for r in sub.itertuples():
        best, best_j = 0.0, None
        for j, tm in enumerate(truth_members):
            if j in taken:
                continue
            o = overlap(r.members, tm, cfg.measure)
            if o >= cfg.threshold and (o > best or (o == best and best_j is not None
                                                     and truth_ids[j] < truth_ids[best_j])):
                best, best_j = o, j
        if best_j is not None:
            taken.add(best_j)
        rows.append({"case_rank": r.case_rank, "matched_case": truth_ids[best_j] if best_j is not None else None,
                     "overlap": best if best_j is not None else 0.0})
    return pd.DataFrame(rows)


def score(sub: pd.DataFrame, truth: pd.DataFrame, cfg: ScorerConfig | None = None) -> dict:
    cfg = cfg or ScorerConfig()
    sub = sub.sort_values("case_rank").reset_index(drop=True)
    m = match(sub, truth, cfg)
    n_truth = len(truth)
    hit = m["matched_case"].notna().to_numpy()
    out: dict = {"n_submitted": len(sub), "n_planted": n_truth,
                 "measure": cfg.measure, "threshold": cfg.threshold}

    for k in cfg.k_values:
        h = hit[:k].sum()
        out[f"P@{k}"] = h / k
        out[f"R@{k}"] = h / n_truth if n_truth else np.nan
        if len(sub) < k:
            out[f"short@{k}"] = True  # fewer cases than k: reported, never padded

    # Average precision over the ranked list, normalised by all planted cases.
    prec_at_hits = [hit[: i + 1].sum() / (i + 1) for i in np.flatnonzero(hit)]
    out["AP"] = float(np.sum(prec_at_hits) / n_truth) if n_truth else np.nan
    out["first_hit_rank"] = int(sub.loc[np.flatnonzero(hit)[0], "case_rank"]) if hit.any() else None
    out["recall"] = hit.sum() / n_truth if n_truth else np.nan
    out["fp_top50"] = int((~hit[:50]).sum())

    truth_by_id = truth.set_index("case_id")
    matched = m[hit].merge(sub[["case_rank", "members", "tags"]], on="case_rank")
    ious, mrec, exact, tag_exact, tag_jacc, tag_any = [], [], [], [], [], []
    for r in matched.itertuples():
        t = truth_by_id.loc[r.matched_case]
        ious.append(overlap(r.members, t.members, "iou"))
        mrec.append(len(r.members & t.members) / len(t.members))
        exact.append(r.members == t.members)
        tag_exact.append(r.tags == t.tags)
        tag_jacc.append(len(r.tags & t.tags) / len(r.tags | t.tags) if (r.tags | t.tags) else 0.0)
        tag_any.append(bool(r.tags & t.tags))
    out["mean_iou"] = float(np.mean(ious)) if ious else np.nan
    out["member_recall"] = float(np.mean(mrec)) if mrec else np.nan
    out["exact_recovery"] = float(np.mean(exact)) if exact else np.nan
    out["tag_exact"] = float(np.mean(tag_exact)) if tag_exact else np.nan
    out["tag_jaccard"] = float(np.mean(tag_jacc)) if tag_jacc else np.nan
    out["tag_any"] = float(np.mean(tag_any)) if tag_any else np.nan

    matched_ids = set(m["matched_case"].dropna())
    by_tag = {}
    for tag in sorted({t for ts in truth["tags"] for t in ts}):
        cases = truth[truth["tags"].map(lambda ts: tag in ts)]
        by_tag[tag] = cases["case_id"].isin(matched_ids).mean()
    out["recall_by_tag"] = by_tag
    out["macro_tag_recall"] = float(np.mean(list(by_tag.values()))) if by_tag else np.nan
    out["worst_tag_recall"] = float(min(by_tag.values())) if by_tag else np.nan
    if "difficulty_tier" in truth.columns:
        out["recall_by_difficulty"] = truth.groupby("difficulty_tier")["case_id"].apply(
            lambda s: s.isin(matched_ids).mean()).to_dict()
    out["_matches"] = m
    return out


def coverage(candidates: list[frozenset], truth: pd.DataFrame, cfg: ScorerConfig | None = None) -> pd.DataFrame:
    """Rank-free coverage: can any candidate match each planted case?

    Separates "not found" from "ranked too low".
    """
    cfg = cfg or ScorerConfig()
    rows = []
    for t in truth.itertuples():
        best = max((overlap(c, t.members, cfg.measure) for c in candidates), default=0.0)
        best_iou = max((overlap(c, t.members, "iou") for c in candidates), default=0.0)
        rows.append({"case_id": t.case_id, "tags": "|".join(sorted(t.tags)), "n_members": len(t.members),
                     "best_overlap": best, "best_iou": best_iou, "covered": best >= cfg.threshold})
    return pd.DataFrame(rows)


def summary_line(res: dict) -> str:
    keys = ["P@10", "P@25", "R@25", "R@50", "R@100", "AP", "first_hit_rank", "mean_iou",
            "macro_tag_recall", "worst_tag_recall", "tag_any", "fp_top50"]
    parts = []
    for k in keys:
        v = res.get(k)
        parts.append(f"{k}={v:.3f}" if isinstance(v, float) else f"{k}={v}")
    return " ".join(parts)


def main(argv=None):
    import argparse

    ap = argparse.ArgumentParser(description="Score a submission CSV against labels (development scorer).")
    ap.add_argument("submission")
    ap.add_argument("labels")
    ap.add_argument("--measure", default="iou")
    ap.add_argument("--threshold", type=float, default=0.5)
    a = ap.parse_args(argv)
    res = score(read_submission(a.submission), load_labels(a.labels), ScorerConfig(a.measure, a.threshold))
    print(summary_line(res))
    print("recall_by_tag:", {k: round(v, 2) for k, v in res["recall_by_tag"].items()})


if __name__ == "__main__":
    main()

In [11]:
%%module meridian.detectors.base
"""Shared types for detectors."""
from __future__ import annotations

from dataclasses import dataclass, field

import numpy as np
import pandas as pd

# The 14 tags used by the starter labels. Fixed vocabulary.
TAGS = (
    "SHARED_ADDRESS",
    "SPONSOR_CONCENTRATION",
    "APPROVER_CONCENTRATION",
    "SPONSOR_APPROVER_COUPLING",
    "DENSE_MULTI_RELATION_CLUSTER",
    "RAPID_ONBOARDING_LINKED",
    "BRIDGE_ENTITY",
    "GEOGRAPHY_LINKED_CONCENTRATION",
    "ACTIVITY_TIMING_CONCENTRATION",
    "COMMON_BENEFICIAL_OWNER",
    "LAYERED_COMPLEX_OWNERSHIP",
    "OWNERSHIP_OPERATIONAL_OVERLAP",
    "OPAQUE_JURISDICTION_ROUTING",
    "OWNERSHIP_CHANGE_STRUCTURAL_SHIFT",
)
OWNERSHIP_TAGS = frozenset({
    "COMMON_BENEFICIAL_OWNER", "LAYERED_COMPLEX_OWNERSHIP", "OWNERSHIP_OPERATIONAL_OVERLAP",
    "OPAQUE_JURISDICTION_ROUTING", "OWNERSHIP_CHANGE_STRUCTURAL_SHIFT",
})
# Tags for single-record policy findings, which the dossier scores separately.
POLICY_TAG = "POLICY_VIOLATION"
OPEN_SET_TAG = "OPEN_SET"


def tail_threshold(values, quantile: float | None, fixed: float) -> float:
    """Threshold from a peer distribution: its `quantile`, so values at or above it
    form the upper tail of this dataset's own distribution. With quantile None, or no
    values, the `fixed` threshold applies."""
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]
    if quantile is None or len(v) == 0:
        return float(fixed)
    return float(np.quantile(v, quantile))


# Vendor peer groups from most to least specific; () is all vendors.
VENDOR_LEVELS = [("party_type", "parent", "country"), ("party_type", "parent", "subregion"),
                 ("party_type", "parent", "region"), ("party_type", "parent"), ("party_type",), ()]


def peer_tail(frame: pd.DataFrame, value: str, levels, min_peers: int = 5, tail: float | None = None) -> pd.DataFrame:
    """For each row: the share of its peers whose `value` is at least as high, and the peer group.

    Peers are the other rows of the most specific group in `levels` that has at
    least `min_peers` of them (Dossier §3.7: a peer residual conditioned on
    role, market and category, backing off to coarser groups); () is all rows.
    A `tail` share (e.g. 0.05) can only be told apart with at least 1/tail
    peers, so the minimum is raised to that.
    """
    if tail:
        min_peers = max(min_peers, int(np.ceil(1 / tail - 1e-9)))
    vals = frame[value].astype(float)
    groups = {}
    for level in levels:
        if level:
            keys = frame[list(level)].apply(tuple, axis=1)
            ok = frame[list(level)].notna().all(axis=1)
            groups[level] = {k: np.sort(g.to_numpy()) for k, g in vals[ok].groupby(keys[ok])}
    every = np.sort(vals.to_numpy())
    rows = []
    for i, row in frame.iterrows():
        x = vals[i]
        for level in levels:
            if level:
                key = tuple(row[c] for c in level)
                peers = groups[level].get(key)
                if peers is None or len(peers) - 1 < min_peers:
                    continue
                name = " x ".join(f"{c} {k}" for c, k in zip(level, key))
            else:
                peers, name = every, "all"
            at_least = len(peers) - np.searchsorted(peers, x, side="left") - 1   # others at least as high
            rows.append((at_least / max(len(peers) - 1, 1), name, len(peers) - 1))
            break
    return pd.DataFrame(rows, index=frame.index, columns=["tail_share", "peer_group", "n_peers"])


def vendor_peers(ctx) -> pd.DataFrame:
    """Peer dimensions per vendor record: party type, main service-category parent, country, subregion, region."""
    if "_vendor_peers" not in ctx.__dict__:
        v = ctx.vendors.set_index("record_id")[["party_type", "country"]].copy()
        parents = ctx.engagements.dropna(subset=["parent"]).groupby("record_id")["parent"].agg(
            lambda s: s.value_counts().sort_index().idxmax())
        v["parent"] = parents.reindex(v.index)
        v["subregion"] = v["country"].map(ctx.markets["subregion"])
        v["region"] = v["country"].map(ctx.markets["region"])
        ctx.__dict__["_vendor_peers"] = v
    return ctx.__dict__["_vendor_peers"]


def peer_note(p) -> str:
    """Benign note for a signal that is not rare among its peers."""
    return f"common among peers: {p.tail_share:.0%} of {p.n_peers} peers ({p.peer_group}) are at least as high"


@dataclass
class Candidate:
    """One detector hit, carrying everything needed later for assembly and explanation.

    `score` is the detector's own strength measure (higher = more unusual); it is
    not comparable across detectors until ranking calibrates it.
    """

    members: frozenset
    tags: tuple
    score: float
    emerged_at: pd.Timestamp | None
    detector: str
    reason: str
    evidence: dict = field(default_factory=dict)
    benign: list = field(default_factory=list)
    features: dict = field(default_factory=dict)
    uses_ownership: bool = False
    # Last date the pattern held; None = still holding at t. Event-like patterns
    # (bursts, spikes) end with their window; structures end when they change.
    active_until: pd.Timestamp | None = None
    # A timeless signal (an outlier score) has no emergence of its own: it adds
    # evidence to a case on the same members but never sets the case's time frame.
    timeless: bool = False
    # Ids of the graph edges that support this candidate (Dossier Appendix C: the
    # evidence edges used by each reason); case packages draw them as a subgraph.
    edges: tuple = ()

    def __post_init__(self):
        self.members = frozenset(self.members)
        self.tags = tuple(self.tags)
        self.edges = tuple(dict.fromkeys(self.edges))
        if self.emerged_at is not None and not pd.isna(self.emerged_at):
            self.emerged_at = pd.Timestamp(self.emerged_at)
        else:
            self.emerged_at = None
        if self.active_until is not None and not pd.isna(self.active_until):
            self.active_until = pd.Timestamp(self.active_until)
        else:
            self.active_until = None


In [12]:
%%module meridian.detectors.relations
"""Entity-level shared-relation items, weighted by rarity.

Each vendor entity carries items such as `sponsor:EMP-..` or `address:<key>`.
Two entities are related when they share an item. Items held by very many
entities (a central buyer, a registered agent, a generic mail domain) are
not expanded into all their pairs; their weight is also low.
"""
from __future__ import annotations

from collections import defaultdict
from itertools import combinations

import numpy as np
import pandas as pd

from ..context import Context

# Relation types that define a structural link between vendors. Category and
# onboarding window are checked pairwise afterwards: they are too common to
# define links on their own.
LINK_TYPES = ("address", "phone", "domain", "sponsor", "approver", "owner", "institution")
# Relations that identify a party rather than reflect a shared business process.
IDENTITY_TYPES = ("address", "phone", "domain", "owner")


# The meta-path behind each relation type: two vendors share an item when their
# instances of the meta-path end on the same node.
METAPATHS = {
    "address": ("LOCATED_AT",),                                  # vendor -> address
    "contact": ("HAS_CONTACT",),                                 # vendor -> phone / email domain
    "sponsor": ("CONTRACTED_UNDER", "SPONSORED_BY"),             # vendor -> engagement -> employee
    "approver": ("CONTRACTED_UNDER", "APPROVED_BY"),             # vendor -> engagement -> employee (financial)
    "category": ("CONTRACTED_UNDER", "FOR_SERVICE"),             # vendor -> engagement -> category (unambiguous)
    "institution": (("SERVES", "SITE_OF", "AFFILIATED_WITH"),),  # vendor -> institution
    # owner: vendor -> registry entity <- (OWNS|CONTROLS)* <- owner, a variable-length
    # upward walk evaluated over each vendor's history (see ownership.OwnershipView)
}


def entity_items(ctx: Context, use_ownership: bool | None = None) -> dict[str, dict[str, set]]:
    """type -> item (the shared end node) -> set of vendor entities, from graph meta-paths."""
    from ..graph_ops import hetero, strip

    use_ownership = ctx.use_ownership if use_ownership is None else use_ownership
    hg = hetero(ctx)
    ent = ctx.entity_of
    items: dict[str, dict[str, set]] = {t: defaultdict(set) for t in LINK_TYPES + ("category",)}

    def fill(kind, mp, order, key=strip):
        mp = mp.sort_values(order, kind="stable")
        k = max(int(c[1:]) for c in mp.columns if c[0] == "n" and c[1:].isdigit())
        for start, end in zip(mp["n0"], mp[f"n{k}"]):
            items[kind][key(end)].add(ent[strip(start)])

    loc = hg.metapath(*METAPATHS["address"]).copy()
    loc["raw"] = [p.get("raw_address") for p in loc["props1"]]
    fill("address", loc, ["n0", "valid_from1", "observed_at1", "raw"],
         key=lambda n: n[len("address:"):])
    con = hg.metapath(*METAPATHS["contact"])
    fill("phone", con[con["n1"].str.startswith("phone:")], ["n0"])
    fill("domain", con[con["n1"].str.startswith("domain:")], ["n0"])
    fill("sponsor", hg.metapath(*METAPATHS["sponsor"]), ["n1"])
    certain = {2: lambda e: e["resolution_confidence"] == 1.0}
    cat = hg.metapath(*METAPATHS["category"], where=certain).sort_values(["n0", "n1"], kind="stable")
    for start, grp in cat.groupby("n0", sort=True):
        for c in pd.unique(grp["n2"]):
            items["category"][strip(c)].add(ent[strip(start)])
    financial = {2: lambda e: [p.get("approval_type") == "financial" for p in e["props"]]}
    fill("approver", hg.metapath(*METAPATHS["approver"], where=financial), ["source_record_id2"])
    fill("institution", hg.metapath(*METAPATHS["institution"]), ["source_record_id1"])
    if use_ownership:
        from .ownership import _view
        ov = _view(ctx)
        for rec in ctx.reg_entity_of:
            for m in ov.snapshots(rec):
                for p in m["persons"]:
                    items["owner"][p].add(ent[rec])
                for s in m["intermediates"]:
                    items["owner"][s].add(ent[rec])
        for rec, p in ctx.reg_person_of.items():
            items["owner"][p].add(ent[rec])
    return {t: dict(v) for t, v in items.items()}


def idf(n: int, total: int) -> float:
    return float(np.log(total / n) / np.log(total)) if n > 0 else 0.0


def pair_links(ctx: Context, items: dict, max_item_size: int = 30) -> dict[tuple, dict[str, list]]:
    """(entity_a, entity_b) -> {type: [shared items]} over LINK_TYPES, skipping hub items."""
    pairs: dict[tuple, dict[str, list]] = defaultdict(lambda: defaultdict(list))
    for t in LINK_TYPES:
        for key, es in items.get(t, {}).items():
            if 2 <= len(es) <= max_item_size:
                for a, b in combinations(sorted(es), 2):
                    pairs[(a, b)][t].append(key)
    return pairs


def entity_onboarding(ctx: Context) -> pd.Series:
    v = ctx.vendors.assign(entity=ctx.vendors["record_id"].map(ctx.entity_of))
    return v.groupby("entity")["onboarding_date"].min()


def entity_country(ctx: Context) -> pd.Series:
    v = ctx.vendors.assign(entity=ctx.vendors["record_id"].map(ctx.entity_of))
    return v.groupby("entity")["country"].first()

In [13]:
%%module meridian.detectors.evidence
"""Look up the graph edges that support a detector's finding.

Each function returns edge ids from the canonical graph; detectors attach them
to their candidates so every case can be traced to the exact edges behind it.
"""
from __future__ import annotations

from collections import defaultdict

import numpy as np

from ..context import Context
from ..graph_ops import hetero
from .relations import METAPATHS

_END_PREFIX = {"address": "address:", "phone": "phone:", "domain": "domain:", "sponsor": "employee:",
               "approver": "employee:", "category": "category:", "institution": "institution:"}


def _index(ctx: Context, name: str, build):
    key = f"_evidence_{name}"
    if key not in ctx.__dict__:
        ctx.__dict__[key] = build()
    return ctx.__dict__[key]


def _positions(ctx: Context, name: str, keys) -> dict:
    """key -> row positions (in table order), built once per context."""
    def build():
        idx = defaultdict(list)
        for i, k in enumerate(keys()):
            idx[k].append(i)
        return idx
    return _index(ctx, name, build)


def _rows(idx: dict, keys) -> np.ndarray:
    """Row positions of all `keys`, in table order."""
    return np.array(sorted(i for k in set(keys) for i in idx.get(k, ())), dtype=int)


def item_edges(ctx: Context, kind: str, item, records) -> list[str]:
    """Edges of the meta-path instances that link `records` to a shared item of type `kind`."""
    if kind == "owner":
        return owner_edges(ctx, records, item)
    steps = METAPATHS["contact" if kind in ("phone", "domain") else kind]
    mp = hetero(ctx).metapath(*steps)
    k = len(steps)
    by_end = _positions(ctx, f"end_{'_'.join(map(str, steps))}", lambda: mp[f"n{k}"])
    hit = mp.iloc[_rows(by_end, [_END_PREFIX[kind] + str(item)])]
    hit = hit[hit["n0"].isin({f"vendor:{r}" for r in records})]
    if kind == "approver":
        hit = hit[[p.get("approval_type") == "financial" for p in hit["props2"]]]
    return [e for i in range(1, k + 1) for e in hit[f"e{i}"]]


def engagement_edges(ctx: Context, engagement_ids, relations=("CONTRACTED_UNDER", "SPONSORED_BY")) -> list[str]:
    """Edges that attach the given engagements to their vendor and sponsor (and other relations)."""
    def build():
        idx = defaultdict(list)
        for rel in ("CONTRACTED_UNDER", "SPONSORED_BY", "APPROVED_BY", "FOR_SERVICE"):
            e = hetero(ctx).edges(rel)
            eng = e["target_node_id"] if rel == "CONTRACTED_UNDER" else e["source_node_id"]
            for n, eid in zip(eng, e["edge_id"]):
                idx[(rel, n[len("engagement:"):])].append(eid)
        return idx
    idx = _index(ctx, "engagements", build)
    return [eid for en in engagement_ids for rel in relations for eid in idx.get((rel, en), [])]


def approval_edges(ctx: Context, engagement_ids, approver: str | None = None) -> list[str]:
    """Financial approval edges on the given engagements (optionally by one approver)."""
    e = hetero(ctx).edges("APPROVED_BY")
    idx = _positions(ctx, "approved_by", lambda: e["source_node_id"])
    e = e.iloc[_rows(idx, (f"engagement:{x}" for x in engagement_ids))]
    keep = np.array([p.get("approval_type") == "financial" for p in e["props"]], dtype=bool)
    if approver is not None:
        keep &= (e["target_node_id"] == f"employee:{approver}").to_numpy()
    return list(e.loc[keep, "edge_id"])


def statement_edges(ctx: Context, statement_ids) -> list[str]:
    """Ownership/control edges published as the given registry statements."""
    def build():
        e = hetero(ctx).edges("OWNS", "CONTROLS")
        idx = defaultdict(list)
        for s, eid in zip(e["source_record_id"], e["edge_id"]):
            idx[s].append(eid)
        return idx
    idx = _index(ctx, "statements", build)
    return [eid for s in statement_ids for eid in idx.get(s, [])]


def registration_edges(ctx: Context, records) -> list[str]:
    """Accepted links from vendor records to registry entities or persons."""
    e = hetero(ctx).edges("REGISTERED_AS", "IS_PERSON")
    idx = _positions(ctx, "registration", lambda: e["source_node_id"])
    e = e.iloc[_rows(idx, (f"vendor:{r}" for r in records))]
    return list(e.loc[np.array([p.get("status") == "accepted" for p in e["props"]], dtype=bool), "edge_id"])


def path_edges(ctx: Context, record: str, paths) -> list[str]:
    """Registration edge plus the ownership edges along the given interest paths."""
    sids = [x["statement_id"] for p in paths for x in p if x.get("party_sid")]
    return registration_edges(ctx, [record]) + statement_edges(ctx, sids)


def owner_edges(ctx: Context, records, owner_sid: str) -> list[str]:
    """Edges by which each record reaches `owner_sid` (registration plus ownership chain)."""
    from .ownership import _view
    ov = _view(ctx)
    out = []
    for r in records:
        if ctx.reg_person_of.get(r) == owner_sid:
            out += registration_edges(ctx, [r])
            continue
        for m in ov.snapshots(r):
            hits = [p for p in m["paths"] if any(x.get("party_sid") == owner_sid for x in p)]
            if hits:
                out += path_edges(ctx, r, [p[:[x.get("party_sid") for x in p].index(owner_sid) + 1] for p in hits])
    return out


def shared_item_edges(ctx: Context, covered: dict, records) -> list[str]:
    """Edges linking `records` to the items they share: {type: item}. Window and category
    coverage without an end node (onboarding window) contribute no edge."""
    out = []
    for kind, item in covered.items():
        if kind in _END_PREFIX or kind == "owner":
            out += item_edges(ctx, kind, item, records)
    return out


def link_edges(ctx: Context, link_ids) -> list[str]:
    e = hetero(ctx).edges("SERVES", "SITE_OF", "AFFILIATED_WITH")
    return list(e.loc[e["source_record_id"].isin(set(link_ids)), "edge_id"])


def identity_edges(ctx: Context, records) -> list[str]:
    e = hetero(ctx).edges("SAME_AS")
    ids = {f"vendor:{r}" for r in records}
    return list(e.loc[e["source_node_id"].isin(ids) & e["target_node_id"].isin(ids), "edge_id"])


def address_edges(ctx: Context, records, since=None, until=None) -> list[str]:
    e = hetero(ctx).edges("LOCATED_AT")
    keep = e["source_node_id"].isin({f"vendor:{r}" for r in records})
    if since is not None:
        keep &= e["valid_from"] >= since
    if until is not None:
        keep &= e["valid_from"] <= until
    return list(e.loc[keep, "edge_id"])

In [14]:
%%module meridian.detectors.shared_address
"""SHARED_ADDRESS: several unrelated vendors registered at one address at the same time."""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..context import Context
from .base import Candidate, peer_note, peer_tail

# Address types where co-location is ordinary; they lower the score, never remove it.
BENIGN_TYPES = {"registered_agent", "coworking", "virtual_office", "hospital_campus", "corporate_hq"}
RARE_TYPES = {"residential"}


def _episodes(ctx: Context, g: pd.DataFrame, k: int) -> list[dict]:
    """Co-location episodes at one address.

    Sweeps the validity intervals. An episode starts on the day the number of
    concurrent vendor entities reaches k and ends on the day it falls below k
    again. Returns, per episode: the peak entity count, the records present at
    the peak, the day k was reached and the day it ended (None if ongoing).
    """
    events = []
    for r in g.itertuples():
        events.append((r.valid_from, 1, r.record_id))
        if pd.notna(r.valid_to):
            events.append((r.valid_to, 0, r.record_id))  # ends sort before starts on the same day
    events.sort(key=lambda x: (x[0], x[1]))
    cur: dict[str, int] = {}
    episodes, ep, below_since = [], None, None
    for d, is_start, rid in events:
        if below_since is not None and d > below_since:  # still below k at the end of that day
            ep["ended"] = below_since
            episodes.append(ep)
            ep = below_since = None
        if is_start:
            cur[rid] = cur.get(rid, 0) + 1
        else:
            cur[rid] -= 1
            if cur[rid] == 0:
                del cur[rid]
        n = ctx.n_entities(cur)
        if n >= k:
            below_since = None
            if ep is None:
                ep = {"peak": 0, "members": set(), "reached": d, "ended": None}
            if n > ep["peak"]:
                ep["peak"], ep["members"] = n, set(cur)
        elif ep is not None and below_since is None:
            below_since = d
    if ep is not None:
        ep["ended"] = below_since
        episodes.append(ep)
    return episodes


def address_instances(ctx: Context) -> pd.DataFrame:
    """Instances of the meta-path VendorRecord -LOCATED_AT-> Address, one row per edge."""
    from ..graph_ops import hetero
    mp = hetero(ctx).metapath("LOCATED_AT")
    a = pd.DataFrame({
        "record_id": mp["n0"].str.slice(len("vendor:")), "addr_key": mp["n1"].str.slice(len("address:")),
        "edge_id": mp["e1"], "valid_from": mp["valid_from1"], "valid_to": mp["valid_to1"],
        "observed_at": mp["observed_at1"], "raw_address": [p.get("raw_address") for p in mp["props1"]],
        "address_type": [p.get("address_type") for p in mp["props1"]],
    })
    return a.sort_values(["record_id", "valid_from", "observed_at", "raw_address"], kind="stable")


def detect(ctx: Context) -> list[Candidate]:
    """Address nodes reached by several vendor entities through overlapping LOCATED_AT edges.

    One candidate per co-location episode: a second group at the same address
    in a later period is a separate pattern.
    """
    k = ctx.param("shared_address", "min_entities", 3)
    a = address_instances(ctx)
    if a.empty:
        return []
    peers = address_peers(ctx, a)
    sizes = a.groupby("addr_key")["record_id"].transform("nunique")
    a = a[sizes >= 2]
    out = []
    for key, g in a.groupby("addr_key"):
        for ep in _episodes(ctx, g, k):
            out.append(_candidate(ctx, key, g, ep, k, peers.loc[key]))
    return out


ADDRESS_LEVELS = [("address_type", "country"), ("address_type", "subregion"), ("address_type", "region"),
                  ("address_type",), ()]


def address_peers(ctx: Context, a: pd.DataFrame) -> pd.DataFrame:
    """Every address's peak number of concurrent vendor entities, against addresses of the
    same type in the same country (backing off to subregion, region, type, all)."""
    rows = []
    for key, g in a.groupby("addr_key"):
        peak = max((ep["peak"] for ep in _episodes(ctx, g, 1)), default=1) if g["record_id"].nunique() > 1 else 1
        types = g["address_type"].dropna()
        rows.append({"addr_key": key, "peak": peak, "country": key.split(":")[0],
                     "address_type": types.mode().iloc[0] if len(types) else "unclassified"})
    f = pd.DataFrame(rows).set_index("addr_key")
    f["subregion"] = f["country"].map(ctx.markets["subregion"])
    f["region"] = f["country"].map(ctx.markets["region"])
    return f.join(peer_tail(f, "peak", ADDRESS_LEVELS, ctx.param("shared_address", "min_peers", 5),
                            1 - ctx.param("shared_address", "peer_quantile", 0.95)))


def _candidate(ctx: Context, key: str, g: pd.DataFrame, ep: dict, k: int, peer) -> Candidate:
    peak, reached, ended = ep["peak"], ep["reached"], ep["ended"]
    members = ctx.expand(ep["members"])
    types = g["address_type"].dropna()
    atype = types.mode().iloc[0] if len(types) else None
    v = ctx.vendors.loc[sorted(members)]
    groups_declared = v["declared_parent_group"].dropna()
    benign = []
    if atype in BENIGN_TYPES:
        benign.append(f"address type {atype}")
    if len(groups_declared) and groups_declared.nunique() == 1 and len(groups_declared) >= 0.5 * len(v):
        benign.append(f"declared group {groups_declared.iloc[0]}")
    # Rarity is judged against comparable addresses; a common level lowers the priority, never removes the signal.
    if peer.tail_share > 1 - ctx.param("shared_address", "peer_quantile", 0.95):
        benign.append(peer_note(peer))
    if atype is None:
        type_w = 1.0
    else:
        type_w = 1.3 if atype in RARE_TYPES else (0.5 if atype in BENIGN_TYPES else 1.0)
    score = float(np.log2(1 + peak) * type_w)
    raw = g["raw_address"].iloc[0]
    # LOCATED_AT edges of the members that were valid at some point during the episode
    during = (g["valid_from"] <= (ended if ended is not None else pd.Timestamp.max)) & (
        g["valid_to"].isna() | (g["valid_to"] >= reached))
    return Candidate(
        members=members, tags=("SHARED_ADDRESS",), score=score, emerged_at=reached, active_until=ended,
        detector="shared_address",
        reason=(f"{peak} vendors registered concurrently at one {atype or 'unclassified'} address "
                f"in {key.split(':')[0]} ('{raw}'); threshold {k} reached {reached.date()}"
                + (f", fell below it {ended.date()}" if ended is not None else "")),
        evidence={"address_key": key, "raw_addresses": sorted(set(g["raw_address"])), "address_type": atype,
                  "peak_entities": peak, "reached": reached, "ended": ended, "peer_group": peer.peer_group,
                  "peer_share_at_least": peer.tail_share},
        benign=benign, features={"peak_entities": peak, "type_weight": type_w, "peer_tail": peer.tail_share},
        edges=tuple(g.loc[g["record_id"].isin(members) & during, "edge_id"]),
    )


def detect_exact(ctx: Context) -> list[Candidate]:
    """Simplest baseline: identical raw address strings, ignoring time. Kept for the representation ladder."""
    a = ctx.b.tp_address_history
    out = []
    for addr, g in a.groupby("raw_address"):
        members = set(g["record_id"])
        if len(members) < 2:
            continue
        out.append(Candidate(members=members, tags=("SHARED_ADDRESS",), score=float(len(members)),
                             emerged_at=g["valid_from"].sort_values().iloc[1], detector="address_exact",
                             reason=f"{len(members)} records share the raw address '{addr}'"))
    return out


In [15]:
%%module meridian.detectors.sponsor
"""SPONSOR_CONCENTRATION: a sponsor takes on many new vendors in a short window, relative to role peers.

v1.3: RAPID_ONBOARDING_LINKED through one sponsor (`detect_burst`): a burst of new vendors far above the
sponsor's own rate."""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..context import Context
from .base import Candidate, tail_threshold

CENTRAL_ROLES = {"category_manager", "procurement_ss", "system_default"}
# Peer groups from most to least specific; () is all sponsors.
PEER_LEVELS = [("role", "scope", "market"), ("role", "scope", "subregion"), ("role", "scope", "region"),
               ("role", "scope"), ("role",), ()]


def scope_level(scope) -> str:
    """Breadth of an employee's market scope: global, region, subregion, multi_market or local."""
    if not isinstance(scope, str) or not scope:
        return "unknown"
    parts = scope.split(";")
    if "*" in parts:
        return "global"
    if any(p.startswith("R:") for p in parts):
        return "region"
    if any(p.startswith("S:") for p in parts):
        return "subregion"
    return "multi_market" if len(parts) > 1 else "local"


def peer_medians(st: pd.DataFrame, value: str, levels, min_peers: int) -> list[tuple[float, str]]:
    """For each row, (median of `value` over its peers excluding itself, peer definition)."""
    out = []
    groups = {}
    for level in levels:
        if level:
            groups[level] = {k: g[value] for k, g in st.groupby(list(level), dropna=True)}
    for i, row in st.iterrows():
        for level in levels:
            if not level:
                others = st[value].drop(i)
                out.append((float(others.median()), f"all sponsors (n={len(others)})"))
                break
            key = tuple(row[c] for c in level)
            if any(pd.isna(k) for k in key):
                continue
            g = groups[level].get(key if len(key) > 1 else key[0])
            if g is None or len(g) - 1 < min_peers:
                continue
            others = g.drop(i)
            out.append((float(others.median()),
                        " x ".join(f"{c} {k}" for c, k in zip(level, key)) + f" (n={len(others)})"))
            break
    return out


def first_engagements(ctx: Context) -> pd.DataFrame:
    """First engagement of each (sponsor, vendor entity) pair.

    Sponsorships are the instances of the meta-path
    VendorRecord -CONTRACTED_UNDER-> Engagement -SPONSORED_BY-> Employee;
    e1 and e2 are the two edges of each instance.
    """
    from ..graph_ops import hetero
    mp = hetero(ctx).metapath("CONTRACTED_UNDER", "SPONSORED_BY")
    inst = pd.DataFrame({"engagement_id": mp["n1"].str.slice(len("engagement:")), "e1": mp["e1"], "e2": mp["e2"]})
    e = ctx.engagements.merge(inst, on="engagement_id", how="inner")
    e["entity"] = e["record_id"].map(ctx.entity_of)
    return (e.sort_values("start_date")
            .groupby(["sponsor_employee_id", "entity"], as_index=False).first())


def best_window(dates: np.ndarray, days: int) -> tuple[int, int]:
    """(start index, count) of the densest window of `days` in sorted dates."""
    best, bi = 0, 0
    for i in range(len(dates)):
        j = np.searchsorted(dates, dates[i] + np.timedelta64(days, "D"), side="right")
        if j - i > best:
            best, bi = j - i, i
    return bi, best


def first_reach(dates: np.ndarray, days: int, k: int):
    """Earliest date at which some window of `days` holds k dates (the k-th date of that window)."""
    for i in range(len(dates) - k + 1):
        if dates[i + k - 1] - dates[i] <= np.timedelta64(days, "D"):
            return pd.Timestamp(dates[i + k - 1])
    return None


def detect(ctx: Context) -> list[Candidate]:
    days = ctx.param("sponsor", "window_days", 60)
    min_new = ctx.param("sponsor", "min_new", 4)
    fe = first_engagements(ctx)

    stats = []
    for sp, g in fe.groupby("sponsor_employee_id"):
        d = g.sort_values("start_date")
        dates = d["start_date"].to_numpy()
        i, n = best_window(dates, days)
        stats.append({"sponsor": sp, "max_new": n, "win": d.iloc[i:i + n], "dates": dates,
                      "role": ctx.role_of.get(sp), "n_total": len(d)})
    st = pd.DataFrame(stats)
    if st.empty:
        return []
    # Each sponsor's densest window against the median of its peers: the most specific
    # group in PEER_LEVELS with at least `min_peers` other sponsors (Dossier §3.7: a peer
    # residual conditioned on role, scope and market). The threshold is a quantile of the ratios.
    roles = ctx.roles.set_index("employee_id")
    st["scope"] = st["sponsor"].map(roles["scope_markets"]).map(scope_level)
    st["market"] = st["sponsor"].map(roles["home_market"])
    st["subregion"] = st["market"].map(ctx.markets["subregion"])
    st["region"] = st["market"].map(ctx.markets["region"])
    peer = peer_medians(st, "max_new", PEER_LEVELS, ctx.param("sponsor", "min_peers", 5))
    st["peer_med"] = [m for m, _ in peer]
    st["peer_def"] = [d for _, d in peer]
    st["ratio"] = st["max_new"] / st["peer_med"].clip(lower=1.0)
    min_ratio = tail_threshold(st["ratio"], ctx.param("sponsor", "ratio_quantile", None),
                               ctx.param("sponsor", "min_ratio", 3.0))
    out = []
    for r in st.itertuples():
        if r.max_new < min_new:
            continue
        peer_med, peer_def, ratio = r.peer_med, r.peer_def, r.ratio
        if ratio < min_ratio:
            continue
        win = r.win
        lo, hi = win["start_date"].min(), win["start_date"].max()
        members = ctx.expand(win["record_id"])
        role_row = ctx.roles.set_index("employee_id").loc[r.sponsor]
        cats = win["cat_primary"].dropna()
        top_cat = cats.mode().iloc[0] if len(cats) else None
        cat_share = float((cats == top_cat).mean()) if len(cats) else 0.0
        benign = []
        if r.role in CENTRAL_ROLES:
            benign.append(f"central role {r.role}")
        scope = str(role_row["scope_markets"])
        if scope == "*" or scope.startswith(("R:", "S:")):
            benign.append(f"regional/global scope {scope}")
        mkt = win["market"].mode().iloc[0]
        same_day = win["start_date"].value_counts().iloc[0] / len(win)
        if same_day >= 0.8:
            benign.append(f"{same_day:.0%} of the new vendors start on one day: batch entry (migration or integration)")
        benign += ctx.benign_events("sponsor", lo, hi, mkt, cats=zip(win["parent"], win["cat_primary"]),
                                    roles=[r.role])
        emerged = first_reach(r.dates, days, min_new)
        out.append(Candidate(
            members=members, tags=("SPONSOR_CONCENTRATION",), score=float(np.log2(ratio) + np.log2(r.max_new)),
            emerged_at=emerged, active_until=hi, detector="sponsor_concentration",
            reason=(f"{r.role} {r.sponsor} ({role_row['home_market']}) took on {r.max_new} new vendors within "
                    f"{days} days ({lo.date()}–{hi.date()}), {ratio:.1f}x the peer median of {peer_med:g} "
                    f"for {peer_def} (threshold {min_ratio:.1f}x); top category {top_cat} ({cat_share:.0%})"),
            evidence={"sponsor": r.sponsor, "role": r.role, "window": (lo, hi), "peer_median": peer_med,
                      "ratio_threshold": min_ratio,
                      "peer_definition": peer_def, "engagements": list(win["engagement_id"])},
            benign=benign,
            features={"max_new": r.max_new, "ratio": ratio, "cat_share": cat_share},
            edges=tuple(win["e1"]) + tuple(win["e2"]),
        ))
    return out


def detect_burst(ctx: Context) -> list[Candidate]:
    """v1.3, RAPID_ONBOARDING_LINKED through one sponsor: within `window_days` a sponsor brings at least `min_new`
    new vendor entities of one country, far more than the sponsor's own rate there over the previous two years.

    Found after the main-data validation labels: the relation detectors skip a sponsor with more than 30 vendors
    as a hub, so a burst under a busy sponsor had nothing to hold it together (VAL-18). The burst is judged
    against the sponsor's own history: the number of new vendors in the window under a Poisson rate from the
    previous 730 days (at least `min_rate` per window), reported when that chance is below `max_p`. A vendor
    counts once, at its first engagement with the company; its sponsor is the sponsor of that engagement.
    """
    from scipy.stats import poisson
    from .relations import entity_country

    days = ctx.param("sponsor_burst", "window_days", 14)
    min_new = ctx.param("sponsor_burst", "min_new", 4)
    max_p = ctx.param("sponsor_burst", "max_p", 1e-3)
    min_rate = ctx.param("sponsor_burst", "min_rate", 0.1)
    fe = first_engagements(ctx)
    first = fe.sort_values("start_date", kind="stable").drop_duplicates("entity")   # each entity's first engagement
    first = first.assign(country=first["entity"].map(entity_country(ctx)))
    found = []
    for (sp, ctry), g in first.groupby(["sponsor_employee_id", "country"]):
        if len(g) < min_new:
            continue
        g = g.sort_values("start_date", kind="stable")
        dates = g["start_date"].to_numpy()
        for i in range(len(dates) - min_new + 1):
            j = int(np.searchsorted(dates, dates[i] + np.timedelta64(days, "D"), side="right"))
            k = j - i
            if k < min_new:
                continue
            lo = pd.Timestamp(dates[i])
            base = int(((dates >= np.datetime64(lo - pd.Timedelta(days=730))) & (dates < dates[i])).sum())
            rate = max(base * days / 730, min_rate)
            p = float(poisson.sf(k - 1, rate))
            if p < max_p:
                found.append((p, sp, ctry, g.iloc[i:j], base, rate))
    # One candidate per burst: the most surprising window, then drop windows that repeat it.
    found.sort(key=lambda x: (x[0], -len(x[3])))
    kept = []
    for f in found:
        ents = set(f[3]["entity"])
        if any(f[1] == k[1] and len(ents & set(k[3]["entity"])) / len(ents | set(k[3]["entity"])) >= 0.5
               for k in kept):
            continue
        kept.append(f)
    out = []
    for p, sp, ctry, win, base, rate in kept:
        lo, hi = win["start_date"].min(), win["start_date"].max()
        members = ctx.expand(win["record_id"])
        role = ctx.role_of.get(sp)
        benign = []
        if role in CENTRAL_ROLES:
            benign.append(f"central role {role}")
        same_day = win["start_date"].value_counts().iloc[0] / len(win)
        if same_day >= 0.8:
            benign.append(f"{same_day:.0%} of the new vendors start on one day: batch entry (migration or integration)")
        benign += ctx.benign_events("onboarding", lo, hi, ctry, cats=zip(win["parent"], win["cat_primary"]))
        out.append(Candidate(
            members=members, tags=("RAPID_ONBOARDING_LINKED",),
            score=float(-np.log10(max(p, 1e-300))), emerged_at=pd.Timestamp(win["start_date"].iloc[min_new - 1]),
            active_until=hi, detector="sponsor_burst",
            reason=(f"{role} {sp} brought {len(win)} new vendors in {ctry} within {(hi - lo).days + 1} days "
                    f"({lo.date()} to {hi.date()}); {base} in the previous two years ({rate:.2f} expected per "
                    f"{days} days, chance {p:.0e})"),
            evidence={"sponsor": sp, "country": ctry, "window": (lo, hi), "previous_two_years": base,
                      "expected": rate, "p": p, "engagements": list(win["engagement_id"])},
            benign=benign, features={"n_new": len(win), "base": base, "p": p},
            edges=tuple(win["e1"]) + tuple(win["e2"]),
        ))
    return out

In [16]:
%%module meridian.detectors.approval
"""SPONSOR_APPROVER_COUPLING and APPROVER_CONCENTRATION.

Pairs: a sponsor's engagements keep going to one approver although several were
eligible. Measured per (sponsor, approver) pair, because on the starter data
one approver sits in two separate planted pairs: an approver-level view would
merge them into one oversized case.

Stars: one approver handles far more of a market's approvals in a category
than an even split among the approvers authorised for them would give
(Dossier §2.4, worked example B: authority is level plus market scope).
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.stats import binom

from ..context import Context
from .base import Candidate, tail_threshold
from .evidence import approval_edges, engagement_edges

CENTRAL_APPROVER_ROLES = {"regional_finance_l3", "system_default"}


def financial_approvals(ctx: Context) -> pd.DataFrame:
    """Financial approvals with a known approver, each with the approvers eligible for it.

    Eligible: role active on the approval date, level at least the value band's
    requirement, scope covering the engagement's market. `in_scope` and
    `under` flag the approval's own approver.
    """
    if "_fin_approvals" in ctx.__dict__:
        return ctx.__dict__["_fin_approvals"]
    ap = ctx.approvals
    fin = ap[(ap["approval_type"] == "financial") & ap["approver_employee_id"].notna()].copy()
    r = ctx.roles[ctx.roles["approver_level"].fillna(0) > 0]
    scope_cache: dict = {}

    def covers(market):
        if market not in scope_cache:
            scope_cache[market] = r[[ctx.in_scope(s, market) for s in r["scope_markets"]]]
        return scope_cache[market]

    eligible = []
    for m, req, d in zip(fin["market"], fin["required_approver_level"], fin["approval_date"]):
        c = covers(m)
        ok = (c["approver_level"] >= (req if pd.notna(req) else 0)) & (c["valid_from"] <= d) & (
            c["valid_to"].isna() | (c["valid_to"] >= d))
        eligible.append(frozenset(c.loc[ok, "employee_id"]))
    fin["eligible"] = eligible
    fin["n_eligible"] = fin["eligible"].map(len)
    fin["in_scope"] = [ctx.in_scope(s, m) for s, m in zip(fin["ap_scope_markets"], fin["market"])]
    fin["under"] = fin["ap_approver_level"] < fin["required_approver_level"]
    ctx.__dict__["_fin_approvals"] = fin
    return fin


def approver_stars(ctx: Context) -> pd.DataFrame:
    """Every (market, category parent, approver) with at least `min_approvals` approvals.

    The expected count gives each approval to its eligible approvers in equal
    shares; `surprise` is -log10 of the binomial tail of the observed count
    against that share. An approver outside the eligible set gets a small floor share.
    """
    if "_approver_stars" in ctx.__dict__:
        return ctx.__dict__["_approver_stars"]
    min_n = ctx.param("approver_star", "min_approvals", 5)
    fin = financial_approvals(ctx)
    rows = []
    for (m, cat), g in fin.groupby(["market", "parent"], dropna=False):
        g = g.drop_duplicates("engagement_id")
        n = len(g)
        if n < min_n:
            continue
        for a, ga in g.groupby("approver_employee_id"):
            k = len(ga)
            if k < min_n:
                continue
            share = float(np.mean([1.0 / len(e) if a in e else 0.0 for e in g["eligible"]]))
            pool = float(g["n_eligible"].median())
            p = min(max(share, 1.0 / max(pool, 1) / 4), 0.99)
            rows.append({"market": m, "parent": cat, "approver": a, "k": k, "n": n, "expected_share": p,
                         "eligible": pool, "surprise": float(-np.log10(max(binom.sf(k - 1, n, p), 1e-300))),
                         "engagements": sorted(ga["engagement_id"]), "records": set(ga["record_id"]),
                         "dates": np.sort(ga["approval_date"].to_numpy()),
                         "out_of_scope": int((~ga["in_scope"]).sum()), "under": int(ga["under"].sum())})
    st = pd.DataFrame(rows)
    q = ctx.param("approver_star", "surprise_quantile", 0.95)
    st["star"] = st["surprise"] >= tail_threshold(st["surprise"], q, 3.0) if len(st) else []
    ctx.__dict__["_approver_stars"] = st
    return st


def split_by_sponsor(ctx: Context, r, max_members: int, min_records: int) -> list[tuple]:
    """v1.1: a star larger than `max_members` records is split into one group per sponsor.

    An approver's whole (market, category) book is a hub, not a case: the
    pattern a reviewer can act on is the set of vendors one sponsor routed to
    that approver. Groups with fewer than `min_records` records are dropped.
    Returns (records, engagements, approval dates, sponsor) per group; one group when no split is needed.
    """
    if max_members is None or len(r.records) <= max_members:
        return [(set(r.records), list(r.engagements), r.dates, None)]
    fin = financial_approvals(ctx)
    fin = fin[(fin["approver_employee_id"] == r.approver) & fin["engagement_id"].isin(r.engagements)]
    e = ctx.engagements[ctx.engagements["engagement_id"].isin(r.engagements)]
    out = []
    for sp, g in e.groupby("sponsor_employee_id"):
        recs = set(g["record_id"])
        if len(recs) >= min_records:
            dates = np.sort(fin.loc[fin["engagement_id"].isin(g["engagement_id"]), "approval_date"].to_numpy())
            out.append((recs, sorted(g["engagement_id"]), dates, sp))
    return out


def sponsor_share(ctx: Context, sponsor, approver) -> tuple[int, int]:
    """v1.2: the sponsor's engagements with a recorded financial approver, and how many of them `approver` approved."""
    ap = ctx.approvals
    fin = ap[(ap["approval_type"] == "financial") & (ap["sponsor_employee_id"] == sponsor) & ap["approver_employee_id"].notna()]
    return fin["engagement_id"].nunique(), fin.loc[fin["approver_employee_id"] == approver, "engagement_id"].nunique()


def detect_stars(ctx: Context) -> list[Candidate]:
    min_eligible = ctx.param("approval", "min_eligible", 3)
    max_members = ctx.param("approver_star", "max_members", None)
    min_records = ctx.param("approver_star", "min_split_records", 3)
    from .policy import review_gaps   # policy follows this module (in the notebook, a later cell)
    st = approver_stars(ctx)
    gaps = set(review_gaps(ctx)["engagement_id"])
    out = []
    for r0 in st[st["star"]].itertuples() if len(st) else []:
        for records, engagements, dates, via in split_by_sponsor(ctx, r0, max_members, min_records):
            r = r0._replace(records=records, engagements=engagements, dates=dates)
            n_gap = len(gaps & set(r.engagements))
            role = ctx.role_of.get(r.approver)
            scope = str(ctx.roles.set_index("employee_id")["scope_markets"].get(r.approver, ""))
            benign = []
            if role in CENTRAL_APPROVER_ROLES or scope == "*" or scope.startswith("R:"):
                benign.append(f"centralised approval role {role} ({scope})")
            if r.eligible < min_eligible:
                benign.append(f"small approver pool ({r.eligible:g})")
            min_n = ctx.param("approver_star", "min_approvals", 5)
            coupled, pair_note = False, ""
            if via is None:
                lead = (f"approver {r.approver} ({role}) handled {r.k} of {r.n} financial approvals for {r.parent} "
                        f"in {r.market}, where ~{r.eligible:g} approvers were authorised (expected share "
                        f"{r.expected_share:.0%}, surprise 10^-{r.surprise:.1f})")
                n_out, n_under = r.out_of_scope, r.under
            else:   # v1.1: a sponsor group leads with the sponsor, since the short reason keeps the first clause only
                fa = financial_approvals(ctx)
                fa = fa[(fa["approver_employee_id"] == r.approver) & fa["engagement_id"].isin(r.engagements)]
                fa = fa.drop_duplicates("engagement_id")
                lead = (f"sponsor {via} ({ctx.role_of.get(via)}) brought {len(r.records)} vendors "
                        f"({len(r.engagements)} engagements, {r.parent} in {r.market}) to approver {r.approver} "
                        f"({role}), who handled {r.k} of {r.n} such financial approvals where ~{r.eligible:g} "
                        f"approvers were authorised (expected share {r.expected_share:.0%}, surprise 10^-{r.surprise:.1f})")
                n_out, n_under = int((~fa["in_scope"]).sum()), int(fa["under"].sum())
                # v1.2: when the sponsor sends most of its approved engagements to this approver, the group is also a
                # sponsor-approver coupling (same share rule as the pair detector), and is tagged with both
                n_s, n_sa = sponsor_share(ctx, via, r.approver)
                coupled = n_s > 0 and n_sa / n_s >= ctx.param("approval", "min_sponsor_share", 0.5)
                pair_note = f"; {n_sa} of the sponsor's {n_s} approved engagements went to this approver" if coupled else ""
            # v1.3: a group without sponsor coupling is reported by detector `approver_hub` (same tag), which ranks with
            # the open-set prior (section 0.1). Every approver case in the labels is a coupled sponsor-approver pair
            # (the starter cases, and 4 of 4 in the main-data validation labels), and none of the 239 uncoupled groups
            # of the validation period matched a labelled case.
            out.append(Candidate(
                members=ctx.expand(r.records),
                tags=("APPROVER_CONCENTRATION",) + (("SPONSOR_APPROVER_COUPLING",) if coupled else ()),
                score=r.surprise + np.log2(1 + n_gap),
                emerged_at=pd.Timestamp(r.dates[min(min_n, len(r.dates)) - 1]), active_until=pd.Timestamp(r.dates[-1]),
                detector="approver_star" if coupled else "approver_hub",
                reason=(lead + pair_note
                        + (f"; {n_out} outside the approver's market scope" if n_out else "")
                        + (f"; {n_under} below the required level" if n_under else "")
                        + (f"; {n_gap} bypassed the compliance review" if n_gap else "")),
                evidence={"approver": r.approver, "market": r.market, "category_parent": r.parent, "sponsor": via,
                          "engagements": r.engagements, "eligible_approvers": r.eligible,
                          "expected_share": r.expected_share, "out_of_scope": r.out_of_scope, "under_authority": r.under,
                          "compliance_review_gaps": n_gap},
                benign=benign,
                features={"k": r.k, "n": r.n, "eligible": r.eligible, "surprise": r.surprise, "review_gaps": n_gap},
                edges=tuple(engagement_edges(ctx, r.engagements) + approval_edges(ctx, r.engagements, approver=r.approver)),
            ))
    return out


def detect(ctx: Context) -> list[Candidate]:
    min_pair = ctx.param("approval", "min_pair", 5)
    min_share = ctx.param("approval", "min_sponsor_share", 0.5)
    min_eligible = ctx.param("approval", "min_eligible", 3)
    ap = ctx.approvals
    fin = ap[(ap["approval_type"] == "financial") & ap["sponsor_employee_id"].notna()].copy()
    fin["under"] = fin["ap_approver_level"] < fin["required_approver_level"]
    fin["out_of_role"] = (fin["approval_date"] < fin["ap_valid_from"]) | (
        fin["ap_valid_to"].notna() & (fin["approval_date"] > fin["ap_valid_to"]))
    known = fin[fin["approver_employee_id"].notna()]
    eng_s = known.groupby("sponsor_employee_id")["engagement_id"].nunique()
    by_market = {m: g for m, g in known.groupby("market")}
    eng_by_sponsor = {s: g for s, g in ctx.engagements.groupby("sponsor_employee_id")}
    from .policy import review_gaps
    gaps = set(review_gaps(ctx)["engagement_id"])

    out = []
    pairs = known.groupby(["sponsor_employee_id", "approver_employee_id"])
    for (s, a), g in pairs:
        n = g["engagement_id"].nunique()
        if n < min_pair:
            continue
        n_s = eng_s[s]
        share = n / n_s
        if share < min_share:
            continue
        mkts = set(g["market"])
        parts = [by_market[m] for m in mkts if m in by_market]
        if any(pd.isna(m) for m in mkts):
            parts.append(known[known["market"].isna()])
        pool = pd.concat(parts)
        pool = pool[pool["sponsor_employee_id"] != s]
        lo, hi = g["approval_date"].min(), g["approval_date"].max()
        period = pool[(pool["approval_date"] >= lo) & (pool["approval_date"] <= hi)]
        eligible = period[period["ap_approver_level"] >= g["required_approver_level"].min()][
            "approver_employee_id"].nunique()
        if eligible < min_eligible:
            continue
        p_a = (period["approver_employee_id"] == a).mean() if len(period) else 1.0 / max(eligible, 1)
        p_a = min(max(p_a, 1.0 / max(eligible, 1) / 4), 0.99)
        surprise = float(-np.log10(max(binom.sf(n - 1, n_s, p_a), 1e-300)))
        n_under = int(g["under"].sum())
        n_oor = int(g["out_of_role"].sum())
        n_oos = int(sum(not ctx.in_scope(sc, m) for sc, m in zip(g["ap_scope_markets"], g["market"])))
        n_gap = len(gaps & set(g["engagement_id"]))   # engagements of the pair that bypassed the compliance review
        stars = approver_stars(ctx)
        star_in = sorted(set(stars.loc[stars["star"] & (stars["approver"] == a) & stars["market"].isin(mkts),
                                       "market"])) if len(stars) else []

        # The sponsor's engagements with no recorded financial approver may belong to the same pair.
        s_all = eng_by_sponsor.get(s, ctx.engagements.iloc[0:0])
        unapproved = s_all[~s_all["engagement_id"].isin(known["engagement_id"])]
        members = ctx.expand(set(g["record_id"]) | set(unapproved["record_id"]))
        dates = np.sort(g["approval_date"].to_numpy())
        emerged = pd.Timestamp(dates[min_pair - 1])
        tags = ["SPONSOR_APPROVER_COUPLING"]
        # Outside documented authority (level or scope), or a star approver in the pair's market.
        if n_under > 0 or n_oos > 0 or star_in:
            tags.append("APPROVER_CONCENTRATION")
        benign = []
        if ctx.role_of.get(s) == "system_default":
            benign.append("system_default sponsor")
        if eligible <= 3:
            benign.append(f"small approver pool ({eligible})")
        out.append(Candidate(
            members=members, tags=tuple(tags), score=surprise + np.log2(1 + n_under) + np.log2(1 + n_oor) + np.log2(1 + n_gap),
            emerged_at=emerged, detector="sponsor_approver_pair",
            reason=(f"sponsor {s} ({ctx.role_of.get(s)}) routed {n} of {n_s} approved engagements to approver "
                    f"{a} ({ctx.role_of.get(a)}) although {eligible} approvers were eligible in "
                    f"{','.join(sorted(mkts))} (expected share {p_a:.0%}, surprise 10^-{surprise:.1f})"
                    + (f"; {n_under} approvals below the required level" if n_under else "")
                    + (f"; {n_oor} approvals outside the approver's role dates" if n_oor else "")
                    + (f"; {n_oos} approvals outside the approver's market scope" if n_oos else "")
                    + (f"; the approver is concentrated across {','.join(star_in)}" if star_in else "")
                    + (f"; {n_gap} bypassed the compliance review" if n_gap else "")),
            evidence={"sponsor": s, "approver": a, "pair_engagements": sorted(set(g["engagement_id"])),
                      "unapproved_engagements": sorted(set(unapproved["engagement_id"])),
                      "eligible_approvers": eligible, "expected_share": p_a,
                      "under_authority": n_under, "out_of_role": n_oor, "out_of_scope": n_oos,
                      "star_markets": star_in, "compliance_review_gaps": n_gap},
            benign=benign,
            features={"pair_n": n, "sponsor_share": share, "eligible": eligible, "surprise": surprise,
                      "under": n_under, "out_of_role": n_oor, "review_gaps": n_gap},
            edges=tuple(engagement_edges(ctx, sorted(set(g["engagement_id"]) | set(unapproved["engagement_id"])))
                        + approval_edges(ctx, sorted(set(g["engagement_id"])), approver=a)),
        ))
    return out


In [17]:
%%module meridian.detectors.cluster
"""DENSE_MULTI_RELATION_CLUSTER and RAPID_ONBOARDING_LINKED."""
from __future__ import annotations

from collections import Counter

import networkx as nx
import numpy as np
import pandas as pd

from ..context import Context
from .base import Candidate
from .evidence import item_edges, shared_item_edges
from .relations import IDENTITY_TYPES, LINK_TYPES, entity_country, entity_items, entity_onboarding, idf, pair_links


def _relations(ctx: Context):
    if "_relations" not in ctx.__dict__:
        items = entity_items(ctx)
        ctx.__dict__["_relations"] = (items, pair_links(ctx, items))
    return ctx.__dict__["_relations"]


def _records(ctx: Context, entities) -> frozenset:
    return frozenset().union(*[ctx.records_of[e] for e in entities])


def coverage(ctx: Context, members: set, items: dict, onb: pd.Series, window_days: int = 30) -> dict:
    """type -> (best item, share of members holding it). Window = onboarding within window_days of the median."""
    out = {}
    n = len(members)
    for t in LINK_TYPES + ("category",):
        cnt = Counter()
        for key, es in items.get(t, {}).items():
            k = len(es & members)
            if k:
                cnt[key] = k
        if cnt:
            key, k = cnt.most_common(1)[0]
            out[t] = (key, k / n)
    d = onb[list(members)]
    med = d.median()
    out["window"] = (str(med.date()), float((abs(d - med) <= pd.Timedelta(days=window_days)).mean()))
    return out


def detect_dense(ctx: Context) -> list[Candidate]:
    min_types = ctx.param("dense_cluster", "min_pair_types", 4)
    min_size = ctx.param("dense_cluster", "min_size", 4)
    min_cover = ctx.param("dense_cluster", "min_cover", 0.8)
    min_cover_types = ctx.param("dense_cluster", "min_covered_types", 4)
    items, pairs = _relations(ctx)
    onb = entity_onboarding(ctx)
    cats = {}
    for key, es in items.get("category", {}).items():
        for e in es:
            cats.setdefault(e, set()).add(key)

    g = nx.Graph()
    for (a, b), types in pairs.items():
        n_types = len(types)
        if cats.get(a, set()) & cats.get(b, set()):
            n_types += 1
        if abs(onb[a] - onb[b]) <= pd.Timedelta(days=30):
            n_types += 1
        if n_types >= min_types:
            g.add_edge(a, b)
    total = len(onb)
    out = []
    for comp in nx.connected_components(g):
        if len(comp) < min_size:
            continue
        cov = coverage(ctx, comp, items, onb)
        covered = {t: v for t, v in cov.items() if v[1] >= min_cover}
        if len(covered) < min_cover_types:
            continue
        rarity = np.mean([idf(len(items[t][v[0]]), total) for t, v in covered.items() if t in items])
        identity = [t for t in covered if t in IDENTITY_TYPES]
        members = _records(ctx, comp)
        v = ctx.vendors.loc[sorted(members)]
        grp = v["declared_parent_group"].dropna()
        benign = []
        if len(grp) and grp.nunique() == 1 and len(grp) >= 0.8 * len(v):
            benign.append(f"declared group {grp.iloc[0]}")
        if not identity:
            benign.append("only process relations shared (sponsor, approver, category, institution): "
                          "consistent with a standardised local business process")
        out.append(Candidate(
            members=members, tags=("DENSE_MULTI_RELATION_CLUSTER",),
            score=float(len(covered) + len(identity) + np.log2(len(comp)) + rarity), emerged_at=onb[list(comp)].max(),
            detector="dense_cluster",
            reason=(f"{len(comp)} vendors in {','.join(sorted(set(v['country'])))} overlap on {len(covered)} relation "
                    f"types ({', '.join(f'{t} {s:.0%}' for t, (_, s) in sorted(covered.items()))})"),
            evidence={"coverage": {t: {"item": k, "share": s} for t, (k, s) in cov.items()}},
            benign=benign,
            features={"size": len(comp), "covered_types": len(covered), "rarity": rarity},
            edges=tuple(shared_item_edges(ctx, {t: k for t, (k, _) in covered.items()}, members)),
        ))
    return out


STRUCTURAL_LINKS = ("sponsor", "address", "owner", "phone", "domain")


def detect_rapid_onboarding(ctx: Context) -> list[Candidate]:
    days = ctx.param("rapid_onboarding", "window_days", 14)
    min_linked = ctx.param("rapid_onboarding", "min_linked", 4)
    items, pairs = _relations(ctx)
    onb = entity_onboarding(ctx)
    country = entity_country(ctx)
    link = {k: v for k, v in pairs.items() if any(t in v for t in STRUCTURAL_LINKS)}
    adj: dict[str, set] = {}
    for a, b in link:
        adj.setdefault(a, set()).add(b)
        adj.setdefault(b, set()).add(a)
    found = []
    for mkt, es in country.groupby(country):
        d = onb[es.index].sort_values()
        dates = d.to_numpy()
        for i in range(len(d)):
            j = np.searchsorted(dates, dates[i] + np.timedelta64(days, "D"), side="right")
            if j - i < min_linked:
                continue
            win = list(d.index[i:j])
            pos = {e: p for p, e in enumerate(win)}
            g = nx.Graph()
            for a_i, a in enumerate(win):
                for b_i in sorted(pos[b] for b in adj.get(a, ()) if pos.get(b, -1) > a_i):
                    g.add_edge(a, win[b_i])
            comps = [c for c in nx.connected_components(g) if len(c) >= min_linked]
            for comp in comps:
                lo = d.iloc[i]
                prior = d[(d >= lo - pd.Timedelta(days=730)) & (d < lo)]
                expected = len(prior) / (730 / days)
                found.append((mkt, frozenset(comp), lo, onb[list(comp)].max(), len(win), expected))
    # One candidate per linked group: keep the strongest window, drop heavily overlapping repeats.
    found.sort(key=lambda x: (-len(x[1]), x[2]))
    kept = []
    for f in found:
        if any(len(f[1] & k[1]) / len(f[1] | k[1]) >= 0.5 for k in kept):
            continue
        kept.append(f)
    out = []
    for mkt, comp, lo, hi, n_win, expected in kept:
        types = Counter(t for a in sorted(comp) for b in sorted(comp) if a < b and (a, b) in link
                        for t in link[(a, b)] if t in STRUCTURAL_LINKS)
        ratio = len(comp) / max(expected, 0.5)
        members = _records(ctx, comp)
        e = ctx.engagements[ctx.engagements["record_id"].isin(members)]
        first = e.sort_values("start_date").groupby("record_id").first()
        benign = ctx.benign_events("onboarding", lo, hi, mkt, cats=zip(first["parent"], first["cat_primary"]))
        out.append(Candidate(
            members=members, tags=("RAPID_ONBOARDING_LINKED",),
            score=float(np.log2(len(comp)) + np.log2(1 + ratio)), emerged_at=hi, active_until=hi,
            detector="rapid_onboarding",
            reason=(f"{len(comp)} linked vendors onboarded in {mkt} within {(hi - lo).days + 1} days "
                    f"({lo.date()}–{hi.date()}); market baseline {expected:.1f} per {days} days "
                    f"({ratio:.1f}x); links: {', '.join(f'{t}×{n}' for t, n in types.most_common())}"),
            evidence={"market": mkt, "window": (lo, hi), "expected_per_window": expected,
                      "link_types": dict(types)},
            benign=benign,
            features={"n_linked": len(comp), "ratio": ratio},
            edges=tuple(eid for a in comp for b in comp if a < b and (a, b) in link
                        for t, keys in link[(a, b)].items() if t in STRUCTURAL_LINKS for k in keys
                        for eid in item_edges(ctx, t, k, ctx.records_of[a] | ctx.records_of[b])),
        ))
    return out

In [18]:
%%module meridian.detectors.market
"""BRIDGE_ENTITY and GEOGRAPHY_LINKED_CONCENTRATION (a market pocket of new intermediaries; v1.3: a cross-border corridor)."""
from __future__ import annotations

from collections import Counter
from itertools import combinations

import networkx as nx
import numpy as np
import pandas as pd

from ..context import Context
from .base import VENDOR_LEVELS, Candidate, peer_note, peer_tail, tail_threshold, vendor_peers
from .evidence import engagement_edges, link_edges

BENIGN_BRIDGE_PARENTS = {"RND", "SCM"}  # CROs, logistics: cross-border by design


def participation(ctx: Context, li: pd.DataFrame) -> dict:
    """record -> (participation coefficient, {community: institutions}) over institution communities.

    Institutions are linked when a vendor serves both; communities are Louvain
    communities of that projection built without the vendor's own links, so the
    vendor cannot merge the communities it bridges (Dossier §3.8). Participation
    is 1 - sum over communities of (links into it / all links)^2.
    Communities are found within the connected parts of the projection that hold
    the vendor's institutions: unrelated parts do not change them, and the cost
    grows with the data instead of with its square.
    """
    by_vendor = li.groupby("record_id")["institution_id"].agg(lambda s: sorted(set(s)))
    pairs = {v: list(combinations(ins, 2)) for v, ins in by_vendor.items() if len(ins) >= 2}
    weight = Counter(p for ps in pairs.values() for p in ps)
    full = nx.Graph()
    full.add_nodes_from(set(li["institution_id"]))
    full.add_edges_from(weight)
    part_of = {n: i for i, c in enumerate(nx.connected_components(full)) for n in c}
    edges_of = {}
    for (a, b), w in weight.items():
        edges_of.setdefault(part_of[a], []).append((a, b, w))
    nodes_of = {}
    for n, i in part_of.items():
        nodes_of.setdefault(i, []).append(n)
    out = {}
    for v, ins in by_vendor.items():
        if len(ins) < 2:
            continue
        own = Counter(pairs[v])
        parts = sorted({part_of[i] for i in ins})
        g = nx.Graph()
        g.add_nodes_from(n for i in parts for n in sorted(nodes_of[i]))
        g.add_weighted_edges_from((a, b, w - own[(a, b)]) for i in parts for a, b, w in edges_of[i]
                                  if w > own[(a, b)])
        comm = {n: i for i, c in enumerate(nx.community.louvain_communities(g, weight="weight",
                                                                            seed=ctx.cfg.get("seed", 0))) for n in c}
        groups: dict = {}
        for i in ins:
            groups.setdefault(comm[i], []).append(i)
        out[v] = (1 - sum((len(x) / len(ins)) ** 2 for x in groups.values()), groups)
    return out


def detect_bridge(ctx: Context) -> list[Candidate]:
    min_foreign = ctx.param("bridge", "min_foreign_links", 2)
    li = ctx.links[ctx.links["link_type"].isin(["SERVES", "SITE_OF"])].copy()
    li["vendor_country"] = li["record_id"].map(ctx.vendors["country"])
    li["foreign"] = li["inst_country"] != li["vendor_country"]
    part = participation(ctx, li)
    # Two links into two communities is what any vendor with two scattered institutions shows.
    min_comms = ctx.param("bridge", "min_communities", 3)
    # Participation and foreign links are judged against comparable vendors (type x category x
    # geography, backing off); Dossier §3.8: comparable entities in similar positions weaken concern.
    tail = 1 - ctx.param("bridge", "peer_quantile", 0.9)
    min_peers = ctx.param("bridge", "min_peers", 5)
    vp = vendor_peers(ctx)
    pf = vp.loc[sorted(part)].assign(participation=[part[r][0] for r in sorted(part)])
    part_peers = peer_tail(pf, "participation", VENDOR_LEVELS, min_peers, tail) if len(pf) else pf
    ff = vp.loc[sorted(set(li["record_id"]))].assign(
        n_foreign=li.groupby("record_id")["foreign"].sum().reindex(sorted(set(li["record_id"]))).to_numpy())
    foreign_peers = peer_tail(ff, "n_foreign", VENDOR_LEVELS, min_peers, tail)
    eng = ctx.engagements
    out = []
    for rec, g in li.groupby("record_id"):
        foreign = g[g["foreign"]]
        p, groups = part.get(rec, (0.0, {}))
        across = len(groups) >= min_comms and part_peers.at[rec, "tail_share"] <= tail
        if len(foreign) < min_foreign and not across:
            continue
        e = eng[eng["record_id"] == rec]
        sponsors = set(e["sponsor_employee_id"].dropna())
        home = ctx.vendors.loc[rec, "country"]
        # A sponsor may have no role yet at t (some engagements name one before the role starts).
        home_of = ctx.roles.set_index("employee_id")["home_market"]
        local = [s for s in sponsors if home_of.get(s) == home]
        single_local = len(sponsors) == 1 and len(local) == 1
        flags = set().union(*e["cat_flags"]) if len(e) else set()
        parents = set(e["parent"].dropna())
        public = float(g["public_sector"].mean())
        countries = sorted(set(g["inst_country"]))
        score = (np.log2(1 + len(foreign)) + public + p + 1.0 * single_local
                 + 0.5 * bool(flags & {"int", "gov"}))
        benign = []
        if len(sponsors) >= 3:
            benign.append(f"{len(sponsors)} sponsors: consistent with a national/regional distributor")
        if parents & BENIGN_BRIDGE_PARENTS:
            benign.append(f"category parent {','.join(sorted(parents & BENIGN_BRIDGE_PARENTS))}: cross-border by design")
        if not across and foreign_peers.at[rec, "tail_share"] > tail:
            benign.append(peer_note(foreign_peers.loc[rec]))
        # The bridge holds from the first foreign link, or from the first link into a second community.
        dates = []
        if len(foreign) >= min_foreign:
            dates.append(foreign["valid_from"].min())
        if across:
            comm_of = {i: c for c, ins in groups.items() for i in ins}
            first = g.sort_values("valid_from").drop_duplicates("institution_id")
            seen = first["institution_id"].map(comm_of).ne(comm_of[first["institution_id"].iloc[0]])
            dates.append(first.loc[seen, "valid_from"].iloc[0])
        emerged = max(min(dates), ctx.vendors.loc[rec, "onboarding_date"])
        out.append(Candidate(
            members=ctx.expand([rec]), tags=("BRIDGE_ENTITY",), score=float(score), emerged_at=emerged,
            detector="bridge",
            reason=(f"vendor in {home} links to institutions in {','.join(countries)} "
                    f"({len(foreign)} foreign links, {public:.0%} public sector); "
                    + (f"links {len(groups)} institution communities that are otherwise separate "
                       f"(participation {p:.2f}, higher than {1 - part_peers.at[rec, 'tail_share']:.0%} of "
                       f"{part_peers.at[rec, 'n_peers']} peers: {part_peers.at[rec, 'peer_group']}); " if across else "")
                    + (f"single local sponsor {next(iter(sponsors))}" if single_local else f"{len(sponsors)} sponsors")),
            evidence={"links": g[["link_id", "institution_id", "inst_country", "link_type", "public_sector",
                                  "valid_from", "valid_to"]].to_dict("records"),
                      "communities": {str(c): ins for c, ins in groups.items()}, "participation": p},
            benign=benign, features={"n_foreign": len(foreign), "public": public, "single_local": single_local,
                                     "participation": p, "n_communities": len(groups)},
            edges=tuple(link_edges(ctx, g["link_id"]) + engagement_edges(ctx, e["engagement_id"])),
        ))
    return out


def detect_geography(ctx: Context) -> list[Candidate]:
    days = ctx.param("geography", "window_days", 365)
    min_new = ctx.param("geography", "min_new", 4)
    e = ctx.engagements.copy()
    e = e[e["cat_flags_possible"].map(lambda f: "int" in f)]
    e["entity"] = e["record_id"].map(ctx.entity_of)
    e["vendor_country"] = e["record_id"].map(ctx.vendors["country"])
    first = e.sort_values("start_date").groupby("entity", as_index=False).first()
    # Every window (a market's new intermediaries within `days` of each first engagement)
    # against the market's earlier yearly rate; the threshold is a quantile of these ratios.
    windows = []
    for mkt, g in first.groupby("vendor_country"):
        d = g.sort_values("start_date").reset_index(drop=True)
        dates = d["start_date"].to_numpy()
        for i in range(len(d)):
            j = np.searchsorted(dates, dates[i] + np.timedelta64(days, "D"), side="right")
            n_new = j - i
            lo = d.loc[i, "start_date"]
            prior = d[d["start_date"] < lo]
            years = max((lo - prior["start_date"].min()).days / 365.0, 1.0) if len(prior) else 1.0
            baseline = len(prior) / years
            windows.append((mkt, i, j, lo, n_new, len(prior), baseline, n_new / max(baseline, 1.0), d))
    min_ratio = tail_threshold([w[7] for w in windows], ctx.param("geography", "ratio_quantile", None),
                               ctx.param("geography", "min_ratio", 3.0))
    found = [w for w in windows if w[4] >= min_new and w[7] >= min_ratio]
    # Keep the strongest window per market.
    best = {}
    for f in found:
        if f[0] not in best or (f[4], f[7]) > (best[f[0]][4], best[f[0]][7]):
            best[f[0]] = f
    out = []
    for mkt, (_, i, j, lo, n_new, n_prior, baseline, ratio, d) in best.items():
        win = d.iloc[i:j]
        top_sp = win["sponsor_employee_id"].value_counts(normalize=True)
        sp_share = float(top_sp.iloc[0]) if len(top_sp) else 0.0
        cpi = ctx.country_index(mkt, "CPI", lo)
        cpi_tier = int(cpi) if cpi is not None and str(cpi).isdigit() else None
        cats = win["cat_primary"].dropna()
        certain = float(win["cat_flags"].map(lambda f: "int" in f).mean())
        score = np.log2(ratio) + np.log2(n_new) + (0.5 * (cpi_tier - 2) if cpi_tier else 0) + sp_share
        benign = ctx.benign_events("onboarding", lo, win["start_date"].max(), mkt,
                                   cats=zip(win["parent"], win["cat_primary"]))
        if n_prior == 0 and len(ctx.vendors[ctx.vendors["country"] == mkt]) <= n_new + 2:
            benign.append("new market entry: no prior vendors of any kind")
        out.append(Candidate(
            members=ctx.expand(win["record_id"]), tags=("GEOGRAPHY_LINKED_CONCENTRATION",), score=float(score),
            emerged_at=pd.Timestamp(d["start_date"].iloc[i + min_new - 1]), active_until=win["start_date"].max(),
            detector="geography",
            reason=(f"{n_new} new intermediaries ({', '.join(sorted(set(cats)))}) in {mkt} within {days} days from "
                    f"{lo.date()} on top of {n_prior} earlier (baseline {baseline:.1f}/yr, {ratio:.1f}x, "
                    f"threshold {min_ratio:.1f}x); "
                    f"CPI tier {cpi_tier}; {sp_share:.0%} under one sponsor; int flag certain for {certain:.0%}"),
            evidence={"market": mkt, "window_start": lo, "engagements": list(win["engagement_id"]),
                      "cpi_tier": cpi_tier, "baseline_per_year": baseline, "ratio_threshold": min_ratio},
            benign=benign, features={"n_new": n_new, "ratio": ratio, "cpi_tier": cpi_tier, "sponsor_share": sp_share},
            edges=tuple(engagement_edges(ctx, win["engagement_id"],
                                         ("CONTRACTED_UNDER", "SPONSORED_BY", "FOR_SERVICE"))),
        ))
    return out


def detect_corridor(ctx: Context) -> list[Candidate]:
    """v1.3: GEOGRAPHY_LINKED_CONCENTRATION through a cross-border corridor (Dossier §2.8: "a group of related third
    parties ... unusually concentrated in one market, cross-border corridor, or jurisdictional path"; "ownership and
    operations structured across specific jurisdictions"). At least `min_entities` vendor entities of one home market
    share an address in another country, and that country is where their payments are remitted (`remit_to_country`):
    their correspondence and their money run through one point abroad. One candidate per address and home market,
    dated when the min_entities-th entity is there; it holds while min_entities are still there.

    Added after the main-data validation labels: GEOGRAPHY_LINKED_CONCENTRATION had the lowest recall (VAL-24: three RU
    vendors with one SG correspondence address, remitting to SG, was submitted only as a shared address and an
    ownership overlap). The full data holds two such groups."""
    from .shared_address import address_instances
    k = ctx.param("corridor", "min_entities", 3)
    a = address_instances(ctx)
    v = ctx.vendors
    a["home"] = a["record_id"].map(v["country"])
    a["remit"] = a["record_id"].map(v["remit_to_country"])
    a["abroad"] = a["addr_key"].str.split(":").str[0]
    a = a[(a["abroad"] != a["home"]) & (a["remit"] == a["abroad"])]
    a["entity"] = a["record_id"].map(lambda r: ctx.entity_of.get(r, r))
    out = []
    for (key, home), g in a.groupby(["addr_key", "home"]):
        first = g.groupby("entity")["valid_from"].min().sort_values()
        if len(first) < k:
            continue
        still = g.loc[g["valid_to"].isna(), "entity"].nunique()
        n, abroad = len(first), g["abroad"].iloc[0]
        cpi = ctx.country_index(home, "CPI", first.iloc[k - 1])
        out.append(Candidate(
            members=ctx.expand(set(g["record_id"])), tags=("GEOGRAPHY_LINKED_CONCENTRATION",), score=float(np.log2(1 + n)),
            emerged_at=first.iloc[k - 1], active_until=None if still >= k else g["valid_to"].dropna().max(),
            detector="corridor",
            reason=(f"{n} vendors of {home} share one address in {abroad} ('{g['raw_address'].iloc[0]}') and remit their "
                    f"payments to {abroad}: a {home}->{abroad} corridor" + (f"; CPI tier of {home}: {cpi}" if cpi is not None else "")),
            evidence={"address_key": key, "home_market": home, "corridor_country": abroad,
                      "raw_addresses": sorted(set(g["raw_address"]))},
            features={"n_entities": n}, edges=tuple(g["edge_id"]),
        ))
    return out


In [19]:
%%module meridian.detectors.activity
"""ACTIVITY_TIMING_CONCENTRATION: synchronised spend spikes under one sponsor; payments after termination.

v1.3: a spike group that chance explains (many vendors under the sponsor) is reported as `portfolio_spikes`."""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..context import Context
from scipy.stats import binom

from .base import Candidate, tail_threshold
from .evidence import engagement_edges


def spend_peaks(ctx: Context, half_window: int = 6, ramp_up: int = 3) -> pd.DataFrame:
    """Months where a vendor's spend is the maximum of its neighbourhood (+-half_window
    months), with the ratio to its own local median (the median of the surrounding
    months, excluding the month itself). The first `ramp_up` months of a vendor's
    payment history are skipped: new vendors ramp up together."""
    key = ("_spend_peaks", half_window, ramp_up)
    if key in ctx.__dict__:
        return ctx.__dict__[key]
    ms = ctx.monthly_spend.sort_values(["record_id", "month"])
    rows = []
    for rec, g in ms.groupby("record_id"):
        if len(g) < 4:
            continue
        s = g.set_index("month")["amount_usd"]
        idx = pd.date_range(s.index.min(), s.index.max(), freq="MS")
        s = s.reindex(idx)  # gaps stay NaN: unknown, not zero
        vals = s.to_numpy()
        for k in range(ramp_up, len(vals)):
            v = vals[k]
            if np.isnan(v):
                continue
            nb = np.concatenate([vals[max(0, k - half_window):k], vals[k + 1:k + 1 + half_window]])
            nb = nb[~np.isnan(nb)]
            if len(nb) < 3:
                continue
            med = np.median(nb)
            if med > 0 and v >= nb.max():
                rows.append((rec, idx[k], v, v / med, med))
    out = pd.DataFrame(rows, columns=["record_id", "month", "amount", "ratio", "local_median"])
    ctx.__dict__[key] = out
    return out


def spike_threshold(ctx: Context) -> float:
    """Spike ratio threshold: the `spike_quantile` of all spend peaks' ratios in this
    dataset (or the fixed `min_spike_ratio` when no quantile is set)."""
    return tail_threshold(spend_peaks(ctx)["ratio"], ctx.param("activity", "spike_quantile", None),
                          ctx.param("activity", "min_spike_ratio", 2.5))


def spend_spikes(ctx: Context, min_ratio: float | None = None) -> pd.DataFrame:
    """Spend peaks at least `min_ratio` x the vendor's local median (default: `spike_threshold`)."""
    p = spend_peaks(ctx)
    thr = spike_threshold(ctx) if min_ratio is None else min_ratio
    return p.loc[p["amount"] >= thr * p["local_median"], ["record_id", "month", "amount", "ratio"]].reset_index(drop=True)


def detect_sync_spikes(ctx: Context) -> list[Candidate]:
    """Vendors under one sponsor whose spend spikes within two months of each other.

    v1.3, after the main-data validation labels: the chance of the group is judged against the size of the
    sponsor's portfolio. With a spike rate q per vendor and two-month window, k of the sponsor's N vendors spike
    together by chance with probability P(Binomial(N, q) >= k). A group below `max_p` stays `sync_spikes`; a more
    likely one (three spikes among 30 vendors is ordinary) is reported as `portfolio_spikes`, ranked with the
    open-set prior. On the main-data VALIDATION period the 117 spike groups matched no labelled case, while the
    labelled starter groups are three of a sponsor's four vendors.
    """
    min_vendors = ctx.param("activity", "min_vendors", 3)
    max_p = ctx.param("activity", "max_p", None)
    spikes = spend_spikes(ctx)
    if spikes.empty:
        return []
    rate = len(spikes) / max(len(ctx.monthly_spend), 1)   # spikes per vendor-month with a payment
    q2 = 1 - (1 - rate) ** 2                                 # a vendor spikes within a two-month window
    eng = ctx.engagements.dropna(subset=["sponsor_employee_id"])
    # A spike belongs to a sponsor when the vendor had an open engagement with that sponsor that month.
    sp = spikes.merge(eng[["record_id", "sponsor_employee_id", "start_date", "end_date", "parent", "market"]],
                      on="record_id")
    sp = sp[(sp["start_date"] <= sp["month"] + pd.offsets.MonthEnd(0))
            & (sp["end_date"].isna() | (sp["end_date"] >= sp["month"]))]
    sp["entity"] = sp["record_id"].map(ctx.entity_of)
    found = []
    for s, g in sp.groupby("sponsor_employee_id"):
        n_vendors_sponsor = eng[eng["sponsor_employee_id"] == s]["record_id"].map(ctx.entity_of).nunique()
        for m in sorted(g["month"].unique()):
            w = g[(g["month"] >= m) & (g["month"] <= m + pd.offsets.MonthBegin(1))]
            ents = set(w["entity"])
            if len(ents) >= min_vendors:
                found.append((s, pd.Timestamp(m), frozenset(ents), w, n_vendors_sponsor))
    found.sort(key=lambda x: (-len(x[2]), x[1]))
    kept = []
    for f in found:
        if any(f[0] == k[0] and len(f[2] & k[2]) / len(f[2] | k[2]) >= 0.5 for k in kept):
            continue
        kept.append(f)
    out = []
    for s, m, ents, w, n_sp in kept:
        members = frozenset().union(*[ctx.records_of[e] for e in ents])
        share = len(ents) / max(n_sp, 1)
        med_ratio = float(w.groupby("entity")["ratio"].max().median())
        parents = set(w["parent"].dropna())
        mkt = w["market"].mode().iloc[0]
        e = ctx.engagements[ctx.engagements["record_id"].isin(members)]
        benign = ctx.benign_events("spend", m, m + pd.offsets.MonthEnd(1), mkt,
                                   cats=zip(e["parent"], e["cat_primary"]))
        if n_sp >= 30:
            benign.append(f"sponsor has {n_sp} vendors: spikes may be seasonal")
        p = float(binom.sf(len(ents) - 1, max(n_sp, len(ents)), q2))
        out.append(Candidate(
            members=members, tags=("ACTIVITY_TIMING_CONCENTRATION",),
            score=float(np.log2(len(ents)) + np.log2(med_ratio) + share), emerged_at=m,
            active_until=m + pd.offsets.MonthEnd(2),
            detector="sync_spikes" if max_p is None or p < max_p else "portfolio_spikes",
            reason=(f"{len(ents)} vendors under sponsor {s} ({ctx.role_of.get(s)}) in {mkt} spike together "
                    f"({med_ratio:.1f}x their own local median) within 2 months of {m.date()}; "
                    f"{share:.0%} of the sponsor's vendors; chance {p:.0e}"),
            evidence={"sponsor": s, "month": m, "spikes": w[["record_id", "month", "amount", "ratio"]].to_dict("records")},
            benign=benign, features={"n_vendors": len(ents), "median_ratio": med_ratio, "sponsor_share": share, "p": p},
            edges=tuple(engagement_edges(ctx, e.loc[e["sponsor_employee_id"] == s, "engagement_id"])),
        ))
    return out


def detect_post_termination(ctx: Context) -> list[Candidate]:
    min_months = ctx.param("activity", "post_termination_min_months", 1)
    tp = ctx.vendors[ctx.vendors["terminated_date"].notna()]
    p = ctx.b.tp_payments_monthly
    p = p[p["record_id"].isin(tp.index)].copy()
    p["term"] = p["record_id"].map(tp["terminated_date"])
    after = p[p["month"] > p["term"] + pd.offsets.MonthEnd(0)]
    out = []
    for rec, g in after.groupby("record_id"):
        months = sorted(g["month"].unique())
        if len(months) < min_months:
            continue
        term = tp.loc[rec, "terminated_date"]
        eng = ctx.engagements[ctx.engagements["record_id"] == rec]
        benign = ctx.benign_events("termination", term - pd.Timedelta(days=180), term + pd.Timedelta(days=180),
                                   ctx.vendors.loc[rec, "country"], units=set(eng["business_unit"]))
        out.append(Candidate(
            members=ctx.expand([rec]), tags=("ACTIVITY_TIMING_CONCENTRATION",),
            score=float(1.5 + np.log2(1 + len(months)) + np.log10(1 + g["amount_usd"].sum()) / 3),
            emerged_at=pd.Timestamp(months[0]), active_until=pd.Timestamp(months[-1]), detector="post_termination",
            reason=(f"terminated on {term.date()} but paid in {len(months)} later month(s) "
                    f"({pd.Timestamp(months[0]).date()}–{pd.Timestamp(months[-1]).date()}), "
                    f"USD {g['amount_usd'].sum():,.0f}"),
            evidence={"terminated": term, "months": [str(pd.Timestamp(x).date()) for x in months],
                      "amount_usd": float(g["amount_usd"].sum())},
            benign=benign, features={"n_months": len(months)},
            edges=tuple(engagement_edges(ctx, sorted(set(g["engagement_id"])), ("CONTRACTED_UNDER",))),
        ))
    return out

In [20]:
%%module meridian.detectors.ownership
"""The five ownership typologies, built on registry R paths.

Ownership structures change over time (on the starter data one vendor was
layered through a trust until 2019 and then acquired through a BVI holdco),
so every vendor is evaluated at each date its upward structure changed while
it was our vendor, not only at t.

Rules followed throughout:
  - a missing path is unknown, never evidence of concealment;
  - registry L accounting parents are not beneficial owners and are not used here;
  - evidence records both valid time and publication date (statementDate).
"""
from __future__ import annotations

from collections import defaultdict

import numpy as np
import pandas as pd

from ..context import Context
from ..resolution import name_key, person_key
from .base import VENDOR_LEVELS, Candidate, peer_note, peer_tail, vendor_peers
from .evidence import item_edges, owner_edges, path_edges, registration_edges, statement_edges

# Jurisdiction lists are a documented modelling choice, not an index value from the
# bundle (FSI and FATF are not loaded in the bundle). "Sink" follows the classic
# offshore financial centres (IMF 2000 OFC list; Garcia-Bernardo et al. 2017,
# "Uncovering Offshore Financial Centers", sink OFCs); "conduit" follows the
# conduit OFCs of the same paper. Conduits carry less weight.
OFFSHORE_SINK = frozenset({
    "JE", "GG", "IM", "BM", "KY", "VG", "BS", "BZ", "SC", "MU", "LI", "MC", "AD", "GI", "TC", "AI", "VC",
    "KN", "AG", "LC", "WS", "VU", "MH", "CW", "BB", "PA",
})
OFFSHORE_CONDUIT = frozenset({"NL", "LU", "IE", "CY", "MT", "CH", "SG", "HK", "AE"})


def _jurisdiction_note(j: str) -> str:
    if j in OFFSHORE_SINK:
        return "sink OFC"
    if j in OFFSHORE_CONDUIT:
        return "conduit OFC"
    return ""


class OwnershipView:
    """Per-vendor ownership snapshots over the vendor's active life."""

    def __init__(self, ctx: Context):
        self.ctx = ctx
        self.nodes = ctx.reg_nodes
        self.dd = ctx.b.tp_due_diligence.set_index("record_id")
        # Cached per view, so the cache goes with its context (a functools cache on the
        # method would keep every context of every run alive).
        self._snapshots: dict = {}

    def node(self, sid):
        return self.nodes.loc[sid] if sid in self.nodes.index else None

    def eval_dates(self, rec: str, sid: str) -> list[pd.Timestamp]:
        v = self.ctx.vendors.loc[rec]
        lo = v["onboarding_date"]
        hi = v["terminated_date"] if pd.notna(v["terminated_date"]) else self.ctx.t
        dates = {lo, hi - pd.Timedelta(days=1) if hi > lo else lo}
        dates |= {d for d in self.ctx.ownership_dates(sid) if lo <= d < hi}
        return sorted(dates)

    def metrics(self, rec: str, sid: str, at) -> dict:
        paths = self.ctx.ownership_paths(sid, at)
        home = self.node(sid)["jurisdiction"]
        inter, persons, jur = set(), {}, {home}
        trust = nominee = cycle = False
        gaps = 0
        for p in paths:
            for x in p:
                if x.get("details") == "trust":
                    trust = True
                if x.get("details") == "nominee":
                    nominee = True
                if x.get("cycle"):
                    cycle = True
                ps = x["party_sid"]
                if ps is None:
                    gaps += 1
                    continue
                n = self.node(ps)
                if n is None:
                    continue
                if n["kind"] == "entity":
                    inter.add(ps)
                    jur.add(n["jurisdiction"])
                    if n["entity_subtype"] == "trust":
                        trust = True
                else:
                    persons[ps] = min(persons.get(ps, 99), len(p))
            last = p[-1]
            if last["party_kind"] == "entity" and not self.ctx.owners_at(at).get(last["party_sid"]):
                gaps += 1
        sink = {s for s in inter if self.node(s)["jurisdiction"] in OFFSHORE_SINK
                and self.node(s)["jurisdiction"] != home}
        conduit = {s for s in inter if self.node(s)["jurisdiction"] in OFFSHORE_CONDUIT
                   and self.node(s)["jurisdiction"] != home}
        peps = {p for p in persons if self.node(p)["pep_status"] == "isPep"}
        layers = max((len(p) for p in paths), default=0)
        return {"at": pd.Timestamp(at), "paths": paths, "layers": layers, "n_paths": len(paths),
                "jurisdictions": jur, "trust": trust, "nominee": nominee, "cycle": cycle, "gaps": gaps,
                "intermediates": inter, "persons": persons, "peps": peps, "sink": sink, "conduit": conduit}

    def snapshots(self, rec: str) -> tuple:
        if rec not in self._snapshots:
            sid = self.ctx.reg_entity_of.get(rec)
            self._snapshots[rec] = () if sid is None else tuple(
                self.metrics(rec, sid, d) for d in self.eval_dates(rec, sid))
        return self._snapshots[rec]

    def path_text(self, rec: str, path: list[dict]) -> str:
        sid = self.ctx.reg_entity_of[rec]
        parts = [f"{self.node(sid)['name']} [{self.node(sid)['jurisdiction']}]"]
        for x in path:
            ps = x["party_sid"]
            n = self.node(ps) if ps else None
            share = f"{x['share']:g}%" if pd.notna(x.get("share")) else (x.get("details") or "?")
            if n is None:
                parts.append(f"<-{share}- (chain stops: {x.get('unspecified_reason')})")
            else:
                tag = n["entity_subtype"] if n["kind"] == "entity" else ("PEP" if n["pep_status"] == "isPep" else "person")
                parts.append(f"<-{share} from {pd.Timestamp(x['start_date']).date() if pd.notna(x['start_date']) else '?'}"
                             f" (published {pd.Timestamp(x['statement_date']).date()})- {n['name']} [{n['jurisdiction']},{tag}]")
        return " ".join(parts)


def _view(ctx: Context) -> OwnershipView:
    if "_ownership_view" not in ctx.__dict__:
        ctx.__dict__["_ownership_view"] = OwnershipView(ctx)
    return ctx.__dict__["_ownership_view"]


def _org_records(ctx: Context):
    return [r for r in ctx.reg_entity_of]


# ---------------------------------------------------------------------------
# LAYERED_COMPLEX_OWNERSHIP
# ---------------------------------------------------------------------------
def complexity(m: dict) -> float:
    return ((m["layers"] - 1) + (len(m["jurisdictions"]) - 1) + 1.5 * m["trust"] + 1.5 * m["nominee"]
            + 0.5 * (m["n_paths"] > 1) + 0.5 * bool(m["sink"]))


def layered_peers(ctx: Context) -> pd.DataFrame:
    """Each registry-linked vendor's highest ownership complexity over its life, against
    comparable vendors (type x category x geography, backing off). Dossier §2.10: complexity
    matters when it is unusual for the peer group."""
    ov = _view(ctx)
    recs = sorted(_org_records(ctx))
    f = vendor_peers(ctx).loc[recs].assign(
        complexity=[max((complexity(m) for m in ov.snapshots(r)), default=0.0) for r in recs])
    return peer_tail(f, "complexity", VENDOR_LEVELS, ctx.param("layered", "min_peers", 5),
                     1 - ctx.param("layered", "peer_quantile", 0.95))


def detect_layered(ctx: Context) -> list[Candidate]:
    min_layers = ctx.param("layered", "min_layers", 4)
    tail = 1 - ctx.param("layered", "peer_quantile", 0.95)
    ov = _view(ctx)
    peers = layered_peers(ctx)
    out = []
    for rec in _org_records(ctx):
        snaps = ov.snapshots(rec)
        hits = [m for m in snaps if m["layers"] >= min_layers
                or (m["layers"] >= 3 and len(m["jurisdictions"]) >= 3)]
        if not hits:
            continue
        best = max(hits, key=complexity)
        later = [m for m in snaps if m["at"] > hits[-1]["at"]]
        ended = later[0]["at"] if later else None  # the structure changed at the next snapshot
        benign = []
        listed = [s for s in best["intermediates"] if ov.node(s)["listed"]]
        funds = [s for s in best["intermediates"] if ov.node(s)["entity_subtype"] == "fund"]
        if listed:
            benign.append("listed entity in the chain")
        if funds:
            benign.append("fund / private-equity structure")
        if best["gaps"]:
            benign.append(f"{best['gaps']} chain gap(s): structure partly unknown")
        if peers.at[rec, "tail_share"] > tail:
            benign.append(peer_note(peers.loc[rec]))
        longest = max(best["paths"], key=len)
        out.append(Candidate(
            members=ctx.expand([rec]), tags=("LAYERED_COMPLEX_OWNERSHIP",), score=complexity(best),
            emerged_at=hits[0]["at"], active_until=ended, detector="layered_ownership",
            reason=(f"owned through {best['layers']} layers across {len(best['jurisdictions'])} jurisdictions "
                    f"({'>'.join(sorted(best['jurisdictions']))}), {best['n_paths']} branch(es)"
                    + (", trust at the top" if best["trust"] else "") + (", nominee" if best["nominee"] else "")
                    + f"; structure as of {best['at'].date()}"),
            evidence={"path": ov.path_text(rec, longest), "as_of": best["at"], "layers": best["layers"],
                      "jurisdictions": sorted(best["jurisdictions"]),
                      "offshore": {s: _jurisdiction_note(ov.node(s)["jurisdiction"])
                                   for s in sorted(best["sink"] | best["conduit"])}},
            benign=benign, uses_ownership=True,
            features={"layers": best["layers"], "n_jur": len(best["jurisdictions"]), "trust": best["trust"]},
            edges=tuple(path_edges(ctx, rec, best["paths"])),
        ))
    return out


# ---------------------------------------------------------------------------
# COMMON_BENEFICIAL_OWNER and OWNERSHIP_OPERATIONAL_OVERLAP
# ---------------------------------------------------------------------------
def person_holdings(ctx: Context) -> dict:
    """person -> {entity: (earliest date the path existed while a vendor, record, path length)}."""
    ov = _view(ctx)
    out: dict = defaultdict(dict)
    for rec in _org_records(ctx):
        ent = ctx.entity_of[rec]
        for m in ov.snapshots(rec):
            for p, dist in m["persons"].items():
                prev = out[p].get(ent)
                if prev is None or m["at"] < prev[0]:
                    out[p][ent] = (m["at"], rec, dist)
    return out


def _name_similarity(names: list[str]) -> float:
    toks = [set(name_key(n).split()) for n in names]
    sims = []
    for i in range(len(toks)):
        for j in range(i + 1, len(toks)):
            u = toks[i] | toks[j]
            sims.append(len(toks[i] & toks[j]) / len(u) if u else 0.0)
    return float(np.mean(sims)) if sims else 0.0


def _rows_of(ctx: Context, name: str, frame: pd.DataFrame, records) -> pd.DataFrame:
    """frame[frame.record_id.isin(records)] through a per-context index (same rows, same order)."""
    cache = ctx.__dict__.setdefault("_rows_by_record", {})
    if name not in cache:
        idx = defaultdict(list)
        for i, r in enumerate(frame["record_id"]):
            idx[r].append(i)
        cache[name] = idx
    return frame.iloc[sorted(i for r in set(records) for i in cache[name].get(r, ()))]


def _operational_links(ctx: Context, records: set) -> dict:
    """Shared addresses, sponsors and contacts among member entities.

    Returns kind -> list of (key, entities, formed) where `formed` is the date
    the second entity joined the shared item (valid time), so the overlap's
    emergence can be dated (both overlaps must hold).
    """
    recs = set(records)
    ent = ctx.entity_of
    rel = defaultdict(list)

    def add(kind, frame, key_col, date_col):
        for key, g in frame.groupby(key_col):
            first = g.assign(e=[ent.get(r) for r in g["record_id"]]).groupby("e")[date_col].min().sort_values()
            if len(first) >= 2:
                rel[kind].append((key, frozenset(first.index), first.iloc[1]))

    a = _rows_of(ctx, "addresses", ctx.addresses, recs)
    add("address", a, "addr_key", "valid_from")
    e = _rows_of(ctx, "engagements", ctx.engagements, recs).dropna(subset=["sponsor_employee_id"])
    add("sponsor", e, "sponsor_employee_id", "start_date")
    c = _rows_of(ctx, "contacts", ctx.contacts, recs).merge(
        ctx.vendors[["record_id", "onboarding_date"]].reset_index(drop=True), on="record_id")
    add("phone", c.dropna(subset=["phone_key"]), "phone_key", "onboarding_date")
    add("domain", c.dropna(subset=["domain_key"]), "domain_key", "onboarding_date")
    return rel


def _personal_vehicle(ctx: Context, expert_records: set, members) -> bool:
    """Is every organisation in the group named after the expert (a personal service company)?

    An expert's own consultancy carrying their name is a disclosed vehicle; an
    unrelated name is what makes the ownership undeclared in practice.
    """
    if not expert_records:
        return False
    toks = {t for r in expert_records for t in person_key(ctx.vendors.loc[r, "legal_name"]).split() if len(t) >= 3}
    orgs = [r for r in members if ctx.vendors.loc[r, "party_type"] == "organization"]
    return bool(orgs) and all(toks & set(name_key(ctx.vendors.loc[r, "legal_name"]).split()) for r in orgs)


_GROUP_WORDS = {"group", "holdings", "holding", "international", "global", "partners"}


def _group_explains(group: str, v: pd.DataFrame) -> bool:
    """Does a declared group plausibly cover the members? Only ~70% of group members
    declare it, so members carrying the group's brand stem also count."""
    stem = [t for t in name_key(group).split() if t not in _GROUP_WORDS][:1]
    covered = v["declared_parent_group"].eq(group) | (
        v["legal_name"].map(lambda n: bool(stem) and stem[0] in name_key(n).split()))
    return bool(covered.mean() >= 0.99)


def detect_common_owner(ctx: Context) -> list[Candidate]:
    min_entities = ctx.param("common_owner", "min_entities", 2)
    ov = _view(ctx)
    holdings = person_holdings(ctx)
    out = []
    for person, ents in holdings.items():
        # An individual expert who is the person also belongs to the group.
        own_records = {r for r, s in ctx.reg_person_of.items() if s == person}
        n_ent = len(ents) + len({ctx.entity_of[r] for r in own_records} - set(ents))
        if n_ent < min_entities:
            continue
        recs = {v[1] for v in ents.values()} | own_records
        members = ctx.expand(recs)
        v = ctx.vendors.loc[sorted(members)]
        groups = v["declared_parent_group"].dropna()
        declared_share = len(groups) / len(v)
        names = list(v.drop_duplicates("legal_name")["legal_name"])
        sim = _name_similarity(names)
        same_group = groups.nunique() == 1 and _group_explains(groups.iloc[0], v)
        pn = ov.node(person)
        dates = sorted(d[0] for d in ents.values())
        k = min(3, len(dates))
        emerged = max(dates[k - 1], *(ctx.vendors.loc[r, "onboarding_date"] for r in own_records)) if own_records \
            else dates[k - 1]
        rel = _operational_links(ctx, set(members))
        benign = []
        if same_group:
            benign.append(f"declared group {groups.iloc[0]} (declared or brand-named members)")
        if any(ov.node(s)["entity_subtype"] == "fund" or ov.node(s)["listed"]
               for m_ in (ov.snapshots(r) for r in recs if r in ctx.reg_entity_of) for snap in m_
               for s in snap["intermediates"]):
            benign.append("fund or listed entity in the chain: possible portfolio holding")
        if sim > 0.5:
            benign.append(f"similar names (token overlap {sim:.2f}): possibly one brand")
        if _personal_vehicle(ctx, own_records, members):
            benign.append("expert's personal service company, named after the expert")
        elif own_records and not rel.get("sponsor"):
            # v1.3, after the main-data validation labels: a group whose owner is also engaged as an individual
            # expert, with no sponsor engaging both, matched none of the labelled cases (310 such findings in the
            # main-data VALIDATION period), so it carries the commonest innocent reading as a benign explanation.
            # When one sponsor engages both the expert and the group it stays a conflict of interest (below; one
            # labelled starter case, CASE-0024).
            benign.append("the owner is also engaged as an individual expert, not by the same sponsor: possibly "
                          "the expert's own practice or consultancy")
        score = 2 * np.log2(n_ent) + (1 - declared_share) + (1 - sim)
        dist = sorted(d[2] for d in ents.values())
        out.append(Candidate(
            members=members, tags=("COMMON_BENEFICIAL_OWNER",), score=float(score), emerged_at=emerged,
            detector="common_owner",
            reason=(f"{n_ent} vendor entities converge on natural person {pn['name']} ({pn['jurisdiction']}"
                    + (", PEP" if pn["pep_status"] == "isPep" else "") + f") via {min(dist)}–{max(dist)}-hop paths; "
                    f"{1 - declared_share:.0%} undeclared, name overlap {sim:.2f}"
                    + (f"; the person is also engaged as expert {sorted(own_records)}" if own_records else "")),
            evidence={"person": person, "person_name": pn["name"],
                      "paths": {rec: ov.path_text(rec, next(p for m in ov.snapshots(rec) for p in m["paths"]
                                                            if p and p[-1]["party_sid"] == person))
                                for _, rec, _ in ents.values()},
                      "expert_records": sorted(own_records)},
            benign=benign, uses_ownership=True,
            features={"n_entities": n_ent, "declared_share": declared_share, "name_sim": sim,
                      "expert_owner": bool(own_records)},
            edges=tuple(owner_edges(ctx, sorted(recs), person)),
        ))

        # OWNERSHIP_OPERATIONAL_OVERLAP: the owner group also overlaps operationally (`rel`, above).
        overlap_ents = set()
        kinds = []
        formed = []
        for kind, items in rel.items():
            for key, es, when in items:
                overlap_ents |= es
                kinds.append(kind)
                formed.append(when)
        if len(overlap_ents) >= 2:
            omembers = ctx.expand({r for r in members if ctx.entity_of.get(r, r) in overlap_ents})
            n_kinds = len(set(kinds))
            # An expert who owns a vendor that the same sponsor engages is a direct conflict of interest.
            expert_conflict = bool(own_records) and "sponsor" in kinds
            # The overlap case emerges once both the common owner and an operational overlap hold.
            o_emerged = max(emerged, min(formed))
            out.append(Candidate(
                members=omembers, tags=("OWNERSHIP_OPERATIONAL_OVERLAP", "COMMON_BENEFICIAL_OWNER"),
                score=float(score + n_kinds + 2 * expert_conflict), emerged_at=o_emerged, detector="ownership_overlap",
                reason=(f"{len(overlap_ents)} entities owned by {pn['name']} also share "
                        f"{', '.join(sorted(set(kinds)))} (overlap since {min(formed).date()})"
                        + ("; the owner is an expert engaged by the same sponsor" if expert_conflict else "")),
                evidence={"person": person, "shared": {k: [(key, sorted(es), when) for key, es, when in rel[k]]
                                                       for k in rel}},
                benign=benign, uses_ownership=True,
                features={"n_entities": len(overlap_ents), "n_kinds": n_kinds},
                edges=tuple(owner_edges(ctx, sorted(omembers & recs), person) + [
                    eid for kind, its in rel.items() for key, es, _ in its
                    for eid in item_edges(ctx, kind, key, {r for e_ in es for r in ctx.records_of[e_]})]),
            ))
    return out


# ---------------------------------------------------------------------------
# OPAQUE_JURISDICTION_ROUTING
# ---------------------------------------------------------------------------
def detect_opaque(ctx: Context) -> list[Candidate]:
    """One candidate per offshore route and owner set, with the period it held.

    A vendor whose route changes (sold from one offshore structure into
    another) yields one candidate per route, so the two periods can become
    separate cases.
    """
    ov = _view(ctx)
    by_route = defaultdict(list)
    for rec in _org_records(ctx):
        snaps = ov.snapshots(rec)
        spans: dict = {}
        for idx, m in enumerate(snaps):
            if not (m["sink"] or m["conduit"]):
                continue
            if not (m["sink"] or m["peps"]):
                continue  # conduit-only holding without a PEP is an ordinary structure
            dd = ov.dd.loc[rec] if rec in ov.dd.index else None
            undisclosed = bool(m["peps"]) and dd is not None and not bool(dd["q_pep_connected"])
            s = 2.0 * bool(m["sink"]) + 1.0 * bool(m["conduit"]) + 2.0 * bool(m["peps"]) + 2.0 * undisclosed
            key = (tuple(sorted(m["sink"] | m["conduit"])), tuple(sorted(m["peps"])))
            span = spans.setdefault(key, {"first": m["at"], "last_idx": idx, "best": None, "score": -1.0})
            span["last_idx"] = idx
            if s > span["score"]:
                span.update(best=dict(m, undisclosed=undisclosed), score=s)
        for key, span in spans.items():
            nxt = span["last_idx"] + 1
            until = snaps[nxt]["at"] if nxt < len(snaps) else None
            by_route[key].append((rec, span["best"], span["score"], span["first"], until))

    out = []
    for (route, peps), items in by_route.items():
        recs = [r for r, *_ in items]
        _, m, s, _, _ = max(items, key=lambda x: x[2])
        juris = sorted({ov.node(x)["jurisdiction"] for x in route})
        benign = []
        if not m["peps"]:
            benign.append("offshore holding without PEP: may be an ordinary holding location")
        if all(ov.node(x)["jurisdiction"] in OFFSHORE_CONDUIT for x in route):
            benign.append("conduit jurisdiction only (common for legitimate holdcos)")
        pep_names = [ov.node(p)["name"] for p in m["peps"]]
        untils = [x[4] for x in items]
        out.append(Candidate(
            members=ctx.expand(recs), tags=("OPAQUE_JURISDICTION_ROUTING",), score=float(s),
            emerged_at=min(x[3] for x in items), active_until=None if None in untils else max(untils),
            detector="opaque_routing",
            reason=(f"{len(recs)} vendor(s) owned via holdco(s) in {','.join(juris)} "
                    f"({', '.join(_jurisdiction_note(j) for j in juris)})"
                    + (f"; ultimate owner is a PEP ({', '.join(pep_names)})" if pep_names else "")
                    + ("; PEP connection not disclosed in due diligence" if m["undisclosed"] else "")),
            evidence={"route": list(route),
                      "paths": {r: ov.path_text(r, max(b["paths"], key=len)) for r, b, *_ in items},
                      "jurisdiction_basis": "IMF 2000 OFC list; Garcia-Bernardo et al. 2017 sink/conduit OFCs"},
            benign=benign, uses_ownership=True,
            features={"sink": bool(m["sink"]), "pep": bool(m["peps"]), "undisclosed": m["undisclosed"]},
            edges=tuple(eid for r, b_, *_ in items for eid in path_edges(ctx, r, b_["paths"])),
        ))
    return out


# ---------------------------------------------------------------------------
# OWNERSHIP_CHANGE_STRUCTURAL_SHIFT
# ---------------------------------------------------------------------------
def detect_change(ctx: Context) -> list[Candidate]:
    days = ctx.param("ownership_change", "window_days", 90)
    min_signals = ctx.param("ownership_change", "min_signals", 2)
    ov = _view(ctx)
    ri = ctx.b.reg_interests
    addr = ctx.addresses
    eng = ctx.engagements
    svc = ctx.b.tp_services
    spend = ctx.monthly_spend
    owned_by = defaultdict(set)  # party -> records whose chain contains it (at any snapshot)
    for rec in _org_records(ctx):
        for m in ov.snapshots(rec):
            for s in m["intermediates"] | set(m["persons"]):
                owned_by[s].add(rec)
    out = []
    for rec, sid in ctx.reg_entity_of.items():
        v = ctx.vendors.loc[rec]
        end = v["terminated_date"] if pd.notna(v["terminated_date"]) else ctx.t
        direct = ri[(ri["subject_sid"] == sid) & ri["party_sid"].notna()]
        for x in direct.itertuples():
            d = x.start_date
            if pd.isna(d) or not (v["onboarding_date"] + pd.Timedelta(days=30) < d < end):
                continue
            w_hi = d + pd.Timedelta(days=days)
            signals = []
            a = addr[addr["record_id"] == rec]
            if ((a["valid_from"] >= d) & (a["valid_from"] <= w_hi)).any():
                signals.append("address")
            e = eng[eng["record_id"] == rec]
            prior_sp = set(e[e["start_date"] < d]["sponsor_employee_id"].dropna())
            new_sp = set(e[(e["start_date"] >= d) & (e["start_date"] <= w_hi)]["sponsor_employee_id"].dropna())
            if new_sp - prior_sp:
                signals.append("sponsor")
            sv = svc[svc["record_id"] == rec]
            prior_cat = set(sv[sv["valid_from"] < d]["category_code"])
            new_cat = set(sv[(sv["valid_from"] >= d) & (sv["valid_from"] <= w_hi)
                             & (sv["taxonomy_version"] == sv["taxonomy_version"].max())]["category_code"])
            if new_cat - prior_cat and not (d <= pd.Timestamp("2024-01-01") <= w_hi):
                signals.append("service_expansion")
            sp = spend[spend["record_id"] == rec]
            before = sp[(sp["month"] >= d - pd.Timedelta(days=180)) & (sp["month"] < d)]["amount_usd"]
            after = sp[(sp["month"] >= d) & (sp["month"] <= d + pd.Timedelta(days=180))]["amount_usd"]
            if len(before) and len(after) and after.mean() >= 2 * before.mean():
                signals.append("spend_jump")
            related = set()
            for r2 in owned_by.get(x.party_sid, set()) - {rec}:
                onb = ctx.vendors.loc[r2, "onboarding_date"]
                if d - pd.Timedelta(days=30) <= onb <= d + pd.Timedelta(days=days + 30):
                    related.add(r2)
            if related:
                signals.append("related_vendors")
            if len(signals) < min_signals:
                continue
            party = ov.node(x.party_sid)
            lag = (x.statement_date - d).days if pd.notna(x.statement_date) else None
            ve = eng[eng["record_id"] == rec]
            benign = ctx.benign_events("onboarding", d, w_hi, v["country"], cats=zip(ve["parent"], ve["cat_primary"])) \
                if related else []
            if party is not None and v["declared_parent_group"] and isinstance(v["declared_parent_group"], str) \
                    and name_key(v["declared_parent_group"]).split()[:1] == name_key(party["name"]).split()[:1]:
                benign.append("new owner matches declared group")
            out.append(Candidate(
                members=ctx.expand({rec} | related), tags=("OWNERSHIP_CHANGE_STRUCTURAL_SHIFT",),
                score=float(len(signals) + 0.5 * len(related)), emerged_at=d, active_until=w_hi,
                detector="ownership_change",
                reason=(f"acquired by {party['name'] if party is not None else x.party_sid} "
                        f"[{party['jurisdiction'] if party is not None else '?'}] on {d.date()} "
                        f"(registry lag {lag}d); within {days} days: {', '.join(signals)}"
                        + (f"; {len(related)} related vendor(s) onboarded" if related else "")),
                evidence={"interest": x.statement_id, "valid_from": d, "published": x.statement_date,
                          "signals": signals, "related": sorted(related)},
                benign=benign, uses_ownership=True,
                features={"n_signals": len(signals), "n_related": len(related)},
                edges=tuple(registration_edges(ctx, [rec]) + statement_edges(ctx, [x.statement_id])
                            + owner_edges(ctx, sorted(related), x.party_sid)),
            ))
    return out


In [21]:
%%module meridian.detectors.policy
"""Single-record policy findings, scored separately from the typologies (Dossier §4).

The FMV card's `country_band` has no market mapping in the bundle, so the
mapping below is our own documented choice, overridable in
the config under params.fmv.band_map.

Compliance review: the approval policy matrix names, per value band, the
category flags that require a compliance review besides the financial
approval. An engagement in such a category with no review, or with a review
dated after the engagement started, bypassed that control.
"""
from __future__ import annotations

import pandas as pd

from ..context import Context
from .base import POLICY_TAG, Candidate
from .evidence import engagement_edges

# Service payments counted against the annual cap. Investigators are paid as
# clinical_services; excluding it finds no breach on the starter data, so it is
# included by default and the narrow set is kept as a sensitivity option.
FEE_CLASSIFICATIONS = ("consulting_fees", "honoraria", "professional_services", "clinical_services")
EU_JP_EXTRA = {"JP", "CA", "AU", "NZ"}


def country_band(ctx: Context, market: str) -> str:
    override = ctx.param("fmv", "band_map", {})
    if market in override:
        return override[market]
    if market == "US":
        return "US"
    if market in EU_JP_EXTRA:
        return "EU_JP"
    if market in ctx.markets.index and ctx.markets.at[market, "region"] == "EUROPE":
        return "EU_JP"
    return "EMERGING"


def review_gaps(ctx: Context) -> pd.DataFrame:
    """Engagements that need a compliance review (policy matrix flags for their value band, on
    categories whose v2 mapping is certain) and have none `grace_days` after starting, or got it
    only after the start. One row per engagement: engagement_id, record_id, start_date, gap, date."""
    if "_review_gaps" in ctx.__dict__:
        return ctx.__dict__["_review_gaps"]
    grace = pd.Timedelta(days=ctx.param("compliance", "grace_days", 30))
    policy = ctx.b.approval_policy_matrix.set_index("value_band")["compliance_review_if_flags"]
    needed = {band: set(str(f).split(";")) - {"", "nan"} for band, f in policy.items()}
    e = ctx.engagements
    e = e[[bool(flags & needed.get(band, set())) for flags, band in zip(e["cat_flags"], e["value_band"])]]
    ap = ctx.b.tp_engagement_approvals
    first = ap[ap["approval_type"] == "compliance_review"].groupby("engagement_id")["approval_date"].min()
    e = e.assign(review=e["engagement_id"].map(first))
    missing = e[e["review"].isna() & (e["start_date"] + grace <= ctx.t)]
    late = e[e["review"].notna() & (e["review"] > e["start_date"])]
    out = pd.concat([missing.assign(gap="missing", date=missing["start_date"] + grace),
                     late.assign(gap="late", date=late["review"])])
    out = out[["engagement_id", "record_id", "start_date", "gap", "date"]].sort_values(["record_id", "date"])
    ctx.__dict__["_review_gaps"] = out.reset_index(drop=True)
    return ctx.__dict__["_review_gaps"]


def detect_compliance(ctx: Context) -> list[Candidate]:
    out = []
    for rec, g in review_gaps(ctx).groupby("record_id"):
        n_missing, n_late = int((g["gap"] == "missing").sum()), int((g["gap"] == "late").sum())
        out.append(Candidate(
            members=ctx.expand([rec]), tags=(POLICY_TAG,), score=float(n_missing + 0.5 * n_late),
            emerged_at=g["date"].min(), detector="compliance_review",
            reason=(f"{len(g)} engagement(s) in categories that require a compliance review bypassed it: "
                    f"{n_missing} without a review {ctx.param('compliance', 'grace_days', 30)} days after the start, "
                    f"{n_late} reviewed only after the start"),
            evidence={"engagements": g[["engagement_id", "start_date", "gap", "date"]].to_dict("records")},
            features={"missing": n_missing, "late": n_late},
            edges=tuple(engagement_edges(ctx, sorted(set(g["engagement_id"])), ("CONTRACTED_UNDER",))),
        ))
    return out


def detect_fmv(ctx: Context) -> list[Candidate]:
    fees = tuple(ctx.param("fmv", "fee_classifications", FEE_CLASSIFICATIONS))
    ind = ctx.vendors[ctx.vendors["party_type"] == "individual"]
    p = ctx.b.tp_payments_monthly
    p = p[p["record_id"].isin(ind.index) & p["expense_classification"].isin(fees)]
    p = p.merge(ctx.engagements[["engagement_id", "start_date"]], on="engagement_id")
    cards = ctx.b.fmv_rate_cards
    caps = cards.groupby(["card_version", "effective_from", "effective_to", "country_band", "expert_role"],
                         dropna=False)["annual_cap_usd"].first().reset_index()

    def cap_for(role, band, when):
        c = caps[(caps["expert_role"] == role) & (caps["country_band"] == band) & (caps["effective_from"] <= when)
                 & (caps["effective_to"].isna() | (caps["effective_to"] >= when))]
        return (float(c["annual_cap_usd"].iloc[0]), c["card_version"].iloc[0]) if len(c) else (None, None)

    out = []
    p["year"] = p["month"].dt.year
    for (rec, year), g in p.groupby(["record_id", "year"]):
        role = ind.loc[rec, "expert_type"]
        band = country_band(ctx, ind.loc[rec, "country"])
        # Card in force on each engagement's date (Dossier §5); the most lenient applies to the year.
        caps_found = [cap_for(role, band, d) for d in g["start_date"].unique()]
        caps_found = [c for c in caps_found if c[0] is not None]
        if not caps_found:
            continue
        cap, version = max(caps_found)
        total = g["amount_usd"].sum()
        if total <= cap:
            continue
        g = g.sort_values("month")
        crossed = g[g["amount_usd"].cumsum() > cap]["month"].iloc[0]
        out.append(Candidate(
            members=ctx.expand([rec]), tags=(POLICY_TAG,), score=float(1.0 + (total / cap - 1)),
            emerged_at=crossed, active_until=pd.Timestamp(f"{year}-12-31"), detector="fmv_cap",
            reason=(f"expert ({role}, {ind.loc[rec, 'country']}, band {band}) paid USD {total:,.0f} in fees in {year}, "
                    f"{total / cap:.2f}x the annual cap of USD {cap:,.0f} (card {version}); cap crossed {crossed.date()}"),
            evidence={"year": int(year), "total_fees": float(total), "annual_cap": cap, "card": version,
                      "band": band, "band_mapping": "own mapping, see policy.country_band"},
            features={"excess_ratio": total / cap},
            edges=tuple(engagement_edges(ctx, sorted(set(g["engagement_id"])), ("CONTRACTED_UNDER",))),
        ))
    return out


In [22]:
%%module meridian.detectors.record
"""Record-level outliers (ladder level A): no relation to any other vendor is used."""
from __future__ import annotations

from ..context import Context
from .base import OPEN_SET_TAG, Candidate


def detect_record_outlier(ctx: Context) -> list[Candidate]:
    from ..features import record_features, record_outlier_scores

    share = ctx.param("record_outlier", "top_share", 0.03)
    feat = record_features(ctx)
    score = record_outlier_scores(feat, ctx.cfg.get("seed", 0))
    top = score.sort_values(ascending=False).head(max(10, int(share * len(score))))
    z = (feat - feat.median()) / (feat.quantile(0.9) - feat.quantile(0.1)).replace(0, 1)
    out = []
    for rec, s in top.items():
        drivers = z.loc[rec].abs().sort_values(ascending=False).head(3)
        out.append(Candidate(
            members=ctx.expand([rec]), tags=(OPEN_SET_TAG,), score=float(s),
            emerged_at=ctx.vendors.loc[rec, "onboarding_date"], detector="record_outlier",
            reason=("record-level outlier (isolation forest on the record's own attributes and payments); "
                    "most unusual: " + ", ".join(f"{k}={feat.at[rec, k]:.2f}" for k in drivers.index)),
            evidence={"outlier_score": float(s), "drivers": {k: float(feat.at[rec, k]) for k in drivers.index}},
            features={"outlier": float(s)}, timeless=True,
        ))
    return out

In [23]:
%%module meridian.detectors.open_set
"""Open-set detection. Evaluated by leave-one-type-out in experiments.py.

The blind test contains patterns that are not in the 14-tag list. These
detectors do not target any known type; their cases carry the tag OPEN_SET:

  rare_sharing_community  communities of the multi-relation vendor graph whose
                          internal, rarity-weighted sharing far exceeds what
                          their members' degrees predict
  entity_change_point     one vendor changes several things at once (address,
                          sponsor, category, spend level) within 90 days
  duplicate_paid          duplicate master records of one entity paid in the same months
  entity_outlier          records far in the tail of one or two explicit features
                          (record attributes, relation degrees, ownership structure)
"""
from __future__ import annotations

from collections import defaultdict

import networkx as nx
import numpy as np
import pandas as pd

from ..context import Context
from .base import OPEN_SET_TAG, Candidate
from .evidence import address_edges, engagement_edges, identity_edges, shared_item_edges
from .relations import IDENTITY_TYPES, LINK_TYPES, entity_items, idf


def _weighted_graph(ctx: Context, max_item_size: int = 30) -> nx.Graph:
    """Entity graph; each shared item adds its rarity (idf) to the edge weight."""
    items = entity_items(ctx)
    total = len(set(ctx.entity_of.values()))
    g = nx.Graph()
    for t in LINK_TYPES:
        for key, es in items.get(t, {}).items():
            if not 2 <= len(es) <= max_item_size:
                continue
            w = idf(len(es), total) * (1.5 if t in IDENTITY_TYPES else 1.0)
            es = sorted(es)
            for i, a in enumerate(es):
                for b in es[i + 1:]:
                    if g.has_edge(a, b):
                        g[a][b]["weight"] += w
                        g[a][b]["types"].add(t)
                    else:
                        g.add_edge(a, b, weight=w, types={t})
    return g


def detect_rare_sharing(ctx: Context) -> list[Candidate]:
    """Groups from Louvain at several resolutions, scored by rare shared items.

    Resolutions are fixed by the community sizes they produce (from market-level
    blobs to groups of a few vendors), not by labels. They are stated per
    `resolution_unit` vendor entities and scaled by the graph's size: at a fixed
    resolution, modularity merges ever larger communities as the graph grows
    (its null model divides by the total edge weight), so the same setting
    would find no small groups on a dataset 20 times larger. A group is kept
    when at least two relation types each have one item held by most members;
    the score sums those items' rarity, weighting identity relations higher.
    """
    unit = ctx.param("open_set", "resolution_unit", 1000)
    total = len(set(ctx.entity_of.values()))
    resolutions = [r * max(1.0, total / unit) for r in ctx.param("open_set", "resolutions", [3, 10, 30])]
    min_size = ctx.param("open_set", "min_size", 3)
    max_size = ctx.param("open_set", "max_size", 15)
    min_cover = ctx.param("open_set", "min_cover", 0.6)
    min_types = ctx.param("open_set", "min_types", 2)
    g = _weighted_graph(ctx)
    if g.number_of_edges() == 0:
        return []
    items = entity_items(ctx)
    onb = ctx.vendors.assign(e=ctx.vendors["record_id"].map(ctx.entity_of)).groupby("e")["onboarding_date"].max()
    groups: list[frozenset] = []
    for res in resolutions:
        for comm in nx.community.louvain_communities(g, weight="weight", resolution=res, seed=ctx.cfg.get("seed", 0)):
            core = frozenset(nx.k_core(g.subgraph(comm), 2).nodes) if len(comm) > 2 else frozenset(comm)
            if min_size <= len(core) <= max_size and all(len(core & x) / len(core | x) < 0.8 for x in groups):
                groups.append(core)
    held: dict = defaultdict(set)   # entity -> (type, item) it holds
    for t in LINK_TYPES:
        for key, es in items.get(t, {}).items():
            for e in es:
                held[e].add((t, key))
    out = []
    for nodes in groups:
        covered = {}
        mine = set().union(*(held.get(e, ()) for e in nodes))
        for t in LINK_TYPES:
            best = max(((len(es & nodes) / len(nodes), key, len(es)) for key, es in
                        ((k, items[t][k]) for tt, k in mine if tt == t) if len(es & nodes) >= 2), default=None)
            if best and best[0] >= min_cover:
                covered[t] = best
        # Single-type groups were tried (one rare identity item held by 3+ vendors): on the
        # starter data they recovered no further planted case and diluted the ranking.
        if len(covered) < min_types:
            continue
        rarity = sum(cov * idf(n, total) * (1.5 if t in IDENTITY_TYPES else 1.0) for t, (cov, _, n) in covered.items())
        identity = set(covered) & set(IDENTITY_TYPES)
        members = frozenset().union(*[ctx.records_of[e] for e in nodes])
        benign = []
        if not identity:
            benign.append("only process relations shared: consistent with a standardised local business process")
        grp = ctx.vendors.loc[sorted(members), "declared_parent_group"].dropna()
        if len(grp) and grp.nunique() == 1 and len(grp) >= 0.8 * len(members):
            benign.append(f"declared group {grp.iloc[0]}")
        out.append(Candidate(
            members=members, tags=(OPEN_SET_TAG,), score=float(rarity + np.log2(len(nodes))),
            emerged_at=onb[list(nodes)].max(), detector="rare_sharing_community",
            reason=(f"{len(nodes)} vendors form a tight group in the relation graph; shared by most members: "
                    + ", ".join(f"{t} {cov:.0%}" for t, (cov, _, _) in sorted(covered.items()))),
            evidence={"shared": {t: {"item": k, "coverage": c, "holders": n} for t, (c, k, n) in covered.items()}},
            benign=benign, features={"rarity": rarity, "n_types": len(covered), "n_identity": len(identity)},
            edges=tuple(shared_item_edges(ctx, {t: k for t, (_, k, _) in covered.items()}, members)),
        ))
    return out


def detect_change_points(ctx: Context) -> list[Candidate]:
    """Several independent changes on one vendor inside one window (ownership-agnostic)."""
    days = ctx.param("open_set", "change_window_days", 90)
    min_kinds = ctx.param("open_set", "min_change_kinds", 3)
    events = []
    a = ctx.addresses.sort_values("valid_from")
    first_addr = a.groupby("record_id")["valid_from"].transform("min")
    for r in a[a["valid_from"] > first_addr].itertuples():
        events.append((r.record_id, r.valid_from, "address"))
    e = ctx.engagements.dropna(subset=["sponsor_employee_id"]).sort_values("start_date")
    first_sp = e.groupby(["record_id", "sponsor_employee_id"])["start_date"].min().reset_index()
    first_any = first_sp.groupby("record_id")["start_date"].transform("min")
    for r in first_sp[first_sp["start_date"] > first_any].itertuples():
        events.append((r.record_id, r.start_date, "sponsor"))
    s = ctx.services
    s = s[s["taxonomy_version"] == "v2"]
    first_cat = s.groupby("record_id")["valid_from"].transform("min")
    for r in s[(s["valid_from"] > first_cat) & (s["valid_from"] > pd.Timestamp("2024-01-31"))].itertuples():
        events.append((r.record_id, r.valid_from, "category"))
    ms = ctx.monthly_spend.sort_values("month")
    for rec, g in ms.groupby("record_id"):
        v = g.set_index("month")["amount_usd"]
        if len(v) < 12:
            continue
        before = v.rolling(6, min_periods=4).median().shift(1)
        after = v[::-1].rolling(6, min_periods=4).median()[::-1]
        jump = (after >= 2.5 * before) & (before > 0)
        if jump.any():
            events.append((rec, jump.idxmax(), "spend_level"))
    ev = pd.DataFrame(events, columns=["record_id", "date", "kind"])
    out = []
    for rec, g in ev.groupby("record_id"):
        g = g.sort_values("date")
        for d in g["date"]:
            w = g[(g["date"] >= d) & (g["date"] <= d + pd.Timedelta(days=days))]
            kinds = set(w["kind"])
            if len(kinds) >= min_kinds:
                benign = ctx.benign_events("category", d, d + pd.Timedelta(days=days)) if "category" in kinds else []
                out.append(Candidate(
                    members=ctx.expand([rec]), tags=(OPEN_SET_TAG,), score=float(len(kinds) + 0.2 * len(w)),
                    emerged_at=d, active_until=d + pd.Timedelta(days=days), detector="entity_change_point",
                    reason=f"within {days} days from {d.date()} the vendor changed {', '.join(sorted(kinds))}",
                    evidence={"events": w.assign(date=w["date"].astype(str)).to_dict("records")},
                    benign=benign, features={"n_kinds": len(kinds)},
                    edges=tuple(address_edges(ctx, [rec], d, d + pd.Timedelta(days=days)) + engagement_edges(
                        ctx, ctx.engagements.loc[(ctx.engagements["record_id"] == rec)
                                                 & ctx.engagements["start_date"].between(d, d + pd.Timedelta(days=days)),
                                                 "engagement_id"])),
                ))
                break
    return out


def detect_duplicate_paid(ctx: Context) -> list[Candidate]:
    """Duplicate master records of one entity that were paid in the same months."""
    em = ctx.entity_map[ctx.entity_map["n_records"] > 1]
    ms = ctx.monthly_spend.merge(em, on="record_id")
    out = []
    for ent, g in ms.groupby("entity_id"):
        per_month = g.groupby("month")["record_id"].nunique()
        both = per_month[per_month >= 2]
        if len(both) == 0:
            continue
        recs = sorted(ctx.records_of[ent])
        out.append(Candidate(
            members=frozenset(recs), tags=(OPEN_SET_TAG,), score=float(np.log2(1 + len(both))),
            emerged_at=both.index.min(), active_until=both.index.max(), detector="duplicate_paid",
            reason=(f"{len(recs)} master records of one entity (merged on "
                    f"{', '.join(sorted(set(ctx.identity_pairs[ctx.identity_pairs['record_a'].isin(recs)]['reason'])))}) "
                    f"were paid in the same {len(both)} month(s), first {both.index.min().date()}"),
            evidence={"months": [str(m.date()) for m in both.index]}, features={"n_months": len(both)},
            edges=tuple(identity_edges(ctx, recs)),
        ))
    return out


def entity_feature_table(ctx: Context) -> pd.DataFrame:
    """Record features plus network position: shares per relation type, foreign
    institution links and (if enabled) ownership structure."""
    from ..features import record_features

    f = record_features(ctx)
    items = entity_items(ctx)
    ent = ctx.entity_of
    for t in LINK_TYPES:
        deg = defaultdict(int)
        for key, es in items.get(t, {}).items():
            if 2 <= len(es) <= 30:
                for e in es:
                    deg[e] += len(es) - 1
        f[f"deg_{t}"] = [deg.get(ent[r], 0) for r in f.index]
    li = ctx.links
    foreign = li[li["inst_country"] != li["record_id"].map(ctx.vendors["country"])].groupby("record_id").size()
    f["foreign_links"] = foreign.reindex(f.index).fillna(0)
    if ctx.use_ownership:
        from .ownership import _view
        ov = _view(ctx)
        snaps = {r: ov.snapshots(r) for r in f.index}
        f["own_layers"] = [max((m["layers"] for m in snaps[r]), default=0) for r in f.index]
        f["own_jurisdictions"] = [max((len(m["jurisdictions"]) for m in snaps[r]), default=0) for r in f.index]
        f["own_sink"] = [float(any(m["sink"] for m in snaps[r])) for r in f.index]
        f["own_pep"] = [float(any(m["peps"] for m in snaps[r])) for r in f.index]
        # Due diligence answered "no PEP" while the registry shows one.
        f["dd_pep_contradiction"] = f["own_pep"] * (1 - f["q_pep_connected"])
    return f


def tail_scores(f: pd.DataFrame, k: int = 2) -> tuple[pd.Series, pd.DataFrame]:
    """Sum of the k largest -log10 upper-tail probabilities per record.

    For each feature, p = share of records with a value at least as high;
    values at or below the median get p = 1. A record is unusual when it sits
    far in the tail of one or two features, which is what a reviewer can check.
    """
    ranks = f.rank(method="max", ascending=False) / len(f)
    ranks = ranks.where(f.gt(f.median()), 1.0)
    nlp = -np.log10(ranks.clip(lower=1.0 / len(f)))
    top = np.sort(nlp.to_numpy(), axis=1)[:, -k:].sum(axis=1)
    return pd.Series(top, index=f.index), nlp


def detect_entity_outlier(ctx: Context) -> list[Candidate]:
    share = ctx.param("open_set", "outlier_top_share", 0.02)
    f = entity_feature_table(ctx)
    score, nlp = tail_scores(f)
    top = score.sort_values(ascending=False).head(max(10, int(share * len(score))))
    out = []
    for rec, s in top.items():
        drivers = nlp.loc[rec].sort_values(ascending=False).head(2)
        out.append(Candidate(
            members=ctx.expand([rec]), tags=(OPEN_SET_TAG,), score=float(s),
            emerged_at=ctx.vendors.loc[rec, "onboarding_date"], detector="entity_outlier",
            reason="far in the tail of: " + ", ".join(
                f"{k}={f.at[rec, k]:.2f} (top {10 ** -v:.1%})" for k, v in drivers.items()),
            evidence={"tail_score": float(s), "drivers": {k: float(f.at[rec, k]) for k in drivers.index}},
            features={"tail": float(s)}, timeless=True,
        ))
    return out


OPEN_SET_DETECTORS = {
    "rare_sharing_community": detect_rare_sharing,
    "entity_change_point": detect_change_points,
    "duplicate_paid": detect_duplicate_paid,
    "entity_outlier": detect_entity_outlier,
}

In [24]:
%%module meridian.detectors.layer2_typologies
"""The new typologies proposed in Layer 2, as optional additions for the 14 / 14+1 / 14+2 / 14+3 comparison.

v1.2 ran the public-institution expert cluster in the default method, with the experts' engagements synchronised
within `window_days` (the pattern of Dossier §3.1 and a component of starter case CASE-0032). v1.3 takes it out of
the default method again: the main-data validation labels use the 14 tags only, and its cases matched none of them.
Variant 14+3 still adds it.

Layer 2 (`layer2_features.NEW_TYPOLOGIES` and its scorecard) proposed new typologies beside the 14; its three
risk-score variants add FMV_CAP_BREACH (14+1), + REONBOARDED_AFTER_TERMINATION (14+2), or
+ PUBLIC_INSTITUTION_EXPERT_CLUSTER + PAYMENT_AFTER_TERMINATION (14+3). Rules follow Layer 2; no labels are read.

The 14-typology method already finds two of them under other tags: `fmv_cap` (POLICY_VIOLATION) and
`post_termination` (ACTIVITY_TIMING_CONCENTRATION). Adding them as typologies gives those findings their own tag,
which changes the submitted tags and the type interleaving of the ranking. The other two are new detectors.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..context import Context
from .base import Candidate
from .evidence import engagement_edges, identity_edges, link_edges

LAYER2_TYPOLOGIES = ("FMV_CAP_BREACH", "REONBOARDED_AFTER_TERMINATION", "PUBLIC_INSTITUTION_EXPERT_CLUSTER",
                     "PAYMENT_AFTER_TERMINATION")
# typologies the 14-typology method already detects under another tag: tag -> detector
RETAG = {"FMV_CAP_BREACH": "fmv_cap", "PAYMENT_AFTER_TERMINATION": "post_termination"}


def _entity(ctx: Context, record: str) -> str:
    return ctx.entity_of.get(record, record)


def detect_public_institution_cluster(ctx: Context) -> list[Candidate]:
    """>= min_experts individual experts affiliated with the same public-sector institution and engaged by the same
    sponsor (Layer 2 N1: AFFILIATED_WITH links of individuals, a shared sponsor, a public-sector institution).
    v1.2: with `window_days`, a group is reported only when at least `min_experts` of its experts joined (affiliated
    and engaged) within that many days of each other: synchronised participation. It then holds all the group's
    experts, and dates from that core (the largest such set, the earliest if tied)."""
    min_experts = ctx.param("public_institution", "min_experts", 3)
    window = ctx.param("public_institution", "window_days", None)
    individuals = set(ctx.vendors.index[ctx.vendors["party_type"] == "individual"])
    li = ctx.links
    li = li[(li["link_type"] == "AFFILIATED_WITH") & li["record_id"].isin(individuals)
            & li["public_sector"].astype(str).str.lower().eq("true")]
    eng = ctx.engagements.dropna(subset=["sponsor_employee_id"])
    first = eng.groupby(["record_id", "sponsor_employee_id"], as_index=False)["start_date"].min()
    pairs = li.merge(first.rename(columns={"start_date": "first_engagement"}), on="record_id")
    pairs["entity"] = pairs["record_id"].map(lambda r: _entity(ctx, r))
    out = []
    for (inst, sponsor), g in pairs.groupby(["institution_id", "sponsor_employee_id"]):
        if g["entity"].nunique() < min_experts:
            continue
        # the group holds from the day its min_experts-th expert is both affiliated and engaged by the sponsor
        joined = g[["valid_from", "first_engagement"]].max(axis=1).groupby(g["entity"]).min().sort_values()
        if window:
            span = pd.Timedelta(days=window)
            counts = [((joined >= t) & (joined <= t + span)).sum() for t in joined]
            t0 = joined.iloc[int(np.argmax(counts))]
            joined = joined[(joined >= t0) & (joined <= t0 + span)]
            if len(joined) < min_experts:
                continue
        n = g["entity"].nunique()
        members = ctx.expand(set(g["record_id"]))
        n_sponsor = eng.loc[eng["sponsor_employee_id"] == sponsor, "record_id"].map(lambda r: _entity(ctx, r)).nunique()
        share = n / max(n_sponsor, 1)
        e = eng[(eng["sponsor_employee_id"] == sponsor) & eng["record_id"].isin(members)]
        out.append(Candidate(
            members=members, tags=("PUBLIC_INSTITUTION_EXPERT_CLUSTER",), score=float(np.log2(n) + share),
            emerged_at=joined.iloc[min_experts - 1], detector="public_institution_cluster",
            reason=(f"{n} individual experts affiliated with public institution {inst} ({g['inst_name'].iloc[0]}, "
                    f"{g['inst_country'].iloc[0]}) are engaged by the same sponsor {sponsor} ({ctx.role_of.get(sponsor)}); "
                    f"{share:.0%} of the sponsor's vendors"),
            evidence={"institution": inst, "sponsor": sponsor, "experts": sorted(set(g["record_id"]))},
            features={"n_experts": n, "sponsor_share": share},
            edges=tuple(link_edges(ctx, g["link_id"]) + engagement_edges(ctx, e["engagement_id"])),
        ))
    return out


def detect_reonboarded(ctx: Context) -> list[Candidate]:
    """A record with the same tax id or phone as a record terminated before it was onboarded (Layer 2 N3)."""
    tp = ctx.vendors
    phone = ctx.contacts.set_index("record_id")["phone_key"]
    keyed = pd.concat([tp["tax_id"].dropna().rename("key").to_frame().assign(kind="tax id"),
                       phone.dropna().rename("key").to_frame().assign(kind="phone")])
    shared: dict[tuple, set] = {}
    for (kind, _), d in keyed.groupby(["kind", "key"]):
        recs = sorted(set(d.index))
        for a in recs:
            term = tp.at[a, "terminated_date"]
            if pd.isna(term):
                continue
            for b in recs:
                if b != a and tp.at[b, "onboarding_date"] > term:
                    shared.setdefault((a, b), set()).add(kind)
    out = []
    for (a, b), kinds in sorted(shared.items()):
        term, onboard = tp.at[a, "terminated_date"], tp.at[b, "onboarding_date"]
        gap = (onboard - term).days
        out.append(Candidate(
            members=ctx.expand([a, b]), tags=("REONBOARDED_AFTER_TERMINATION",),
            score=float(len(kinds) + 0.5 * ("tax id" in kinds) + 1 / (1 + gap / 365)), emerged_at=onboard,
            detector="reonboarded_after_termination",
            reason=(f"{b} onboarded {onboard.date()}, {gap} days after {a} was terminated ({term.date()}); "
                    f"same {' and '.join(sorted(kinds))}"),
            evidence={"terminated_record": a, "new_record": b, "shared": sorted(kinds), "gap_days": gap},
            features={"gap_days": gap, "n_shared": len(kinds)},
            edges=tuple(identity_edges(ctx, [a, b])),
        ))
    return out


NEW_DETECTORS = {"REONBOARDED_AFTER_TERMINATION": detect_reonboarded,
                 "PUBLIC_INSTITUTION_EXPERT_CLUSTER": detect_public_institution_cluster}


def apply(ctx: Context, cands: list, added, exclude=()) -> list:
    """Candidates of the 14-typology method with the given Layer 2 typologies added."""
    unknown = set(added) - set(LAYER2_TYPOLOGIES)
    if unknown:
        raise ValueError(f"unknown Layer 2 typologies {sorted(unknown)}")
    for tag, det in RETAG.items():
        if tag in added:
            for c in cands:
                if c.detector == det:
                    c.tags = (tag,)
    have = {c.detector for c in cands}   # v1.2: the expert cluster already runs in the default method
    extra = [c for tag, fn in NEW_DETECTORS.items() if tag in added for c in fn(ctx)]
    return cands + [c for c in extra if c.detector not in exclude and c.detector not in have]

In [25]:
%%module meridian.detectors package
"""Detector registry. Each detector takes a Context and returns a list of Candidates."""
from __future__ import annotations

from . import activity, approval, cluster, layer2_typologies, market, ownership, policy, record, shared_address, sponsor
from .open_set import OPEN_SET_DETECTORS

# Typed detectors and policy detectors. The exact-address baseline
# (the first, simplest detector) is kept for the representation ladder but not run by default.
DETECTORS = {
    "shared_address": shared_address.detect,
    "sponsor_concentration": sponsor.detect,
    "sponsor_burst": sponsor.detect_burst,   # v1.3: RAPID_ONBOARDING_LINKED through one sponsor's burst
    "sponsor_approver_pair": approval.detect,
    "approver_star": approval.detect_stars,
    "dense_cluster": cluster.detect_dense,
    "rapid_onboarding": cluster.detect_rapid_onboarding,
    "bridge": market.detect_bridge,
    "geography": market.detect_geography,
    "corridor": market.detect_corridor,   # v1.3: GEOGRAPHY_LINKED_CONCENTRATION through a cross-border corridor
    "sync_spikes": activity.detect_sync_spikes,
    "post_termination": activity.detect_post_termination,
    "common_owner": ownership.detect_common_owner,
    "layered_ownership": ownership.detect_layered,
    "opaque_routing": ownership.detect_opaque,
    "ownership_change": ownership.detect_change,
    # v1.3: the public-institution expert cluster (v1.2) is no longer part of the method: the main-data validation
    # labels use the 14 tags only, and its cases matched none of them. Variant 14+3 still adds it (section 7).
    "fmv_cap": policy.detect_fmv,
    "compliance_review": policy.detect_compliance,
}
OPTIONAL = {
    "address_exact": shared_address.detect_exact,
    "record_outlier": record.detect_record_outlier,
}
OWNERSHIP_DETECTORS = {"common_owner", "layered_ownership", "opaque_routing", "ownership_change"}


def registry() -> dict:
    return {**DETECTORS, **OPTIONAL, **OPEN_SET_DETECTORS}


def run_detectors(ctx, names="all", exclude=()) -> list:
    """Run the selected detectors. `exclude` drops candidates by detector name
    (a function can emit several, e.g. common_owner and ownership_overlap)."""
    reg = registry()
    if names in (None, "all"):
        selected = {**DETECTORS, **(OPEN_SET_DETECTORS if ctx.cfg.get("features", {}).get("open_set", True) else {})}
    else:
        selected = {n: reg[n] for n in names}
    out = []
    for name, fn in selected.items():
        if not ctx.use_ownership and name in OWNERSHIP_DETECTORS:
            continue
        cands = fn(ctx)
        if not ctx.use_ownership:
            cands = [c for c in cands if not c.uses_ownership]
        out.extend(c for c in cands if c.detector not in exclude)
    added = ctx.cfg.get("layer2_typologies") or ()
    if added:   # the Layer 2 typology variants of section 7; the default method adds none
        out = layer2_typologies.apply(ctx, out, added, exclude)
    return out

In [26]:
%%module meridian.features
"""Explicit features for the representation ladder.

Record features (level A) use only the record itself, its own engagements'
attributes and payments, and reference tables. Case features describe any
member set at four levels:

  A  aggregated record features of the members
  B  deterministic relation rules among the members (address, sponsor, approver, timing, market)
  C  graph structure of the members in the multi-relation vendor graph
  D  ownership (registry R paths)

Nothing here reads labels.
"""
from __future__ import annotations

from collections import Counter
from functools import cached_property

import networkx as nx
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler

from .context import Context
from .detectors.policy import FEE_CLASSIFICATIONS, country_band

LEVELS = ("A", "B", "C", "D")


# ---------------------------------------------------------------------------
# Record-level features (A)
# ---------------------------------------------------------------------------
def record_features(ctx: Context) -> pd.DataFrame:
    t = ctx.t
    tp = ctx.vendors
    f = pd.DataFrame(index=tp.index)
    dd = ctx.b.tp_due_diligence.set_index("record_id").reindex(tp.index)
    f["dd_enhanced"] = dd["dd_level"].eq("enhanced").astype(float)
    f["risk_rating"] = dd["risk_rating"].map({"low": 0, "medium": 1, "high": 2}).fillna(0)
    for q in ["q_gov_ownership", "q_pep_connected", "q_public_official_interaction", "q_former_public_official",
              "adverse_media"]:
        f[q] = dd[q].astype(float).fillna(0)
    f["screen_hit"] = dd["screening_result"].eq("potential_match_cleared").astype(float)
    f["individual"] = tp["party_type"].eq("individual").astype(float)
    f["remit_mismatch"] = (tp["remit_to_country"] != tp["country"]).astype(float)
    f["no_lei"] = tp["lei"].isna().astype(float)
    f["no_registry_no"] = tp["registry_no"].isna().astype(float)
    f["no_declared_group"] = tp["declared_parent_group"].isna().astype(float)
    f["tenure_years"] = (t - tp["onboarding_date"]).dt.days / 365.25
    f["terminated"] = tp["terminated_date"].notna().astype(float)
    cpi = {c: ctx.country_index(c, "CPI") for c in tp["country"].unique()}
    f["cpi_tier"] = tp["country"].map(lambda c: float(cpi[c]) if cpi[c] is not None and str(cpi[c]).isdigit() else 2.0)

    a = ctx.addresses.sort_values("valid_from")
    cur = a.groupby("record_id").last()
    f["n_addresses"] = a.groupby("record_id").size().reindex(tp.index).fillna(0)
    f["addr_residential"] = cur["address_type"].eq("residential").reindex(tp.index, fill_value=False).astype(float)
    f["addr_virtual"] = cur["address_type"].isin(["virtual_office", "po_box"]).reindex(tp.index, fill_value=False).astype(float)
    f["addr_agent"] = cur["address_type"].eq("registered_agent").reindex(tp.index, fill_value=False).astype(float)
    f["addr_foreign"] = (cur["address_country"] != tp["country"].reindex(cur.index)).reindex(
        tp.index, fill_value=False).astype(float)

    e = ctx.engagements.copy()
    e["vendor_country"] = e["record_id"].map(tp["country"])
    g = e.groupby("record_id")
    f["n_engagements"] = g.size().reindex(tp.index).fillna(0)
    f["max_value_band"] = g["value_band"].max().reindex(tp.index).fillna(0)
    for flag in ["int", "gov", "hcp", "flow"]:
        f[f"share_{flag}"] = g["cat_flags_possible"].apply(lambda s: np.mean([flag in x for x in s])).reindex(tp.index).fillna(0)
    f["n_categories"] = g["cat_primary"].nunique().reindex(tp.index).fillna(0)
    f["share_cross_border"] = g.apply(lambda d: (d["market"] != d["vendor_country"]).mean(),
                                      include_groups=False).reindex(tp.index).fillna(0)
    f["share_no_contract"] = g["contract_type"].apply(lambda s: (s == "none").mean()).reindex(tp.index).fillna(0)

    p = ctx.b.tp_payments_monthly.copy()
    p = p.merge(e[["engagement_id", "value_band", "end_date"]], on="engagement_id", how="left")
    pg = p.groupby("record_id")
    monthly = p.groupby(["record_id", "month"])["amount_usd"].sum()
    mg = monthly.groupby(level=0)
    f["log_total_spend"] = np.log1p(pg["amount_usd"].sum()).reindex(tp.index).fillna(0)
    f["n_paid_months"] = mg.size().reindex(tp.index).fillna(0)
    f["spend_spikiness"] = (mg.max() / mg.median().clip(lower=1)).reindex(tp.index).fillna(0)
    f["round_share"] = pg["round_amount_share"].mean().reindex(tp.index).fillna(0)
    f["manual_share"] = (pg["manual_payment_count"].sum() / pg["payment_count"].sum().clip(lower=1)).reindex(tp.index).fillna(0)
    nonpo = p.assign(np_amt=np.where(p["channel"] != "po_backed", p["amount_usd"], 0.0)).groupby("record_id")
    f["non_po_share"] = (nonpo["np_amt"].sum() / nonpo["amount_usd"].sum().clip(lower=1)).reindex(tp.index).fillna(0)
    f["invoice_over_band"] = (p["max_invoice_band"] > p["value_band"]).groupby(p["record_id"]).sum().reindex(tp.index).fillna(0)
    f["foreign_payer_share"] = (p["paying_market"] != p["vendor_country"]).groupby(p["record_id"]).mean().reindex(tp.index).fillna(0)
    term = tp["terminated_date"]
    p["term"] = p["record_id"].map(term)
    f["months_after_termination"] = (p["month"] > p["term"] + pd.offsets.MonthEnd(0)).groupby(
        p["record_id"]).sum().reindex(tp.index).fillna(0)
    f["months_after_contract_end"] = (p["end_date"].notna() & (p["month"] > p["end_date"] + pd.offsets.MonthEnd(1))).groupby(
        p["record_id"]).sum().reindex(tp.index).fillna(0)

    # FMV: worst annual fee total relative to the cap (individuals only).
    ind = tp[tp["party_type"] == "individual"]
    pf = p[p["record_id"].isin(ind.index) & p["expense_classification"].isin(FEE_CLASSIFICATIONS)].copy()
    pf["year"] = pf["month"].dt.year
    cards = ctx.b.fmv_rate_cards
    ratio = {}
    for (rec, year), s in pf.groupby(["record_id", "year"])["amount_usd"]:
        band = country_band(ctx, ind.loc[rec, "country"])
        version = "v2" if year >= 2024 else "v1"
        cap = cards[(cards["card_version"] == version) & (cards["country_band"] == band)
                    & (cards["expert_role"] == ind.loc[rec, "expert_type"])]["annual_cap_usd"]
        if len(cap):
            ratio[rec] = max(ratio.get(rec, 0.0), s.sum() / float(cap.iloc[0]))
    f["fmv_ratio"] = pd.Series(ratio).reindex(tp.index).fillna(0)
    return f.astype(float)


def record_outlier_scores(feat: pd.DataFrame, seed: int = 0) -> pd.Series:
    """Isolation-forest outlier score on record features (higher = more unusual)."""
    x = RobustScaler(quantile_range=(10, 90)).fit_transform(feat.to_numpy())
    x = np.clip(x, -10, 10)
    iso = IsolationForest(n_estimators=300, random_state=seed).fit(x)
    return pd.Series(-iso.score_samples(x), index=feat.index)


# ---------------------------------------------------------------------------
# Case-level features (A-D)
# ---------------------------------------------------------------------------
class CaseFeaturizer:
    """Features of arbitrary member sets, grouped by ladder level."""

    def __init__(self, ctx: Context, seed: int = 0):
        self.ctx = ctx
        self.seed = seed

    @cached_property
    def rec(self) -> pd.DataFrame:
        f = record_features(self.ctx)
        f["outlier"] = record_outlier_scores(f, self.seed)
        return f

    @cached_property
    def relations(self):
        from .detectors.relations import entity_items, pair_links
        items = entity_items(self.ctx, use_ownership=False)
        return items, pair_links(self.ctx, items)

    @cached_property
    def graph(self) -> nx.Graph:
        """Entity graph weighted by the number of shared (non-owner) relation types."""
        _, pairs = self.relations
        g = nx.Graph()
        for (a, b), types in pairs.items():
            g.add_edge(a, b, weight=len(types))
        return g

    @cached_property
    def communities(self) -> dict:
        comms = nx.community.louvain_communities(self.graph, weight="weight", seed=self.seed)
        return {n: i for i, c in enumerate(comms) for n in c}

    @cached_property
    def sponsor_burst(self) -> dict:
        from .detectors.sponsor import best_window, first_engagements
        fe = first_engagements(self.ctx)
        out = {}
        for sp, g in fe.groupby("sponsor_employee_id"):
            out[sp] = best_window(np.sort(g["start_date"].to_numpy()), 60)[1]
        role = self.ctx.role_of
        med = pd.Series(out).groupby(lambda s: role.get(s)).median()
        return {s: n / max(med.get(role.get(s), 1.0), 1.0) for s, n in out.items()}

    @cached_property
    def fin_approvals(self) -> pd.DataFrame:
        ap = self.ctx.approvals
        fin = ap[(ap["approval_type"] == "financial")].copy()
        fin["under"] = fin["ap_approver_level"] < fin["required_approver_level"]
        return fin

    @cached_property
    def spikes(self) -> pd.DataFrame:
        from .detectors.activity import spend_spikes
        return spend_spikes(self.ctx)

    @cached_property
    def ownership(self):
        from .detectors.ownership import _view, person_holdings
        return _view(self.ctx), person_holdings(self.ctx)

    # ------------------------------------------------------------------
    def level_a(self, members: frozenset) -> dict:
        r = self.rec.loc[sorted(members)]
        return {
            "A_size": float(np.log2(1 + len(members))),
            "A_outlier_max": r["outlier"].max(), "A_outlier_mean": r["outlier"].mean(),
            "A_fmv_ratio": r["fmv_ratio"].max(), "A_after_term": r["months_after_termination"].max(),
            "A_after_end": r["months_after_contract_end"].max(), "A_risk": r["risk_rating"].mean(),
            "A_share_int": r["share_int"].mean(), "A_cpi": r["cpi_tier"].mean(),
            "A_residential": r["addr_residential"].mean(), "A_tenure": r["tenure_years"].min(),
            "A_spiky": r["spend_spikiness"].max(), "A_individual": r["individual"].mean(),
        }

    def level_b(self, members: frozenset) -> dict:
        ctx = self.ctx
        n = len(members)
        a = ctx.addresses[ctx.addresses["record_id"].isin(members)]
        addr_share = a.groupby("addr_key")["record_id"].nunique().max() / n if len(a) else 0.0
        e = ctx.engagements[ctx.engagements["record_id"].isin(members)]
        sp_counts = e.dropna(subset=["sponsor_employee_id"]).groupby("sponsor_employee_id")["record_id"].nunique()
        top_sp = sp_counts.idxmax() if len(sp_counts) else None
        fin = self.fin_approvals[self.fin_approvals["record_id"].isin(members)]
        pair = fin.dropna(subset=["approver_employee_id", "sponsor_employee_id"]).groupby(
            ["sponsor_employee_id", "approver_employee_id"])["record_id"].nunique()
        onb = ctx.vendors.loc[sorted(members), "onboarding_date"]
        span = (onb.max() - onb.min()).days
        sk = self.spikes[self.spikes["record_id"].isin(members)]
        sync = 0.0
        if len(sk):
            by_month = sk.groupby(sk["month"].dt.to_period("M"))["record_id"].nunique()
            roll = by_month.rolling(2, min_periods=1).sum() if len(by_month) > 1 else by_month
            sync = float(roll.max()) / n
        li = ctx.links[ctx.links["record_id"].isin(members)]
        foreign = (li["inst_country"] != li["record_id"].map(ctx.vendors["country"])).sum()
        cats = e["cat_primary"].dropna()
        return {
            "B_addr_share": addr_share,
            "B_sponsor_share": sp_counts.max() / n if len(sp_counts) else 0.0,
            "B_sponsor_burst": self.sponsor_burst.get(top_sp, 0.0) if top_sp else 0.0,
            "B_pair_share": pair.max() / n if len(pair) else 0.0,
            "B_under_authority": float(fin["under"].sum()),
            "B_onboard_span": float(np.log1p(span)),
            "B_market_share": ctx.vendors.loc[sorted(members), "country"].value_counts(normalize=True).iloc[0],
            "B_category_share": cats.value_counts(normalize=True).iloc[0] if len(cats) else 0.0,
            "B_sync_spikes": sync,
            "B_foreign_links": float(foreign),
        }

    def level_c(self, members: frozenset) -> dict:
        ctx = self.ctx
        ents = {ctx.entity_of[r] for r in members}
        g = self.graph
        inside = [e for e in ents if e in g]
        sub = g.subgraph(inside)
        k = len(ents)
        w_in = sum(d["weight"] for _, _, d in sub.edges(data=True))
        w_all = sum(d["weight"] for e in inside for _, _, d in g.edges(e, data=True))
        possible = k * (k - 1) / 2
        comm = Counter(self.communities.get(e) for e in inside)
        return {
            "C_density": sub.number_of_edges() / possible if possible else 0.0,
            "C_mean_types": w_in / possible if possible else 0.0,
            "C_conductance": 1 - (2 * w_in / w_all) if w_all else 1.0,
            "C_community_purity": comm.most_common(1)[0][1] / k if comm else 0.0,
            "C_max_degree": float(max((g.degree(e) for e in inside), default=0)),
        }

    def level_d(self, members: frozenset) -> dict:
        ctx = self.ctx
        ov, holdings = self.ownership
        ents = {ctx.entity_of[r] for r in members}
        best_share = 0.0
        for person, held in holdings.items():
            share = len(set(held) & ents) / len(ents)
            best_share = max(best_share, share if len(set(held) & ents) >= 1 else 0.0)
        layers = jur = sink = pep = undisclosed = 0.0
        for r in members:
            for m in ov.snapshots(r):
                layers = max(layers, m["layers"])
                jur = max(jur, len(m["jurisdictions"]))
                sink = max(sink, float(bool(m["sink"])))
                pep = max(pep, float(bool(m["peps"])))
                if m["peps"] and not bool(ov.dd.loc[r, "q_pep_connected"]):
                    undisclosed = 1.0
        linked = sum(r in ctx.reg_entity_of or r in ctx.reg_person_of for r in members) / len(members)
        return {"D_common_owner_share": best_share if len(ents) > 1 else 0.0, "D_layers": float(layers),
                "D_jurisdictions": float(jur), "D_sink": sink, "D_pep": pep, "D_pep_undisclosed": undisclosed,
                "D_registry_linked": linked}

    def features(self, members: frozenset, level: str = "D") -> dict:
        out = self.level_a(members)
        if level in ("B", "C", "D"):
            out.update(self.level_b(members))
        if level in ("C", "D"):
            out.update(self.level_c(members))
        if level == "D":
            out.update(self.level_d(members))
        return out

    def matrix(self, member_sets: list[frozenset], level: str) -> pd.DataFrame:
        return pd.DataFrame([self.features(m, level) for m in member_sets]).astype(float).fillna(0.0)

In [27]:
%%module meridian.cases
"""Case assembly: merge candidates, emergence dates, benign explanations, window filter."""
from __future__ import annotations

from dataclasses import dataclass, field

import pandas as pd

from .detectors.base import OPEN_SET_TAG, Candidate


@dataclass
class Case:
    members: frozenset
    tags: tuple
    score: float                      # max raw detector score (not comparable across detectors)
    emerged_at: pd.Timestamp | None
    candidates: list = field(default_factory=list)
    support: list = field(default_factory=list)   # candidates inside the case that add tags but not members
    calibrated: float = 0.0
    benign_factor: float = 1.0
    rank: int | None = None                        # position in the submitted list, None if cut
    tag_scores: dict = field(default_factory=dict)  # tag -> strongest calibrated evidence for it
    effective: float = 0.0                         # score that set the rank (after type decay and penalties)
    lead_detector: str = ""                        # detector whose evidence scores highest
    lead_raw_score: float = 0.0                    # that detector's own raw score (its own scale)
    selected_tags: tuple = ()                      # v1.2: the submitted tags, set by ranking before the order
    window_basis: str = ""                         # v1.3: why the case is in the period: emerged, grew, finding, standing
    window_weight: float = 1.0                     # v1.3: ranking weight from the period filter (standing cases < 1)

    @property
    def lead(self) -> Candidate:
        return self.candidates[0]

    @property
    def detectors(self) -> tuple:
        return tuple(sorted({c.detector for c in self.candidates + self.support}))

    @property
    def primary_tag(self) -> str:
        return self.selected_tags[0] if self.selected_tags else self.lead.tags[0]

    @property
    def reason(self) -> str:
        parts = list(dict.fromkeys(c.reason for c in self.candidates + self.support))
        if self.benign:
            parts.append("benign to check: " + "; ".join(self.benign))
        return " | ".join(parts)

    @property
    def edges(self) -> tuple:
        """Graph edges behind the case: the union of its candidates' evidence edges, by edge id."""
        return tuple(sorted({e for c in self.candidates + self.support for e in c.edges}))

    @property
    def benign(self) -> list:
        return list(dict.fromkeys(b for c in self.candidates for b in c.benign))


def _iou(a: frozenset, b: frozenset) -> float:
    return len(a & b) / len(a | b)


def _close(a, b, days: int) -> bool:
    if a is None or b is None:
        return True
    return abs((a - b).days) <= days


def _concurrent(case: "Case", c: Candidate, days: int) -> bool:
    """Same time frame: emergence dates within `days`, or one pattern was still
    holding when the other emerged. Two patterns on the same vendors whose
    periods do not meet (a layered structure that ended when the vendor was
    sold) stay separate cases."""
    if _close(case.emerged_at, c.emerged_at, days):
        return True
    for a in case.candidates:
        if a.timeless:
            continue
        first, second = (a, c) if (a.emerged_at or pd.Timestamp.min) <= (c.emerged_at or pd.Timestamp.min) else (c, a)
        if second.emerged_at is None or first.active_until is None or second.emerged_at < first.active_until:
            return True
    return False


def _edge_share(a: Candidate, b: Candidate) -> float:
    """Share of the smaller evidence-edge set that both candidates cite."""
    if not a.edges or not b.edges:
        return 0.0
    ea, eb = set(a.edges), set(b.edges)
    return len(ea & eb) / min(len(ea), len(eb))


def _percentiles(cands: list[Candidate]) -> dict[int, float]:
    by_det: dict[str, list] = {}
    for c in cands:
        by_det.setdefault(c.detector, []).append(c.score)
    return {id(c): sum(s <= c.score for s in by_det[c.detector]) / len(by_det[c.detector]) for c in cands}


def assemble(cands: list[Candidate], merge_iou: float = 0.8, union_iou: float = 0.5, max_growth: float = 1.5,
             merge_days: int = 365, support_days: int = 365, evidence_union: float | None = None,
             best_match: bool = True, open_set_grows: bool = False, typed_first: bool = False,
             single_into_pair: bool = False) -> list[Case]:
    """Merge candidates that describe the same group at about the same time.

    Candidates are visited by within-detector percentile (raw scores are not
    comparable across detectors). A candidate joins an existing case when the
    emergence dates are within merge_days and either
      - IoU >= merge_iou: same members; the case keeps its boundary, or
      - it comes from a different detector than the case's candidates and
        either both it and the case's lead are groups with IoU >= union_iou
        (union within max_growth of the lead), or it overlaps one of the
        case's candidates and shares at least `evidence_union` of the smaller
        evidence-edge set with it (the same graph structure seen by two
        detectors, e.g. a layered chain and the common owner at its top): the
        case takes the union of members. Planted multi-typology cases are unions of what several
        relations show.
    Among the cases a candidate qualifies for it joins the one with the highest
    member IoU (`best_match`), not the first one found.
    Open-set candidates (communities, outliers) corroborate a case but never
    add members to it: they have no typed pattern that defines the group, and
    a community usually reaches past the group it overlaps.
    The time condition (`_concurrent`) keeps apart two patterns on the same
    vendor whose periods do not meet (on the starter data one vendor carries a
    layered-ownership case that ended when it was sold, and the ownership-change
    case of that sale), while an owner group and the address overlap it later
    develops stay one case. Tags are the union, the emergence date is
    the earliest, and every candidate's evidence is kept.

    With `typed_first`, open-set candidates are visited after typed ones, so a
    community never sets the boundary of a case that typed findings then join.

    Candidates whose members are a strict subset of a larger case (e.g. a
    single-record ownership finding inside an owner group) stay as their own
    case and also add their tag to the larger case as support, if
    they carry no benign flag and score at or above their detector's median.
    With `single_into_pair`, a one-vendor typed case inside exactly one
    two-vendor typed case of the same time frame joins it instead (IoU 0.5,
    the union threshold): the vendor's own findings describe the pair.
    """
    pct = _percentiles(cands)
    cases: list[Case] = []
    # Only cases that share a record with a candidate can take it: index cases by
    # the records of their candidates (open-set candidates may reach past the members).
    by_rec: dict[str, set] = {}

    def near(c: Candidate) -> list[int]:
        return sorted(set().union(*(by_rec.get(m, ()) for m in c.members)))

    def add(i: int, c: Candidate) -> None:
        for m in c.members:
            by_rec.setdefault(m, set()).add(i)

    # Timed candidates first, so timeless ones only attach to cases whose time frame is already set.
    def open_set(c: Candidate) -> bool:
        return typed_first and OPEN_SET_TAG in c.tags

    for c in sorted(cands, key=lambda c: (c.timeless, open_set(c), -pct[id(c)], -c.score, c.detector,
                                          sorted(c.members))):
        if c.timeless:
            hosts = [i for i in near(c) if _iou(cases[i].members, c.members) >= merge_iou]
            if hosts:
                h = max(hosts, key=lambda i: _iou(cases[i].members, c.members)) if best_match else hosts[0]
                cases[h].candidates.append(c)
                cases[h].tags = tuple(dict.fromkeys(cases[h].tags + c.tags))
                add(h, c)
                continue
        grows = open_set_grows or OPEN_SET_TAG not in c.tags
        target, target_iou, target_union = None, -1.0, False
        for i in near(c):
            case = cases[i]
            if not _concurrent(case, c, merge_days):
                continue
            iou = _iou(case.members, c.members)
            # Union is judged against the lead candidate and capped, so that groups
            # sharing one hub (a sponsor, an owner) do not chain into one big case.
            lead = case.lead.members
            groups = (len(c.members) > 1 and len(lead) > 1 and _iou(lead, c.members) >= union_iou
                      and len(case.members | c.members) <= max_growth * len(lead))
            same_structure = evidence_union is not None and any(
                x.members & c.members and _edge_share(x, c) >= evidence_union for x in case.candidates)
            union = c.detector not in {x.detector for x in case.candidates} and (groups or same_structure)
            if (iou >= merge_iou or union) and iou > target_iou:
                target, target_iou, target_union = i, iou, union and iou < merge_iou
                if not best_match:
                    break
        if target is None:
            cases.append(Case(members=c.members, tags=c.tags, score=c.score,
                              emerged_at=c.emerged_at, candidates=[c]))
            add(len(cases) - 1, c)
            continue
        case = cases[target]
        case.candidates.append(c)
        add(target, c)
        if target_union and grows:
            case.members = case.members | c.members
        case.tags = tuple(dict.fromkeys(case.tags + c.tags))
        case.score = max(case.score, c.score)
        if c.emerged_at is not None and (case.emerged_at is None or c.emerged_at < case.emerged_at):
            case.emerged_at = c.emerged_at
    if single_into_pair:
        cases = _singles_into_pairs(cases, merge_days)
    cand_by_rec: dict[str, list] = {}
    for j, c in enumerate(cands):
        for m in c.members:
            cand_by_rec.setdefault(m, []).append(j)
    for case in cases:
        if len(case.members) < 2:
            continue
        for j in sorted(set().union(*(cand_by_rec.get(m, ()) for m in case.members))):
            c = cands[j]
            if (c.members < case.members and c not in case.candidates and not c.benign and pct[id(c)] >= 0.5
                    and _close(case.emerged_at, c.emerged_at, support_days)):
                case.support.append(c)
                case.tags = tuple(dict.fromkeys(case.tags + c.tags))
    return cases


def _singles_into_pairs(cases: list[Case], days: int) -> list[Case]:
    typed = {id(c) for c in cases if OPEN_SET_TAG not in c.lead.tags}
    pairs = [c for c in cases if id(c) in typed and len(c.members) == 2]
    out = []
    for case in cases:
        if len(case.members) == 1 and id(case) in typed:
            hosts = [p for p in pairs if case.members < p.members and _concurrent(p, case.lead, days)]
            if len(hosts) == 1:
                host = hosts[0]
                host.candidates.extend(case.candidates)
                host.support.extend(case.support)
                host.tags = tuple(dict.fromkeys(host.tags + case.tags))
                host.score = max(host.score, case.score)
                if case.emerged_at is not None and (host.emerged_at is None or case.emerged_at < host.emerged_at):
                    host.emerged_at = case.emerged_at
                continue
        out.append(case)
    return out


def benign_weight(benign: list, per_item: float = 0.7, floor: float = 0.3) -> float:
    """Down-weight for documented benign explanations. Signals are never removed."""
    return max(floor, per_item ** len(benign))


def in_window(cases: list[Case], window, joined: dict | None = None, finding_cover: float | None = None,
              standing: float | None = None, finding_weight: float = 1.0) -> list[Case]:
    """Keep cases that emerged or grew in [start, end]. Undated cases are dropped.

    A case emerged in the window when its emergence date falls in it. A case
    that emerged earlier grew in the window when one of its vendor entities was
    first onboarded in it (`joined`: record -> its entity's first onboarding);
    such an entity cannot have been part of the pattern before the window. A
    new record of an entity already in the case does not count.

    v1.3, after the main-data validation labels (8 of the 43 validation cases were found but dated before the
    period and dropped here):
      - `finding_cover`: a case also emerged in the window when one of its typed findings that covers the case
        (member IoU >= finding_cover) emerged in it. A case takes the earliest date of everything merged into it,
        so a shared address formed in the window on vendors with an older relation was dated by the older relation
        (VAL-01, VAL-02). Such a case ranks with weight `finding_weight`: the structure is older than the period,
        and on the starter TEST period two old owner groups with a new finding otherwise rank above labelled cases.
      - `standing`: a case that emerged earlier and whose covering typed finding still holds at the window start is
        kept too, with ranking weight `standing`. A static structure (an ownership chain, a foreign correspondence
        address) has no date inside the period, yet the labels place some of them there (VAL-09, VAL-24, VAL-33,
        VAL-35, VAL-36); the weight keeps them behind the cases that emerged in the period.
    Each kept case records why (`window_basis`) and its weight (`window_weight`).
    """
    for c in cases:
        c.window_basis, c.window_weight = "", 1.0
    if not window:
        return cases
    lo, hi = (pd.Timestamp(x) for x in window)

    def grew(c: Case) -> bool:
        return joined is not None and any(lo <= joined.get(m, pd.NaT) <= hi for m in c.members)

    def covering(c: Case) -> list:
        if finding_cover is None:
            return []
        return [x for x in c.candidates if OPEN_SET_TAG not in x.tags and not x.timeless and x.emerged_at is not None
                and _iou(x.members, c.members) >= finding_cover]

    out = []
    for c in cases:
        if c.emerged_at is None or c.emerged_at > hi:
            continue
        if c.emerged_at >= lo:
            basis = "emerged"
        elif grew(c):
            basis = "grew"
        elif any(lo <= x.emerged_at <= hi for x in covering(c)):
            basis = "finding"
        elif standing and any(x.emerged_at <= hi and (x.active_until is None or x.active_until >= lo)
                              for x in covering(c)):
            basis = "standing"
        else:
            continue
        c.window_basis, c.window_weight = basis, {"standing": standing, "finding": finding_weight}.get(basis, 1.0)
        out.append(c)
    return out


In [28]:
%%module meridian.ranking
"""Unified ranking: calibration, type interleaving, submission table.

Raw detector scores have different scales. Each candidate inside a case gets
    s = hit_rate(detector) * (0.5 + 0.5 * within-detector percentile) * benign weight
where hit_rate is either a fixed prior (no labels) or estimated from labelled
cases by `calibrate.py`, and the benign weight only reflects explanations for
that candidate's own signal (a coworking address does not explain an ownership
convergence). The case takes its best candidate plus part of the noisy-OR gain
from other detectors that corroborate it.

Each piece of evidence counts once: candidates are taken strongest first, and
each one adds to the noisy-OR only in proportion to the evidence edges no
stronger candidate has already used (a common-owner finding adds nothing to
an ownership-overlap finding built on the same ownership edges). Candidates
without edges (record-level outlier scores) are independent evidence.

A detector with fewer than `min_pool` candidates cannot rank them meaningfully
against each other, so their percentile is a fixed 0.5 instead of 1.0 for a
lone candidate.

The ranked list then favours type diversity: a case's effective score decays
with the number of already selected cases of the same primary tag, and is
halved when it nearly duplicates a selected case.

v1.2 (after the client's second score: tag accuracy 0.79 -> 0.74, and "reduce
the volume of weak or overlapping cases"):
  - tags come only from detectors whose own members cover the case (member
    IoU >= tag_fit), and a typology that fits implies the general ones it is
    built from (SUBSUMES), which are then not submitted as the reason;
  - the type interleaving uses the submitted primary tag;
  - a near-duplicate of a chosen case is dropped instead of halved, and a case
    with open-set evidence only is submitted when two open-set detectors agree.

v1.3 (after the main-data validation labels):
  - within-detector percentiles are taken against every case assembled at the
    snapshot (`pool`), not only the period's cases, so a candidate is ranked
    against everything its detector finds;
  - a case kept by the period filter as a standing structure is down-weighted
    by its `window_weight`;
  - a cross-border corridor (GEOGRAPHY_LINKED_CONCENTRATION) implies the shared
    address and the common owner it is built from.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from .cases import Case, _iou, benign_weight

SUBMISSION_COLUMNS = ["case_rank", "member_record_ids", "typology_tags", "score", "reason_codes"]

# Label-free starting point: every detector gets the same prior, so the order
# comes only from within-detector strength, benign factors and interleaving.
DEFAULT_PRIOR = 0.5


def percentiles(cases: list[Case], min_pool: int = 5, pool: list[Case] | None = None) -> dict[int, float]:
    """Within-detector percentile of each candidate's raw score, over all cases
    (v1.3: over the cases of `pool` when given, the cases assembled at the snapshot).

    Detectors with fewer than `min_pool` candidates get 0.5: with one or two
    candidates the percentile would reflect pool size, not evidence strength.
    """
    by_det: dict[str, list] = {}
    for case in (pool if pool is not None else cases):
        for c in case.candidates:
            by_det.setdefault(c.detector, []).append(c.score)
    arrays = {d: np.sort(np.array(v)) for d, v in by_det.items()}
    out = {}
    for case in cases:
        for c in case.candidates:
            a = arrays.get(c.detector)
            out[id(c)] = (float(np.searchsorted(a, c.score, side="right") / len(a))
                          if a is not None and len(a) >= max(min_pool, 2) else 0.5)
    return out


def corroboration_values(per_cand: list[tuple], count_once: bool = True) -> list[float]:
    """Per-detector values entering the noisy-OR, strongest first.

    per_cand: (value, detector, edges) for each candidate. With count_once,
    a candidate's value is scaled by the share of its edges not already used by
    a stronger candidate, and each detector keeps its best scaled value.
    """
    used: set = set()
    best: dict[str, float] = {}
    for v, det, edges in sorted(per_cand, key=lambda x: -x[0]):
        if count_once and edges:
            new = sum(e not in used for e in edges) / len(edges)
            used.update(edges)
            v = v * new
        best[det] = max(best.get(det, 0.0), v)
    return sorted(best.values(), reverse=True)


def calibrate_cases(cases: list[Case], hit_rates: dict | None = None, prior: float = DEFAULT_PRIOR,
                    per_case: list[dict] | None = None, corroboration: float = 0.5,
                    detector_priors: dict | None = None, count_once: bool = True, min_pool: int = 5,
                    benign_per_item: float = 0.7, benign_floor: float = 0.3, tag_fit: float | None = None,
                    pool: list[Case] | None = None) -> None:
    """Set `calibrated` and `tag_scores` on every case. `per_case` (aligned with cases) overrides `hit_rates`."""
    hit_rates = hit_rates or {}
    pct = percentiles(cases, min_pool, pool)
    for i, case in enumerate(cases):
        table = per_case[i] if per_case is not None else {**(detector_priors or {}), **hit_rates}
        per_det: dict[str, tuple] = {}
        per_cand = []
        case.tag_scores = {}
        case.fit_tag_scores = {}   # v1.2: tag evidence from candidates that cover the case
        for c in case.candidates + case.support:
            w = benign_weight(c.benign, benign_per_item, benign_floor)
            v = table.get(c.detector, prior) * (0.5 + 0.5 * pct.get(id(c), 0.5)) * w
            for t in c.tags:
                case.tag_scores[t] = max(case.tag_scores.get(t, 0.0), v)
            if c in case.support:
                continue
            if tag_fit is not None and _iou(c.members, case.members) >= tag_fit:
                for t in c.tags:
                    case.fit_tag_scores[t] = max(case.fit_tag_scores.get(t, 0.0), v)
            per_cand.append((v, c.detector, c.edges))
            if v > per_det.get(c.detector, (0.0, 1.0))[0]:
                per_det[c.detector] = (v, w)
        best, best_w = max(per_det.values())
        lead = max((c for c in case.candidates if c.detector in per_det and per_det[c.detector][0] == best),
                   key=lambda c: c.score)
        case.lead_detector, case.lead_raw_score = lead.detector, float(lead.score)
        vals = corroboration_values(per_cand, count_once)
        noisy_or = 1 - float(np.prod([1 - v for v in vals]))
        case.calibrated = best + corroboration * (noisy_or - best)
        case.benign_factor = best_w


def size_weight(n: int, max_size: int | None, power: float = 1.0) -> float:
    """v1.1: a case larger than `max_size` members is down-weighted by (max_size / n) ** power.

    A review case is what one analyst can check; a portfolio of dozens of vendors
    is a hub to be explained, not a case. Signals are never removed."""
    if not max_size or n <= max_size:
        return 1.0
    return (max_size / n) ** power


def order(cases: list[Case], interleave: bool = True, decay: float = 0.85, dup_penalty: float = 0.5,
          dup_days: int = 365, contained: float = 0.5, max_cases: int | None = None,
          drop_duplicates: bool = False, dropped: list | None = None) -> list[Case]:
    remaining = sorted(cases, key=lambda c: (-c.calibrated, sorted(c.members)))
    if not interleave:
        for c in remaining:
            c.effective = c.calibrated
        return remaining[:max_cases] if max_cases else remaining
    def same_time(c: Case, s: Case) -> bool:
        return c.emerged_at is None or s.emerged_at is None or abs((c.emerged_at - s.emerged_at).days) <= dup_days

    def near_duplicate(c: Case, s: Case) -> bool:
        return _iou(c.members, s.members) >= 0.5 and same_time(c, s)

    def duplicates(c: Case, s: Case) -> bool:
        # Near-duplicates and fragments of a chosen case add little for a reviewer
        # (and cannot match a second planted case under one-to-one matching).
        return ((_iou(c.members, s.members) >= 0.5 or len(c.members & s.members) >= contained * len(c.members)) and
                same_time(c, s))

    chosen: list[Case] = []
    tag_count: dict[str, int] = {}
    dup = [False] * len(remaining)   # duplicates some chosen case (updated as cases are chosen)
    limit = max_cases or len(remaining)
    while remaining and len(chosen) < limit:
        best_i, best_v = 0, -1.0
        for i, c in enumerate(remaining):
            v = c.calibrated * decay ** tag_count.get(c.primary_tag, 0)
            if dup[i]:
                v *= dup_penalty
            if v > best_v:
                best_i, best_v = i, v
        c = remaining.pop(best_i)
        dup.pop(best_i)
        c.effective = best_v
        chosen.append(c)
        tag_count[c.primary_tag] = tag_count.get(c.primary_tag, 0) + 1
        dup = [d or duplicates(r, c) for d, r in zip(dup, remaining)]
        if drop_duplicates:   # v1.2: a near-duplicate of a chosen case is dropped (a fragment is still halved)
            gone = [near_duplicate(r, c) for r in remaining]
            if dropped is not None:
                dropped.extend((r, c) for r, g in zip(remaining, gone) if g)
            remaining = [r for r, g in zip(remaining, gone) if not g]
            dup = [d for d, g in zip(dup, gone) if not g]
    return chosen


# v1.2: a typology that fits the case implies the general typologies it is built from. The general tag is then not
# the reason for the case, and is not submitted: a dense cluster is made of shared owners, addresses, sponsors and
# approvers; an ownership change brings vendors under the new owner; an ownership-operational overlap is a common
# owner plus a shared address or sponsor; a rapid onboarding, a synchronised spend spike, a market pocket, a
# sponsor-approver coupling and a public-institution expert cluster are each defined on one sponsor's vendors.
SUBSUMES = {
    "DENSE_MULTI_RELATION_CLUSTER": ("SHARED_ADDRESS", "SPONSOR_CONCENTRATION", "APPROVER_CONCENTRATION",
                                     "SPONSOR_APPROVER_COUPLING", "COMMON_BENEFICIAL_OWNER",
                                     "OWNERSHIP_OPERATIONAL_OVERLAP", "RAPID_ONBOARDING_LINKED"),
    "OWNERSHIP_CHANGE_STRUCTURAL_SHIFT": ("COMMON_BENEFICIAL_OWNER", "OWNERSHIP_OPERATIONAL_OVERLAP"),
    "OWNERSHIP_OPERATIONAL_OVERLAP": ("SHARED_ADDRESS", "SPONSOR_CONCENTRATION"),
    "RAPID_ONBOARDING_LINKED": ("SPONSOR_CONCENTRATION",),
    "ACTIVITY_TIMING_CONCENTRATION": ("SPONSOR_CONCENTRATION",),
    # v1.3: a cross-border corridor is a shared foreign address of commonly owned vendors (VAL-24 is labelled
    # GEOGRAPHY_LINKED_CONCENTRATION alone); a market pocket is defined on new vendors, which a sponsor brings
    "GEOGRAPHY_LINKED_CONCENTRATION": ("SPONSOR_CONCENTRATION", "SHARED_ADDRESS", "COMMON_BENEFICIAL_OWNER",
                                       "OWNERSHIP_OPERATIONAL_OVERLAP"),
    "SPONSOR_APPROVER_COUPLING": ("SPONSOR_CONCENTRATION",),
    "PUBLIC_INSTITUTION_EXPERT_CLUSTER": ("SPONSOR_CONCENTRATION",),
}
# Tag pairs submitted together (the overlap is a common owner plus operations; the coupling a pair at a concentrated
# approver): the more specific one leads, so it sets the primary tag and the first reason.
LEADS = {"OWNERSHIP_OPERATIONAL_OVERLAP": "COMMON_BENEFICIAL_OWNER", "SPONSOR_APPROVER_COUPLING": "APPROVER_CONCENTRATION"}


def select_tags(case: Case, ratio: float = 0.85, fit: bool = False) -> tuple:
    """Tags whose strongest evidence is within `ratio` of the case's strongest, strongest first.

    OPEN_SET is a fallback label: it is dropped when a typed tag remains.
    v1.2 (`fit`): only typed tags from candidates that cover the case count, the tags a fitting typology
    subsumes are dropped, and the ratio is taken among the rest.
    """
    from .detectors.base import OPEN_SET_TAG
    if fit:
        typed = {t: v for t, v in getattr(case, "fit_tag_scores", {}).items() if t != OPEN_SET_TAG}
        if typed:
            implied = {g for t in typed for g in SUBSUMES.get(t, ())}
            typed = {t: v for t, v in typed.items() if t not in implied}
            best = max(typed.values())
            sel = sorted((t for t, v in typed.items() if v >= ratio * best), key=lambda t: (-typed[t], t))
            for a, b in LEADS.items():
                if a in sel and b in sel and sel.index(a) > sel.index(b):
                    sel.remove(a)
                    sel.insert(sel.index(b), a)
            return tuple(sel)
    ts = case.tag_scores or {t: 1.0 for t in case.tags}
    best = max(ts.values())
    keep = sorted((t for t, v in ts.items() if v >= ratio * best), key=lambda t: -ts[t])
    typed = [t for t in keep if t != OPEN_SET_TAG]
    return tuple(typed or keep)


def short_reason(case: Case, tags: tuple = (), max_items: int = 3, max_len: int = 300) -> str:
    """A one-line reason for the submission: the first clause of the strongest candidates' reasons.
    The full reasons stay in ranked_cases.csv and the case packages.
    v1.2: the reasons behind the submitted tags come first, the primary tag's first, so the reason
    matches the tag."""
    def first(c):
        if tags and tags[0] in c.tags:
            return (0, -_iou(c.members, case.members))
        if set(c.tags) & set(tags):
            return (1, -_iou(c.members, case.members))
        return (2, float(c.detector != case.lead_detector))
    seen, parts = set(), []
    for c in sorted(case.candidates, key=first):
        if c.detector in seen:
            continue
        seen.add(c.detector)
        parts.append(c.reason.split(";")[0].strip())
        if len(parts) == max_items:
            break
    text = "; ".join(parts)
    if case.benign:
        text += f"; {len(case.benign)} benign explanation(s) to check"
    return text if len(text) <= max_len else text[: max_len - 3].rstrip() + "..."


def to_submission(ordered: list[Case], tag_ratio: float = 0.85) -> pd.DataFrame:
    """Submission rows. `score` is the effective score that set the rank, so it never
    increases down the list; `case_score` is the case's own score before type
    interleaving and the duplicate penalty."""
    rows = []
    for i, c in enumerate(ordered, start=1):
        tags = c.selected_tags or select_tags(c, tag_ratio)
        rows.append({
            "case_rank": i,
            "member_record_ids": "|".join(sorted(c.members)),
            "typology_tags": "|".join(tags),
            "score": round(float(c.effective), 4),
            "reason_codes": short_reason(c, tags),
            "case_score": round(float(c.calibrated), 4),
            "all_tags": "|".join(c.tags),
            "detectors": "|".join(c.detectors),
            "lead_detector": c.lead_detector,
            "lead_raw_score": round(c.lead_raw_score, 3),
            "emerged_at": c.emerged_at.date().isoformat() if c.emerged_at is not None else "",
            "window_basis": getattr(c, "window_basis", ""),
            "benign_factor": round(c.benign_factor, 3),
            "reason_full": c.reason,
        })
    return pd.DataFrame(rows, columns=SUBMISSION_COLUMNS + [
        "case_score", "all_tags", "detectors", "lead_detector", "lead_raw_score", "emerged_at", "window_basis",
        "benign_factor", "reason_full"])


def rank(cases: list[Case], cfg: dict | None = None, hit_rates: dict | None = None,
         per_case: list[dict] | None = None, pool: list[Case] | None = None) -> pd.DataFrame:
    """Rank the cases. `pool` (v1.3): the cases assembled at the snapshot, for the within-detector percentiles."""
    from .detectors.base import OPEN_SET_TAG
    cfg = cfg or {}
    calibrate_cases(cases, hit_rates, cfg.get("prior", DEFAULT_PRIOR), per_case, cfg.get("corroboration", 0.5),
                    cfg.get("detector_priors"), cfg.get("count_evidence_once", True), cfg.get("min_pool", 5),
                    cfg.get("benign_per_item", 0.7), cfg.get("benign_floor", 0.3), cfg.get("tag_fit"), pool)
    for c in cases:
        c.calibrated *= size_weight(len(c.members), cfg.get("max_size"), cfg.get("size_power", 1.0))
        c.calibrated *= getattr(c, "window_weight", 1.0)   # v1.3: standing structures rank behind
        c.selected_tags = select_tags(c, cfg.get("tag_ratio", 0.85), cfg.get("tag_fit") is not None)
    weak = []
    k = cfg.get("open_set_min_detectors")
    if k:   # v1.2: open-set evidence alone (no typology) needs k open-set detectors to agree
        is_weak = [all(set(x.tags) <= {OPEN_SET_TAG} for x in c.candidates + c.support) and len(c.detectors) < k
                   for c in cases]
        weak = [c for c, w in zip(cases, is_weak) if w]
        cases_in = [c for c, w in zip(cases, is_weak) if not w]
    else:
        cases_in = cases
    dups: list = []
    ordered = order(cases_in, interleave=cfg.get("interleave", True), decay=cfg.get("decay", 0.85),
                    dup_penalty=cfg.get("dup_penalty", 0.5), contained=cfg.get("contained", 0.5),
                    max_cases=cfg.get("max_cases"), drop_duplicates=cfg.get("drop_duplicates", False), dropped=dups)
    for c in cases:
        c.rank = None
    for i, c in enumerate(ordered, start=1):
        c.rank = i
    sub = to_submission(ordered, cfg.get("tag_ratio", 0.85))
    # v1.2: the cases left out, with why (results only, not the deliverable)
    sub.dropped = pd.DataFrame(
        [{"member_record_ids": "|".join(sorted(c.members)), "detectors": "|".join(c.detectors),
          "why": "open-set evidence from one detector only", "duplicate_of_rank": None, "case_score": round(c.calibrated, 4)}
         for c in weak]
        + [{"member_record_ids": "|".join(sorted(c.members)), "detectors": "|".join(c.detectors),
            "why": "near-duplicate of a higher case (member IoU >= 0.5)", "duplicate_of_rank": s.rank,
            "case_score": round(c.calibrated, 4)} for c, s in dups],
        columns=["member_record_ids", "detectors", "why", "duplicate_of_rank", "case_score"])
    return sub

In [29]:
%%module meridian.calibrate
"""Detector hit-rate calibration from labelled cases.

Reads labels, so like scorer.py it sits on the evaluation side: nothing here
feeds candidate generation.

A case is a hit when some planted case overlaps it at the scorer threshold
(rank-free). Hit rates are Beta-smoothed per lead detector.

`crossfit` avoids in-sample optimism when calibrating and evaluating on the
same labels: planted cases are split into folds; a case that matches a planted
case in fold f is scored with rates estimated without fold f, and all other
cases with the mean over folds.
"""
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

from .cases import Case
from .scorer import ScorerConfig, overlap


def _best_match(case: Case, truth: pd.DataFrame, cfg: ScorerConfig):
    best, best_id = 0.0, None
    for t in truth.itertuples():
        o = overlap(case.members, t.members, cfg.measure)
        if o > best:
            best, best_id = o, t.case_id
    return best_id if best >= cfg.threshold else None


def _rates(cases, matched, exclude: set, a: float, b: float) -> dict:
    counts: dict[str, list] = {}
    for c, m in zip(cases, matched):
        if m in exclude:
            continue
        h = counts.setdefault(c.lead.detector, [0, 0])
        h[0] += m is not None
        h[1] += 1
    return {d: (h + a) / (n + a + b) for d, (h, n) in counts.items()}


def fit(cases: list[Case], truth: pd.DataFrame, cfg: ScorerConfig | None = None, a: float = 1.0,
        b: float = 1.0) -> dict:
    cfg = cfg or ScorerConfig()
    matched = [_best_match(c, truth, cfg) for c in cases]
    return _rates(cases, matched, set(), a, b)


def crossfit(cases: list[Case], truth: pd.DataFrame, cfg: ScorerConfig | None = None, k: int = 3,
             seed: int = 0, a: float = 1.0, b: float = 1.0) -> list[dict]:
    """Per-case hit-rate tables (one dict per case, aligned with `cases`)."""
    cfg = cfg or ScorerConfig()
    matched = [_best_match(c, truth, cfg) for c in cases]
    ids = np.array(sorted(truth["case_id"]))
    rng = np.random.default_rng(seed)
    folds = np.array_split(rng.permutation(ids), k)
    fold_of = {cid: f for f, fold in enumerate(folds) for cid in fold}
    tables = [_rates(cases, matched, set(folds[f]), a, b) for f in range(k)]
    detectors = {c.lead.detector for c in cases}
    mean_table = {d: float(np.mean([t.get(d, a / (a + b)) for t in tables])) for d in detectors}
    return [tables[fold_of[m]] if m is not None else mean_table for m in matched]


def save(rates: dict, path: str | Path, meta: dict | None = None) -> None:
    Path(path).write_text(json.dumps({"hit_rates": rates, "meta": meta or {}}, indent=2, default=str))


def load(path: str | Path) -> dict:
    return json.loads(Path(path).read_text())["hit_rates"]

In [30]:
%%module meridian.run
"""Command-line entry point: one command from raw data to submission and local score.

    meridian --config configs/default.yaml [--out artifacts/runs/NAME] [--set key=value ...]

A config may start with `extends: <file>`: it then holds only the keys that differ
from that file (configs/validation.yaml and configs/test.yaml extend default.yaml).
"""
from __future__ import annotations

import argparse
import hashlib
import json
import platform
import subprocess
import time
from pathlib import Path

import pandas as pd
import yaml

from . import __version__
from .cases import assemble, in_window
from .checks import run_checks, to_markdown
from .context import Context
from .data import as_of, load_bundle
from .detectors import run_detectors
from .ranking import SUBMISSION_COLUMNS, rank

ROOT = Path.cwd()


# Keys that say which data to run on and which cases to keep; every other key is part of
# the method, which must not change between the validation and the test run.
DATA_KEYS = ("bundle", "labels", "snapshot", "window")


def _merge(base: dict, over: dict) -> dict:
    out = dict(base)
    for k, v in over.items():
        out[k] = _merge(out[k], v) if isinstance(v, dict) and isinstance(out.get(k), dict) else v
    return out


def load_config(path, overrides=()) -> dict:
    path = Path(path)
    with open(path) as fh:
        cfg = yaml.safe_load(fh) or {}
    parent = cfg.pop("extends", None)
    if parent:
        cfg = _merge(load_config(path.parent / parent), cfg)
    for item in overrides:
        key, val = item.split("=", 1)
        node = cfg
        parts = key.split(".")
        for p in parts[:-1]:
            node = node.setdefault(p, {})
        node[parts[-1]] = yaml.safe_load(val)
    return cfg


def method_hash(cfg: dict) -> str:
    """Hash of the method settings: the config without the data keys."""
    method = {k: v for k, v in cfg.items() if k not in DATA_KEYS}
    return hashlib.sha256(json.dumps(method, sort_keys=True, default=str).encode()).hexdigest()[:12]


def _git_commit() -> str:
    try:
        return subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, capture_output=True,
                              text=True, check=True).stdout.strip()
    except Exception:
        return "unknown"


def _calibration(cases, cfg: dict):
    """Return (hit_rates, per_case) for the configured calibration mode.

    prior     no labels: every detector gets the same prior
    file      hit rates fitted earlier and saved with calibrate.save
    insample  fit on the configured labels and rank with the same labels (optimistic)
    crossfit  fit on the labels, cross-fitted by planted case (for honest starter evaluation)
    """
    rcfg = cfg.get("ranking", {})
    mode = rcfg.get("calibration", "prior")
    if mode == "prior":
        return None, None
    from . import calibrate
    if mode == "file":
        return calibrate.load(ROOT / rcfg["calibration_file"]), None
    from .scorer import ScorerConfig, load_labels
    truth = load_labels(ROOT / cfg["labels"])
    scfg = ScorerConfig(**cfg.get("scorer", {}))
    if mode == "insample":
        return calibrate.fit(cases, truth, scfg), None
    if mode == "crossfit":
        return None, calibrate.crossfit(cases, truth, scfg, k=rcfg.get("folds", 3), seed=cfg.get("seed", 0))
    raise ValueError(f"unknown calibration mode {mode}")


def pipeline(cfg: dict, bundle=None):
    """Run detection end to end. Returns (submission, cases, candidates, context, checks).

    Source tables -> as_of(t) view -> normalisation and entity resolution -> canonical
    heterogeneous graph -> detectors. Detectors read only the graph (plus reference
    tables); `ctx.graph` is the graph they ran on.
    """
    b = bundle if bundle is not None else load_bundle(ROOT / cfg["bundle"])
    checks = run_checks(b)
    view = as_of(b, cfg["snapshot"])
    gcfg = cfg.get("graph", {})
    ctx = Context.for_detection(view, cfg, gcfg.get("drop_relations", ()), gcfg.get("drop_activity", False))
    cands = run_detectors(ctx, cfg.get("detectors", "all"), cfg.get("exclude_detectors", ()))
    joined = None
    if cfg.get("window_grown", True):
        v = ctx.vendors.assign(entity=ctx.vendors["record_id"].map(ctx.entity_of))
        joined = dict(zip(v["record_id"], v.groupby("entity")["onboarding_date"].transform("min")))
    ctx.policy_sub = None
    if not cfg.get("ranking", {}).get("include_policy", True):
        from .detectors.base import POLICY_TAG
        policy = [c for c in cands if POLICY_TAG in c.tags]
        cands = [c for c in cands if POLICY_TAG not in c.tags]
        # v1.1: the single-record policy findings, scored separately by the sponsor (Dossier §4),
        # are ranked on their own: by within-detector strength, no interleaving, no merging into cases.
        if cfg["ranking"].get("policy_list", False):
            pcases = in_window(assemble(policy, merge_iou=1.0, union_iou=1.1), cfg.get("window"), joined)
            ctx.policy_sub = rank(pcases, {"interleave": False}) if pcases else pd.DataFrame(columns=SUBMISSION_COLUMNS)
    all_cases = assemble(cands, **cfg.get("cases", {}))
    # v1.3: a case is also in the period when a covering finding emerged in it, or (down-weighted) when it still stands
    cases = in_window(all_cases, cfg.get("window"), joined, cfg.get("window_findings"), cfg.get("window_standing"),
                      cfg.get("window_finding_weight", 1.0))
    ctx.all_cases = all_cases   # every case assembled at the snapshot, before the period filter
    hit_rates, per_case = _calibration(cases, cfg)
    pool = all_cases if cfg.get("ranking", {}).get("percentile_pool") == "snapshot" else None
    sub = rank(cases, cfg.get("ranking", {}), hit_rates, per_case, pool=pool)
    return sub, cases, cands, ctx, checks


def evaluate(sub: pd.DataFrame, cands, cfg: dict) -> dict:
    from .scorer import ScorerConfig, coverage, load_labels, prepare_submission, score

    truth = load_labels(ROOT / cfg["labels"])
    scfg = ScorerConfig(**cfg.get("scorer", {}))
    res = score(prepare_submission(sub[SUBMISSION_COLUMNS]), truth, scfg)
    cov = coverage([c.members for c in cands], truth, scfg)
    return {"score": res, "coverage": cov}


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--config", default=str(ROOT / "configs" / "default.yaml"))
    ap.add_argument("--out", default=None, help="output directory (default artifacts/runs/<timestamp>)")
    ap.add_argument("--set", action="append", default=[], help="override a config key, e.g. features.ownership=false")
    ap.add_argument("--no-score", action="store_true", help="skip the local scorer")
    ap.add_argument("--packages", type=int, default=10, help="write analyst case packages for the top N cases")
    a = ap.parse_args(argv)

    cfg = load_config(a.config, a.set)
    t0 = time.time()
    sub, cases, cands, ctx, checks = pipeline(cfg)
    runtime = time.time() - t0

    out = Path(a.out) if a.out else ROOT / "artifacts" / "runs" / time.strftime("%Y%m%d-%H%M%S")
    out.mkdir(parents=True, exist_ok=True)
    sub[SUBMISSION_COLUMNS].to_csv(out / "submission.csv", index=False)
    sub.to_csv(out / "ranked_cases.csv", index=False)
    (out / "checks.md").write_text(to_markdown(checks))
    # Resolution lists for manual review.
    ctx.reg_matches.to_csv(out / "registry_matches.csv", index=False)
    ctx.identity_pairs.to_csv(out / "identity_pairs.csv", index=False)
    ctx.entity_map[ctx.entity_map["n_records"] > 1].to_csv(out / "resolved_entities.csv", index=False)
    with open(out / "cases.jsonl", "w") as fh:
        for c in cases:
            fh.write(json.dumps({
                "members": sorted(c.members), "tags": list(c.tags), "score": c.score,
                "emerged_at": c.emerged_at.date().isoformat() if c.emerged_at is not None else None,
                "detectors": list(c.detectors), "reason": c.reason, "benign": c.benign,
                "calibrated": c.calibrated, "benign_factor": c.benign_factor, "edges": list(c.edges),
                "evidence": [cd.evidence for cd in c.candidates + c.support],
            }, default=str, sort_keys=True) + "\n")

    meta = {
        "version": __version__, "git_commit": _git_commit(), "method_hash": method_hash(cfg), "config": cfg,
        "input_hashes": ctx.b.input_hashes, "snapshot": str(ctx.t.date()),
        "runtime_sec": round(runtime, 2), "python": platform.python_version(),
        "pandas": pd.__version__, "n_candidates": len(cands), "n_cases": len(cases),
        "n_submitted": len(sub),
        "checks": checks["status"].value_counts().to_dict(),
    }
    print(f"candidates={len(cands)} cases={len(cases)} submitted={len(sub)} runtime={runtime:.1f}s -> {out}")

    # The canonical heterogeneous graph behind this run, with its content hash.
    from .graph import export, validate
    g = ctx.graph
    meta["graph"] = export(g, out / "graph")
    problems = validate(g)
    meta["graph"]["problems"] = problems
    print(f"graph {meta['graph']['graph_hash']}: {len(g.nodes)} nodes, {len(g.edges)} edges"
          + (f"; problems: {problems}" if problems else "") + f" -> {out / 'graph'}")

    if a.packages:
        from .explain import config_hash, write_packages
        ordered = sorted((c for c in cases if c.rank is not None), key=lambda c: c.rank)
        pk = write_packages(ctx, ordered, out / "cases", {
            "snapshot": str(ctx.t.date()), "run": out.name, "config_hash": config_hash(cfg),
            "git_commit": meta["git_commit"], "input_hashes": ctx.b.input_hashes,
            "graph_hash": meta["graph"]["graph_hash"]}, top=a.packages)
        print(f"case packages for the top {min(a.packages, len(ordered))} cases -> {pk}")

    if not a.no_score and cfg.get("labels"):
        from .scorer import summary_line
        ev = evaluate(sub, cands, cfg)
        res = ev["score"]
        cov = ev["coverage"]
        cov.to_csv(out / "coverage.csv", index=False)
        res.pop("_matches").to_csv(out / "matches.csv", index=False)
        meta["score"] = res
        meta["coverage"] = float(cov["covered"].mean())
        print(f"[dev scorer, {Path(cfg['labels']).name}]", summary_line(res))
        print(f"candidate coverage={cov['covered'].mean():.3f} "
              f"({int(cov['covered'].sum())}/{len(cov)} planted cases reachable by some candidate)")
        from .scorer import load_labels, shortcut_checks
        sc = shortcut_checks(load_labels(ROOT / cfg["labels"]), ctx.b)
        sc.to_csv(out / "shortcuts.csv", index=False)
        meta["shortcuts"] = int(sc["all_planted"].sum())
    (out / "run.json").write_text(json.dumps(meta, indent=2, default=str))


if __name__ == "__main__":
    main()

### 0.1 Method settings

The detectors, thresholds, case assembly and ranking. Data paths are set in the next cell; the method hash covers everything else, so a changed setting gives a different hash.

In [31]:
CONFIG_YAML = r'''
# Default run configuration. Paths are relative to the repository root.
bundle: student_bundle
labels: student_bundle/labelled_cases_TRAINING_SET.parquet   # read by the scorer and calibrate only
snapshot: "2026-06-30"
# Only cases emerging inside this window are submitted. null = no filter
# (the starter labels carry no emergence dates, so starter runs use null).
# Validation and test runs set it in configs/validation.yaml and configs/test.yaml.
window: null
window_grown: true       # also keep older cases that gained a vendor entity first onboarded in the window,
window_findings: 0.5     # v1.3: ...or in which a typed finding covering the case (member IoU >= 0.5) emerged
window_finding_weight: 0.5  # v1.3: ...ranked with weight 0.5 (the structure itself is older than the window),
window_standing: 0.1     # v1.3: ...or whose covering finding still holds at the window start (weight 0.1)
seed: 0                  # Louvain communities and isolation forests; see stability.md for other seeds

scorer:
  measure: iou
  threshold: 0.5

features:
  ownership: true        # switch for the ownership layer (registry R)
  open_set: true         # generic detectors for patterns outside the 14 tags

detectors: all           # or a list of detector names

cases:
  merge_iou: 0.8         # same members: merge, keep the boundary
  union_iou: 0.5         # overlapping groups from different detectors: merge into the union
  max_growth: 1.5        # ...as long as the union stays within 1.5x the lead candidate
  evidence_union: null   # e.g. 0.8: ...or when they share this much of the smaller evidence-edge set
  merge_days: 365        # only when they emerged within this many days of each other
  support_days: 365
  best_match: true       # join the qualifying case with the highest member IoU, not the first
  open_set_grows: false  # open-set candidates corroborate a case but never add members
  typed_first: true      # ...and never set its boundary: they are visited after typed candidates
  single_into_pair: true # a one-vendor case inside exactly one two-vendor case of the same time joins it

ranking:
  max_cases: null        # submit every case assembled (no truncation); section 7 compares with the first 100
  calibration: prior     # prior | file | insample | crossfit
  calibration_file: artifacts/calibration.json
  folds: 3
  prior: 0.5
  corroboration: 0.5     # share of the noisy-OR gain credited for other detectors on the same case
  count_evidence_once: true  # other detectors corroborate only with evidence edges not already counted
  min_pool: 5            # detectors with fewer candidates get percentile 0.5
  detector_priors:       # label-free design choice: open-set cases fill in behind typed ones
    rare_sharing_community: 0.3
    entity_change_point: 0.3
    duplicate_paid: 0.3
    entity_outlier: 0.3
    approver_hub: 0.3    # v1.3: approver groups without sponsor coupling (meridian.detectors.approval)
    portfolio_spikes: 0.3  # v1.3: spike groups that chance explains in a large sponsor portfolio (activity)
  percentile_pool: period    # v1.3 tested `snapshot` (percentiles against every case at the snapshot): no gain with the other v1.3 changes
  interleave: true
  tag_ratio: 0.8         # submit tags whose evidence is within this share of the strongest (v1.2: was 0.85)...
  tag_fit: 0.75          # ...from detectors whose own members cover the case (member IoU >= 0.75), most specific first (v1.2)
  decay: 0.95            # effective score x decay^(cases already chosen with the same primary tag) (v1.1: was 0.85)
  dup_penalty: 0.5       # x this when a case nearly duplicates a chosen one (IoU >= 0.5)...
  contained: 0.5         # ...or when half its members already sit in a chosen case (a 2-vendor case sharing one)
  drop_duplicates: true  # v1.2: a near-duplicate (IoU >= 0.5) is dropped instead; a fragment is still halved
  open_set_min_detectors: 2  # v1.2: a case with open-set evidence only is submitted when two open-set detectors agree
  include_policy: false  # v1.1 (was true): single-record policy findings leave the main queue...
  policy_list: true      # ...and are submitted as their own ranked list
  max_size: 15           # v1.1: cases above 15 members are down-weighted by (15 / n) ** size_power
  size_power: 1.0
  benign_per_item: 0.5   # v1.1 (was 0.7): weight per documented benign explanation...
  benign_floor: 0.15     # ...down to this floor (was 0.3)

# Detector parameters. Ratio thresholds (*_quantile) are quantiles of each dataset's own peer
# distribution; the others are fixed rules (see "Thresholds" in reports/results.md).
params:
  shared_address: {min_entities: 3, peer_quantile: 0.95, min_peers: 5}
  sponsor: {window_days: 60, min_new: 4, ratio_quantile: 0.95}
  approval: {min_pair: 5, min_sponsor_share: 0.5, min_eligible: 3}
  approver_star: {min_approvals: 5, surprise_quantile: 0.95, max_members: 15, min_split_records: 3}  # v1.1: split by sponsor above 15
  public_institution: {min_experts: 3, window_days: 90}   # v1.2; v1.3: variant 14+3 only
  dense_cluster: {min_pair_types: 4, min_size: 4, min_cover: 0.8, min_covered_types: 4}
  rapid_onboarding: {window_days: 14, min_linked: 4}
  sponsor_burst: {window_days: 14, min_new: 4, max_p: 0.001, min_rate: 0.1}   # v1.3
  bridge: {min_foreign_links: 2, min_communities: 3, peer_quantile: 0.90, min_peers: 5}
  geography: {window_days: 365, min_new: 4, ratio_quantile: 0.90}
  corridor: {min_entities: 3}   # v1.3: vendors of one market sharing an address abroad, in their remit country
  activity: {spike_quantile: 0.85, min_vendors: 3, post_termination_min_months: 1, max_p: 0.001}  # v1.3: max_p
  common_owner: {min_entities: 2}
  layered: {min_layers: 4, peer_quantile: 0.95, min_peers: 5}
  ownership_change: {window_days: 90, min_signals: 2}
  fmv: {band_map: {}}
  compliance: {grace_days: 30}
  open_set: {resolutions: [3, 10, 30], resolution_unit: 1000, min_size: 3, max_size: 15, min_cover: 0.6, min_types: 2,
             change_window_days: 90, min_change_kinds: 3, outlier_top_share: 0.02}
'''

from meridian.data import load_bundle
from meridian.graph import export, validate
from meridian.ranking import SUBMISSION_COLUMNS
from meridian.resolution import name_match_check
from meridian.run import method_hash, pipeline
from meridian.scorer import PERIODS, ScorerConfig, case_periods, load_labels, overlap, prepare_submission, score

DATA_DIR = "student_bundle"             # the data; point this at the main data when it is released
LABELS_FILE = f"{DATA_DIR}/labelled_cases_TRAINING_SET.parquet"
VERSION = "v1.3"                        # method version (README.md); outputs go to results/graph/<VERSION>/
OUT = ROOT / "results" / "graph" / VERSION      # metrics, case landing, graph export
OUT.mkdir(parents=True, exist_ok=True)
CASE_DIR = OUT / "cases"                        # every ranked case file, in the submission format
(CASE_DIR / "starter").mkdir(parents=True, exist_ok=True)
DELIVERABLE = ROOT / "output" / VERSION / "submission_TEST.csv"  # the client files (section 8), one folder per version

CFG = yaml.safe_load(CONFIG_YAML)
CFG["bundle"] = DATA_DIR
BUNDLE = load_bundle(ROOT / DATA_DIR)
LABELS = load_labels(ROOT / LABELS_FILE)        # read by the scorer only, never by detection
SCORER = ScorerConfig(**CFG["scorer"])
T0 = time.time()
print(f"{VERSION}, method hash {method_hash(CFG)}; {len(BUNDLE.third_parties):,} vendor records; {len(LABELS)} labelled cases; "
      f"scorer: {SCORER.measure} >= {SCORER.threshold}")

v1.3, method hash 80e0a80a439a; 1,002 vendor records; 33 labelled cases; scorer: iou >= 0.5


In [32]:
PERIOD = case_periods(LABELS, BUNDLE.third_parties).set_index("case_id")
CASES_IN = {p: list(PERIOD.index[PERIOD.period == p]) for p in PERIODS}
display(PERIOD.assign(tags=LABELS.set_index("case_id").typology_tags).reset_index())
print({p: len(c) for p, c in CASES_IN.items()})

frozen = ROOT / "results" / "layer1_frozen" / "case_periods.csv"
if frozen.exists():   # the split used by Layers 1 and 2
    other = pd.read_csv(frozen).set_index("case_id").period
    print("same periods as results/layer1_frozen/case_periods.csv:", bool((other.reindex(PERIOD.index) == PERIOD.period).all()))

,case_id,emerged,period,basis,tags
0,CASE-0001,2019-12-15,TRAINING,last member onboarded,SHARED_ADDRESS
1,CASE-0002,2022-02-07,TRAINING,last member onboarded,SHARED_ADDRESS
2,CASE-0003,2025-03-14,VALIDATION,last member onboarded,SHARED_ADDRESS
3,CASE-0004,2019-10-29,TRAINING,last member onboarded,SPONSOR_CONCENTRATION
4,CASE-0005,2020-11-13,TRAINING,last member onboarded,SPONSOR_CONCENTRATION
5,CASE-0006,2026-06-30,TEST,last member onboarded,APPROVER_CONCENTRATION|SPONSOR_APPROVER_COUPLING
6,CASE-0007,2026-02-10,TEST,last member onboarded,APPROVER_CONCENTRATION|SPONSOR_APPROVER_COUPLING
7,CASE-0008,2025-11-16,TEST,last member onboarded,DENSE_MULTI_RELATION_CLUSTER
8,CASE-0009,2020-04-14,TRAINING,last member onboarded,DENSE_MULTI_RELATION_CLUSTER
9,CASE-0010,2025-10-02,TEST,last member onboarded,RAPID_ONBOARDING_LINKED


{'TRAINING': 16, 'VALIDATION': 7, 'TEST': 10}
same periods as results/layer1_frozen/case_periods.csv: True


## 2. The graph at the snapshot

Normalisation and entity resolution read the source tables to build the graph; after that, every detector reads only the graph plus reference tables (service taxonomy, approval policy, FMV rate cards, country context, business events).
- **Nodes:** vendor records, resolved entities, addresses, contact points, declared groups, engagements, employees, service categories, institutions, registry legal entities, natural persons, accounting parents.
- **Edges** keep their valid time, observation time, source table and record; edges that depend on entity resolution keep the match confidence. Uncertain matches stay as candidates.
- **Monthly payments** are an activity table on engagement nodes (a time series, not a relation).
- **Ultimate owners** (Dossier §7.3–7.4) are a derived edge `ULTIMATE_OWNER`, vendor → natural person: the OWNS/CONTROLS paths at the snapshot that end in a person, with the indirect share (shares multiplied along a path, summed over paths; empty when a hop is control without a share), the path depths and every statement behind it. Each path's terminal reason (natural person, owner stated unknown, listed exemption, no owner recorded, accounting parent only, cycle, depth cap) is kept in `ownership_paths.parquet`; a missing owner is never read as concealment. Detection does not read these edges: the ownership detectors walk the paths themselves over each vendor's history, which a snapshot edge cannot hold (one starter vendor was layered through a trust until 2019, then sold).
- **Not made into edges:** markets, paying entities and roles. A market or paying entity would be a hub linking every vendor booked or paid there, which is not evidence of a link between them; they stay attributes (engagement market, role home market and scope) and the payments table, where the detectors already use them (approver scope, foreign payer share).
- The same inputs and configuration give the same graph hash.

The next cell runs the whole pipeline at the snapshot (2026-06-30, no period filter); sections 2–5 read its outputs.

In [33]:
SUB_ALL, CASES_ALL, CANDS_ALL, CTX_ALL, CHECKS = pipeline(CFG, BUNDLE)
G = CTX_ALL.graph
info = export(G, OUT / "graph")
print(f"graph hash {info['graph_hash']}: {len(G.nodes):,} nodes, {len(G.edges):,} edges, "
      f"{len(G.activity):,} activity rows; validation problems: {validate(G) or 'none'}")
nodes = G.nodes.node_type.value_counts().rename("nodes")
edges = G.edges.relation_type.value_counts().rename("edges")
types = pd.concat([nodes.reset_index(), edges.reset_index()], axis=1)
types["nodes"] = types["nodes"].astype("Int64").astype(str).replace("<NA>", "")
display(types.fillna(""))
uo = G.edges[G.edges.relation_type == "ULTIMATE_OWNER"]
share_known = uo.relation_properties.map(lambda p: "indirect_share" in json.loads(p)).mean()
print(f"ULTIMATE_OWNER (derived): {len(uo)} edges for {uo.source_node_id.nunique()} vendors, "
      f"indirect share known for {share_known:.0%}")
print("where the ownership paths at the snapshot stop:",
      G.ownership_paths.terminal_reason.value_counts().to_dict())
print("data checks (Appendix B):", CHECKS.status.value_counts().to_dict())
display(CHECKS[CHECKS.status != "ok"])

graph hash 115402efb65ac068: 10,089 nodes, 20,828 edges, 34,467 activity rows; validation problems: none


,node_type,nodes,relation_type,edges
0,Engagement,2638,APPROVED_BY,3709
1,ContactPoint,1759,FOR_SERVICE,3601
2,Employee,1502,CONTRACTED_UNDER,2638
3,VendorRecord,1002,SPONSORED_BY,2466
4,Address,993,PROVIDES_SERVICE,2382
5,ResolvedEntity,980,HAS_CONTACT,2002
6,LegalEntity,508,LOCATED_AT,1091
7,NaturalPerson,373,PART_OF,1002
8,HealthcareInstitution,213,OWNS,568
9,AccountingParent,58,ULTIMATE_OWNER,383


ULTIMATE_OWNER (derived): 383 edges for 249 vendors, indirect share known for 97%
where the ownership paths at the snapshot stop: {'natural_person': 383, 'no_owner_recorded': 57, 'accounting_parent_only': 49, 'stated:exempt_listed': 28, 'stated:unknown_parent': 5}
data checks (Appendix B): {'ok': 78, 'warn': 7}


,area,check,status,value,detail
19,contract,null_rate:tp_engagements.sponsor_employee_id,warn,0.065,dictionary implies ~3%
20,contract,null_rate:tp_engagement_approvals.approver_employee_id,warn,0.137,dictionary implies ~5%
21,contract,null_rate:third_parties.lei,warn,0.673,dictionary implies ~75%
22,contract,null_rate:tp_address_history.address_type,warn,0.379,dictionary implies ~40%
57,referential,registry_L.child_lei in third_parties.lei,warn,75,children not in the vendor master
63,temporal,from<=to:third_parties,warn,1,onboarding_date <= terminated_date
78,contract,country_risk_context indices,warn,"BO_REGISTER_ACCESS,CPI,SANCTIONS",FSI and FATF expected when loaded


## 3. Entity resolution

- **Duplicate vendor records** are merged on a shared tax id, register number or LEI, or on the same name in the same country when an address or phone also agrees and no identifier conflicts.
- **Vendor → register:** register number, then LEI, then the transliterated name within the jurisdiction (accepted only when unique there). A fuzzy name fallback catches typos, truncations and abbreviations in the register; it is accepted only when the vendor's rarest word matches and the best name leads the next by 0.05.
- **Label-free check:** re-match the vendors already linked by an identifier using their name alone, and count how often the name picks the same entity.

In [34]:
m = CTX_ALL.reg_matches
display(m.groupby(["method", "status"]).size().rename("links").reset_index())
print("name matching against the identifier links:", name_match_check(BUNDLE))
em = CTX_ALL.entity_map
print(f"{len(em):,} records -> {em.entity_id.nunique():,} entities; {int((em.groupby('entity_id').size() > 1).sum())} merged")

,method,status,links
0,lei,accepted,65
1,name_fuzzy,accepted,7
2,name_jurisdiction,accepted,122
3,name_jurisdiction,candidate,3
4,person_name_nationality,accepted,17
5,registry_no,accepted,145


name matching against the identifier links: {'identifier_linked': 210, 'name_fuzzy_accepted': 9, 'name_fuzzy_agrees': 9, 'name_jurisdiction_accepted': 195, 'name_jurisdiction_agrees': 195, 'candidate_only': 4, 'no_name_match': 2}
1,002 records -> 980 entities; 22 merged


## 4. Detectors

Each detector looks for one typology as a pattern over typed relations (meta-paths) and compares it with peers. Thresholds that compare with peers are quantiles of the data's own distribution, so they are recomputed on new data without labels; the others are fixed rules from the typology definitions.

**Peer groups** (Dossier §3.7: a peer residual conditioned on role, market and category). A statistic is compared with the most specific group that is large enough, backing off to coarser ones: address type × country → subregion → region → type → all for addresses; party type × service-category parent × country → subregion → region → parent → type → all for vendors; role × scope × market → … → all for sponsors. A group needs at least 5 peers, and at least 1/tail peers when a tail share is tested (20 for a 5% tail), since a smaller group cannot show it. Where a threshold was already relative (bridge participation) it is taken within the peer group; where the dossier gives a fixed floor (three vendors at an address, layered depth) the floor stays, and a level that is common among peers becomes a benign explanation: it lowers the priority and never removes the signal.

| detector | typology | main statistic | peers / threshold |
|---|---|---|---|
| `shared_address` | SHARED_ADDRESS | vendor entities at one address at the same time | ≥ 3; benign when not in the top 5% of addresses of its type and country |
| `sponsor_concentration` | SPONSOR_CONCENTRATION | a sponsor's new vendors in 60 days ÷ peer median | peers: role × scope × market, backing off to all sponsors (≥ 5 others); top 5% |
| `sponsor_burst` (v1.3) | RAPID_ONBOARDING_LINKED | a sponsor's new vendors of one country within 14 days, against the sponsor's own rate there over the previous two years (Poisson) | ≥ 4 new vendors and a chance below 0.001 |
| `sponsor_approver_pair` | SPONSOR_APPROVER_COUPLING (+ APPROVER_CONCENTRATION) | binomial surprise of a pair's shared engagements against the approver's share of the market pool | ≥ 5 engagements, ≥ 50% of the sponsor's, ≥ 3 eligible approvers |
| `approver_star`, `approver_hub` | APPROVER_CONCENTRATION (+ SPONSOR_APPROVER_COUPLING) | an approver's approvals in a market and category against an even split among the approvers authorised (level and scope); a group without sponsor coupling is an `approver_hub` (v1.3) | top 5%; `approver_hub` ranks with the open-set prior 0.3 (v1.3) |
| `dense_cluster` | DENSE_MULTI_RELATION_CLUSTER | vendor pairs sharing ≥ 4 relation types, groups covering them | ≥ 4 members |
| `rapid_onboarding` | RAPID_ONBOARDING_LINKED | linked vendors onboarded within 14 days | ≥ 4 |
| `bridge` | BRIDGE_ENTITY | links to foreign institutions, or participation across institution communities (found without the vendor's own links) | ≥ 2 foreign links (benign when not in the top 10% of peer vendors); or participation in the top 10% of peer vendors and ≥ 3 communities |
| `geography`, `corridor` | GEOGRAPHY_LINKED_CONCENTRATION | new intermediaries in a market in 365 days ÷ earlier yearly rate; v1.3: vendors of one market sharing an address abroad in their remit-to country | top 10%; corridor ≥ 3 entities |
| `sync_spikes`, `post_termination` | ACTIVITY_TIMING_CONCENTRATION | related vendors whose spend peaks together; payments after termination | top 15% of peaks; v1.3: a group that chance explains among the sponsor's vendors (binomial, chance ≥ 0.001) is `portfolio_spikes`, prior 0.3 |
| `common_owner`, `ownership_overlap` | COMMON_BENEFICIAL_OWNER, OWNERSHIP_OPERATIONAL_OVERLAP | entities converging on one owner; plus shared operations | ≥ 2 entities; v1.3: benign note when the owner is also engaged as an individual expert and no sponsor engages both |
| `layered_ownership`, `opaque_routing`, `ownership_change` | LAYERED_COMPLEX_OWNERSHIP, OPAQUE_JURISDICTION_ROUTING, OWNERSHIP_CHANGE_STRUCTURAL_SHIFT | ownership chain depth and jurisdictions; offshore routing; changes after a new owner | fixed rules; layered benign when its complexity is not in the top 5% of peer vendors |
| `compliance_review` | POLICY_VIOLATION (scored separately, Dossier §4) | engagements in categories the policy matrix flags for a compliance review (by value band) with no review 30 days after the start, or a review dated after it | ≥ 1; also corroborates sponsor–approver pairs and approver stars |
| open-set detectors | (none) | communities of rare shared relations, change points, entity outliers | prior 0.3: they fill in behind typed evidence |

Cases enter a period's list when they emerged in it, grew in it (a vendor entity first onboarded in it), or (v1.3) when a typed finding covering them emerged in it (weight 0.5); a structure still holding at the period start is listed behind them (weight 0.1). Section 9 gives the reason: 8 of the 43 main-data validation cases were found and then dropped as older than the period.

Cases: findings on overlapping members from different detectors merge into their union (capped at 1.5× the lead) when their active periods meet; each piece of evidence counts once in the score; benign explanations down-weight only the signal they explain.

In [35]:
per_det = pd.Series([c.detector for c in CANDS_ALL]).value_counts().rename("candidates")
display(per_det.to_frame().T)
print(f"{len(CANDS_ALL)} candidates -> {len(CASES_ALL)} cases -> {len(SUB_ALL)} submitted")

,rare_sharing_community,common_owner,ownership_overlap,entity_outlier,opaque_routing,sponsor_concentration,sponsor_approver_pair,shared_address,sponsor_burst,rapid_onboarding,approver_star,approver_hub,bridge,dense_cluster,layered_ownership,entity_change_point,duplicate_paid,geography,sync_spikes,ownership_change,post_termination,portfolio_spikes
candidates,69,41,26,20,18,16,13,12,10,10,9,8,7,7,7,6,6,5,4,3,1,1


299 candidates -> 186 cases -> 112 submitted


## 5. Results

### 5.1 All 33 cases at the snapshot

The whole history at 2026-06-30, scored against all 33 labelled cases. This mixes the periods; section 5.2 is the protocol the sponsor uses.

In [36]:
KEYS = ["n_submitted", "n_planted", "P@10", "P@25", "P@50", "R@25", "R@50", "R@100", "AP", "first_hit_rank",
        "mean_iou", "member_recall", "exact_recovery", "tag_exact", "tag_jaccard", "macro_tag_recall",
        "worst_tag_recall", "fp_top50"]


def evaluate(sub, truth):
    res = score(prepare_submission(sub[SUBMISSION_COLUMNS]), truth, SCORER)
    matches = res.pop("_matches")
    return {k: res[k] for k in KEYS}, matches


def landing(sub, matches, truth, period):
    """rank and IoU of the submitted case matched to each labelled case (blank: not matched)"""
    hit = matches.dropna(subset=["matched_case"]).set_index("matched_case")
    members = dict(zip(sub.case_rank, sub.member_record_ids.str.split("|").map(frozenset)))
    rows = []
    for t in truth.itertuples():
        r = hit.case_rank.get(t.case_id)
        rows.append({"period": period, "case_id": t.case_id, "tags": t.typology_tags, "members": len(t.members),
                     "rank": r, "IoU": round(overlap(members[r], t.members, "iou"), 2) if r is not None else None})
    return pd.DataFrame(rows).sort_values("rank")


ALL, matches = evaluate(SUB_ALL, LABELS)
SUB_ALL[SUBMISSION_COLUMNS].to_csv(CASE_DIR / "starter" / "submission_ALL.csv", index=False)
pd.DataFrame([ALL]).to_csv(OUT / "metrics_all.csv", index=False)
display(pd.DataFrame([ALL]).round(3))

,n_submitted,n_planted,P@10,P@25,P@50,R@25,R@50,R@100,AP,first_hit_rank,mean_iou,member_recall,exact_recovery,tag_exact,tag_jaccard,macro_tag_recall,worst_tag_recall,fp_top50
0,112,33,1.0,0.96,0.64,0.727,0.97,1.0,0.946,1,0.947,0.964,0.727,0.909,0.919,1.0,1.0,18


### 5.2 By period

Each period runs at its end date with its window (none for TRAINING) and is scored against the cases of that period only, with members not yet onboarded at the end date removed.

In [37]:
def run_period(name):
    lo, hi = PERIODS[name]
    cfg = copy.deepcopy(CFG)
    cfg["snapshot"], cfg["window"] = hi, ([lo, hi] if lo else None)
    sub, _, _, ctx, _ = pipeline(cfg, BUNDLE)
    POLICY[name] = ctx.policy_sub
    onboarded = BUNDLE.third_parties.set_index("record_id").onboarding_date
    present = set(onboarded.index[onboarded <= pd.Timestamp(hi)])
    truth = LABELS[LABELS.case_id.isin(CASES_IN[name])].copy()
    truth["members"] = truth.members.map(lambda m: m & present)
    truth = truth[truth.members.map(len) > 0]
    res, matches = evaluate(sub, truth)
    return sub, res, landing(sub, matches, truth, name), truth


BY_PERIOD, LANDING, SUBS, TRUTHS, POLICY = {}, [], {}, {}, {}
for p in PERIODS:
    SUBS[p], BY_PERIOD[p], land, TRUTHS[p] = run_period(p)
    SUBS[p][SUBMISSION_COLUMNS].to_csv(CASE_DIR / "starter" / f"submission_{p}.csv", index=False)
    if POLICY.get(p) is not None:
        POLICY[p][SUBMISSION_COLUMNS].to_csv(CASE_DIR / "starter" / f"policy_{p}.csv", index=False)
    LANDING.append(land)
LANDING = pd.concat(LANDING, ignore_index=True)
metrics = pd.DataFrame(BY_PERIOD).T
metrics.insert(2, "found", LANDING.groupby("period")["rank"].count().reindex(metrics.index))
metrics.to_csv(OUT / "metrics_by_period.csv")
LANDING.to_csv(OUT / "case_landing.csv", index=False)
display(metrics[["n_planted", "found", "n_submitted", "AP", "P@10", "P@25", "R@25", "R@50", "R@100", "mean_iou",
                 "exact_recovery", "tag_exact", "fp_top50"]].round(3))

,n_planted,found,n_submitted,AP,P@10,P@25,R@25,R@50,R@100,mean_iou,exact_recovery,tag_exact,fp_top50
TRAINING,16.0,16,74.0,0.893,0.9,0.60,0.938,1.0,1.0,0.977,0.875,1.0,34.0
VALIDATION,7.0,7,65.0,0.934,0.6,0.28,1.000,1.0,1.0,1.000,1.000,1.0,43.0
TEST,10.0,10,85.0,0.881,0.8,0.40,1.000,1.0,1.0,0.864,0.300,0.7,40.0


In [38]:
# where each labelled case lands in its own period
display(LANDING)

,period,case_id,tags,members,rank,IoU
0,TRAINING,CASE-0009,DENSE_MULTI_RELATION_CLUSTER,7,1,0.88
1,TRAINING,CASE-0005,SPONSOR_CONCENTRATION,10,2,1.00
2,TRAINING,CASE-0023,LAYERED_COMPLEX_OWNERSHIP,1,3,1.00
3,TRAINING,CASE-0021,COMMON_BENEFICIAL_OWNER,4,4,1.00
4,TRAINING,CASE-0002,SHARED_ADDRESS,6,5,1.00
5,TRAINING,CASE-0004,SPONSOR_CONCENTRATION,10,6,1.00
6,TRAINING,CASE-0029,OWNERSHIP_CHANGE_STRUCTURAL_SHIFT,3,7,1.00
7,TRAINING,CASE-0027,OPAQUE_JURISDICTION_ROUTING,1,8,1.00
8,TRAINING,CASE-0022,LAYERED_COMPLEX_OWNERSHIP,1,9,1.00
9,TRAINING,CASE-0028,OPAQUE_JURISDICTION_ROUTING,1,12,1.00


### 5.3 Matching threshold

The official overlap threshold is not published. The same submissions, rescored at other thresholds; "> 0.5" is the strict rule (an IoU of exactly 0.5 does not match), as used in Layers 1 and 2.

In [39]:
rows = {}
for label, th in [("IoU >= 0.3", 0.3), ("IoU >= 0.5", 0.5), ("IoU > 0.5", 0.5 + 1e-9), ("IoU >= 0.7", 0.7)]:
    cfg = ScorerConfig(SCORER.measure, th)
    rows[label] = {**{p: score(prepare_submission(SUBS[p][SUBMISSION_COLUMNS]), TRUTHS[p], cfg)["AP"] for p in PERIODS},
                   "ALL": score(prepare_submission(SUB_ALL[SUBMISSION_COLUMNS]), LABELS, cfg)["AP"]}
THRESHOLDS = pd.DataFrame(rows).T
THRESHOLDS.to_csv(OUT / "ap_by_threshold.csv")
print("AP by matching threshold")
display(THRESHOLDS.round(3))

AP by matching threshold


,TRAINING,VALIDATION,TEST,ALL
IoU >= 0.3,0.918,0.934,0.881,0.950
IoU >= 0.5,0.893,0.934,0.881,0.946
IoU > 0.5,0.893,0.934,0.803,0.883
IoU >= 0.7,0.893,0.934,0.655,0.840


In [40]:
print(f"total run time {time.time() - T0:.0f} s; outputs in {OUT.relative_to(ROOT)}/")

total run time 35 s; outputs in results\graph\v1.3/


## 6. Findings

**Recovery.** Every labelled case of each period is in the top 50 of that period's queue: 16/16, 7/7 and 10/10 (R@50 = 1.0), with the first hit at rank 1 in every period. AP is 0.893 (TRAINING), 0.934 (VALIDATION) and 0.881 (TEST), and 0.946 on all 33 cases at the snapshot (v1.2: 0.864, 0.929, 0.856 and 0.924). The v1.3 changes were made on the main-data validation labels (section 9); here they move labelled cases by a few places at most. In TRAINING eight cases move up one place, because six approver groups without sponsor coupling now rank with the open-set prior; in TEST CASE-0012 moves from 13 to 9 and CASE-0010 from 7 to 6, and CASE-0017 and CASE-0014 move down one place each. Changes 6–8 of v1.3 (spike groups, sponsor bursts, expert owners; section 9.4) move two cases: in TRAINING CASE-0009 rises from 5 to 1, since a sponsor burst corroborates it (its boundary widens by one vendor, IoU 1.00 → 0.88), and in VALIDATION CASE-0011 rises from 14 to 13.

**The period filter (v1.3).** Each submitted case records why it belongs to its period (`window_basis` in the ranked case lists): it emerged in the period, it grew in it, a finding that covers it emerged in it, or it is a standing structure (older, still holding at the period start). The two new reasons make the starter lists longer: VALIDATION holds 16 cases that emerged, 2 that grew and 47 standing structures (65 cases; v1.2: 19), TEST 26 that emerged, 5 that grew, 3 with a covering finding and 51 standing structures (85; v1.2: 31). No labelled starter case needs the new reasons. The added cases rank behind the period's own (covering findings from rank 17 in TEST, standing structures from rank 19 in VALIDATION and 35 in TEST), which is why the false positives in the top 50 rise (VALIDATION 12 → 43, TEST 21 → 40): v1.2's lists were shorter than 50. This data sets the weight of a covering finding: at weight 1.0 instead of 0.5, two older owner groups with a new finding rank 4th and 5th in TEST and AP falls to 0.770.

**Cases that grew.** CASE-0026 (VALIDATION) and CASE-0006, CASE-0007 and CASE-0032 (TEST) formed before their period and gained members in it. They are submitted because they grew in the period; dated by their earliest finding alone, they would fall outside the window.

**Case boundaries.** Matched cases are exact for 14 of 16 TRAINING cases (CASE-0009, above, and CASE-0018) and all 7 VALIDATION cases, but for only 3 of 10 TEST cases (mean IoU 0.86). Case assembly is that of v1.0.
- CASE-0032 (10 members) is the union of a sponsor group (6) and a sponsor–approver pair (5) that share one vendor. The two stay separate cases, and the pair matches it at IoU exactly 0.50.
- Under the strict "> 0.5" rule used in Layers 1 and 2, TEST AP is 0.803, and 0.655 at IoU ≥ 0.7 (section 5.3). TRAINING and VALIDATION do not change.

**Tags.** As in v1.2, the submitted tag set is exactly right for all 16 TRAINING and all 7 VALIDATION matched cases, and for 7 of 10 TEST cases. A tag comes only from a detector whose own members cover the case, and a typology that fits replaces the general ones it is built from (`SUBSUMES` in `meridian.ranking`). The three TEST cases still wrong are the composite cases with three to six labelled tags (CASE-0031, CASE-0032, CASE-0033).

**Left out (v1.2, unchanged).** A near-duplicate of a higher case and a case whose only evidence is one open-set detector are not submitted. No labelled starter case is among them. (On the main-data validation labels three composite cases are; keeping them was tested and not adopted, section 9.4.)

**Approver groups without coupling, corridors, expert clusters (v1.3).** 6, 2 and 4 approver groups without sponsor coupling (TRAINING, VALIDATION, TEST) are reported by `approver_hub` and rank with the open-set prior; no labelled starter case is one. The starter data has no cross-border corridor. The public-institution expert cluster of v1.2 no longer runs (variant 14+3 only); on the starter data it found only the experts inside CASE-0032, which a higher case already matched.

**Spike groups, sponsor bursts, expert owners (v1.3).** The starter spike groups are small: at the snapshot four of the five have a chance below 0.001 and stay `sync_spikes`, among them the labelled CASE-0017 and CASE-0018 (three of a sponsor's four vendors each); one is `portfolio_spikes`. The sponsor-burst detector finds 10 bursts in the whole starter history: they match CASE-0008, CASE-0009, CASE-0010 and CASE-0011 (IoU 0.88–1.00), cover half of CASE-0004, CASE-0005 or CASE-0032, and one (six vendors entered on 2024-05-15, all on that day, the first day of an acquisition integration: two benign notes) matches nothing. In the starter data they corroborate cases already found. Common-owner groups whose owner is an engaged expert with no sponsor engaging both get a benign note; the labelled CASE-0024, where one sponsor engages the expert and the group, keeps its conflict-of-interest reading.

**Peer groups, ultimate owners, compliance review.** The starter groups are small, so most comparisons back off to coarse peer groups and change nothing. One labelled case is affected: the four vendors of CASE-0003 at one commercial address in Morocco. 7% of the 29 commercial addresses in its subregion host as many vendors at once (CASE-0001's address in Algeria is one of them), so the case carries a benign note "common among peers" and moves down. The 383 ULTIMATE_OWNER edges change no result, since detection does not read them. The compliance-review check finds nothing: every review the policy matrix requires is present, and dated before the engagement starts.

**Caveat.** The detectors, case rules and tag rules were designed while looking at these 33 cases, and the v1.3 changes while looking at the 43 main-data validation cases (section 9): both are development data. The honest estimate is the sponsor's blind score of the TEST list.

## 7. Adding the Layer 2 typologies: 14 / 14+1 / 14+2 / 14+3

Layer 2 proposed new typologies beside the 14 and compared three risk-score variants (`results/layer2_frozen/risk_score_weights.csv`). The same four variants of this method:

| variant | typologies added to the 14 |
|---|---|
| **14** | none: the method of sections 0–6 (it already runs `fmv_cap` as POLICY_VIOLATION and `post_termination` under ACTIVITY_TIMING_CONCENTRATION) |
| **14+1** | FMV_CAP_BREACH |
| **14+2** | FMV_CAP_BREACH, REONBOARDED_AFTER_TERMINATION |
| **14+3** | FMV_CAP_BREACH, PUBLIC_INSTITUTION_EXPERT_CLUSTER, PAYMENT_AFTER_TERMINATION |

How each is added (module `meridian.detectors.layer2_typologies`, rules as in Layer 2, no labels):
- **FMV_CAP_BREACH** and **PAYMENT_AFTER_TERMINATION** are already found by `fmv_cap` and `post_termination`; as typologies their findings take their own tag. This changes the submitted tags and the type interleaving of the ranking, not the cases found.
- **REONBOARDED_AFTER_TERMINATION** (new detector): a record with the same tax id or phone as a record terminated before it was onboarded.
- **PUBLIC_INSTITUTION_EXPERT_CLUSTER** (new detector): ≥ 3 individual experts affiliated with the same public-sector institution and engaged by the same sponsor. It needs three of the experts to join within 90 days. v1.2 made it part of the method itself (every variant); v1.3 takes it out again (the main-data validation labels use the 14 tags only), so only 14+3 runs it.

Every variant runs each period as in section 5.2 and is scored the same way, **twice**: on **all the cases it assembles** (no truncation, the method's submission) and on its first 100 cases (the earlier setting). AP and recall are normalised by all labelled cases, so cases beyond rank 100 can only add to them; P@k and R@k for k ≤ 100 and the false positives in the top 50 are the same for both. Case files are written to `results/graph/<VERSION>/cases/starter/layer2_variants/all_cases/` and `.../top_100/`. Reported: AP at IoU ≥ 0.5 (the headline) and at the strict IoU > 0.5 of Layers 1 and 2, mean IoU of the matched cases, cases found, exact recovery, tag accuracy, false positives in the top 50, and how many submitted cases carry an added tag. As in Layer 2, the variant is **chosen on VALIDATION** (AP, then mean IoU) and TEST is reported; from v1.3 the choice that sets the deliverable is made on the main-data VALIDATION labels (section 8). No starter case carries an added tag, so on this data an added typology can only move other cases down the queue or change tags; what it finds is shown in the last table.

In [41]:
VARIANTS = {   # the Layer 2 risk-score variants (results/layer2_frozen/risk_score_weights.csv)
    "14": [],
    "14+1": ["FMV_CAP_BREACH"],
    "14+2": ["FMV_CAP_BREACH", "REONBOARDED_AFTER_TERMINATION"],
    "14+3": ["FMV_CAP_BREACH", "PUBLIC_INSTITUTION_EXPERT_CLUSTER", "PAYMENT_AFTER_TERMINATION"],
}
# Submission lengths compared: the first 100 cases (the earlier setting), and every case the method assembles
# (the method's setting, max_cases: null). The ranking picks cases one at a time, best first, so the first 100
# of the full list is exactly what a 100-case submission would hold.
LIMITS = {"top 100": 100, "all cases": None}
assert CFG["ranking"]["max_cases"] is None
STRICT = ScorerConfig(SCORER.measure, SCORER.threshold + 1e-9)   # IoU > 0.5, as in Layers 1 and 2
NEW_DETECTORS = ("reonboarded_after_termination", "public_institution_cluster")


def variant_config(added, snapshot=None, window=None, max_cases="method"):
    cfg = copy.deepcopy(CFG)
    if added:   # the 14-typology config is left untouched, so its method hash stays that of section 0.1
        cfg["layer2_typologies"] = list(added)
    if snapshot is not None:
        cfg["snapshot"] = snapshot
    cfg["window"] = window
    if max_cases != "method":
        cfg["ranking"]["max_cases"] = max_cases
    return cfg


def period_truth(bundle, labels, case_ids, hi):
    """labelled cases of a period, without members not yet onboarded at its end"""
    onboarded = bundle.third_parties.set_index("record_id").onboarding_date
    present = set(onboarded.index[onboarded <= pd.Timestamp(hi)])
    truth = labels[labels.case_id.isin(case_ids)].copy()
    truth["members"] = truth.members.map(lambda m: m & present)
    return truth[truth.members.map(len) > 0]


def has_added_tag(sub, added):
    return sub["all_tags"].str.split("|").map(lambda t: bool(set(t) & set(added)))


def score_variant(sub, truth, added):
    res, matches = evaluate(sub, truth)
    res["found"] = int(matches["matched_case"].notna().sum())
    res["recall"] = res["found"] / res["n_planted"] if res["n_planted"] else np.nan
    res["AP (IoU > 0.5)"] = score(prepare_submission(sub[SUBMISSION_COLUMNS]), truth, STRICT)["AP"]
    res["cases with an added tag"] = int(has_added_tag(sub, added).sum())
    return res, matches


def variant_runs(bundle, variants=VARIANTS):
    """Each variant on each period (period end, period window) and at the snapshot, without truncation."""
    for name, added in variants.items():
        for period, (lo, hi) in [("ALL", (None, CFG["snapshot"]))] + list(PERIODS.items()):
            sub, cases, cands, ctx, _ = pipeline(variant_config(added, hi, [lo, hi] if lo else None, max_cases=None), bundle)
            sub.policy = ctx.policy_sub   # v1.1: the separate policy list travels with the submission
            yield name, added, period, sub, cases, cands
        print(f"{name}: done ({time.time() - T0:.0f} s)", flush=True)


def write_cases(sub, out_dir, name, period):
    for limit_name, limit in LIMITS.items():
        d = out_dir / limit_name.replace(" ", "_")
        d.mkdir(parents=True, exist_ok=True)
        (sub.head(limit) if limit else sub)[SUBMISSION_COLUMNS].to_csv(d / f"submission_{name}_{period}.csv", index=False)
    if getattr(sub, "policy", None) is not None:
        sub.policy[SUBMISSION_COLUMNS].to_csv(out_dir / f"policy_{name}_{period}.csv", index=False)


def run_variants(bundle, labels, cases_in, variants=VARIANTS, out_dir=None):
    """Every variant scored on its top 100 cases and on all its cases."""
    rows, lands, subs = [], [], {}
    for name, added, period, sub, cases, cands in variant_runs(bundle, variants):
        truth = labels if period == "ALL" else period_truth(bundle, labels, cases_in[period], hi=(CFG["snapshot"] if period == "ALL" else PERIODS[period][1]))
        for limit_name, limit in LIMITS.items():
            s = sub.head(limit) if limit else sub
            res, matches = score_variant(s, truth, added)
            rows.append({"variant": name, "period": period, "cases": limit_name,
                         "method hash": method_hash(variant_config(added, max_cases=limit)), "cases assembled": len(cases),
                         "new-detector candidates": sum(c.detector in NEW_DETECTORS for c in cands), **res})
            lands.append(landing(s, matches, truth, period).assign(variant=name, cases=limit_name))
        subs[(name, period)] = sub
        if out_dir is not None:
            write_cases(sub, out_dir, name, period)
    return pd.DataFrame(rows), pd.concat(lands, ignore_index=True), subs


VAR_DIR = OUT / "layer2_variants"                       # metrics
VAR_CASES = CASE_DIR / "starter" / "layer2_variants"   # case files: top_100/ and all_cases/
VAR_DIR.mkdir(exist_ok=True)
VAR_METRICS, VAR_LANDING, VAR_SUBS = run_variants(BUNDLE, LABELS, CASES_IN, out_dir=VAR_CASES)
VAR_METRICS.to_csv(VAR_DIR / "metrics.csv", index=False)
VAR_LANDING.to_csv(VAR_DIR / "case_landing.csv", index=False)

# the 14-typology runs here are the submissions of sections 5.1-5.2
assert VAR_SUBS[("14", "ALL")]["member_record_ids"].tolist() == SUB_ALL["member_record_ids"].tolist()
for p_ in PERIODS:
    assert VAR_SUBS[("14", p_)]["member_record_ids"].tolist() == SUBS[p_]["member_record_ids"].tolist()

SHOW = ["AP", "AP (IoU > 0.5)", "mean_iou", "found", "n_planted", "recall", "exact_recovery", "tag_exact", "P@10",
        "fp_top50", "n_submitted", "cases with an added tag"]
for limit_name in LIMITS:
    print(f"\n{limit_name}" + (" (the earlier 100-case setting)" if LIMITS[limit_name] else
                              " (the method's submission: every case assembled; P@k, R@k for k <= 100 and fp_top50 equal the top 100)"))
    t = VAR_METRICS[VAR_METRICS.cases == limit_name].set_index(["period", "variant"])[SHOW].reindex(["ALL", *PERIODS], level=0)
    display(t.round(3))

14: done (68 s)


14+1: done (99 s)


14+2: done (133 s)


14+3: done (164 s)



top 100 (the earlier 100-case setting)


AP  AP (IoU > 0.5)  mean_iou  found  n_planted  recall  exact_recovery  tag_exact  P@10  fp_top50  n_submitted  cases with an added tag
period     variant                                                                                                                                            
ALL        14       0.946           0.883     0.947     33         33     1.0           0.727      0.909   1.0        18          100                        0
           14+1     0.944           0.881     0.947     33         33     1.0           0.727      0.909   1.0        18          100                        3
           14+2     0.877           0.812     0.947     33         33     1.0           0.727      0.909   1.0        19          100                       16
           14+3     0.932           0.883     0.947     33         33     1.0           0.727      0.879   1.0        18          100                        5
TRAINING   14       0.893           0.893     0.977     16         16     1.0           0.875      1.000   0.9        34           74                        0
           14+1     0.893           0.893     0.977     16         16     1.0           0.875      1.000   0.9        34           74                        0
           14+2     0.836           0.836     0.977     16         16     1.0           0.875      1.000   0.9        34           93                       20
           14+3     0.893           0.893     0.977     16         16     1.0           0.875      1.000   0.9        34           74                        0
VALIDATION 14       0.934           0.934     1.000      7          7     1.0           1.000      1.000   0.6        43           65                        0
           14+1     0.929           0.929     1.000      7          7     1.0           1.000      1.000   0.6        43           66                        2
           14+2     0.929           0.929     1.000      7          7     1.0           1.000      1.000   0.6        43           84                       21
           14+3     0.929           0.929     1.000      7          7     1.0           1.000      0.857   0.6        43           66                        3
TEST       14       0.881           0.803     0.864     10         10     1.0           0.300      0.700   0.8        40           85                        0
           14+1     0.881           0.803     0.864     10         10     1.0           0.300      0.700   0.8        40           87                        3
           14+2     0.827           0.749     0.864     10         10     1.0           0.300      0.700   0.7        40          100                       21
           14+3     0.881           0.803     0.864     10         10     1.0           0.300      0.700   0.8        40           87                        4


all cases (the method's submission: every case assembled; P@k, R@k for k <= 100 and fp_top50 equal the top 100)


AP  AP (IoU > 0.5)  mean_iou  found  n_planted  recall  exact_recovery  tag_exact  P@10  fp_top50  n_submitted  cases with an added tag
period     variant                                                                                                                                            
ALL        14       0.946           0.883     0.947     33         33     1.0           0.727      0.909   1.0        18          112                        0
           14+1     0.944           0.881     0.947     33         33     1.0           0.727      0.909   1.0        18          114                        3
           14+2     0.877           0.812     0.947     33         33     1.0           0.727      0.909   1.0        19          141                       31
           14+3     0.932           0.883     0.947     33         33     1.0           0.727      0.879   1.0        18          114                        5
TRAINING   14       0.893           0.893     0.977     16         16     1.0           0.875      1.000   0.9        34           74                        0
           14+1     0.893           0.893     0.977     16         16     1.0           0.875      1.000   0.9        34           74                        0
           14+2     0.836           0.836     0.977     16         16     1.0           0.875      1.000   0.9        34           93                       20
           14+3     0.893           0.893     0.977     16         16     1.0           0.875      1.000   0.9        34           74                        0
VALIDATION 14       0.934           0.934     1.000      7          7     1.0           1.000      1.000   0.6        43           65                        0
           14+1     0.929           0.929     1.000      7          7     1.0           1.000      1.000   0.6        43           66                        2
           14+2     0.929           0.929     1.000      7          7     1.0           1.000      1.000   0.6        43           84                       21
           14+3     0.929           0.929     1.000      7          7     1.0           1.000      0.857   0.6        43           66                        3
TEST       14       0.881           0.803     0.864     10         10     1.0           0.300      0.700   0.8        40           85                        0
           14+1     0.881           0.803     0.864     10         10     1.0           0.300      0.700   0.8        40           87                        3
           14+2     0.827           0.749     0.864     10         10     1.0           0.300      0.700   0.7        40          112                       29
           14+3     0.881           0.803     0.864     10         10     1.0           0.300      0.700   0.8        40           87                        4

In [42]:
# choose on VALIDATION (AP, then mean IoU), report TEST; for each submission length
for limit_name in LIMITS:
    m = VAR_METRICS[VAR_METRICS.cases == limit_name]
    val = m[m.period == "VALIDATION"].set_index("variant")
    test = m[m.period == "TEST"].set_index("variant")
    chosen = val.sort_values(["AP", "mean_iou"], ascending=False).index[0]
    if LIMITS[limit_name] is None:   # the method's submission length
        CHOSEN_VARIANT = chosen
    print(f"{limit_name}: chosen on VALIDATION {chosen} (AP {val.at[chosen, 'AP']:.3f}, mean IoU {val.at[chosen, 'mean_iou']:.3f})"
          f" -> TEST AP {test.at[chosen, 'AP']:.3f} (IoU > 0.5: {test.at[chosen, 'AP (IoU > 0.5)']:.3f}), "
          f"mean IoU {test.at[chosen, 'mean_iou']:.3f}, found {test.at[chosen, 'found']}/{test.at[chosen, 'n_planted']}")
    ties = val.index[(val["AP"] == val.at[chosen, "AP"]) & (val["mean_iou"] == val.at[chosen, "mean_iou"])]
    if len(ties) > 1:
        print(f"   VALIDATION does not separate {', '.join(ties)}: the first listed is kept")

# where each labelled case lands in each variant (only placements that differ between the variants)
land = VAR_LANDING[VAR_LANDING.cases == "all cases"]   # with no truncation every found case has its rank
rank = land.pivot_table(index=["period", "case_id"], columns="variant", values="rank", aggfunc="first")
iou = land.pivot_table(index=["period", "case_id"], columns="variant", values="IoU", aggfunc="first")
changed = (rank.nunique(axis=1, dropna=False) > 1) | (iou.nunique(axis=1, dropna=False) > 1)
print(f"{int(changed.sum())} of {len(rank)} labelled case placements differ between the variants")
if changed.any():
    display(rank[changed].join(iou[changed], lsuffix=" rank", rsuffix=" IoU"))

# what the added typologies submit (at the snapshot, all periods together, within the top 100)
added_rows = []
for name, added in VARIANTS.items():
    sub = VAR_SUBS[(name, "ALL")].head(LIMITS["top 100"])
    added_rows += [{"variant": name, "rank": r.case_rank, "tags": r.typology_tags, "members": r.member_record_ids.count("|") + 1,
                    "reason": r.reason_codes} for r in sub[has_added_tag(sub, added)].itertuples()]
if added_rows:
    display(pd.DataFrame(added_rows))

top 100: chosen on VALIDATION 14 (AP 0.934, mean IoU 1.000) -> TEST AP 0.881 (IoU > 0.5: 0.803), mean IoU 0.864, found 10/10
all cases: chosen on VALIDATION 14 (AP 0.934, mean IoU 1.000) -> TEST AP 0.881 (IoU > 0.5: 0.803), mean IoU 0.864, found 10/10
37 of 66 labelled case placements differ between the variants


variant               14 rank  14+1 rank  14+2 rank  14+3 rank  14 IoU  14+1 IoU  14+2 IoU  14+3 IoU
period     case_id                                                                                  
ALL        CASE-0001       16         16         18         17    1.00      1.00      1.00      1.00
           CASE-0003       53         54         64         54    1.00      1.00      1.00      1.00
           CASE-0004       12         12         13         13    1.00      1.00      1.00      1.00
           CASE-0006       25         25         31         25    0.92      0.92      0.92      0.92
           CASE-0007       30         30         38         31    0.83      0.83      0.83      0.83
           CASE-0010       17         17         19         18    0.83      0.83      0.83      0.83
           CASE-0011       41         42         49         42    1.00      1.00      1.00      1.00
           CASE-0012       29         29         33         30    1.00      1.00      1.00      1.00
           CASE-0014       22         22         25         22    0.86      0.86      0.86      0.86
           CASE-0016       19         19         21         20    1.00      1.00      1.00      1.00
           CASE-0017       33         33         37         29    0.80      0.80      0.80      0.80
           CASE-0018       36         37         42         35    0.75      0.75      0.75      0.75
           CASE-0020       32         32         35         33    1.00      1.00      1.00      1.00
           CASE-0021       24         24         27         24    1.00      1.00      1.00      1.00
           CASE-0022       15         15         16         16    1.00      1.00      1.00      1.00
           CASE-0024       44         45         51         45    1.00      1.00      1.00      1.00
           CASE-0026       21         21         23         21    1.00      1.00      1.00      1.00
           CASE-0027       18         18         20         19    1.00      1.00      1.00      1.00
           CASE-0028       23         23         26         23    1.00      1.00      1.00      1.00
           CASE-0029       13         13         14         14    1.00      1.00      1.00      1.00
           CASE-0030       26         26         28         26    1.00      1.00      1.00      1.00
           CASE-0033       14         14         15         15    1.00      1.00      1.00      1.00
TEST       CASE-0006        8          8          9          8    0.92      0.92      0.92      0.92
           CASE-0007       12         12         14         12    0.83      0.83      0.83      0.83
           CASE-0012        9          9         10          9    1.00      1.00      1.00      1.00
           CASE-0014       11         11         13         11    0.86      0.86      0.86      0.86
           CASE-0017       10         10         12         10    0.80      0.80      0.80      0.80
TRAINING   CASE-0001       14         14         16         14    1.00      1.00      1.00      1.00
           CASE-0015       17         17         20         17    1.00      1.00      1.00      1.00
           CASE-0018       20         20         24         20    0.75      0.75      0.75      0.75
           CASE-0020       18         18         21         18    1.00      1.00      1.00      1.00
           CASE-0022        9          9         10          9    1.00      1.00      1.00      1.00
           CASE-0024       28         28         33         28    1.00      1.00      1.00      1.00
           CASE-0027        8          8          9          8    1.00      1.00      1.00      1.00
           CASE-0028       12         12         13         12    1.00      1.00      1.00      1.00
           CASE-0030       15         15         17         15    1.00      1.00      1.00      1.00
VALIDATION CASE-0011       13         14         14         14    1.00      1.00      1.00      1.00

,variant,rank,tags,members,reason
0,14+1,34,FMV_CAP_BREACH,1,"expert (investigator_kol, US, band US) paid USD 109,202 in fees in 2025, 1.16x the annual cap of USD 94,500 (card v2)"
1,14+1,50,SPONSOR_APPROVER_COUPLING|APPROVER_CONCENTRATION,5,"sponsor EMP-69759B36 (medical_lead) brought 5 vendors (5 engagements, RND in US) to approver EMP-5B8FCF3B (finance_a..."
2,14+1,94,FMV_CAP_BREACH,7,7 vendors form a tight group in the relation graph; 1 benign explanation(s) to check
3,14+2,12,SPONSOR_APPROVER_COUPLING|APPROVER_CONCENTRATION,4,"sponsor EMP-720ECEB1 (supply_buyer) brought 4 vendors (4 engagements, SCM in US) to approver EMP-5B8FCF3B (finance_a..."
4,14+2,17,REONBOARDED_AFTER_TERMINATION,2,"TP-A7844ED9 onboarded 2024-01-29, 570 days after TP-30A81111 was terminated (2022-07-08)"
5,14+2,24,REONBOARDED_AFTER_TERMINATION,2,"TP-95B087F6 onboarded 2022-06-23, 166 days after TP-BEADE2CA was terminated (2022-01-08)"
6,14+2,30,REONBOARDED_AFTER_TERMINATION,2,"TP-AD84C526 onboarded 2021-12-23, 180 days after TP-E7E104B7 was terminated (2021-06-26)"
7,14+2,36,REONBOARDED_AFTER_TERMINATION,2,"TP-CF42D63D onboarded 2025-11-06, 216 days after TP-CAE7E587 was terminated (2025-04-04)"
8,14+2,39,FMV_CAP_BREACH,1,"expert (investigator_kol, US, band US) paid USD 109,202 in fees in 2025, 1.16x the annual cap of USD 94,500 (card v2)"
9,14+2,40,REONBOARDED_AFTER_TERMINATION,2,"TP-ECE4A275 onboarded 2026-04-12, 278 days after TP-C8CEE0BA was terminated (2025-07-08)"


### 7.1 Findings on the starter data

**The method submits every case it assembles, except near-duplicates and lone open-set findings (v1.2); older structures that still stand come last (v1.3).** The comparison with the first 100 cases is kept below.

**The four variants find the same cases with the same boundaries.** Every variant finds 16/16, 7/7 and 10/10, with the same mean IoU (0.977 / 1.000 / 0.864) and exact recovery. The added typologies (FMV_CAP_BREACH, REONBOARDED_AFTER_TERMINATION, PUBLIC_INSTITUTION_EXPERT_CLUSTER, PAYMENT_AFTER_TERMINATION) cover no labelled starter case, so they can only move cases down the queue and change tags.

**AP by variant** (IoU ≥ 0.5; IoU > 0.5 in brackets; all cases):

| variant | TRAINING | VALIDATION | TEST | all 33 at the snapshot |
|---|---|---|---|---|
| 14 | 0.893 | **0.934** | 0.881 (0.803) | 0.946 |
| 14+1 | 0.893 | 0.929 | 0.881 (0.803) | 0.944 |
| 14+2 | 0.836 | 0.929 | 0.827 (0.749) | 0.877 |
| 14+3 | 0.893 | 0.929 | 0.881 (0.803) | 0.932 |

**Cases submitted** (all cases):

| variant | TRAINING | VALIDATION | TEST | all periods together |
|---|---|---|---|---|
| 14 | 74 | 65 | 85 | 112 |
| 14+1, 14+3 | 74 | 66 | 87 | 114 |
| 14+2 | 93 | 84 | 112 | 141 |

- **14+1 and 14+3 are one place behind 14 on VALIDATION.** FMV_CAP_BREACH is a typology tag, so under these variants the `fmv_cap` finding stays in the main queue and pushes CASE-0011 from 13 to 14. 14+3 also tags the payment after termination (CASE-0016) PAYMENT_AFTER_TERMINATION, which costs tag accuracy on VALIDATION (1.000 → 0.857).
- **14+2 is the weakest.** Its REONBOARDED_AFTER_TERMINATION pairs (largely duplicate records of one vendor, as Layer 2 found) push labelled cases down: TRAINING AP −0.057, TEST −0.054.
- **The starter VALIDATION cases choose 14** (AP 0.934 against 0.929). From v1.3 the deliverable's variant is chosen on the 43 main-data VALIDATION cases instead (section 8), among the variants whose added typologies occur in those labels; the labels use the 14 tags only, so 14 is chosen there too (it also has the highest AP there, 0.419 against 0.413 for 14+1 and 14+3).

**Reading.** On the starter data the added typologies add no recovery. REONBOARDED_AFTER_TERMINATION as defined mainly finds duplicate records and costs AP; it would need a rule separating a re-entered record from a real re-onboarding before it is used.

## 8. The full dataset

The same four variants on the full data in `Full Dataset/` (same file layout as `student_bundle/`: all tables, `registry_R/statements.json`, `registry_L_parents.csv`). The method is not changed for it.

**The full data carries no labelled cases for TRAINING and TEST** (TEST is scored blind by the sponsor). **From v1.3 it has labels for VALIDATION**: the sponsor released the 43 validation cases (`Full Dataset/validation_student_safe.csv`, 2026-11). The cell scores every variant's VALIDATION list against them and chooses the variant there (AP, then mean IoU), no longer on the 7 starter VALIDATION cases, among the variants whose added typologies occur in the labels: a typology the sponsor does not label cannot be the right reason for a case. Section 9 analyses the labels in detail. **Its record ids reuse the starter's ids for different records** (all 1,002 starter ids occur in it, with other names, countries and tax ids), so the starter labels must never be scored against it. Without labels the cell also compares what the variants submit:
- per variant and period: candidates, cases, submitted cases, cases carrying an added tag, OPEN_SET cases;
- how far each variant's queue agrees with the 14-typology queue: of its top 25 / 50 / 100 cases, how many match a case of the 14 queue one-to-one at IoU ≥ 0.5;
- the primary tags submitted.

**The deliverable `output/v1.3/submission_TEST.csv`** is the TEST-period case list of the variant chosen on the main-data VALIDATION labels, every case assembled except near-duplicates and lone open-set findings (listed with the reason in `results/graph/v1.3/full_dataset/dropped_TEST.csv`), with `policy_findings_TEST.csv` beside it; they are the only files written to `output/v1.3/`. All case files (every variant and period, all cases and the first 100, and the policy lists) are written to `results/graph/v1.3/cases/full_dataset/`, metrics to `results/graph/v1.3/full_dataset/`. If labelled cases are added to the folder later (`labelled_cases*.parquet` or `.csv`), each variant is scored by period exactly as in section 7. The cell checks the folder first and stops with the list of missing files when the data is incomplete.

In [43]:
from meridian.data import OPTIONAL_TABLES, TABLES

FULL_DIR = ROOT / "Full Dataset"
required = [t for t in TABLES if t not in OPTIONAL_TABLES]
missing = [t for t in required if not any((FULL_DIR / f"{t}{ext}").exists() for ext in (".parquet", ".csv"))]
missing += [p for p in ["registry_L_parents.csv", "registry_R/statements.json"] if not (FULL_DIR / p).exists()]
FULL_METRICS = None
if missing:
    print(f"{FULL_DIR.name}/ is incomplete: {len(missing)} of {len(required) + 2} required files are missing; "
          f"the full-dataset run is skipped.\nmissing: {', '.join(missing)}")
else:
    FULL = load_bundle(FULL_DIR)
    FULL_OUT = OUT / "full_dataset"                 # metrics
    FULL_CASES = CASE_DIR / "full_dataset"          # case files: all_cases/ and top_100/
    FULL_OUT.mkdir(exist_ok=True)
    label_files = sorted(FULL_DIR.glob("labelled_cases*.parquet")) or sorted(FULL_DIR.glob("labelled_cases*.csv"))
    print(f"full dataset: {len(FULL.third_parties):,} vendor records; labels: {label_files[0].name if label_files else 'none'}")
    if label_files:
        FULL_LABELS = load_labels(label_files[0])
        full_period = case_periods(FULL_LABELS, FULL.third_parties).set_index("case_id")
        full_cases_in = {p: list(full_period.index[full_period.period == p]) for p in PERIODS}
        print({p: len(c) for p, c in full_cases_in.items()})
        FULL_METRICS, FULL_LANDING, FULL_SUBS = run_variants(FULL, FULL_LABELS, full_cases_in, out_dir=FULL_CASES)
        FULL_METRICS.to_csv(FULL_OUT / "metrics.csv", index=False)
        FULL_LANDING.to_csv(FULL_OUT / "case_landing.csv", index=False)
        for limit_name in LIMITS:
            print(limit_name)
            display(FULL_METRICS[FULL_METRICS.cases == limit_name].set_index(["period", "variant"])[SHOW]
                    .reindex(["ALL", *PERIODS], level=0).round(3))
    else:
        def agreement(sub, ref, k=None, threshold=0.5):
            """cases of the top k of `sub` matched one-to-one to a case of the top k of `ref` at IoU >= threshold"""
            a = list(sub["member_record_ids"].str.split("|").map(frozenset)[:k])
            b = list(ref["member_record_ids"].str.split("|").map(frozenset)[:k])
            taken, n = set(), 0
            for m in a:
                best = max(((overlap(m, x, "iou"), j) for j, x in enumerate(b) if j not in taken), default=(0, None))
                if best[1] is not None and best[0] >= threshold:
                    taken.add(best[1]); n += 1
            return n

        rows, FULL_SUBS = [], {}
        for name, added, period, sub, cases, cands in variant_runs(FULL):
            write_cases(sub, FULL_CASES, name, period)
            FULL_SUBS[(name, period)] = sub
            top = sub.head(LIMITS["top 100"])
            rows.append({"variant": name, "period": period, "candidates": len(cands), "cases (all)": len(sub),
                         "added-tag cases (top 100)": int(has_added_tag(top, added).sum()),
                         "added-tag cases (all)": int(has_added_tag(sub, added).sum()),
                         "OPEN_SET cases (top 100)": int(top["typology_tags"].str.contains("OPEN_SET").sum()),
                         "OPEN_SET cases (all)": int(sub["typology_tags"].str.contains("OPEN_SET").sum())})
        for r in rows:
            sub, ref = FULL_SUBS[(r["variant"], r["period"])], FULL_SUBS[("14", r["period"])]
            for k in (25, 50, 100):
                r[f"top {k} shared with 14"] = agreement(sub, ref, k)
        FULL_METRICS = pd.DataFrame(rows)
        FULL_METRICS.to_csv(FULL_OUT / "submitted.csv", index=False)
        print("no labels in the full dataset: case files written (top_100/ and all_cases/), nothing scored")
        display(FULL_METRICS.set_index(["period", "variant"]).reindex(["ALL", *PERIODS], level=0))
        primary = pd.DataFrame({name: FULL_SUBS[(name, "ALL")]["typology_tags"].str.split("|").str[0].value_counts()
                                for name in VARIANTS}).fillna(0).astype(int)
        print("primary tag of all cases at the snapshot")
        display(primary.sort_values("14", ascending=False))
    # v1.3: with the sponsor's main-data validation labels, the variant is chosen on them (AP, then mean IoU)
    # instead of on the 7 starter VALIDATION cases; members onboarded after the period end are left out, as in section 7
    VAL_FILE = FULL_DIR / "validation_student_safe.csv"
    if VAL_FILE.exists():
        VAL_LABELS = load_labels(VAL_FILE)
        VAL_TRUTH = period_truth(FULL, VAL_LABELS, list(VAL_LABELS.case_id), PERIODS["VALIDATION"][1])
        vrows = []
        for name, added in VARIANTS.items():
            res, _ = score_variant(FULL_SUBS[(name, "VALIDATION")], VAL_TRUTH, added)
            vrows.append({"variant": name, **res})
        FULL_VAL_METRICS = pd.DataFrame(vrows).set_index("variant")
        FULL_VAL_METRICS.to_csv(FULL_OUT / "validation_metrics.csv")
        print(f"main-data VALIDATION labels ({VAL_FILE.name}): {len(VAL_TRUTH)} cases")
        display(FULL_VAL_METRICS[SHOW].round(3))
        # a variant whose added typologies occur in none of the labels cannot submit them as the right reason (the
        # sponsor scores typology accuracy): only variants whose added tags are in the labels' vocabulary are eligible
        label_tags = set().union(*VAL_LABELS.tags)
        eligible = [name for name, added in VARIANTS.items() if set(added) <= label_tags]
        starter_choice = CHOSEN_VARIANT
        CHOSEN_VARIANT = FULL_VAL_METRICS.loc[eligible].sort_values(["AP", "mean_iou"], ascending=False).index[0]
        best_any = FULL_VAL_METRICS.sort_values(["AP", "mean_iou"], ascending=False).index[0]
        print(f"typologies in the labels: {len(label_tags)}; eligible variants: {', '.join(eligible)}")
        print(f"chosen on the main-data VALIDATION labels: {CHOSEN_VARIANT} (on the starter VALIDATION cases: {starter_choice})"
              + (f"; highest AP overall: {best_any} (AP {FULL_VAL_METRICS.at[best_any, 'AP']:.3f}, tag exact "
                 f"{FULL_VAL_METRICS.at[best_any, 'tag_exact']:.3f}), whose added tags are not in the labels"
                 if best_any != CHOSEN_VARIANT else ""))
    # the deliverable: the TEST-period cases of the variant chosen on VALIDATION, every case assembled
    DELIVERABLE.parent.mkdir(parents=True, exist_ok=True)
    for old in DELIVERABLE.parent.iterdir():   # output/<VERSION>/ holds this version's two files only
        if old.is_file():
            old.unlink()
    FULL_SUBS[(CHOSEN_VARIANT, "TEST")][SUBMISSION_COLUMNS].to_csv(DELIVERABLE, index=False)
    pol = getattr(FULL_SUBS[(CHOSEN_VARIANT, "TEST")], "policy", None)
    if pol is not None:   # v1.1: the policy findings, as their own list
        pol[SUBMISSION_COLUMNS].to_csv(DELIVERABLE.with_name("policy_findings_TEST.csv"), index=False)
        print(f"policy findings: {DELIVERABLE.with_name('policy_findings_TEST.csv').relative_to(ROOT)}, {len(pol)} findings")
    dropped = getattr(FULL_SUBS[(CHOSEN_VARIANT, "TEST")], "dropped", None)
    if dropped is not None:   # v1.2: the cases left out of the deliverable, and why (results only)
        dropped.to_csv(FULL_OUT / "dropped_TEST.csv", index=False)
        print(f"left out: {len(dropped)} cases {dropped['why'].value_counts().to_dict()}, "
              f"listed in {(FULL_OUT / 'dropped_TEST.csv').relative_to(ROOT)}")
    print(f"deliverable: {DELIVERABLE.relative_to(ROOT)} = variant {CHOSEN_VARIANT}, TEST period, "
          f"{len(FULL_SUBS[(CHOSEN_VARIANT, 'TEST')])} cases (every case assembled, less those left out)")

full dataset: 21,260 vendor records; labels: none


14: done (1121 s)


14+1: done (2295 s)


14+2: done (3303 s)


14+3: done (4272 s)


no labels in the full dataset: case files written (top_100/ and all_cases/), nothing scored


... (2 outputs left out)

,14,14+1,14+2,14+3
typology_tags,,,,
APPROVER_CONCENTRATION,774,774,770,774
ACTIVITY_TIMING_CONCENTRATION,467,467,466,463
SHARED_ADDRESS,406,406,401,406
COMMON_BENEFICIAL_OWNER,340,340,340,340
OWNERSHIP_OPERATIONAL_OVERLAP,231,231,230,231
SPONSOR_APPROVER_COUPLING,163,163,163,163
OPAQUE_JURISDICTION_ROUTING,151,151,151,151
SPONSOR_CONCENTRATION,150,150,150,149
RAPID_ONBOARDING_LINKED,122,122,122,120


main-data VALIDATION labels (validation_student_safe.csv): 43 cases


,AP,AP (IoU > 0.5),mean_iou,found,n_planted,recall,exact_recovery,tag_exact,P@10,fp_top50,n_submitted,cases with an added tag
variant,,,,,,,,,,,,
14,0.419,0.409,0.9,33,43,0.767,0.636,0.758,0.9,30,1457,0
14+1,0.413,0.402,0.9,33,43,0.767,0.636,0.758,0.9,30,1460,3
14+2,0.331,0.322,0.9,33,43,0.767,0.636,0.758,0.7,31,2329,895
14+3,0.413,0.402,0.9,33,43,0.767,0.636,0.697,0.9,30,1464,9


typologies in the labels: 14; eligible variants: 14
chosen on the main-data VALIDATION labels: 14 (on the starter VALIDATION cases: 14)
policy findings: output\v1.3\policy_findings_TEST.csv, 15 findings
left out: 428 cases {'open-set evidence from one detector only': 401, 'near-duplicate of a higher case (member IoU >= 0.5)': 27}, listed in results\graph\v1.3\full_dataset\dropped_TEST.csv
deliverable: output\v1.3\submission_TEST.csv = variant 14, TEST period, 1897 cases (every case assembled, less those left out)


## 9. The main-data validation labels (v1.3)

In November 2026 the sponsor released the labelled cases of the VALIDATION period on the main data: 43 cases with their member records, typology tags, difficulty tier, market and emergence date (`Full Dataset/validation_student_safe.csv`). The TEST labels stay blind. How they are used, following the sponsor's boundaries:

- **Development, error analysis and calibration only.** They are read by the scorer (`meridian.scorer.load_labels`) and by this section; detection, case assembly and ranking never read them. The settings changed after reading them are listed, with the issue, the expected effect and the overfitting risk, in section 9.4 and `README.md`.
- **`difficulty_tier` is not a feature.** It is reported below only to stratify recall.
- **`market` is not a risk signal.** No rate of labelled cases by market enters the method.
- **Temporal validity.** Each case is scored on the VALIDATION run, whose graph holds only what was observable at the period end (2025-06-30). 16 of the 199 member records were onboarded after that date (the case grew later); they are left out of the case, as for the starter cases in sections 5.2 and 7. The emergence date sets each case's period (all 43 fall in VALIDATION).

9.1 scores the VALIDATION lists of v1.1, v1.2 (the archived main-data runs) and this version; 9.2 shows, for every case, where it lands or where it is lost; 9.3 gives the uncertainty of the change in AP.

In [44]:
# 9.1 The VALIDATION lists of v1.1, v1.2 and this version against the main-data validation labels
VAL_FILE = ROOT / "Full Dataset" / "validation_student_safe.csv"
VAL_OUT = OUT / "full_dataset"
if not VAL_FILE.exists() or "FULL_SUBS" not in globals():
    print("no main-data validation labels or no full-data run (section 8): section 9 is skipped")
else:
    VAL_LABELS = load_labels(VAL_FILE)
    vp = case_periods(VAL_LABELS, FULL.third_parties)
    assert (vp.period == "VALIDATION").all() and (vp.basis == "label emergence date").all()
    VAL_TRUTH = period_truth(FULL, VAL_LABELS, list(VAL_LABELS.case_id), PERIODS["VALIDATION"][1])
    n_all, n_obs = sum(map(len, VAL_LABELS.members)), sum(map(len, VAL_TRUTH.members))
    print(f"{len(VAL_TRUTH)} validation cases; {n_all} member records, {n_all - n_obs} of them onboarded after "
          f"{PERIODS['VALIDATION'][1]} and left out")
    archived = {v: ROOT / "results" / "graph" / v / "cases" / "full_dataset" / "all_cases" / "submission_14_VALIDATION.csv"
                for v in ["v1.1", "v1.2"]}
    VAL_LISTS = {v: pd.read_csv(p) for v, p in archived.items() if p.exists()}
    VAL_LISTS[VERSION] = FULL_SUBS[(CHOSEN_VARIANT, "VALIDATION")]
    rows, by_tag, by_tier, VAL_LANDING = {}, {}, {}, {}
    for v, s in VAL_LISTS.items():
        res = score(prepare_submission(s[SUBMISSION_COLUMNS]), VAL_TRUTH, SCORER)
        m = res.pop("_matches")
        rows[v] = {**{k: res[k] for k in KEYS}, "found": int(m.matched_case.notna().sum()), "recall": res["recall"],
                   "AP (IoU > 0.5)": score(prepare_submission(s[SUBMISSION_COLUMNS]), VAL_TRUTH, STRICT)["AP"]}
        by_tag[v], by_tier[v] = res["recall_by_tag"], res["recall_by_difficulty"]
        VAL_LANDING[v] = landing(s, m, VAL_TRUTH, "VALIDATION").set_index("case_id")
    VAL_COMPARE = pd.DataFrame(rows).T[["n_submitted", "AP", "AP (IoU > 0.5)", "P@10", "P@25", "P@50", "R@50", "R@100",
                                         "found", "recall", "mean_iou", "member_recall", "exact_recovery", "tag_exact",
                                         "tag_jaccard", "fp_top50"]]
    n_tag = {t: int(VAL_TRUTH.tags.map(lambda ts: t in ts).sum()) for t in by_tag[VERSION]}
    VAL_BY_TAG = pd.DataFrame(by_tag).assign(cases=pd.Series(n_tag))[["cases", *VAL_LISTS]]
    VAL_BY_TIER = pd.DataFrame(by_tier).assign(cases=VAL_TRUTH.difficulty_tier.value_counts())[["cases", *VAL_LISTS]]
    VAL_COMPARE.to_csv(VAL_OUT / "validation_compare.csv")
    VAL_BY_TAG.to_csv(VAL_OUT / "validation_recall_by_tag.csv")
    VAL_BY_TIER.to_csv(VAL_OUT / "validation_recall_by_difficulty.csv")
    print("scores on the 43 validation cases (VALIDATION run: graph at 2025-06-30, the period's cases)")
    display(VAL_COMPARE.astype(float).round(3))
    print("recall by typology (a case counts for each of its tags)")
    display(VAL_BY_TAG.round(2))
    print("recall by difficulty tier (sponsor metadata: stratified evaluation only, never a feature)")
    display(VAL_BY_TIER.round(2))

43 validation cases; 199 member records, 16 of them onboarded after 2025-06-30 and left out
scores on the 43 validation cases (VALIDATION run: graph at 2025-06-30, the period's cases)


,n_submitted,AP,AP (IoU > 0.5),P@10,P@25,P@50,R@50,R@100,found,recall,mean_iou,member_recall,exact_recovery,tag_exact,tag_jaccard,fp_top50
v1.1,956.0,0.298,0.290,0.7,0.44,0.36,0.419,0.465,27.0,0.628,0.847,0.891,0.519,0.444,0.506,32.0
v1.2,622.0,0.308,0.302,0.8,0.44,0.38,0.442,0.465,24.0,0.558,0.877,0.919,0.583,0.625,0.663,31.0
v1.3,1457.0,0.419,0.409,0.9,0.64,0.40,0.465,0.488,33.0,0.767,0.900,0.936,0.636,0.758,0.770,30.0


recall by typology (a case counts for each of its tags)


,cases,v1.1,v1.2,v1.3
ACTIVITY_TIMING_CONCENTRATION,5,0.40,0.40,0.40
APPROVER_CONCENTRATION,4,0.50,0.50,0.75
BRIDGE_ENTITY,4,1.00,0.50,0.50
COMMON_BENEFICIAL_OWNER,7,0.71,0.57,0.86
DENSE_MULTI_RELATION_CLUSTER,7,0.86,0.71,0.71
GEOGRAPHY_LINKED_CONCENTRATION,3,0.33,0.33,0.67
LAYERED_COMPLEX_OWNERSHIP,4,0.75,0.25,0.50
OPAQUE_JURISDICTION_ROUTING,3,0.67,0.33,0.67
OWNERSHIP_CHANGE_STRUCTURAL_SHIFT,4,1.00,0.75,0.75
OWNERSHIP_OPERATIONAL_OVERLAP,5,0.80,0.40,0.60


recall by difficulty tier (sponsor metadata: stratified evaluation only, never a feature)


,cases,v1.1,v1.2,v1.3
easy,12,0.75,0.75,1.00
hard,9,0.56,0.22,0.22
medium,22,0.59,0.59,0.86


In [45]:
# 9.2 Where each validation case lands in this version, and where the ones not found are lost
if "VAL_TRUTH" in globals():
    lo, hi = PERIODS["VALIDATION"]
    v_sub, v_cases, v_cands, v_ctx, _ = pipeline(variant_config(VARIANTS[CHOSEN_VARIANT], hi, [lo, hi], max_cases=None), FULL)
    assert v_sub["member_record_ids"].tolist() == VAL_LISTS[VERSION]["member_record_ids"].tolist()

    def best(t, groups):
        return max(((overlap(t, g, "iou"), j) for j, g in enumerate(groups)), default=(0.0, None))

    groups = {"candidate": [c.members for c in v_cands], "case": [c.members for c in v_ctx.all_cases],
              "in period": [c.members for c in v_cases]}
    basis = dict(zip(v_sub.case_rank, v_sub.window_basis))
    rows = []
    for t in VAL_TRUTH.itertuples():
        b = {k: best(t.members, g) for k, g in groups.items()}
        r = {v: VAL_LANDING[v].at[t.case_id, "rank"] for v in VAL_LISTS}
        if pd.notna(r[VERSION]):
            where = f"found ({basis[int(r[VERSION])]})"
        elif b["in period"][0] >= 0.5:
            where = "in the period, left out (near-duplicate or lone open-set finding)"
        elif b["case"][0] >= 0.5:
            where = "assembled, not in the period"
        elif b["candidate"][0] >= 0.5:
            where = "found, merged into a larger case"
        elif b["candidate"][0] > 0:
            where = "only partly found (boundary)"
        else:
            where = "not found"
        c = v_cands[b["candidate"][1]] if b["candidate"][1] is not None else None
        rows.append({"case": t.case_id, "tags": t.typology_tags, "tier": t.difficulty_tier, "emerged": t.emergence_date.date(),
                     "members": len(t.members), **{f"rank {v}": r[v] for v in VAL_LISTS},
                     "IoU": VAL_LANDING[VERSION].at[t.case_id, "IoU"], "where": where,
                     "best candidate": f"{c.detector} ({b['candidate'][0]:.2f})" if c is not None else ""})
    VAL_CASES = pd.DataFrame(rows)
    VAL_CASES.to_csv(VAL_OUT / "validation_case_landing.csv", index=False)
    display(VAL_CASES)
    print(VAL_CASES["where"].str.replace(r" \(.*\)", "", regex=True).value_counts().to_string())

,case,tags,tier,emerged,members,rank v1.1,rank v1.2,rank v1.3,IoU,where,best candidate
0,VAL-01,SHARED_ADDRESS,medium,2024-10-04,3,NaN,NaN,402.0,1.00,found (finding),shared_address (1.00)
1,VAL-02,SHARED_ADDRESS,easy,2024-10-02,4,NaN,NaN,347.0,1.00,found (finding),rare_sharing_community (1.00)
2,VAL-03,SHARED_ADDRESS,easy,2024-10-27,6,12.0,10.0,11.0,1.00,found (emerged),shared_address (1.00)
3,VAL-04,SHARED_ADDRESS,hard,2024-09-17,3,NaN,NaN,NaN,NaN,only partly found (boundary),sponsor_approver_pair (0.29)
4,VAL-05,SPONSOR_CONCENTRATION,easy,2025-02-17,10,110.0,113.0,94.0,1.00,found (emerged),sponsor_concentration (1.00)
5,VAL-06,SPONSOR_CONCENTRATION,medium,2025-03-01,7,195.0,191.0,186.0,0.86,found (emerged),sponsor_concentration (0.86)
6,VAL-07,SPONSOR_CONCENTRATION,medium,2024-09-16,7,42.0,47.0,45.0,0.71,found (emerged),sponsor_concentration (0.71)
7,VAL-08,APPROVER_CONCENTRATION|SPONSOR_APPROVER_COUPLING,medium,2024-12-31,8,32.0,48.0,30.0,0.88,found (emerged),sponsor_approver_pair (0.88)
8,VAL-09,APPROVER_CONCENTRATION|SPONSOR_APPROVER_COUPLING,medium,2024-09-23,12,NaN,NaN,513.0,0.83,found (standing),sponsor_approver_pair (0.83)
9,VAL-10,APPROVER_CONCENTRATION|SPONSOR_APPROVER_COUPLING,hard,2025-02-26,7,NaN,NaN,NaN,NaN,only partly found (boundary),rare_sharing_community (0.20)


where
found                      33
only partly found           5
in the period, left out     3
not found                   2


In [46]:
# 9.3 How certain is the change? AP of v1.2 and this version over 2,000 bootstrap resamples of the 43 cases
if "VAL_TRUTH" in globals() and "v1.2" in VAL_LISTS:
    def ap_parts(v):
        """per labelled case: rank of its match (inf if none); AP of a resample = mean over its cases of
        hit * (resampled hits ranked at or above it) / rank"""
        return VAL_LANDING[v]["rank"].reindex(VAL_TRUTH.case_id).fillna(np.inf).to_numpy()
    rng = np.random.default_rng(0)
    ranks = {v: ap_parts(v) for v in ["v1.2", VERSION]}
    def ap(rk, idx):
        r = rk[idx]
        hits = np.sort(r[np.isfinite(r)])
        return float(np.sum(np.arange(1, len(hits) + 1) / hits) / len(idx)) if len(hits) else 0.0
    n = len(VAL_TRUTH)
    draws = [rng.integers(0, n, n) for _ in range(2000)]
    diff = np.array([ap(ranks[VERSION], d) - ap(ranks["v1.2"], d) for d in draws])
    full = np.arange(n)
    print(f"AP v1.2 {ap(ranks['v1.2'], full):.3f} -> {VERSION} {ap(ranks[VERSION], full):.3f} "
          f"(rank-based, matches as above); bootstrap change {diff.mean():+.3f}, 90% interval "
          f"[{np.quantile(diff, 0.05):+.3f}, {np.quantile(diff, 0.95):+.3f}], better in {np.mean(diff > 0):.0%} of resamples")

AP v1.2 0.308 -> v1.3 0.419 (rank-based, matches as above); bootstrap change +0.113, 90% interval [+0.045, +0.195], better in 100% of resamples


### 9.4 Changes made after reading the validation labels

The sponsor asked that every material change made after reviewing the validation labels state the issue observed, the change, the expected improvement and the overfitting risk. Each effect below is measured on the main-data VALIDATION run by removing that one change from the final v1.3 (AP over the 43 cases; "found" = cases matched at IoU ≥ 0.5). Rows 1–5 answer the client's score of v1.2; rows 6–8 followed an error analysis of the list with rows 1–5 only (AP 0.374): its false positives in the first 50 and the cases still missed.

| # | validation issue observed | change made (setting) | expected improvement | overfitting risk |
|---|---|---|---|---|
| 1 | 8 of the 43 cases are found by a detector (candidate IoU 0.75–1.0) but dated before the period and dropped: a case took the earliest date of everything merged into it, and some planted structures (an ownership chain, a foreign correspondence address) carry no date inside the period at all | **period filter** (`meridian.cases.in_window`): a case also belongs to the period when a typed finding that covers it (member IoU ≥ 0.5) emerged in it, ranked with weight 0.5; a case whose covering finding still holds at the period start is submitted behind the period's own cases, with weight 0.1 (`window_findings`, `window_finding_weight`, `window_standing`) | found 25 → 33; every typology's recall ≥ 0.40; AP 0.391 → 0.419. The standing cases themselves rank 254th–557th; most of the AP comes from their entering each detector's percentile distribution, which reorders the period's own cases | **medium.** The rule follows the labels' emergence dates, which the data cannot reproduce for static structures. The weights are set by rule, not tuned: standing weight 0.5 scores 0.431 on these labels but costs the starter VALIDATION period (0.934 → 0.896); finding weight 1.0 scores 0.427 but puts two older owner groups above labelled starter TEST cases (0.881 → 0.793). The cost is volume: older structures are appended to every list, most of them not cases of the period |
| 2 | `approver_star` leads 239 cases in the VALIDATION period and none matches a labelled case; every labelled approver case is a coupled sponsor–approver pair (4 of 4 here, all starter cases) | an approver group without sponsor coupling is reported by `approver_hub` and ranks with the open-set prior 0.3 (`detector_priors`) | AP 0.391 → 0.419, P@10 0.8 → 0.9, P@25 0.60 → 0.64; starter TRAINING 0.864 → 0.893 | **low–medium.** The rule rests on every labelled approver case of both label sets; the prior is the method's existing open-set prior, not fitted |
| 3 | GEOGRAPHY_LINKED_CONCENTRATION has the lowest recall (v1.2: 1 of 3; 0 of 3 counting members onboarded after the period). VAL-24, three RU vendors with one SG correspondence address that remit to SG, is found only as a shared address and an ownership overlap | a **cross-border corridor** detector (`corridor`): at least 3 vendor entities of one market share an address abroad that is also their remit-to country; a fitting geography tag replaces the shared-address and ownership tags it is built from (`SUBSUMES`) | VAL-24 carries the right tag: tag exact 0.727 → 0.758; geography 1 → 2 of 3 found (with the period filter) | **high per case, small in reach.** One labelled example; the rule fires on 2 groups in the whole data (RU→SG, which is VAL-24, and MA→PA, unlabelled) |
| 4 | the labels use the 14 tags only; the v1.2 expert-cluster cases match none | `public_institution_cluster` leaves the method (variant 14+3 still runs it) | neutral on validation; 3 TEST cases that cannot carry a right tag leave the list | low |
| 5 | the variant is now chosen on these labels | **variant choice** (section 8): AP, then mean IoU, among the variants whose added typologies occur in the labels (a typology the sponsor does not label cannot be the right reason for a case); this leaves **14**, which also has the highest AP (0.419; 14+1 and 14+3 0.413) | the deliverable is chosen on 43 main-data cases instead of 7 starter cases | low: the rule uses the labels' vocabulary, not their scores |
| 6 | `sync_spikes` leads 117 cases in the VALIDATION period and none matches a labelled case, yet five of them sit in the first 50 of the list. They are three to six vendors out of 20 to 30 under one sponsor; the labelled starter spike groups are three of a sponsor's four vendors | the chance of a spike group is judged against the sponsor's portfolio: P(Binomial(N, q) ≥ k) for k of the sponsor's N vendors, q the data's two-month spike rate. A group with chance ≥ 0.001 is reported as `portfolio_spikes` and ranks with the open-set prior 0.3 (`activity.max_p`, `detector_priors`) | AP 0.382 → 0.419, P@10 0.8 → 0.9, P@25 0.56 → 0.64; the starter periods do not change (their labelled groups have chance 5·10⁻⁵) | **low–medium.** A significance rule, not fitted: 0.001 and 0.0001 give the same lists; 0.01 keeps more of the groups (AP 0.391 against 0.411 when tested). It rests on 117 unmatched groups and two labelled starter groups; it does not find the activity-timing cases still missed |
| 7 | VAL-18, four CO vendors onboarded under one sponsor within 11 days, is missed: the relation detectors skip a sponsor with more than 30 vendors as a hub, so nothing linked the four | a **sponsor-burst** detector (`sponsor_burst`, RAPID_ONBOARDING_LINKED): within 14 days a sponsor brings at least 4 new vendor entities of one country, with a chance below 0.001 under a Poisson rate from the sponsor's own previous two years there (`sponsor_burst` parameters) | found 32 → 33 (VAL-18, IoU 0.80); AP 0.413 → 0.419. Of the 6 bursts in the VALIDATION period, 4 belong to labelled cases (VAL-05, VAL-14, VAL-18, VAL-20); in the starter data it matches CASE-0008 to CASE-0011 (IoU 0.88–1.00) | **medium.** Designed after one missed case, but it is a generic rate test and rare: 85 bursts in the main data's whole history at the VALIDATION snapshot, 3 in the TEST period. Two of the VALIDATION bursts (VAL-05, VAL-18) carry an acquisition-integration benign note; their cases rank 94th and 102nd. It widens one starter case by a vendor (CASE-0009, IoU 1.00 → 0.88) |
| 8 | 310 common-owner and ownership-overlap findings in the VALIDATION period have an owner who is also engaged as an individual expert, with no sponsor engaging both; none matches a labelled case | such a finding carries a benign note, "possibly the expert's own practice or consultancy" (weight 0.5). When one sponsor engages the expert and the group it stays a conflict of interest, as before (`detect_common_owner`) | AP 0.417 → 0.419; starter VALIDATION 0.929 → 0.934 | **medium.** The note on every expert owner scored higher here (AP 0.427, P@25 0.68), but the labelled starter CASE-0024 is a same-sponsor expert group (TRAINING 0.893 → 0.872); the narrower rule costs neither label set |

All eight together: AP 0.308 (v1.2) → 0.419, found 24 → 33 (section 9.1). Adding rows 6–8 to rows 1–5: AP 0.374 → 0.419, P@10 0.8 → 0.9, P@25 0.56 → 0.64; the starter periods 0.893 / 0.929 / 0.881 → 0.893 / 0.934 / 0.881.

**Tested and not adopted:**

| tested | effect on validation | decision |
|---|---|---|
| type interleaving of v1.1 (the client's P@k fell v1.1 → v1.2) | AP 0.374 → 0.344, P@25 0.56 → 0.48 (with rows 1–5 only) | not adopted: v1.2's interleaving is better on the labels; the TEST hint is one to three cases |
| detector hit-rate calibration from the labels (in-sample, cross-fitted, shrunk) | AP 0.28–0.34, against 0.34 with equal priors | not adopted: a detector-level rate overrides the within-detector order |
| a case-level ranker on label-free case features, cross-validated by labelled case | AP 0.22–0.26 out of fold | not adopted: 32 positives are too few |
| fixed lower priors for detectors with few hits (ownership overlap, layered ownership, bridge 0.3; common owner 0.7) | AP 0.346–0.365 against 0.374 (with rows 1–5 only) | not adopted |
| domestic bridges with the open-set prior (every labelled bridge links institutions abroad) | AP 0.411 → 0.407; starter VALIDATION 0.929 → 0.908 (with fewer than 5 cross-border bridges their percentile falls to 0.5) | not adopted |
| keep the lone open-set cases (undo v1.2's rule) | found 33 → 36 (three composite cases), AP +0.002, tag exact 0.758 → 0.694 | not adopted: the cases carry the OPEN_SET tag, a wrong reason |
| a looser spend-spike threshold (activity timing) | VAL-28 found only at the 0.65 quantile, among 320 spike groups, with chance 0.05 | not adopted |
| one global setting at a time (corroboration 0.75 / 1.0, finding weight 0.75, type decay 0.9, benign weight 0.6, hub prior 0.2) | AP 0.422–0.435 against 0.427 (tested with the broader expert-owner note of row 8); all but one cost a starter period (TRAINING by 0.008–0.024, VALIDATION or TEST by 0.020); the approver-hub prior 0.2 costs nothing and gains 0.004 | not adopted: the two label sets disagree, and +0.004 is within one case |
| the acquisition-integration benign note (VAL-05, VAL-06, VAL-18 and VAL-23 carry it and rank 94th–186th) | not changed: acquisition windows cover about half of the VALIDATION period, but the TEST period has none | — |
| within-detector percentiles against every case at the snapshot | AP 0.374 → 0.372 (with rows 1–5 only) | not adopted (kept as the option `percentile_pool`) |